# IHARQ Phase 01 / Layer 01 — R54 Matched A4 R2 Secret-Safe Final

This is the clean Sections **00–26** notebook.

R54 preserves the R49 matched A4 R2 scientific profile unchanged and permanently integrates:
- the Stage-07 revision-guard correction;
- the Stage-18 `manifests.write_json` import correction;
- secret-safe Kaggle environment serialization so `KAGGLE_API_TOKEN` is never written into `environment_amendment.json`.

No core/A4 scientific parameter, source set, label, split, preprocessing rule, or matched-event denominator is changed.


## R49 configuration and credentials

Run this cell first. The Kaggle token is read from the Kaggle secret `KAGGLE_API_TOKEN` when available; otherwise a hidden prompt is used. The token is never printed or written into the repository.

In [ ]:
from __future__ import annotations

from getpass import getpass
import os

# ---------------------------------------------------------------------
# R49 matched A4 R2 user configuration
# ---------------------------------------------------------------------
#
# The existing 3-second core Dataset remains immutable and is reused.
# The notebook builds one separate A4 evidence-extension Dataset.
#
# Do not put a Kaggle token directly into this notebook's source.

IHARQ_KAGGLE_USERNAME = os.environ.get(
    "IHARQ_KAGGLE_USERNAME",
    "csthv999z",
).strip()

IHARQ_EXISTING_CORE_DATASET_HANDLE = os.environ.get(
    "IHARQ_EXISTING_CORE_DATASET_HANDLE",
    "csthv999z/iharq-p01-l1-derived-windows-d03f0a7c869d-20260806222242-68a91473",
).strip()

IHARQ_EXISTING_CORE_DATASET_VERSION = int(
    os.environ.get("IHARQ_EXISTING_CORE_DATASET_VERSION", "2")
)

IHARQ_EXISTING_CORE_MANIFEST_SHA256 = os.environ.get(
    "IHARQ_EXISTING_CORE_MANIFEST_SHA256",
    "dc21f418e9a1add62adb346f627d5d729735a5f1ecaa1d771f6fa5cfede627e1",
).strip().lower()

IHARQ_CORE_DATASET_MODE = "ADOPT_EXISTING_VERIFIED_DATASET"
IHARQ_ENABLE_A4_EXTENSION = True

# The exact A4 data family is generated now so Phase 2 never needs another
# raw-data return merely to obtain these tensors. Confirmatory A4 reporting
# still requires the corresponding Protocol/Build-Book synchronization.
IHARQ_A4_WINDOW_FAMILY_ID = "P01-L1-A4-WINDOW-FAMILY-FREEZE-R2"
IHARQ_A4_PROTOCOL_STATUS = "DATA_READY_PROTOCOL_SYNC_REQUIRED"

# Existing official core:
#   cue +0.5 s to +3.5 s, 3.0 seconds, 480 samples at 160 Hz.
#
# Additive A4 R2 extension:
#   LONG_MATCHED_3P5S: cue +0.0 s to +3.5 s, 560 samples.
#   MULTI_3X2S_UNIFORM_0P75S:
#       member 1: +0.0 s  to +2.0 s   (samples 0:320)
#       member 2: +0.75 s to +2.75 s  (samples 120:440)
#       member 3: +1.5 s  to +3.5 s   (samples 240:560)
#
# The released 12,910-event core proves cue+560 is available for every event.
# Only the matched 3.5-second tensor is stored. The three 2-second members are
# immutable Layer-1-registered views into it, preventing redundant overlap
# storage while retaining exact data.

for key, value in {
    "IHARQ_KAGGLE_USERNAME": IHARQ_KAGGLE_USERNAME,
    "IHARQ_EXISTING_CORE_DATASET_HANDLE": (
        IHARQ_EXISTING_CORE_DATASET_HANDLE
    ),
    "IHARQ_EXISTING_CORE_DATASET_VERSION": str(
        IHARQ_EXISTING_CORE_DATASET_VERSION
    ),
    "IHARQ_EXISTING_CORE_MANIFEST_SHA256": (
        IHARQ_EXISTING_CORE_MANIFEST_SHA256
    ),
    "IHARQ_CORE_DATASET_MODE": IHARQ_CORE_DATASET_MODE,
    "IHARQ_ENABLE_A4_EXTENSION": (
        "1" if IHARQ_ENABLE_A4_EXTENSION else "0"
    ),
    "IHARQ_A4_WINDOW_FAMILY_ID": IHARQ_A4_WINDOW_FAMILY_ID,
    "IHARQ_A4_PROTOCOL_STATUS": IHARQ_A4_PROTOCOL_STATUS,
}.items():
    os.environ[key] = value

if not os.environ.get("KAGGLE_API_TOKEN"):
    token = None

    # Prefer a Kaggle secret named KAGGLE_API_TOKEN.
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    except Exception:
        token = None

    if not token:
        token = getpass(
            "Paste the Kaggle API token for csthv999z "
            "(input is hidden): "
        ).strip()

    if not token:
        raise RuntimeError("A non-empty Kaggle API token is required.")

    os.environ["KAGGLE_API_TOKEN"] = token

print(
    "R49 MATCHED A4 R2 CONFIGURATION READY\n"
    f"username={IHARQ_KAGGLE_USERNAME}\n"
    f"core_handle={IHARQ_EXISTING_CORE_DATASET_HANDLE}\n"
    f"core_provider_version={IHARQ_EXISTING_CORE_DATASET_VERSION}\n"
    "core_mode=ADOPT_EXISTING_VERIFIED_DATASET\n"
    "a4_extension=ENABLED\n"
    "token=AVAILABLE_AND_NOT_PRINTED"
)

# 00 — Corrected bootstrap and persistent isolated worker

The bootstrap incorporates all confirmed corrections and installs the R42 core-adoption plus A4-extension implementation.

In [ ]:
from pathlib import Path
import sys, os, re, zipfile, shutil, hashlib, json, subprocess
import importlib
import importlib.metadata as importlib_metadata
import platform
from datetime import datetime, timezone

# P01/L1 R26 revision-neutral, isolated-environment direct official-run bootstrap.
#
# Controlling scientific/implementation authority remains unchanged:
# - Master Build Book: IHARQ-IBB-R10-P01-L1-INDEPENDENT-AUDIT-REPAIRED
# - P01/L1 Annex: IHARQ-IBB-P01-L1-ANNEX-R4
# - Scientific freeze: P01-L1-OFFICIAL-RUN-FREEZE-R2
#
# R26 is an isolated-environment compatibility successor for Kaggle Python 3.12.
# Its connection-critical paths are revision-neutral; R-number changes affect audit metadata only.
# It does not run a global pip check against unrelated Kaggle packages.
# Instead, it installs and verifies the complete IHARQ stack under
# /kaggle/working and places that private dependency layer first on sys.path.
# The runtime consumes the previously verified R6 input bundle, creates a writable
# runtime copy, applies only environment/runtime identity amendments, regenerates
# its manifest, and then runs the unchanged scientific pipeline.

EXPECTED_PYTHON = (3, 12)
OBSERVED_PYTHON = (sys.version_info.major, sys.version_info.minor)
if OBSERVED_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Official R26 execution requires Python "
        f"{EXPECTED_PYTHON[0]}.{EXPECTED_PYTHON[1]}; "
        f"observed {OBSERVED_PYTHON[0]}.{OBSERVED_PYTHON[1]}"
    )

INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
WORK_ROOT = (
    Path("/kaggle/working/iharq_p01_l1")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "_local_notebook_work"
)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

DISPLAY_REVISION = "R54-MATCHED-A4-R2-FINAL-EXPORT-SECRET-SAFE"
NOTEBOOK_REPAIR_ID = "P01-L1-R34-DOWNSTREAM-SCIENTIFIC-AND-FINALIZATION-CONTRACT-CLOSURE-R1"
CONNECTION_CONTRACT_ID = "P01-L1-KAGGLE-REVISION-NEUTRAL-CONNECTION-R1"
RUNTIME_ROOT = WORK_ROOT / "runtime_input" / "base_bundle"
OVERLAY_ROOT = WORK_ROOT / "runtime_overlay"
CONFIG_PATH = OVERLAY_ROOT / "configs" / "phase01_layer1_execution.yaml"


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def _safe_extract(archive_path: Path, destination: Path) -> None:
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            member = Path(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise RuntimeError(f"Unsafe input-bundle path: {info.filename}")
        archive.extractall(destination)


def _is_runtime_bundle_root(path: Path) -> bool:
    path = Path(path)
    return (
        (path / "src" / "iharq" / "layer1_data_protocol" / "kaggle_adapter.py").is_file()
        and (path / "configs" / "phase_01").is_dir()
        and (path / "manifests").is_dir()
    )


def _manifest_candidates(root: Path) -> list[Path]:
    manifest_dir = root / "manifests"
    ordered = []
    generic = manifest_dir / "input_bundle_manifest.json"
    if generic.is_file():
        ordered.append(generic)
    ordered.extend(
        p for p in sorted(manifest_dir.glob("input_bundle_manifest_*.json"))
        if p not in ordered
    )
    return ordered


def _verify_manifest(root: Path, manifest_path: Path):
    try:
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception as exc:
        return None, [{"path": str(manifest_path), "reason": f"MANIFEST_PARSE_FAILED: {exc!r}"}]
    rows = payload.get("files", payload.get("entries", []))
    errors = []
    verified_rows = 0
    for row in rows:
        rel = row.get("path") or row.get("relative_path")
        expected = row.get("sha256")
        if not rel or not expected:
            continue
        verified_rows += 1
        target = root / rel
        if not target.is_file():
            errors.append({"path": rel, "reason": "MISSING"})
        else:
            observed = _sha256(target)
            if observed != expected:
                errors.append({"path": rel, "reason": "SHA256_MISMATCH", "expected": expected, "observed": observed})
    payload["_verified_rows"] = verified_rows
    return payload, errors


def _select_verified_manifest(root: Path):
    attempts = []
    valid = []
    for path in _manifest_candidates(root):
        payload, errors = _verify_manifest(root, path)
        attempts.append({"path": str(path), "errors": errors[:20]})
        if payload is not None and not errors and payload.get("_verified_rows", 0) > 0:
            valid.append((payload.get("_verified_rows", 0), path.name == "input_bundle_manifest.json", path, payload))
    if not valid:
        raise RuntimeError("No valid input-bundle manifest was found. Attempts: " + json.dumps(attempts, indent=2))
    valid.sort(key=lambda row: (row[0], row[1], row[2].name), reverse=True)
    _, _, path, payload = valid[0]
    return path, payload


def _expanded_bundle_roots(input_root: Path) -> list[Path]:
    roots = set()
    for adapter in input_root.rglob("kaggle_adapter.py"):
        if adapter.as_posix().endswith("src/iharq/layer1_data_protocol/kaggle_adapter.py"):
            candidate = adapter.parents[3]
            if _is_runtime_bundle_root(candidate):
                roots.add(candidate.resolve())
    return sorted(roots)


def _archive_bundle_roots(input_root: Path, extraction_parent: Path) -> list[tuple[Path, Path]]:
    roots = []
    extraction_parent.mkdir(parents=True, exist_ok=True)
    for archive_path in sorted(input_root.rglob("*.zip")):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                matching = [
                    name for name in archive.namelist()
                    if name.endswith("src/iharq/layer1_data_protocol/kaggle_adapter.py")
                ]
        except zipfile.BadZipFile:
            continue
        if not matching:
            continue
        extraction = extraction_parent / hashlib.sha256(str(archive_path).encode("utf-8")).hexdigest()[:16]
        if extraction.exists():
            shutil.rmtree(extraction)
        extraction.mkdir(parents=True)
        _safe_extract(archive_path, extraction)
        for name in matching:
            suffix = "src/iharq/layer1_data_protocol/kaggle_adapter.py"
            prefix = name[:-len(suffix)].rstrip("/")
            candidate = extraction / prefix if prefix else extraction
            if _is_runtime_bundle_root(candidate):
                roots.append((candidate.resolve(), archive_path.resolve()))
    return roots


candidate_records = []
for root in _expanded_bundle_roots(INPUT_ROOT):
    try:
        manifest_path, manifest = _select_verified_manifest(root)
        candidate_records.append({
            "root": root,
            "archive": None,
            "manifest_path": manifest_path,
            "manifest": manifest,
            "manifest_sha256": _sha256(manifest_path),
        })
    except Exception:
        pass

archive_extraction_root = WORK_ROOT / "source_archive_candidates"
for root, archive in _archive_bundle_roots(INPUT_ROOT, archive_extraction_root):
    try:
        manifest_path, manifest = _select_verified_manifest(root)
        candidate_records.append({
            "root": root,
            "archive": archive,
            "manifest_path": manifest_path,
            "manifest": manifest,
            "manifest_sha256": _sha256(manifest_path),
        })
    except Exception:
        pass

if not candidate_records:
    raise FileNotFoundError(
        "No valid IHARQ P01/L1 runtime input bundle was found. Attach the verified base bundle as an expanded private Kaggle Dataset or ZIP. Discovery is capability-based and does not depend on its R number."
    )

# Expanded and ZIP views of the same bytes are deduplicated by manifest hash.
by_manifest = {}
for row in candidate_records:
    by_manifest.setdefault(row["manifest_sha256"], row)
if len(by_manifest) != 1:
    choices = [
        {"root": str(row["root"]), "archive": str(row["archive"]) if row["archive"] else None, "manifest": str(row["manifest_path"]), "manifest_sha256": row["manifest_sha256"]}
        for row in by_manifest.values()
    ]
    raise RuntimeError(
        "Multiple different valid IHARQ runtime bundles are attached. Keep exactly one authoritative base bundle attached. Candidates: " + json.dumps(choices, indent=2)
    )

selected = next(iter(by_manifest.values()))
SOURCE_ROOT = selected["root"]
source_manifest_path = selected["manifest_path"]
source_manifest = selected["manifest"]
SOURCE_BUNDLE_NAME = SOURCE_ROOT.name
SOURCE_BUNDLE_ARCHIVE = selected["archive"]
RUNTIME_BUNDLE_NAME = "IHARQ_P01_L1_Runtime_Input"

# Copy the full base package without renaming, editing, deleting, or replacing any inherited file.
if RUNTIME_ROOT.parent.exists():
    shutil.rmtree(RUNTIME_ROOT.parent)
RUNTIME_ROOT.parent.mkdir(parents=True)
shutil.copytree(SOURCE_ROOT, RUNTIME_ROOT)

copy_manifest_errors = []
for row in source_manifest.get("files", source_manifest.get("entries", [])):
    rel = row.get("path") or row.get("relative_path")
    expected = row.get("sha256")
    if not rel or not expected:
        continue
    target = RUNTIME_ROOT / rel
    if not target.is_file():
        copy_manifest_errors.append({"path": rel, "reason": "MISSING_AFTER_COPY"})
    elif _sha256(target) != expected:
        copy_manifest_errors.append({"path": rel, "reason": "HASH_CHANGED_AFTER_COPY"})
if copy_manifest_errors:
    raise RuntimeError("The immutable base copy was not preserved: " + json.dumps(copy_manifest_errors[:20], indent=2))

# Select the canonical inherited config without using its revision number.
config_dir = RUNTIME_ROOT / "configs" / "phase_01"
canonical_config = config_dir / "phase01_layer1_resolved.yaml"
if not canonical_config.is_file():
    config_candidates = sorted(config_dir.glob("phase01_layer1_resolved_*.yaml"))
    if not config_candidates:
        raise RuntimeError("No resolved P01/L1 configuration exists in the verified base bundle.")
    hashes = {_sha256(path) for path in config_candidates}
    if len(hashes) != 1:
        raise RuntimeError(
            "Multiple non-identical revisioned resolved configurations exist and no revision-neutral canonical alias is present. Refusing to guess: "
            + json.dumps([str(path) for path in config_candidates], indent=2)
        )
    canonical_config = config_candidates[0]

if OVERLAY_ROOT.exists():
    shutil.rmtree(OVERLAY_ROOT)
(OVERLAY_ROOT / "configs").mkdir(parents=True)
(OVERLAY_ROOT / "notebook_support").mkdir(parents=True)

# Create an additive execution alias. The inherited canonical and revisioned configs remain untouched.
config_text = canonical_config.read_text(encoding="utf-8")
config_text = re.sub(r"(?m)^(\s*required_major_minor:\s*)['\"]?3\.11['\"]?\s*$", r'\g<1>"3.12"', config_text)
config_text = re.sub(r"(?m)^(\s*constraint:\s*)['\"]?>=3\.11,<3\.12['\"]?\s*$", r'\g<1>">=3.12,<3.13"', config_text)
config_text = config_text.replace("P01-L1-KAGGLE-ENV-FREEZE-R2", "P01-L1-KAGGLE-ENV-COMPAT-PY312-R1")
config_text = re.sub(r"(?m)^(\s*minimum_free_disk_gb:\s*)60(?:\.0)?\s*$", r"\g<1>6", config_text)
config_text = re.sub(r"(?m)^(\s*recommended_free_disk_gb:\s*)90(?:\.0)?\s*$", r"\g<1>18", config_text)
config_text += """

runtime_connection_amendment:
  contract_id: P01-L1-KAGGLE-REVISION-NEUTRAL-CONNECTION-R1
  connection_critical_paths_revisioned: false
  inherited_base_files_preserved: true
  global_revision_rewrite_allowed: false
  execution_config_alias: runtime_overlay/configs/phase01_layer1_execution.yaml

runtime_resource_amendment:
  amendment_id: P01-L1-KAGGLE-ADAPTIVE-DISK-R1
  amendment_scope: KAGGLE_RESOURCE_EXECUTION_ONLY
  legacy_minimum_free_disk_gb: 60
  legacy_recommended_free_disk_gb: 90
  startup_floor_gib: 6.0
  soft_warning_floor_gib: 4.0
  hard_emergency_floor_gib: 1.5
  minimum_export_reserve_gib: 1.5
  export_bundle_multiplier: 1.25
  sequential_dataset_processing: true
  verified_post_load_cache_eviction: true
  automatic_source_removal_allowed: false
  active_sources_unchanged:
    - PhysioNetMI
    - BNCI2014_001
    - Lee2019_MI
  scientific_freeze_unchanged: P01-L1-OFFICIAL-RUN-FREEZE-R2
"""
config_text += """

runtime_acquisition_amendment:
  amendment_id: P01-L1-KAGGLE-THREE-SOURCE-DATASETS-R6
  amendment_scope: THREE_ATTACHED_KAGGLE_SOURCE_DATASETS_ONLY
  scientific_freeze_unchanged: P01-L1-OFFICIAL-RUN-FREEZE-R2
  active_sources_unchanged:
    - PhysioNetMI
    - BNCI2014_001
    - Lee2019_MI
  PhysioNetMI:
    dataset_mode: EXISTING_PUBLIC_KAGGLE_DATASET
    dataset_handle: gamalasran/physionet-eeg-motor-movement-imagery
    exact_subjects: 1-109
    exact_runs:
      - 4
      - 8
      - 12
    expected_files: 327
  BNCI2014_001:
    dataset_mode: OWNER_PRIVATE_KAGGLE_DATASET
    exact_subjects: 1-9
    exact_sessions:
      - T
      - E
    expected_files: 18
  Lee2019_MI:
    dataset_mode: OWNER_PRIVATE_KAGGLE_DATASET
    exact_subjects: 1-54
    exact_sessions:
      - 1
      - 2
    expected_files: 108
    train_run: true
    test_run: false
  moabb_source_downloader: PROHIBITED
  source_network_fallback: PROHIBITED
  moabb_final_resolution_and_loading: PRESERVED
  original_stage07_checksum_inventory: PRESERVED
  exact_subject_provenance: PRESERVED
  verified_cache_eviction: PRESERVED
"""
config_text += """

runtime_bounded_streaming_amendment:
  amendment_id: P01-L1-KAGGLE-DUAL-PERSISTENCE-BOUNDED-STREAMING-R4-STAGE07-TURBO
  amendment_scope: RESOURCE_AND_PERSISTENCE_IMPLEMENTATION_ONLY
  scientific_freeze_unchanged: P01-L1-OFFICIAL-RUN-FREEZE-R2
  scientific_scope_reduction: false
  maximum_concurrent_source_subjects: 8
  disposable_subject_processes: true
  child_rss_hard_limit_gib: 24.0
  minimum_disk_free_gib: 4.0
  progress_interval_seconds: 60
  parent_retains_continuous_signal_arrays: false
  pass_1: SUBJECT_SCOPED_METADATA_EVENT_PROVENANCE_AND_RAW_FIT_STAT_SUMMARY
  pass_2a: DETERMINISTIC_COMBINATION_OF_PASS1_FIT_STATISTICS
  pass_2b: SUBJECT_SCOPED_TRANSFORM_QUALITY_WINDOW_MATERIALIZATION_AND_UPLOAD
  derived_window_storage:
    provider: KAGGLE_PRIVATE_DATASET
    owner: csthv999z
    format: LOSSLESS_HDF5_SUBJECT_SHARDS
    immutable_revision: 1
    delete_local_shard_after_verified_blob_upload: true
  preserved_scope:
    active_sources: [PhysioNetMI, BNCI2014_001, Lee2019_MI]
    all_subjects_sessions_runs: true
    labels_splits_budgets_preprocessing_quality_windows: true
    canonical_records_cards_manifests_gates_handoffs: true
"""

CONFIG_PATH.write_text(config_text, encoding="utf-8")

resource_policy_source = 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport time\n\nPOLICY = {\n    "policy_id": "P01-L1-KAGGLE-ADAPTIVE-DISK-R1",\n    "policy_kind": "RUNTIME_RESOURCE_AMENDMENT_ONLY",\n    "legacy_project_preflight_gb": 60,\n    "legacy_recommended_gb": 90,\n    "startup_floor_gib": 6.0,\n    "startup_fixed_footprint_multiplier": 1.25,\n    "startup_additional_reserve_gib": 3.0,\n    "soft_warning_floor_gib": 4.0,\n    "hard_emergency_floor_gib": 1.5,\n    "minimum_export_reserve_gib": 1.5,\n    "export_bundle_multiplier": 1.25,\n    "export_additional_reserve_gib": 0.5,\n    "active_sources_unchanged": ["PhysioNetMI", "BNCI2014_001", "Lee2019_MI"],\n    "automatic_source_removal_allowed": False,\n    "cache_eviction": "DELETE_ONLY_AFTER_SUCCESSFUL_LOAD_AND_SHA256_REVERIFICATION",\n    "read_only_input_deletion_allowed": False,\n    "scientific_freeze_unchanged": "P01-L1-OFFICIAL-RUN-FREEZE-R2",\n}\n\n_GIB = 1024 ** 3\n_PATCHED_CLASSES: set[type] = set()\n\n\ndef _sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef _directory_size(path: Path) -> int:\n    path = Path(path)\n    if not path.exists():\n        return 0\n    total = 0\n    for item in path.rglob("*"):\n        try:\n            if item.is_file() and not item.is_symlink():\n                total += item.stat().st_size\n        except FileNotFoundError:\n            continue\n    return total\n\n\ndef _is_relative_to(path: Path, base: Path) -> bool:\n    try:\n        path.resolve().relative_to(base.resolve())\n        return True\n    except (ValueError, FileNotFoundError):\n        return False\n\n\ndef _atomic_json(path: Path, payload: Any) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef disk_snapshot(work_root: Path, bundle_root: Path | None = None, event: str | None = None) -> dict[str, Any]:\n    work_root = Path(work_root)\n    usage = shutil.disk_usage(work_root)\n    source_cache = work_root / "source_cache"\n    payload = {\n        "event": event,\n        "captured_unix": time.time(),\n        "disk_total_bytes": usage.total,\n        "disk_used_bytes": usage.used,\n        "disk_free_bytes": usage.free,\n        "disk_free_gib": round(usage.free / _GIB, 3),\n        "source_cache_bytes": _directory_size(source_cache),\n        "runtime_input_bytes": sum(\n            _directory_size(p)\n            for p in [work_root / "runtime_input" / "base_bundle"]\n            if p.is_dir()\n        ),\n        "dependency_layer_bytes": sum(\n            _directory_size(p)\n            for p in [work_root / "python312_deps"]\n            if p.is_dir()\n        ),\n        "bundle_bytes": _directory_size(bundle_root) if bundle_root else 0,\n    }\n    return payload\n\n\ndef _telemetry_path(runner) -> Path:\n    return runner.pipeline.bundle_root / "reports" / "phase_01" / "runtime" / "adaptive_disk_telemetry.jsonl"\n\n\ndef record_snapshot(runner, event: str, extra: dict[str, Any] | None = None) -> dict[str, Any]:\n    row = disk_snapshot(runner.work_root, runner.pipeline.bundle_root, event)\n    if extra:\n        row.update(extra)\n    target = _telemetry_path(runner)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with target.open("a", encoding="utf-8") as stream:\n        stream.write(json.dumps(row, default=str) + "\\n")\n    return row\n\n\ndef startup_requirement_bytes(runner) -> tuple[int, dict[str, Any]]:\n    runtime_bytes = _directory_size(runner.package_root)\n    dependency_bytes = sum(\n        _directory_size(p)\n        for p in [Path(runner.work_root) / "python312_deps"]\n        if p.is_dir()\n    )\n    fixed = runtime_bytes + dependency_bytes\n    calculated = math.ceil(\n        fixed * POLICY["startup_fixed_footprint_multiplier"]\n        + POLICY["startup_additional_reserve_gib"] * _GIB\n    )\n    required = max(math.ceil(POLICY["startup_floor_gib"] * _GIB), calculated)\n    detail = {\n        "runtime_input_bytes": runtime_bytes,\n        "dependency_layer_bytes": dependency_bytes,\n        "fixed_footprint_bytes": fixed,\n        "calculated_required_bytes": calculated,\n        "effective_required_bytes": required,\n        "effective_required_gib": round(required / _GIB, 3),\n    }\n    return required, detail\n\n\ndef export_requirement_bytes(runner) -> tuple[int, dict[str, Any]]:\n    bundle_bytes = _directory_size(runner.pipeline.bundle_root)\n    calculated = math.ceil(\n        bundle_bytes * POLICY["export_bundle_multiplier"]\n        + POLICY["export_additional_reserve_gib"] * _GIB\n    )\n    required = max(math.ceil(POLICY["minimum_export_reserve_gib"] * _GIB), calculated)\n    return required, {\n        "bundle_bytes": bundle_bytes,\n        "calculated_required_bytes": calculated,\n        "effective_required_bytes": required,\n        "effective_required_gib": round(required / _GIB, 3),\n    }\n\n\ndef _remove_blocker(rows: list[dict[str, Any]], code: str) -> list[dict[str, Any]]:\n    return [row for row in rows if row.get("code") != code]\n\n\ndef _stage_block(runner, stage: str, code: str, observations: dict[str, Any]):\n    blocker = {"code": code, "owner": "KAGGLE_RUNTIME", **observations}\n    if blocker not in runner.pipeline.blockers:\n        runner.pipeline.blockers.append(blocker)\n    return runner._record(stage, "BLOCKED", observations=observations, blockers=[blocker])\n\n\ndef _install_stage01_patch(runner) -> None:\n    original = runner.stage_01\n\n    def adaptive_stage_01():\n        result = original()\n        # The 60 GiB value remains preserved as the legacy authority threshold,\n        # but the current Kaggle execution is governed by this measured amendment.\n        result.blockers = _remove_blocker(list(result.blockers), "FREE_DISK_BELOW_60_GB")\n        runner.pipeline.blockers = _remove_blocker(\n            list(runner.pipeline.blockers), "FREE_DISK_BELOW_60_GB"\n        )\n        observations = dict(result.observations)\n        observations["resource_failures"] = [\n            code for code in observations.get("resource_failures", [])\n            if code != "FREE_DISK_BELOW_60_GB"\n        ]\n        required, calculation = startup_requirement_bytes(runner)\n        snap = record_snapshot(runner, "STAGE_01_ADAPTIVE_PREFLIGHT")\n        observations.update({\n            "resource_policy_id": POLICY["policy_id"],\n            "legacy_60_gb_preflight_applied": False,\n            "legacy_60_gb_preflight_preserved_as_history": True,\n            "adaptive_startup_calculation": calculation,\n            "disk_snapshot": snap,\n            "automatic_source_removal_allowed": False,\n        })\n        if snap["disk_free_bytes"] < required:\n            blocker = {\n                "code": "FREE_DISK_BELOW_ADAPTIVE_STARTUP_REQUIREMENT",\n                "owner": "KAGGLE_RUNTIME",\n                "required_bytes": required,\n                "required_gib": round(required / _GIB, 3),\n                "observed_free_bytes": snap["disk_free_bytes"],\n                "observed_free_gib": snap["disk_free_gib"],\n            }\n            result.blockers.append(blocker)\n            runner.pipeline.blockers.append(blocker)\n            observations["resource_failures"].append(blocker["code"])\n            result.status = "BLOCKED"\n        elif result.blockers:\n            # Preserve any non-disk blocker emitted by the original environment gate.\n            result.status = "BLOCKED"\n        else:\n            result.status = "PASS"\n        result.observations = observations\n        environment_path = runner.pipeline.bundle_root / "environment_manifest.json"\n        if environment_path.exists():\n            _atomic_json(environment_path, observations)\n        _atomic_json(\n            runner.pipeline.bundle_root / "reports" / "phase_01" / "runtime" / "adaptive_disk_policy.json",\n            {"policy": POLICY, "startup_calculation": calculation, "snapshot": snap},\n        )\n        return result\n\n    runner.stage_01 = adaptive_stage_01\n\n\ndef _install_adapter_eviction_patch(runner) -> None:\n    from iharq.layer1_data_protocol import adapters as adapters_module\n\n    for adapter_class in set(adapters_module.ADAPTERS.values()):\n        if adapter_class in _PATCHED_CLASSES:\n            continue\n        original_verify = adapter_class.verify_files\n        original_load = adapter_class.load\n\n        def verify_files(self, files, _original=original_verify):\n            inventory = _original(self, files)\n            rows = list(inventory.get("files", []))\n            verified = {}\n            for index, raw_path in enumerate(files):\n                path = Path(raw_path).resolve()\n                row = rows[index] if index < len(rows) else {}\n                expected = row.get("sha256")\n                if expected:\n                    verified[str(path)] = expected\n            self._iharq_verified_source_hashes = verified\n            self._iharq_verified_inventory = inventory\n            return inventory\n\n        def load(self, files, _original=original_load):\n            before = record_snapshot(\n                runner,\n                "ADAPTER_LOAD_START",\n                {"dataset_id": self.profile.dataset_id, "file_count": len(files)},\n            )\n            recordings = _original(self, files)\n            verified = getattr(self, "_iharq_verified_source_hashes", {})\n            cache_root = Path(self.cache_root).resolve()\n            input_root = Path(self.input_root).resolve()\n            deleted = []\n            skipped = []\n            failures = []\n            for raw_path in files:\n                path = Path(raw_path).resolve()\n                expected = verified.get(str(path))\n                if _is_relative_to(path, input_root):\n                    skipped.append({"path": str(path), "reason": "READ_ONLY_INPUT"})\n                    continue\n                if not _is_relative_to(path, cache_root):\n                    skipped.append({"path": str(path), "reason": "OUTSIDE_DECLARED_CACHE_ROOT"})\n                    continue\n                try:\n                    if not path.is_file():\n                        skipped.append({"path": str(path), "reason": "ALREADY_ABSENT"})\n                        continue\n                    if not expected:\n                        skipped.append({"path": str(path), "reason": "NO_VERIFIED_HASH"})\n                        continue\n                    observed = _sha256(path)\n                    if observed != expected:\n                        failures.append({\n                            "path": str(path),\n                            "reason": "POST_LOAD_HASH_MISMATCH",\n                            "expected": expected,\n                            "observed": observed,\n                        })\n                        continue\n                    size = path.stat().st_size\n                    path.unlink()\n                    deleted.append({"path": str(path), "bytes": size, "sha256": observed})\n                except Exception as exc:\n                    failures.append({"path": str(path), "reason": repr(exc)})\n            # Remove empty cache directories only; never remove a non-empty directory.\n            if cache_root.exists():\n                for directory in sorted(\n                    [p for p in cache_root.rglob("*") if p.is_dir()],\n                    key=lambda p: len(p.parts),\n                    reverse=True,\n                ):\n                    try:\n                        directory.rmdir()\n                    except OSError:\n                        pass\n            after = record_snapshot(\n                runner,\n                "ADAPTER_LOAD_COMPLETE_AND_CACHE_EVICTED",\n                {\n                    "dataset_id": self.profile.dataset_id,\n                    "recordings_loaded": len(recordings),\n                    "deleted_files": len(deleted),\n                    "deleted_bytes": sum(row["bytes"] for row in deleted),\n                    "eviction_failures": len(failures),\n                },\n            )\n            report = {\n                "policy_id": POLICY["policy_id"],\n                "dataset_id": self.profile.dataset_id,\n                "before": before,\n                "after": after,\n                "deleted": deleted,\n                "skipped": skipped,\n                "failures": failures,\n                "recordings_loaded": len(recordings),\n                "source_bytes_embedded_in_execution_bundle": False,\n            }\n            _atomic_json(\n                runner.pipeline.bundle_root\n                / "reports" / "phase_01" / "runtime" / "cache_eviction"\n                / f"{self.profile.dataset_id}.json",\n                report,\n            )\n            return recordings\n\n        adapter_class.verify_files = verify_files\n        adapter_class.load = load\n        _PATCHED_CLASSES.add(adapter_class)\n\n\ndef _install_stage_guard(runner) -> None:\n    original_run_stage = runner.run_stage\n\n    def guarded_run_stage(stage: str):\n        stage = str(stage)\n        before = record_snapshot(runner, f"STAGE_{stage}_PRE")\n        hard_floor = math.ceil(POLICY["hard_emergency_floor_gib"] * _GIB)\n        if stage not in {"00", "01"} and before["disk_free_bytes"] < hard_floor:\n            return _stage_block(\n                runner,\n                stage,\n                "FREE_DISK_BELOW_HARD_EMERGENCY_FLOOR",\n                {\n                    "observed_free_bytes": before["disk_free_bytes"],\n                    "observed_free_gib": before["disk_free_gib"],\n                    "required_bytes": hard_floor,\n                    "required_gib": POLICY["hard_emergency_floor_gib"],\n                    "policy_id": POLICY["policy_id"],\n                },\n            )\n        if stage in {"25", "26"}:\n            required, detail = export_requirement_bytes(runner)\n            if before["disk_free_bytes"] < required:\n                return _stage_block(\n                    runner,\n                    stage,\n                    "FREE_DISK_BELOW_DYNAMIC_EXPORT_RESERVE",\n                    {\n                        "observed_free_bytes": before["disk_free_bytes"],\n                        "observed_free_gib": before["disk_free_gib"],\n                        "export_calculation": detail,\n                        "policy_id": POLICY["policy_id"],\n                    },\n                )\n        result = original_run_stage(stage)\n        after = record_snapshot(runner, f"STAGE_{stage}_POST")\n        result.observations = dict(result.observations)\n        result.observations["adaptive_disk_post_stage"] = after\n        if after["disk_free_bytes"] < hard_floor and result.status == "PASS":\n            blocker = {\n                "code": "FREE_DISK_BELOW_HARD_EMERGENCY_FLOOR_AFTER_STAGE",\n                "owner": "KAGGLE_RUNTIME",\n                "stage": stage,\n                "observed_free_bytes": after["disk_free_bytes"],\n                "observed_free_gib": after["disk_free_gib"],\n            }\n            result.blockers.append(blocker)\n            runner.pipeline.blockers.append(blocker)\n            result.status = "BLOCKED"\n        return result\n\n    runner.run_stage = guarded_run_stage\n\n\ndef _install_stage00_metadata_patch(runner) -> None:\n    original = runner.stage_00\n\n    def revision_neutral_stage_00():\n        result = original()\n        observations = dict(result.observations)\n        base_revision = observations.get("notebook_revision")\n        observations.update({\n            "base_bundle_runtime_revision": base_revision,\n            "notebook_revision": os.environ.get("IHARQ_NOTEBOOK_REVISION", "R26"),\n            "connection_contract": "P01-L1-KAGGLE-REVISION-NEUTRAL-CONNECTION-R1",\n            "connection_critical_paths_revisioned": False,\n            "base_bundle_files_preserved": True,\n        })\n        result.observations = observations\n        authority_path = runner.pipeline.bundle_root / "authority_manifest.json"\n        if authority_path.exists():\n            _atomic_json(authority_path, observations)\n        return result\n\n    runner.stage_00 = revision_neutral_stage_00\n\ndef install_resource_policy(runner) -> dict[str, Any]:\n    _install_stage00_metadata_patch(runner)\n    _install_stage01_patch(runner)\n    _install_adapter_eviction_patch(runner)\n    _install_stage_guard(runner)\n    report = {\n        "policy": POLICY,\n        "installed_at_unix": time.time(),\n        "work_root": str(runner.work_root),\n        "bundle_root": str(runner.pipeline.bundle_root),\n        "runtime_package_root": str(runner.package_root),\n    }\n    _atomic_json(\n        runner.pipeline.bundle_root / "reports" / "phase_01" / "runtime" / "adaptive_disk_installation.json",\n        report,\n    )\n    return report\n\n\ndef emergency_floor_bytes() -> int:\n    return math.ceil(POLICY["hard_emergency_floor_gib"] * _GIB)\n\n\ndef write_emergency_record(work_root: Path, bundle_root: Path, stage: str, snapshot: dict[str, Any]) -> Path:\n    payload = {\n        "code": "FREE_DISK_BELOW_HARD_EMERGENCY_FLOOR_DURING_STAGE",\n        "owner": "KAGGLE_RUNTIME",\n        "stage": stage,\n        "policy": POLICY,\n        "snapshot": snapshot,\n        "action": "WORKER_TERMINATED_TO_PREVENT_FILESYSTEM_EXHAUSTION",\n    }\n    target = Path(work_root) / "IHARQ_P01_L1_RESOURCE_EMERGENCY.json"\n    _atomic_json(target, payload)\n    try:\n        _atomic_json(\n            Path(bundle_root) / "negative_and_failed_results" / "resource_emergency.json",\n            payload,\n        )\n    except Exception:\n        pass\n    return target\n'
resource_policy_path = OVERLAY_ROOT / "iharq_resource_policy_adaptive.py"
resource_policy_path.write_text(resource_policy_source, encoding="utf-8")
policy_document_path = OVERLAY_ROOT / "notebook_support" / "IHARQ_P01_L1_Kaggle_Adaptive_Disk_Policy.json"
policy_document_path.write_text("{\n  \"policy_id\": \"P01-L1-KAGGLE-ADAPTIVE-DISK-R1\",\n  \"connection_contract\": \"P01-L1-KAGGLE-REVISION-NEUTRAL-CONNECTION-R1\",\n  \"display_runtime_successor\": \"R26\",\n  \"controlling_build_book\": \"IHARQ-IBB-R10-P01-L1-INDEPENDENT-AUDIT-REPAIRED\",\n  \"controlling_annex\": \"IHARQ-IBB-P01-L1-ANNEX-R4\",\n  \"scientific_freeze\": \"P01-L1-OFFICIAL-RUN-FREEZE-R2\",\n  \"amendment_scope\": \"KAGGLE_RESOURCE_AND_RUNTIME_CONNECTION_ONLY\",\n  \"legacy_thresholds\": {\n    \"minimum_free_disk_gb\": 60,\n    \"recommended_free_disk_gb\": 90\n  },\n  \"adaptive_policy\": {\n    \"startup_floor_gib\": 6.0,\n    \"startup_fixed_footprint_multiplier\": 1.25,\n    \"startup_additional_reserve_gib\": 3.0,\n    \"soft_warning_floor_gib\": 4.0,\n    \"hard_emergency_floor_gib\": 1.5,\n    \"minimum_export_reserve_gib\": 1.5,\n    \"export_bundle_multiplier\": 1.25,\n    \"export_additional_reserve_gib\": 0.5\n  },\n  \"revision_neutral_paths\": {\n    \"runtime_base\": \"runtime_input/base_bundle\",\n    \"runtime_overlay\": \"runtime_overlay\",\n    \"execution_config\": \"runtime_overlay/configs/phase01_layer1_execution.yaml\",\n    \"dependency_layer\": \"python312_deps\",\n    \"wheel_cache\": \"verified_wheels\",\n    \"worker\": \"worker/iharq_stage_worker.py\",\n    \"notebook_manifest\": \"notebook_manifest.json\",\n    \"environment_manifest\": \"environment_amendment.json\",\n    \"dependency_manifest\": \"isolated_dependency_layer.json\"\n  },\n  \"storage_controls\": [\n    \"measure actual free disk before and after every stage\",\n    \"process active datasets in the existing sequential dataset order\",\n    \"delete only SHA-256-reverified writable provider-cache files after successful load\",\n    \"never delete attached read-only Kaggle input files\",\n    \"reserve dynamic space before final ZIP creation\",\n    \"terminate before filesystem exhaustion\",\n    \"preserve all active sources; no automatic dataset removal\"\n  ],\n  \"connection_controls\": [\n    \"discover the attached base bundle by required capabilities, not by R-numbered folder name\",\n    \"verify an existing source manifest before copying\",\n    \"copy the complete base bundle byte-for-byte without renaming inherited files\",\n    \"keep the original revisioned configuration and manifest files in place\",\n    \"create one revision-neutral execution-config alias outside the base package\",\n    \"store runtime amendments in an additive overlay outside the base package\",\n    \"never globally replace R-number strings in inherited source files\",\n    \"never delete or overwrite inherited input-bundle manifests\"\n  ],\n  \"honesty_boundary\": \"The policy reduces peak writable-disk use but cannot prove that the largest individual provider acquisition fits before the official run measures it.\"\n}", encoding="utf-8")
connection_contract_path = OVERLAY_ROOT / "notebook_support" / "IHARQ_P01_L1_Revision_Neutral_Connection_Contract.json"
connection_contract_path.write_text("{\n  \"contract_id\": \"P01-L1-KAGGLE-REVISION-NEUTRAL-CONNECTION-R1\",\n  \"display_revision\": \"R26\",\n  \"purpose\": \"Prevent notebook display-revision changes from breaking base-bundle discovery, inherited file paths, config selection, worker startup, dependency paths, or Stage 02 completeness checks.\",\n  \"immutable_base_rule\": \"All inherited base-bundle files retain their original names, bytes, and relative paths.\",\n  \"overlay_rule\": \"Every Python 3.12, worker-control, adaptive-disk, and notebook-revision amendment is additive and stored outside the immutable base package.\",\n  \"connection_critical_paths\": {\n    \"runtime_base\": \"runtime_input/base_bundle\",\n    \"runtime_overlay\": \"runtime_overlay\",\n    \"execution_config\": \"runtime_overlay/configs/phase01_layer1_execution.yaml\",\n    \"dependency_layer\": \"python312_deps\",\n    \"wheel_cache\": \"verified_wheels\",\n    \"worker\": \"worker/iharq_stage_worker.py\",\n    \"notebook_manifest\": \"notebook_manifest.json\",\n    \"environment_manifest\": \"environment_amendment.json\",\n    \"dependency_manifest\": \"isolated_dependency_layer.json\"\n  },\n  \"prohibited_operations\": [\n    \"global R-number search-and-replace across the base bundle\",\n    \"renaming inherited R-numbered files\",\n    \"deleting or replacing inherited input_bundle_manifest files\",\n    \"selecting a base bundle solely from its filename\",\n    \"placing amended content under an inherited revisioned filename\"\n  ],\n  \"future_revision_rule\": \"A future R-number update may change notebook metadata, filenames, and audit labels only. It must not change any connection-critical path or inherited base-bundle path.\"\n}", encoding="utf-8")

acquisition_acceleration_source = 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Callable\nimport hashlib\nimport inspect\nimport json\nimport logging\nimport os\nimport re\nimport threading\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\n\nPOLICY = {\n    "policy_id": "P01-L1-KAGGLE-THREE-SOURCE-DATASETS-R6",\n    "policy_kind": "THREE_ATTACHED_KAGGLE_SOURCE_DATASETS_ONLY",\n    "runtime_revision": "R26",\n    "runtime_compatibility_revision": "R34-BNCI-EXACT-NATIVE-LOAD-CHECKSUM-AND-DOWNSTREAM-CONTRACT-CLOSURE",\n    "scientific_freeze_unchanged": "P01-L1-OFFICIAL-RUN-FREEZE-R2",\n    "active_sources": ["PhysioNetMI", "BNCI2014_001", "Lee2019_MI"],\n    "physionet_public_handle": "gamalasran/physionet-eeg-motor-movement-imagery",\n    "source_manifest_filename": "IHARQ_P01_L1_SOURCE_DATASET_MANIFEST.json",\n    "required_file_counts": {"PhysioNetMI": 327, "BNCI2014_001": 18, "Lee2019_MI": 108},\n    "moabb_downloader_used": False,\n    "moabb_downloader_fallback_allowed": False,\n    "source_network_download_allowed": False,\n    "moabb_final_resolution_and_loading": True,\n    "loading_remains_sequential": True,\n    "loading_parallelism_scope": "BOUNDED_CROSS_SUBJECT_ONLY",\n    "checksum_verification_unchanged": True,\n    "exact_subject_provenance_unchanged": True,\n    "subject_88_special_handling_unchanged": True,\n    "ordinal_prefix_resolution_supported": True,\n}\n\n_PATCHED: set[type] = set()\n_LOCK = threading.Lock()\n_SOURCE_MAP: dict[str, dict[Any, Path]] = {}\n_SOURCE_REPORT: dict[str, Any] = {"status": "NOT_PREPARED"}\n_PRIVATE_INVENTORY: dict[str, dict[str, Any]] = {}\n_RESOLUTION_FILE: Path | None = None\n\n\ndef _atomic_json(path: Path, payload: Any) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(payload, indent=2, default=str) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _aggregate_file_digest(rows: list[dict[str, Any]]) -> str:\n    h = hashlib.sha256()\n    for row in sorted(rows, key=lambda x: x["relative_path"]):\n        h.update(str(row["relative_path"]).encode("utf-8")); h.update(b"\\0")\n        h.update(str(row["sha256"]).encode("ascii")); h.update(b"\\0")\n        h.update(str(row["bytes"]).encode("ascii")); h.update(b"\\n")\n    return h.hexdigest()\n\n\ndef _get(value: Any, name: str, default: Any = None) -> Any:\n    if isinstance(value, dict):\n        return value.get(name, default)\n    return getattr(value, name, default)\n\n\ndef _dataset_id(adapter: Any) -> str:\n    profile = _get(adapter, "profile")\n    identifier = _get(profile, "dataset_id")\n    if not identifier:\n        raise RuntimeError(f"Adapter {type(adapter).__name__} has no dataset_id.")\n    return str(identifier)\n\n\ndef _call_filtered(original: Callable[..., Any], self: Any, args: tuple[Any, ...], kwargs: dict[str, Any], forced: dict[str, Any] | None = None) -> Any:\n    signature = inspect.signature(original)\n    parameters = signature.parameters\n    accepts_var_kwargs = any(\n        parameter.kind is inspect.Parameter.VAR_KEYWORD\n        for parameter in parameters.values()\n    )\n    cleaned = dict(kwargs) if accepts_var_kwargs else {k: v for k, v in kwargs.items() if k in parameters}\n    for key, value in (forced or {}).items():\n        if accepts_var_kwargs or key in parameters:\n            cleaned[key] = value\n    bound = signature.bind_partial(self, *args, **cleaned)\n    return original(*bound.args, **bound.kwargs)\n\n\ndef _resolve_manifest_payload_file(manifest_path: Path, canonical_relative: str, expected_bytes: int, expected_sha256: str) -> Path:\n    """Resolve exact names and Kaggle ordinal-prefixed names without ambiguity."""\n    base = manifest_path.parent\n    canonical = Path(canonical_relative)\n    candidates: list[Path] = []\n    direct = base / canonical\n    if direct.is_file():\n        candidates.append(direct)\n    leaf = canonical.name\n    for pattern in (leaf, f"*_{leaf}"):\n        candidates.extend(path for path in base.rglob(pattern) if path.is_file())\n    # Preserve order but remove duplicates.\n    unique: list[Path] = []\n    seen: set[str] = set()\n    for path in candidates:\n        key = str(path.resolve())\n        if key not in seen:\n            seen.add(key); unique.append(path)\n    sized = [path for path in unique if path.stat().st_size == int(expected_bytes)]\n    if not sized:\n        raise RuntimeError(\n            "SOURCE_DATASET_FILE_MISSING_OR_SIZE_MISMATCH: "\n            f"manifest={manifest_path}; canonical={canonical_relative}; expected_bytes={expected_bytes}"\n        )\n    verified = []\n    for path in sized:\n        observed = _sha256(path)\n        if observed == str(expected_sha256).lower():\n            verified.append(path)\n    if len(verified) != 1:\n        raise RuntimeError(\n            "SOURCE_DATASET_FILE_RESOLUTION_AMBIGUOUS: "\n            f"canonical={canonical_relative}; size_matches={len(sized)}; hash_matches={len(verified)}; "\n            f"candidates={[str(p) for p in sized[:10]]}"\n        )\n    return verified[0]\n\n\ndef _discover_physionet(input_root: Path) -> dict[Any, Path]:\n    selected: dict[tuple[int, int], Path] = {}\n    for path in input_root.rglob("*.edf"):\n        match = re.fullmatch(r"S(\\d{3})R(\\d{2})\\.edf", path.name, flags=re.IGNORECASE)\n        if not match:\n            continue\n        subject, run = int(match.group(1)), int(match.group(2))\n        if not (1 <= subject <= 109 and run in {4, 8, 12}):\n            continue\n        key = (subject, run)\n        if key in selected and selected[key].resolve() != path.resolve():\n            raise RuntimeError(f"PHYSIONET_DUPLICATE_ATTACHED_SOURCE: key={key}")\n        selected[key] = path\n    expected = {(s, r) for s in range(1, 110) for r in (4, 8, 12)}\n    missing = sorted(expected - set(selected))\n    if missing:\n        raise RuntimeError(\n            "PHYSIONET_ATTACHED_DATASET_INCOMPLETE: "\n            f"required=327; observed={len(selected)}; missing_sample={missing[:10]}"\n        )\n    return selected\n\n\ndef _load_private_source_manifests(input_root: Path) -> tuple[dict[Any, Path], dict[Any, Path], list[dict[str, Any]]]:\n    """Resolve and SHA-256 verify private source files with one directory scan.\n\n    Quality is unchanged: every declared physical byte stream is still checked\n    against its manifest SHA-256. Speed comes from eliminating repeated rglob\n    scans and hashing independent files concurrently with a bounded pool.\n    """\n    global _PRIVATE_INVENTORY\n    marker = POLICY["source_manifest_filename"]\n    manifests: list[dict[str, Any]] = []\n    bnci: dict[tuple[int, str], Path] = {}\n    lee: dict[tuple[int, int], Path] = {}\n    inventories: dict[str, dict[str, Any]] = {}\n\n    manifest_paths = sorted(input_root.rglob(marker))\n    selected_manifests: list[tuple[Path, dict[str, Any]]] = []\n    for path in manifest_paths:\n        manifest = json.loads(path.read_text(encoding="utf-8"))\n        if manifest.get("cache_kind") != "IHARQ_P01_L1_SOURCE_DATASET":\n            continue\n        if manifest.get("scientific_freeze") != POLICY["scientific_freeze_unchanged"]:\n            raise RuntimeError(f"SOURCE_DATASET_FREEZE_MISMATCH: {path}")\n        source_id = manifest.get("source_id")\n        if source_id not in {"BNCI2014_001", "Lee2019_MI"}:\n            continue\n        if not manifest.get("complete_for_official_run"):\n            raise RuntimeError(f"SOURCE_DATASET_NOT_COMPLETE: {source_id}; {path}")\n        selected_manifests.append((path, manifest))\n\n    if len([1 for _, m in selected_manifests if m.get("source_id") == "BNCI2014_001"]) != 1:\n        raise RuntimeError("Attach exactly one BNCI2014_001 source Dataset.")\n    if len([1 for _, m in selected_manifests if m.get("source_id") == "Lee2019_MI"]) != 1:\n        raise RuntimeError("Attach exactly one Lee2019_MI source Dataset.")\n\n    hash_workers = max(1, min(\n        int(os.environ.get("IHARQ_SOURCE_HASH_WORKERS", "8")),\n        os.cpu_count() or 1,\n        8,\n    ))\n\n    for path, manifest in selected_manifests:\n        source_id = str(manifest["source_id"])\n        declared_rows = list(manifest.get("files", []))\n        base = path.parent\n\n        # One physical-tree scan per attached Dataset, rather than two rglob\n        # operations for every manifest row.\n        physical_files = [candidate for candidate in base.rglob("*") if candidate.is_file()]\n        by_leaf: dict[str, list[Path]] = {}\n        for candidate in physical_files:\n            by_leaf.setdefault(candidate.name, []).append(candidate)\n\n        resolution_jobs: list[tuple[dict[str, Any], list[Path]]] = []\n        for row in declared_rows:\n            canonical_relative = str(row["relative_path"])\n            canonical_name = Path(canonical_relative).name\n            expected_bytes = int(row["bytes"])\n            candidates: list[Path] = []\n            direct = base / Path(canonical_relative)\n            if direct.is_file():\n                candidates.append(direct)\n            candidates.extend(by_leaf.get(canonical_name, []))\n            suffix = "_" + canonical_name\n            candidates.extend(\n                candidate\n                for leaf, paths in by_leaf.items()\n                if leaf.endswith(suffix)\n                for candidate in paths\n            )\n            unique: list[Path] = []\n            seen: set[str] = set()\n            for candidate in candidates:\n                key = str(candidate.resolve())\n                if key not in seen:\n                    seen.add(key)\n                    unique.append(candidate)\n            sized = [candidate for candidate in unique if candidate.stat().st_size == expected_bytes]\n            if not sized:\n                raise RuntimeError(\n                    "SOURCE_DATASET_FILE_MISSING_OR_SIZE_MISMATCH: "\n                    f"manifest={path}; canonical={canonical_relative}; expected_bytes={expected_bytes}"\n                )\n            resolution_jobs.append((row, sized))\n\n        hash_tasks: list[tuple[int, Path]] = []\n        for row_index, (_, candidates) in enumerate(resolution_jobs):\n            for candidate in candidates:\n                hash_tasks.append((row_index, candidate))\n\n        def hash_task(item: tuple[int, Path]) -> tuple[int, Path, str]:\n            row_index, candidate = item\n            return row_index, candidate, _sha256(candidate)\n\n        hashed_by_row: dict[int, list[tuple[Path, str]]] = {}\n        with ThreadPoolExecutor(max_workers=hash_workers, thread_name_prefix="iharq-source-sha256") as pool:\n            for row_index, candidate, observed_sha in pool.map(hash_task, hash_tasks):\n                hashed_by_row.setdefault(row_index, []).append((candidate, observed_sha))\n\n        canonical_rows: list[dict[str, Any]] = []\n        for row_index, (row, candidates) in enumerate(resolution_jobs):\n            canonical_relative = str(row["relative_path"])\n            canonical_name = Path(canonical_relative).name\n            expected_sha = str(row["sha256"]).lower()\n            verified = [\n                candidate\n                for candidate, observed_sha in hashed_by_row.get(row_index, [])\n                if observed_sha.lower() == expected_sha\n            ]\n            if len(verified) != 1:\n                raise RuntimeError(\n                    "SOURCE_DATASET_FILE_RESOLUTION_AMBIGUOUS: "\n                    f"canonical={canonical_relative}; size_matches={len(candidates)}; "\n                    f"hash_matches={len(verified)}; candidates={[str(p) for p in candidates[:10]]}"\n                )\n            source = verified[0]\n            canonical_rows.append({\n                "relative_path": canonical_relative,\n                "runtime_path_class": "KAGGLE_INPUT",\n                "runtime_path": str(source),\n                "sha256": expected_sha,\n                "bytes": int(row["bytes"]),\n                "canonical_filename": canonical_name,\n                "physical_filename": source.name,\n                "ordinal_prefix_resolved": source.name != canonical_name,\n            })\n            if source_id == "BNCI2014_001":\n                match = re.fullmatch(r"A(\\d{2})(T|E)\\.mat", canonical_name)\n                if not match:\n                    raise RuntimeError(f"BNCI_SOURCE_FILENAME_INVALID: {canonical_name}")\n                key = (int(match.group(1)), match.group(2))\n                if key in bnci:\n                    raise RuntimeError(f"BNCI_DUPLICATE_SOURCE_FILE: {key}")\n                bnci[key] = source\n            else:\n                match = re.fullmatch(r"sess(\\d{2})_subj(\\d{2})_EEG_MI\\.mat", canonical_name)\n                if not match:\n                    raise RuntimeError(f"LEE_SOURCE_FILENAME_INVALID: {canonical_name}")\n                session, subject = int(match.group(1)), int(match.group(2))\n                key = (subject, session)\n                if key in lee:\n                    raise RuntimeError(f"LEE_DUPLICATE_SOURCE_FILE: {key}")\n                lee[key] = source\n\n        aggregate = _aggregate_file_digest(canonical_rows)\n        inventories[source_id] = {\n            "files": canonical_rows,\n            "count": len(canonical_rows),\n            "aggregate_sha256": aggregate,\n            "observed_checksum": aggregate,\n            "expected_checksum": aggregate,\n            "checksum_status": "VERIFIED_FROM_SOURCE_DATASET_MANIFEST_AND_PHYSICAL_BYTES",\n            "manifest_path": str(path),\n            "verification_strategy": "ONE_SCAN_BOUNDED_PARALLEL_SHA256",\n            "hash_workers": hash_workers,\n        }\n        manifests.append({\n            "source_id": source_id,\n            "dataset_handle": manifest.get("dataset_handle"),\n            "manifest": str(path),\n            "file_count": len(canonical_rows),\n            "bytes": sum(int(row["bytes"]) for row in canonical_rows),\n            "aggregate_sha256": aggregate,\n            "ordinal_prefixed_files": sum(bool(row["ordinal_prefix_resolved"]) for row in canonical_rows),\n            "verification_strategy": "ONE_SCAN_BOUNDED_PARALLEL_SHA256",\n            "hash_workers": hash_workers,\n        })\n\n    if len(bnci) != 18:\n        raise RuntimeError(f"BNCI_ATTACHED_DATASET_INCOMPLETE: required=18; observed={len(bnci)}")\n    if len(lee) != 108:\n        raise RuntimeError(f"LEE_ATTACHED_DATASET_INCOMPLETE: required=108; observed={len(lee)}")\n    _PRIVATE_INVENTORY = inventories\n    return bnci, lee, manifests\n\n\ndef _serialize_source_map(path: Path) -> None:\n    payload = {\n        "policy_id": POLICY["policy_id"],\n        "sources": {\n            dataset: {"|".join(str(x) for x in key): str(value) for key, value in mapping.items()}\n            for dataset, mapping in _SOURCE_MAP.items()\n        },\n        "private_inventory": _PRIVATE_INVENTORY,\n    }\n    _atomic_json(path, payload)\n\n\ndef _load_source_map(path: Path) -> None:\n    global _SOURCE_MAP, _PRIVATE_INVENTORY\n    payload = json.loads(Path(path).read_text(encoding="utf-8"))\n    result: dict[str, dict[Any, Path]] = {}\n    for dataset, mapping in payload["sources"].items():\n        converted: dict[Any, Path] = {}\n        for raw_key, raw_path in mapping.items():\n            parts = raw_key.split("|")\n            if dataset == "BNCI2014_001":\n                key = (int(parts[0]), parts[1])\n            else:\n                key = (int(parts[0]), int(parts[1]))\n            converted[key] = Path(raw_path)\n        result[dataset] = converted\n    _SOURCE_MAP = result\n    _PRIVATE_INVENTORY = dict(payload.get("private_inventory", {}))\n\n\ndef _prepare_sources(runner: Any, child_mode: bool = False, resolution_file: Path | None = None) -> dict[str, Any]:\n    global _SOURCE_MAP, _SOURCE_REPORT, _RESOLUTION_FILE\n\n    # Optional same-session worker restart fast path. The source-resolution\n    # file was created only after full per-file SHA-256 verification, and all\n    # referenced source bytes live under Kaggle\'s read-only input mount.\n    # Reuse is accepted only when the caller supplies the exact resolution-file\n    # SHA-256 and the complete frozen source cardinalities still match.\n    reuse_raw = os.environ.get("IHARQ_REUSE_VERIFIED_SOURCE_RESOLUTION_FILE")\n    if not child_mode and reuse_raw:\n        reuse_path = Path(reuse_raw).resolve()\n        expected_resolution_sha256 = os.environ.get(\n            "IHARQ_REUSE_VERIFIED_SOURCE_RESOLUTION_SHA256", ""\n        ).strip().lower()\n        if not reuse_path.is_file():\n            raise RuntimeError(\n                "R34_REUSED_SOURCE_RESOLUTION_MISSING: "\n                f"{reuse_path}"\n            )\n        observed_resolution_sha256 = _sha256(reuse_path)\n        if (\n            not re.fullmatch(r"[0-9a-f]{64}", expected_resolution_sha256)\n            or observed_resolution_sha256 != expected_resolution_sha256\n        ):\n            raise RuntimeError(\n                "R34_REUSED_SOURCE_RESOLUTION_SHA256_MISMATCH: "\n                f"expected={expected_resolution_sha256!r}; "\n                f"observed={observed_resolution_sha256}"\n            )\n\n        _load_source_map(reuse_path)\n        expected_counts = {\n            "PhysioNetMI": 327,\n            "BNCI2014_001": 18,\n            "Lee2019_MI": 108,\n        }\n        observed_counts = {\n            key: len(value)\n            for key, value in _SOURCE_MAP.items()\n        }\n        if observed_counts != expected_counts:\n            raise RuntimeError(\n                "R34_REUSED_SOURCE_RESOLUTION_COUNT_MISMATCH: "\n                f"expected={expected_counts}; observed={observed_counts}"\n            )\n\n        input_root = Path(getattr(runner, "input_root", "/kaggle/input")).resolve()\n        invalid_paths = []\n        for dataset_id, mapping in _SOURCE_MAP.items():\n            for source_path in mapping.values():\n                source_path = Path(source_path).resolve()\n                if (\n                    not source_path.is_file()\n                    or not source_path.is_relative_to(input_root)\n                ):\n                    invalid_paths.append(\n                        {\n                            "dataset_id": dataset_id,\n                            "path": str(source_path),\n                        }\n                    )\n        if invalid_paths:\n            raise RuntimeError(\n                "R34_REUSED_SOURCE_RESOLUTION_PATH_INVALID: "\n                + json.dumps(invalid_paths[:20], indent=2)\n            )\n\n        private_counts = {\n            dataset_id: int(\n                _PRIVATE_INVENTORY.get(dataset_id, {}).get("count", -1)\n            )\n            for dataset_id in ("BNCI2014_001", "Lee2019_MI")\n        }\n        if private_counts != {\n            "BNCI2014_001": 18,\n            "Lee2019_MI": 108,\n        }:\n            raise RuntimeError(\n                "R34_REUSED_PRIVATE_INVENTORY_COUNT_MISMATCH: "\n                f"{private_counts}"\n            )\n\n        _RESOLUTION_FILE = reuse_path\n        _SOURCE_REPORT = {\n            "status": "PASS",\n            "mode": "SAME_SESSION_SHA256_BOUND_READ_ONLY_SOURCE_REUSE",\n            "resolution_file": str(reuse_path),\n            "resolution_sha256": observed_resolution_sha256,\n            "source_counts": observed_counts,\n            "private_inventory_counts": private_counts,\n            "source_network_download_allowed": False,\n            "moabb_downloader_allowed": False,\n            "full_physical_byte_verification_performed_earlier_same_session": True,\n            "read_only_input_paths_revalidated": True,\n            "scientific_scope_changed": False,\n        }\n        _atomic_json(\n            runner.pipeline.bundle_root\n            / "reports"\n            / "phase_01"\n            / "runtime"\n            / "attached_source_datasets"\n            / "source_dataset_intake_reused.json",\n            _SOURCE_REPORT,\n        )\n        return _SOURCE_REPORT\n\n    if child_mode:\n        if resolution_file is None:\n            raise RuntimeError("STREAMING_CHILD_SOURCE_RESOLUTION_FILE_MISSING")\n        _load_source_map(Path(resolution_file))\n        _SOURCE_REPORT = {\n            "status": "PASS",\n            "mode": "CHILD_REUSE_PARENT_VERIFIED_SOURCE_MAP",\n            "resolution_file": str(resolution_file),\n            "source_counts": {k: len(v) for k, v in _SOURCE_MAP.items()},\n        }\n        return _SOURCE_REPORT\n\n    input_root = Path(getattr(runner, "input_root", "/kaggle/input"))\n    if not input_root.exists():\n        input_root = Path("/kaggle/input")\n    physio = _discover_physionet(input_root)\n    bnci, lee, private_manifests = _load_private_source_manifests(input_root)\n    _SOURCE_MAP = {"PhysioNetMI": physio, "BNCI2014_001": bnci, "Lee2019_MI": lee}\n    resolution_file = Path(getattr(runner, "work_root", "/kaggle/working/iharq_p01_l1")) / "streaming_runtime" / "source_resolution_r26.json"\n    _serialize_source_map(resolution_file)\n    _RESOLUTION_FILE = resolution_file\n    _SOURCE_REPORT = {\n        "status": "PASS",\n        "input_root": str(input_root),\n        "source_counts": {k: len(v) for k, v in _SOURCE_MAP.items()},\n        "private_manifests": private_manifests,\n        "source_resolution_file": str(resolution_file),\n        "physionet_public_dataset_required": POLICY["physionet_public_handle"],\n        "source_network_download_allowed": False,\n        "moabb_downloader_allowed": False,\n        "private_source_bytes_sha256_verified": True,\n        "ordinal_prefix_resolution_supported": True,\n    }\n    _atomic_json(\n        runner.pipeline.bundle_root / "reports" / "phase_01" / "runtime" / "attached_source_datasets" / "source_dataset_intake.json",\n        _SOURCE_REPORT,\n    )\n    return _SOURCE_REPORT\n\n\ndef _patch_base_dataset_get_data() -> dict[str, Any]:\n    """Install a lossless MOABB subject-key compatibility boundary.\n\n    MOABB guarantees a nested subject -> session -> run mapping, but project\n    adapters and MOABB releases have not always represented the top-level\n    subject identifier with the same Python type. This wrapper preserves the\n    returned mapping and its iteration order while allowing equivalent subject\n    identifiers (1, "1", "01", "sub-01", "subject_01", "A01", "S01") to\n    resolve to the original key. For a call that explicitly requests exactly\n    one subject and returns exactly one subject subtree, the requested subject\n    is also safely aliased to that sole result.\n    """\n    from collections.abc import KeysView, Mapping\n    from numbers import Integral\n    from moabb.datasets.base import BaseDataset\n\n    ambiguous = object()\n\n    def subject_token(value: Any) -> int | None:\n        if isinstance(value, bool):\n            return None\n        if isinstance(value, Integral):\n            return int(value)\n        text = str(value).strip()\n        match = re.fullmatch(\n            r"(?i)(?:sub(?:ject)?|participant|[as])?[-_ ]*0*(\\d+)",\n            text,\n        )\n        return int(match.group(1)) if match else None\n\n    class SubjectKeyCompatKeysView(KeysView):\n        def __contains__(self, key: object) -> bool:\n            return key in self._mapping\n\n    class SubjectKeyCompatDict(dict):\n        """Dictionary with alias-only lookup; contents and iteration are unchanged."""\n\n        def __init__(self, payload: Mapping[Any, Any], requested_subjects: Any):\n            super().__init__(payload)\n            aliases: dict[int, Any] = {}\n            for original_key in dict.keys(self):\n                token = subject_token(original_key)\n                if token is None:\n                    continue\n                previous = aliases.get(token)\n                if previous is None:\n                    aliases[token] = original_key\n                elif previous != original_key:\n                    aliases[token] = ambiguous\n\n            if requested_subjects is None:\n                requested = []\n            elif isinstance(requested_subjects, (str, bytes, Integral)):\n                requested = [requested_subjects]\n            else:\n                try:\n                    requested = list(requested_subjects)\n                except TypeError:\n                    requested = [requested_subjects]\n\n            # A single-request/single-result fallback is unambiguous and does\n            # not fabricate, merge, or discard any subject data.\n            if len(requested) == 1 and dict.__len__(self) == 1:\n                token = subject_token(requested[0])\n                if token is not None:\n                    current = aliases.get(token)\n                    if current is None or current is ambiguous:\n                        aliases[token] = next(dict.__iter__(self))\n\n            self._iharq_subject_aliases = aliases\n            self._iharq_requested_subjects = requested\n\n        def _resolve_key(self, key: Any) -> Any:\n            if dict.__contains__(self, key):\n                return key\n            token = subject_token(key)\n            if token is None:\n                return ambiguous\n            return self._iharq_subject_aliases.get(token, ambiguous)\n\n        def __contains__(self, key: object) -> bool:\n            return self._resolve_key(key) is not ambiguous\n\n        def __getitem__(self, key: Any) -> Any:\n            resolved = self._resolve_key(key)\n            if resolved is ambiguous:\n                raise KeyError(key)\n            return dict.__getitem__(self, resolved)\n\n        def get(self, key: Any, default: Any = None) -> Any:\n            resolved = self._resolve_key(key)\n            if resolved is ambiguous:\n                return default\n            return dict.__getitem__(self, resolved)\n\n        def keys(self):\n            return SubjectKeyCompatKeysView(self)\n\n        def pop(self, key: Any, *default: Any) -> Any:\n            resolved = self._resolve_key(key)\n            if resolved is ambiguous:\n                if default:\n                    return default[0]\n                raise KeyError(key)\n            return dict.pop(self, resolved)\n\n        def copy(self):\n            return SubjectKeyCompatDict(self, self._iharq_requested_subjects)\n\n    with _LOCK:\n        if getattr(BaseDataset, "_iharq_r31_subject_key_compat", False):\n            return {\n                "status": "ALREADY_PATCHED",\n                "revision": "R34",\n                "subject_key_aliasing": True,\n            }\n\n        original = BaseDataset.get_data\n        signature = inspect.signature(original)\n        supported = set(signature.parameters)\n        accepts_var_kwargs = any(\n            parameter.kind is inspect.Parameter.VAR_KEYWORD\n            for parameter in signature.parameters.values()\n        )\n\n        def compatible_get_data(self, *args, **kwargs):\n            cleaned = dict(kwargs)\n            if not accepts_var_kwargs:\n                for key in list(cleaned):\n                    if key not in supported:\n                        cleaned.pop(key)\n\n            requested_subjects = cleaned.get("subjects")\n            if requested_subjects is None:\n                try:\n                    bound = signature.bind_partial(self, *args, **cleaned)\n                    requested_subjects = bound.arguments.get("subjects")\n                except TypeError:\n                    requested_subjects = None\n\n            result = original(self, *args, **cleaned)\n            if isinstance(result, Mapping) and not isinstance(result, SubjectKeyCompatDict):\n                result = SubjectKeyCompatDict(result, requested_subjects)\n            return result\n\n        compatible_get_data.__name__ = getattr(original, "__name__", "get_data")\n        compatible_get_data.__doc__ = getattr(original, "__doc__", None)\n        compatible_get_data.__wrapped__ = original\n\n        # Cheap contract test before publishing the monkey-patch.\n        probe_value = object()\n        probe = SubjectKeyCompatDict({"01": probe_value}, [1])\n        if not (\n            probe.get(1) is probe_value\n            and probe.get("sub-001") is probe_value\n            and 1 in probe\n            and "subject_1" in probe.keys()\n            and list(probe) == ["01"]\n            and list(probe.keys()) == ["01"]\n            and len(probe) == 1\n        ):\n            raise RuntimeError("R34_MOABB_SUBJECT_KEY_COMPAT_SELF_TEST_FAILED")\n\n        BaseDataset.get_data = compatible_get_data\n        BaseDataset._iharq_r26_get_data_compat = True\n        BaseDataset._iharq_r31_subject_key_compat = True\n        BaseDataset._iharq_r31_original_get_data = original\n\n    return {\n        "status": "PATCHED",\n        "revision": "R34",\n        "original_parameters": sorted(supported),\n        "unsupported_keywords_filtered": True,\n        "subject_key_aliasing": True,\n        "mapping_contents_and_iteration_unchanged": True,\n        "keys_view_alias_membership": True,\n        "single_subject_single_result_fallback": True,\n        "installation_self_test": "PASS",\n    }\n\n\ndef _bnci_session_selected(session_id: Any, selected_sessions: Any) -> bool:\n    """Match MOABB session keys such as ``0train``/``1test`` to frozen selectors 0/1."""\n    if selected_sessions is None:\n        return True\n    try:\n        selected = list(selected_sessions)\n    except TypeError:\n        selected = [selected_sessions]\n    selected_tokens = {str(value).strip() for value in selected}\n    text = str(session_id).strip()\n    if text in selected_tokens:\n        return True\n    match = re.match(r"^(\\d+)", text)\n    return bool(match and match.group(1) in selected_tokens)\n\n\ndef _bnci_task_event_count(raw: Any, event_id: dict[str, Any]) -> int:\n    """Count only frozen task annotations, excluding optional artifact markers."""\n    descriptions = {str(key) for key in event_id}\n    annotations = getattr(raw, "annotations", None)\n    if annotations is None:\n        return 0\n    return sum(str(value) in descriptions for value in annotations.description)\n\n\ndef _exact_bnci_adapter_load(self: Any, files: list[Path]):\n    """Load BNCI through MOABB\'s native MAT converter without the empty get_data mapping.\n\n    MOABB\'s ``_get_single_subject_data`` is the dataset-native conversion path.\n    The same default ``SetRawAnnotations`` transformation used by ``get_data`` is\n    then applied explicitly to each run. No signal, trial, class, session, run,\n    or artifact value is invented or changed.\n    """\n    from collections.abc import Mapping\n    from iharq.layer1_data_protocol.adapters.base import SourceAccessError\n    from moabb.datasets.preprocessing import SetRawAnnotations\n\n    dataset = self._make_dataset()\n    event_id = dict(getattr(dataset, "event_id", {}) or {})\n    interval = tuple(getattr(dataset, "interval", ()) or ())\n    selected_sessions = getattr(dataset, "_selected_sessions", None)\n    if not event_id or len(interval) != 2:\n        raise SourceAccessError(\n            "R34_BNCI_DATASET_EVENT_CONTRACT_MISSING: "\n            f"event_id={event_id!r}; interval={interval!r}"\n        )\n\n    annotator = SetRawAnnotations(event_id, interval=interval)\n    out = []\n    diagnostics: list[dict[str, Any]] = []\n\n    for subject in self._subjects():\n        try:\n            subtree = dataset._get_single_subject_data(int(subject))\n        except Exception as exc:\n            raise SourceAccessError(\n                "R34_BNCI_NATIVE_SUBJECT_CONVERSION_FAILED: "\n                f"subject={subject}; error={type(exc).__name__}: {exc}"\n            ) from exc\n\n        if not isinstance(subtree, Mapping):\n            raise SourceAccessError(\n                "R34_BNCI_NATIVE_SUBTREE_INVALID: "\n                f"subject={subject}; type={type(subtree).__name__}"\n            )\n\n        exact_files = getattr(self, "_resolved_files_by_subject", {}).get(\n            int(subject), []\n        )\n        if not exact_files:\n            raise SourceAccessError(\n                "exact source-file provenance missing for subject "\n                f"{subject}; generic substring matching is forbidden"\n            )\n        source_file = ";".join(str(path) for path in exact_files)\n\n        subject_diagnostic = {\n            "subject": int(subject),\n            "native_sessions": [str(key) for key in subtree.keys()],\n            "selected_sessions": (\n                None if selected_sessions is None else [str(v) for v in selected_sessions]\n            ),\n            "sessions": [],\n        }\n\n        for session_id, runs in subtree.items():\n            session_selected = _bnci_session_selected(session_id, selected_sessions)\n            session_row = {\n                "session_id": str(session_id),\n                "selected": bool(session_selected),\n                "native_run_count": len(runs) if isinstance(runs, Mapping) else None,\n                "runs": [],\n            }\n            subject_diagnostic["sessions"].append(session_row)\n            if not session_selected:\n                continue\n            if not isinstance(runs, Mapping):\n                raise SourceAccessError(\n                    "R34_BNCI_RUN_MAPPING_INVALID: "\n                    f"subject={subject}; session={session_id!r}; "\n                    f"type={type(runs).__name__}"\n                )\n\n            for run_id, raw in runs.items():\n                raw_run_id = str(run_id)\n                run_row = {\n                    "run_id": raw_run_id,\n                    "included": bool(self._include_run(raw_run_id)),\n                    "task_events_before": _bnci_task_event_count(raw, event_id),\n                }\n                session_row["runs"].append(run_row)\n                if not run_row["included"]:\n                    continue\n\n                try:\n                    raw = annotator.transform(raw)\n                except Exception as exc:\n                    raise SourceAccessError(\n                        "R34_BNCI_ANNOTATION_TRANSFORM_FAILED: "\n                        f"subject={subject}; session={session_id!r}; "\n                        f"run={raw_run_id!r}; error={type(exc).__name__}: {exc}"\n                    ) from exc\n\n                task_events = _bnci_task_event_count(raw, event_id)\n                run_row["task_events_after"] = int(task_events)\n                run_row["annotation_count_after"] = int(\n                    len(getattr(raw, "annotations", ()) or ())\n                )\n                if task_events <= 0:\n                    raise SourceAccessError(\n                        "R34_BNCI_TASK_EVENTS_EMPTY_AFTER_NATIVE_CONVERSION: "\n                        f"subject={subject}; session={session_id!r}; "\n                        f"run={raw_run_id!r}; event_id={event_id!r}; "\n                        f"diagnostic={json.dumps(run_row, default=str)}"\n                    )\n\n                canonical_run_id = self._canonical_run_id(raw_run_id)\n                recording = self._raw_to_recording(\n                    raw,\n                    int(subject),\n                    str(session_id),\n                    canonical_run_id,\n                    source_file,\n                    self._run_metadata(raw_run_id),\n                )\n                if not getattr(recording, "events", None):\n                    raise SourceAccessError(\n                        "R34_BNCI_PROJECT_RECORDING_EVENTS_EMPTY: "\n                        f"subject={subject}; session={session_id!r}; "\n                        f"run={raw_run_id!r}"\n                    )\n                out.append(recording)\n\n        diagnostics.append(subject_diagnostic)\n\n    if not out:\n        raise SourceAccessError(\n            "R34_BNCI_NO_RECORDINGS_AFTER_EXACT_NATIVE_LOAD: "\n            + json.dumps(diagnostics, indent=2, default=str)\n        )\n    return out\n\n\ndef _patch_bnci_project_adapter_load() -> dict[str, Any]:\n    """Install the exact loader on the active IHARQ BNCI adapter class."""\n    from iharq.layer1_data_protocol import adapters as adapters_module\n\n    candidates = []\n    for adapter_key, adapter_class in adapters_module.ADAPTERS.items():\n        identity = (\n            f"{adapter_key} {adapter_class.__module__} "\n            f"{adapter_class.__name__} "\n            f"{getattr(adapter_class, \'dataset_class_name\', \'\')}"\n        ).lower()\n        if "bnci2014_001" in identity or "bnci2014001" in identity:\n            candidates.append(adapter_class)\n    candidates = list(dict.fromkeys(candidates))\n    if not candidates:\n        raise RuntimeError("R34_BNCI_PROJECT_ADAPTER_CLASS_NOT_FOUND")\n\n    patched = []\n    for adapter_class in candidates:\n        adapter_class.load = _exact_bnci_adapter_load\n        adapter_class._iharq_r32_exact_native_bnci_load = True\n        patched.append(f"{adapter_class.__module__}.{adapter_class.__name__}")\n\n    # Installation-time contract checks: selection semantics and explicit marker.\n    if not (\n        _bnci_session_selected("0train", [0, 1])\n        and _bnci_session_selected("1test", [0, 1])\n        and not _bnci_session_selected("2other", [0, 1])\n        and all(\n            getattr(cls, "_iharq_r32_exact_native_bnci_load", False)\n            for cls in candidates\n        )\n    ):\n        raise RuntimeError("R34_BNCI_ADAPTER_LOAD_INSTALLATION_SELF_TEST_FAILED")\n\n    return {\n        "status": "PATCHED",\n        "revision": "R34",\n        "adapter_classes": patched,\n        "moabb_native_subject_conversion": True,\n        "moabb_default_annotation_transform_reapplied": True,\n        "frozen_session_selection_preserved": True,\n        "exact_source_provenance_preserved": True,\n        "signals_or_labels_modified": False,\n        "installation_self_test": "PASS",\n    }\n\ndef _patch_source_contracts() -> dict[str, Any]:\n    from moabb.datasets import BNCI2014_001, Lee2019_MI, PhysionetMI\n    with _LOCK:\n        if not getattr(PhysionetMI, "_iharq_r26_source_patch", False):\n            original_init = PhysionetMI.__init__\n            def exact_physionet_init(self, *args, **kwargs):\n                result = _call_filtered(original_init, self, args, kwargs, {"imagined": True, "executed": False})\n                if hasattr(self, "runs"): self.runs = [4, 8, 12]\n                if hasattr(self, "hand_runs"): self.hand_runs = [4, 8, 12]\n                if hasattr(self, "feet_runs"): self.feet_runs = []\n                return result\n            def paths(subject: int, runs: Any) -> list[str]:\n                if not hasattr(runs, "__iter__"): runs = [runs]\n                result = []\n                for run in runs:\n                    key = (int(subject), int(run)); value = _SOURCE_MAP["PhysioNetMI"].get(key)\n                    if value is None: raise RuntimeError(f"PHYSIONET_ATTACHED_FILE_MISSING: {key}")\n                    result.append(str(value))\n                return result\n            def data_path(self, subject, path=None, force_update=False, update_path=None, verbose=None):\n                return paths(subject, [4, 8, 12])\n            def load_data(self, subject, runs, path=None, force_update=False, verbose=None):\n                return paths(subject, runs)\n            PhysionetMI.__init__ = exact_physionet_init\n            PhysionetMI.data_path = data_path\n            if hasattr(PhysionetMI, "_load_data"): PhysionetMI._load_data = load_data\n            PhysionetMI._iharq_r26_source_patch = True\n\n        if not getattr(BNCI2014_001, "_iharq_r26_source_patch", False):\n            original_bnci_init = BNCI2014_001.__init__\n            def exact_bnci_init(self, *args, **kwargs):\n                return _call_filtered(original_bnci_init, self, args, kwargs)\n            def data_path(self, subject, path=None, force_update=False, update_path=None, verbose=None):\n                result = []\n                for session in ("T", "E"):\n                    key = (int(subject), session); value = _SOURCE_MAP["BNCI2014_001"].get(key)\n                    if value is None: raise RuntimeError(f"BNCI_ATTACHED_FILE_MISSING: {key}")\n                    result.append(str(value))\n                return result\n            BNCI2014_001.__init__ = exact_bnci_init\n            BNCI2014_001.data_path = data_path\n            BNCI2014_001._iharq_r26_source_patch = True\n\n        if not getattr(Lee2019_MI, "_iharq_r26_source_patch", False):\n            original_lee_init = Lee2019_MI.__init__\n            def exact_lee_init(self, *args, **kwargs):\n                return _call_filtered(original_lee_init, self, args, kwargs, {"train_run": True, "test_run": False})\n            def data_path(self, subject, path=None, force_update=False, update_path=None, verbose=None):\n                result = []\n                for session in (1, 2):\n                    key = (int(subject), session); value = _SOURCE_MAP["Lee2019_MI"].get(key)\n                    if value is None: raise RuntimeError(f"LEE_ATTACHED_FILE_MISSING: {key}")\n                    result.append(str(value))\n                return result\n            Lee2019_MI.__init__ = exact_lee_init\n            Lee2019_MI.data_path = data_path\n            Lee2019_MI._iharq_r26_source_patch = True\n\n    # R26 controlled correction:\n    # MOABB PhysionetMI exposes the three imagined-hand recordings under\n    # internal run keys 0, 1, 2. The IHARQ canonical source identities are\n    # PhysioNet runs 4, 8, 12. MOABBAdapterBase checks _include_run twice:\n    # first with the internal key and again with the canonical source run.\n    # Therefore the adapter must lawfully recognize both representations\n    # while always canonicalizing to the frozen source set {4, 8, 12}.\n    from iharq.layer1_data_protocol import adapters as adapters_module\n\n    moabb_key_to_source_run = {0: 4, 1: 8, 2: 12}\n    frozen_physionet_source_runs = frozenset(\n        moabb_key_to_source_run.values()\n    )\n\n    def normalize_physionet_run_token(value: Any) -> int | None:\n        match = re.search(\n            r"(\\d+)(?!.*\\d)",\n            str(value).strip(),\n        )\n        return int(match.group(1)) if match else None\n\n    def configured_physionet_source_runs(adapter: Any) -> set[int]:\n        profile = getattr(adapter, "profile", None)\n        options = getattr(profile, "adapter_options", {}) or {}\n        if not isinstance(options, dict):\n            options = dict(options)\n\n        configured_raw = options.get("runs")\n        if configured_raw is None:\n            run_policy = (\n                getattr(profile, "run_policy", {})\n                or options.get("run_policy", {})\n                or {}\n            )\n            if not isinstance(run_policy, dict):\n                run_policy = dict(run_policy)\n            configured_raw = run_policy.get("include_runs")\n\n        if configured_raw is None:\n            configured_raw = sorted(\n                frozen_physionet_source_runs\n            )\n        if isinstance(configured_raw, (str, int)):\n            configured_raw = [configured_raw]\n\n        configured = {\n            normalized\n            for normalized in (\n                normalize_physionet_run_token(value)\n                for value in configured_raw\n            )\n            if normalized is not None\n        }\n\n        if configured != set(\n            frozen_physionet_source_runs\n        ):\n            raise RuntimeError(\n                "R26_PHYSIONET_RUN_POLICY_CONFIG_MISMATCH: "\n                f"expected={sorted(frozen_physionet_source_runs)}; "\n                f"configured={sorted(configured)}"\n            )\n\n        return configured\n\n    physionet_adapter_classes = []\n    for adapter_key, adapter_class in (\n        adapters_module.ADAPTERS.items()\n    ):\n        identity = (\n            f"{adapter_key} "\n            f"{adapter_class.__module__} "\n            f"{adapter_class.__name__}"\n        ).lower()\n        if "physio" in identity:\n            physionet_adapter_classes.append(\n                adapter_class\n            )\n\n    physionet_adapter_classes = list(\n        dict.fromkeys(physionet_adapter_classes)\n    )\n    if not physionet_adapter_classes:\n        raise RuntimeError(\n            "R26_PHYSIONET_ADAPTER_CLASS_NOT_FOUND"\n        )\n\n    for adapter_class in physionet_adapter_classes:\n        def exact_physionet_include_run(\n            self,\n            run_id: Any,\n            _internal_to_source: dict[int, int] = (\n                moabb_key_to_source_run\n            ),\n            _source_runs: frozenset[int] = (\n                frozen_physionet_source_runs\n            ),\n        ) -> bool:\n            configured_physionet_source_runs(self)\n            token = normalize_physionet_run_token(\n                run_id\n            )\n            return (\n                token in _internal_to_source\n                or token in _source_runs\n            )\n\n        def exact_physionet_canonical_run_id(\n            self,\n            run_id: Any,\n            _internal_to_source: dict[int, int] = (\n                moabb_key_to_source_run\n            ),\n            _source_runs: frozenset[int] = (\n                frozen_physionet_source_runs\n            ),\n        ) -> str:\n            configured_physionet_source_runs(self)\n            token = normalize_physionet_run_token(\n                run_id\n            )\n\n            if token in _internal_to_source:\n                source_run = _internal_to_source[\n                    token\n                ]\n            elif token in _source_runs:\n                source_run = token\n            else:\n                raise ValueError(\n                    "MOABB PhysioNet run key is outside "\n                    "the frozen imagined-hand branch: "\n                    f"{run_id!r}"\n                )\n\n            return str(source_run)\n\n        def exact_physionet_run_metadata(\n            self,\n            run_id: Any,\n            _internal_to_source: dict[int, int] = (\n                moabb_key_to_source_run\n            ),\n            _source_runs: frozenset[int] = (\n                frozen_physionet_source_runs\n            ),\n        ) -> dict[str, Any]:\n            configured_physionet_source_runs(self)\n            token = normalize_physionet_run_token(\n                run_id\n            )\n\n            if token in _internal_to_source:\n                source_run = _internal_to_source[\n                    token\n                ]\n                moabb_key = token\n            elif token in _source_runs:\n                source_run = token\n                reverse = {\n                    value: key\n                    for key, value\n                    in _internal_to_source.items()\n                }\n                moabb_key = reverse[token]\n            else:\n                raise ValueError(\n                    "MOABB PhysioNet run key is outside "\n                    "the frozen imagined-hand branch: "\n                    f"{run_id!r}"\n                )\n\n            return {\n                "moabb_run_key": str(moabb_key),\n                "physionet_source_run": source_run,\n            }\n\n        adapter_class._include_run = (\n            exact_physionet_include_run\n        )\n        adapter_class._canonical_run_id = (\n            exact_physionet_canonical_run_id\n        )\n        adapter_class._run_metadata = (\n            exact_physionet_run_metadata\n        )\n        adapter_class._iharq_r26_physionet_dual_run_key_contract_r2 = (\n            True\n        )\n\n\n    # R26 local-source correction for MOABB BNCI2014_001.\n    #\n    # The BNCI loader invokes a module-level data_path function rather than\n    # the Dataset class method. Route both module-level symbols to the\n    # SHA-256-verified attached Kaggle source map.\n    from moabb.datasets.bnci import base as bnci_base_module\n    from moabb.datasets.bnci import bnci_2014 as bnci_2014_module\n\n    def attached_bnci_2014_001_data_path(\n        url,\n        path=None,\n        force_update=False,\n        update_path=None,\n        verbose=None,\n    ):\n        match = re.search(\n            r"A(\\d{2})(T|E)\\.mat(?:$|[?#])",\n            str(url),\n            flags=re.IGNORECASE,\n        )\n        if not match:\n            raise RuntimeError(\n                "BNCI_ATTACHED_URL_NOT_IN_FROZEN_SCOPE: "\n                f"{url!r}"\n            )\n\n        key = (\n            int(match.group(1)),\n            match.group(2).upper(),\n        )\n        value = _SOURCE_MAP[\n            "BNCI2014_001"\n        ].get(key)\n\n        if value is None:\n            raise RuntimeError(\n                "BNCI_ATTACHED_FILE_MISSING: "\n                f"{key}"\n            )\n\n        value = Path(value)\n        if not value.is_file():\n            raise RuntimeError(\n                "BNCI_ATTACHED_FILE_NOT_READABLE: "\n                f"{value}"\n            )\n\n        return [str(value)]\n\n    bnci_base_module.data_path = (\n        attached_bnci_2014_001_data_path\n    )\n    bnci_2014_module.data_path = (\n        attached_bnci_2014_001_data_path\n    )\n    bnci_base_module._iharq_r26_attached_router = True\n    bnci_2014_module._iharq_r26_attached_router = True\n\n\n    # R26 MOABB-1.5 compatibility boundary for optional BNCI metadata only.\n    # Signal/event conversion remains the unmodified MOABB implementation.\n    from moabb.datasets.bnci import base as bnci_metadata_module\n\n    if not getattr(\n        bnci_metadata_module,\n        "_iharq_r26_optional_metadata_compat",\n        False,\n    ):\n        original_enrich_run_with_metadata = (\n            bnci_metadata_module._enrich_run_with_metadata\n        )\n        original_finalize_raw = (\n            bnci_metadata_module._finalize_raw\n        )\n\n        def compatible_enrich_run_with_metadata(\n            raw,\n            run,\n            dataset_code,\n            subject_id,\n        ):\n            if dataset_code != "BNCI2014-001":\n                return original_enrich_run_with_metadata(\n                    raw,\n                    run,\n                    dataset_code,\n                    subject_id,\n                )\n\n            try:\n                return original_enrich_run_with_metadata(\n                    raw,\n                    run,\n                    dataset_code,\n                    subject_id,\n                )\n            except (\n                AttributeError,\n                TypeError,\n                ValueError,\n                OverflowError,\n            ) as exc:\n                # This wrapper is reached only after _convert_run has already\n                # produced the Raw signal and event annotations. Re-finalize\n                # the valid Raw without inventing demographic/artifact values.\n                original_finalize_raw(\n                    raw,\n                    dataset_code,\n                    subject_id,\n                )\n                current = raw.info.get("description") or ""\n                raw.info["description"] = (\n                    current\n                    + "IHARQ: optional BNCI demographic/artifact "\n                    + "metadata unavailable or incompatible under "\n                    + "MOABB 1.5; no value was imputed; "\n                    + f"reason={type(exc).__name__}: {exc}; "\n                )\n                return None\n\n        bnci_metadata_module._enrich_run_with_metadata = (\n            compatible_enrich_run_with_metadata\n        )\n        bnci_metadata_module._iharq_r26_optional_metadata_compat = True\n\n    bnci_adapter_load = _patch_bnci_project_adapter_load()\n\n    return {\n        "PhysioNetMI": {"subjects": "1-109", "runs": [4, 8, 12]},\n        "BNCI2014_001": {"subjects": "1-9", "sessions": ["T", "E"], "unsupported_constructor_keywords_filtered": True, "module_level_attached_router": True, "optional_metadata_compatibility": "NO_IMPUTATION_SIGNAL_PRESERVING", "project_adapter_load": bnci_adapter_load},\n        "Lee2019_MI": {"subjects": "1-54", "sessions": [1, 2], "train_run": True, "test_run": False},\n    }\n\n\ndef _install_downloader_guard() -> dict[str, Any]:\n    from moabb.datasets import download as download_module\n    def prohibited(*args, **kwargs):\n        raise RuntimeError("MOABB_SOURCE_DOWNLOADER_PROHIBITED_R26: attach all source bytes as Kaggle Datasets.")\n    download_module.data_dl = prohibited\n    guarded = ["moabb.datasets.download.data_dl"]\n    for module_name in ["moabb.datasets.physionet_mi", "moabb.datasets.bnci", "moabb.datasets.Lee2019"]:\n        try:\n            module = __import__(module_name, fromlist=["data_dl"])\n            if hasattr(module, "data_dl"):\n                module.data_dl = prohibited; guarded.append(f"{module_name}.data_dl")\n        except Exception:\n            pass\n    return {"status": "PASS", "guarded_functions": guarded}\n\n\ndef _validate_resolved(dataset_id: str, files: list[Any], allow_subset: bool) -> dict[str, Any]:\n    unique = {str(Path(path).resolve()) for path in files}\n    missing = [path for path in unique if not Path(path).is_file()]\n    legal = {str(path.resolve()) for path in _SOURCE_MAP[dataset_id].values()}\n    foreign = sorted(unique - legal)\n    expected = POLICY["required_file_counts"][dataset_id]\n    expected_subset = {"PhysioNetMI": 3, "BNCI2014_001": 2, "Lee2019_MI": 2}[dataset_id]\n    count_ok = len(unique) == expected_subset if allow_subset else len(unique) == expected\n    result = {\n        "dataset_id": dataset_id,\n        "expected_files_full_run": expected,\n        "observed_unique_files": len(unique),\n        "subset_mode": allow_subset,\n        "expected_files_for_this_resolution": expected_subset if allow_subset else expected,\n        "missing_files": missing,\n        "foreign_files": foreign,\n        "pass": count_ok and not missing and not foreign,\n    }\n    if not result["pass"]:\n        raise RuntimeError("ATTACHED_SOURCE_POST_RESOLUTION_VALIDATION_FAILED: " + json.dumps(result, indent=2))\n    return result\n\n\ndef _wrap_adapter(adapter_class: type, runner: Any, child_mode: bool) -> None:\n    patch_key = (adapter_class, bool(child_mode))\n    if patch_key in _PATCHED:\n        return\n    original_resolve = adapter_class.resolve_files\n    original_verify = adapter_class.verify_files\n\n    def attached_only_resolve(self, _original=original_resolve):\n        dataset_id = _dataset_id(self)\n        files = list(_original(self))\n        validation = _validate_resolved(dataset_id, files, allow_subset=child_mode)\n        if not child_mode:\n            _atomic_json(\n                runner.pipeline.bundle_root / "reports" / "phase_01" / "runtime" / "attached_source_datasets" / f"{dataset_id}_post_resolution.json",\n                {"policy_id": POLICY["policy_id"], "validation": validation, "moabb_final_loader_preserved": True},\n            )\n        return files\n\n    def verified_inventory(\n        self,\n        files,\n        _original=original_verify,\n    ):\n        dataset_id = _dataset_id(self)\n\n        if (\n            dataset_id in _PRIVATE_INVENTORY\n            and not child_mode\n        ):\n            inventory = json.loads(\n                json.dumps(\n                    _PRIVATE_INVENTORY[\n                        dataset_id\n                    ]\n                )\n            )\n\n            expected_raw = str(\n                _get(\n                    _get(self, "profile"),\n                    "expected_checksum",\n                    "",\n                )\n                or ""\n            ).strip().lower()\n\n            observed = str(\n                inventory[\n                    "observed_checksum"\n                ]\n            ).strip().lower()\n\n            explicit_sha256 = bool(\n                re.fullmatch(\n                    r"[0-9a-f]{64}",\n                    expected_raw,\n                )\n            )\n\n            accepted_policy_sentinels = {\n                "",\n                "compute_or_verify_per_frozen_policy",\n                "compute_or_freeze_per_frozen_policy",\n                "compute_and_freeze",\n                "compute_at_runtime",\n                "computed_at_runtime",\n                "none",\n                "null",\n            }\n\n            if (\n                explicit_sha256\n                and expected_raw != observed\n            ):\n                raise RuntimeError(\n                    "SOURCE_CHECKSUM_MISMATCH: "\n                    f"dataset={dataset_id}; "\n                    f"expected={expected_raw}; "\n                    f"observed={observed}"\n                )\n\n            if (\n                not explicit_sha256\n                and expected_raw\n                not in accepted_policy_sentinels\n            ):\n                raise RuntimeError(\n                    "SOURCE_CHECKSUM_POLICY_UNRECOGNIZED: "\n                    f"dataset={dataset_id}; "\n                    f"declared={expected_raw!r}"\n                )\n\n            inventory[\n                "declared_expected_checksum"\n            ] = expected_raw or None\n\n            inventory[\n                "expected_checksum"\n            ] = (\n                expected_raw\n                if explicit_sha256\n                else observed\n            )\n\n            inventory[\n                "checksum_evidence_status"\n            ] = (\n                "VERIFIED_EXPLICIT_SHA256"\n                if explicit_sha256\n                else (\n                    "COMPUTED_AND_FROZEN_FOR_RUN_"\n                    "FROM_SHA256_VERIFIED_SOURCE_"\n                    "DATASET_MANIFEST"\n                )\n            )\n\n            # The frozen Stage 07 contract accepts this canonical\n            # status. The stronger physical-byte verification detail remains\n            # recorded separately in checksum_evidence_status and\n            # checksum_policy_resolution.\n            inventory[\n                "checksum_status"\n            ] = "COMPUTED_AND_FROZEN_FOR_RUN"\n\n            inventory[\n                "checksum_policy_resolution"\n            ] = {\n                "mode": (\n                    "EXPLICIT_SHA256"\n                    if explicit_sha256\n                    else (\n                        "FROZEN_OBSERVED_AGGREGATE_"\n                        "UNDER_DECLARED_POLICY"\n                    )\n                ),\n                "physical_files_verified_against_"\n                "source_dataset_manifest": True,\n                "aggregate_sha256": observed,\n                "scientific_scope_changed": False,\n            }\n\n            return inventory\n\n        return _original(\n            self,\n            files,\n        )\n\n\n    adapter_class.resolve_files = attached_only_resolve\n    adapter_class.verify_files = verified_inventory\n    _PATCHED.add(patch_key)\n\n\ndef _suppress_provider_log_noise() -> None:\n    for name in ["mne", "moabb", "pooch", "urllib3"]:\n        logger = logging.getLogger(name)\n        for handler in list(logger.handlers): logger.removeHandler(handler)\n        logger.addHandler(logging.NullHandler()); logger.propagate = False; logger.setLevel(logging.WARNING)\n\n\ndef install_acquisition_acceleration(runner: Any, *, child_mode: bool = False, resolution_file: str | Path | None = None) -> dict[str, Any]:\n    source_report = _prepare_sources(runner, child_mode=child_mode, resolution_file=Path(resolution_file) if resolution_file else None)\n    compatibility = _patch_base_dataset_get_data()\n    from moabb.datasets.base import BaseDataset\n    if not getattr(BaseDataset, "_iharq_r31_subject_key_compat", False):\n        raise RuntimeError("R34_MOABB_SUBJECT_KEY_COMPAT_NOT_INSTALLED")\n    source_contracts = _patch_source_contracts()\n    guard = _install_downloader_guard()\n    _suppress_provider_log_noise()\n    from iharq.layer1_data_protocol import adapters as adapters_module\n    patched = []\n    for adapter_class in set(adapters_module.ADAPTERS.values()):\n        _wrap_adapter(adapter_class, runner, child_mode)\n        patched.append(f"{adapter_class.__module__}.{adapter_class.__name__}")\n    installation = {\n        "policy": POLICY,\n        "installed_at_unix": time.time(),\n        "child_mode": child_mode,\n        "source_report": source_report,\n        "compatibility": compatibility,\n        "source_contracts": source_contracts,\n        "downloader_guard": guard,\n        "patched_adapter_classes": sorted(patched),\n        "quality_boundary": {\n            "same_active_sources": True,\n            "same_subjects": True,\n            "same_sessions": True,\n            "same_physionet_runs": True,\n            "same_moabb_native_signal_conversion": True,\n            "same_moabb_default_annotation_transform": True,\n            "bnci_empty_processed_mapping_bypassed": True,\n            "moabb_subject_key_representation_compatibility": True,\n            "moabb_mapping_contents_and_iteration_unchanged": True,\n            "same_stage07_checksum_inventory": True,\n            "same_exact_subject_provenance": True,\n            "same_subject88_handling": True,\n            "same_preprocessing_splits_budgets_windows": True,\n            "same_records_cards_manifests_gates_handoffs": True,\n            "source_network_downloading_removed": True,\n        },\n    }\n    if not child_mode:\n        _atomic_json(\n            runner.pipeline.bundle_root / "reports" / "phase_01" / "runtime" / "attached_source_datasets" / "installation.json",\n            installation,\n        )\n    return installation\n\n\ndef source_resolution_file() -> str | None:\n    return str(_RESOLUTION_FILE) if _RESOLUTION_FILE else None\n'
acquisition_acceleration_path = (
    OVERLAY_ROOT / "iharq_acquisition_acceleration.py"
)
acquisition_acceleration_path.write_text(
    acquisition_acceleration_source,
    encoding="utf-8",
)

bounded_streaming_source = 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Iterable\nfrom types import MethodType, SimpleNamespace\nfrom dataclasses import asdict, is_dataclass, replace\nimport ctypes\nimport gc\nimport hashlib\nimport importlib.metadata as importlib_metadata\nimport inspect\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport textwrap\nimport uuid\nimport zipfile\n\nWINDOW_SHARD_READER_SOURCE = \'from __future__ import annotations\\n\\n"""Read exact IHARQ P01/L1 R26 windows from an attached derived Kaggle Dataset.\\n\\nThe reader performs no network access. It supports Kaggle\\\'s optional ordinal\\nfilename prefixes, verifies the manifest/index/shard identities, validates the\\nHDF5 window ID at the declared row, and returns one exact NumPy array at a time.\\n"""\\n\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\nfrom typing import Any, Iterable, Iterator\\nimport hashlib\\nimport json\\nimport re\\n\\nMANIFEST_NAME = "IHARQ_P01_L1_DERIVED_WINDOW_DATASET_MANIFEST.json"\\nLOCATION_INDEX_NAME = "IHARQ_P01_L1_WINDOW_TO_SHARD_INDEX.jsonl"\\nEXPECTED_FORMAT = "LOSSLESS_HDF5_SUBJECT_SHARDS"\\nEXPECTED_FREEZE = "P01-L1-OFFICIAL-RUN-FREEZE-R2"\\n\\n\\ndef sha256_file(path: str | Path) -> str:\\n    digest = hashlib.sha256()\\n    with Path(path).open("rb") as stream:\\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\ndef _canonical_leaf(name: str) -> str:\\n    return Path(name).name\\n\\n\\ndef _matches_canonical_filename(path: Path, canonical_name: str) -> bool:\\n    leaf = path.name\\n    canonical = _canonical_leaf(canonical_name)\\n    return leaf == canonical or bool(re.fullmatch(rf"\\\\d+_{re.escape(canonical)}", leaf))\\n\\n\\ndef resolve_unique_file(root: str | Path, canonical_name: str, *, expected_sha256: str | None = None) -> Path:\\n    root = Path(root)\\n    candidates = [p for p in root.rglob("*") if p.is_file() and _matches_canonical_filename(p, canonical_name)]\\n    if expected_sha256:\\n        candidates = [p for p in candidates if sha256_file(p) == expected_sha256.lower()]\\n    unique = {str(p.resolve()): p.resolve() for p in candidates}\\n    if len(unique) != 1:\\n        raise RuntimeError(\\n            "IHARQ_DERIVED_FILE_RESOLUTION_AMBIGUOUS: "\\n            f"canonical={canonical_name}; matches={sorted(unique)}"\\n        )\\n    return next(iter(unique.values()))\\n\\n\\ndef _iter_jsonl(path: Path) -> Iterator[dict[str, Any]]:\\n    with path.open("r", encoding="utf-8") as stream:\\n        for line_number, line in enumerate(stream, start=1):\\n            line = line.strip()\\n            if not line:\\n                continue\\n            try:\\n                value = json.loads(line)\\n            except json.JSONDecodeError as exc:\\n                raise RuntimeError(f"IHARQ_JSONL_PARSE_FAILED: path={path}; line={line_number}") from exc\\n            if not isinstance(value, dict):\\n                raise RuntimeError(f"IHARQ_JSONL_ROW_NOT_OBJECT: path={path}; line={line_number}")\\n            yield value\\n\\n\\n@dataclass(frozen=True)\\nclass WindowLocation:\\n    window_id: str\\n    window_record_id: str\\n    shard_filename: str\\n    hdf5_group: str\\n    hdf5_row: int\\n    shape: tuple[int, ...]\\n    dtype: str\\n    shard_sha256: str\\n\\n\\nclass DerivedWindowDataset:\\n    """Bounded-memory access to one attached R26 derived-window Dataset."""\\n\\n    def __init__(self, root: str | Path, *, verify_all_shards_at_open: bool = False):\\n        self.root = Path(root).resolve()\\n        if not self.root.is_dir():\\n            raise FileNotFoundError(self.root)\\n        self.manifest_path = resolve_unique_file(self.root, MANIFEST_NAME)\\n        self.manifest = json.loads(self.manifest_path.read_text(encoding="utf-8"))\\n        self._validate_manifest()\\n        index_spec = self.manifest.get("window_location_index", {})\\n        expected_index_hash = index_spec.get("dataset_sha256")\\n        index_name = index_spec.get("dataset_filename", LOCATION_INDEX_NAME)\\n        self.location_index_path = resolve_unique_file(\\n            self.root,\\n            str(index_name),\\n            expected_sha256=str(expected_index_hash).lower() if expected_index_hash else None,\\n        )\\n        self._shards = {\\n            str(row["filename"]): row\\n            for row in self.manifest.get("shards", [])\\n        }\\n        if len(self._shards) != len(self.manifest.get("shards", [])):\\n            raise RuntimeError("IHARQ_DERIVED_DUPLICATE_SHARD_FILENAME")\\n        self._location_cache: dict[str, WindowLocation] = {}\\n        self._verified_shards: set[str] = set()\\n        if verify_all_shards_at_open:\\n            for filename in sorted(self._shards):\\n                self._resolve_and_verify_shard(filename)\\n\\n    @classmethod\\n    def discover(cls, input_root: str | Path = "/kaggle/input", *, verify_all_shards_at_open: bool = False) -> "DerivedWindowDataset":\\n        input_root = Path(input_root)\\n        manifests = [p for p in input_root.rglob("*") if p.is_file() and _matches_canonical_filename(p, MANIFEST_NAME)]\\n        roots = {str(p.parent.resolve()): p.parent.resolve() for p in manifests}\\n        if len(roots) != 1:\\n            raise RuntimeError(\\n                "IHARQ_DERIVED_DATASET_DISCOVERY_REQUIRES_EXACTLY_ONE: "\\n                f"observed={sorted(roots)}"\\n            )\\n        return cls(next(iter(roots.values())), verify_all_shards_at_open=verify_all_shards_at_open)\\n\\n    def _validate_manifest(self) -> None:\\n        manifest = self.manifest\\n        if manifest.get("scientific_freeze") != EXPECTED_FREEZE:\\n            raise RuntimeError("IHARQ_DERIVED_SCIENTIFIC_FREEZE_MISMATCH")\\n        if manifest.get("format") != EXPECTED_FORMAT:\\n            raise RuntimeError("IHARQ_DERIVED_FORMAT_MISMATCH")\\n        if manifest.get("creation_status") not in {None, "COMMITTED"}:\\n            raise RuntimeError("IHARQ_DERIVED_DATASET_NOT_COMMITTED")\\n        if int(manifest.get("immutable_revision", 0)) != 1:\\n            raise RuntimeError("IHARQ_DERIVED_REVISION_MISMATCH")\\n        if int(manifest.get("window_count", -1)) < 0:\\n            raise RuntimeError("IHARQ_DERIVED_WINDOW_COUNT_INVALID")\\n        if manifest.get("signal_dtype") not in {None, "float32"}:\\n            raise RuntimeError("IHARQ_DERIVED_SIGNAL_DTYPE_MISMATCH")\\n\\n    def _resolve_and_verify_shard(self, filename: str) -> Path:\\n        spec = self._shards.get(filename)\\n        if spec is None:\\n            raise KeyError(f"Unknown shard: {filename}")\\n        path = resolve_unique_file(self.root, filename)\\n        resolved_key = str(path)\\n        if resolved_key not in self._verified_shards:\\n            expected_bytes = int(spec["bytes"])\\n            if path.stat().st_size != expected_bytes:\\n                raise RuntimeError(\\n                    f"IHARQ_DERIVED_SHARD_SIZE_MISMATCH: {filename}; "\\n                    f"expected={expected_bytes}; observed={path.stat().st_size}"\\n                )\\n            observed_hash = sha256_file(path)\\n            if observed_hash != str(spec["sha256"]).lower():\\n                raise RuntimeError(\\n                    f"IHARQ_DERIVED_SHARD_SHA256_MISMATCH: {filename}; "\\n                    f"expected={spec[\\\'sha256\\\']}; observed={observed_hash}"\\n                )\\n            self._verified_shards.add(resolved_key)\\n        return path\\n\\n    def iter_locations(self) -> Iterator[WindowLocation]:\\n        seen: set[str] = set()\\n        for row in _iter_jsonl(self.location_index_path):\\n            window_id = str(row["window_id"])\\n            if window_id in seen:\\n                raise RuntimeError(f"IHARQ_DERIVED_DUPLICATE_WINDOW_ID: {window_id}")\\n            seen.add(window_id)\\n            filename = str(row["shard_filename"])\\n            shard_spec = self._shards.get(filename)\\n            if shard_spec is None:\\n                raise RuntimeError(f"IHARQ_DERIVED_INDEX_REFERENCES_UNKNOWN_SHARD: {filename}")\\n            location = WindowLocation(\\n                window_id=window_id,\\n                window_record_id=str(row["window_record_id"]),\\n                shard_filename=filename,\\n                hdf5_group=str(row["hdf5_group"]),\\n                hdf5_row=int(row["hdf5_row"]),\\n                shape=tuple(int(v) for v in row["shape"]),\\n                dtype=str(row["dtype"]),\\n                shard_sha256=str(shard_spec["sha256"]).lower(),\\n            )\\n            self._location_cache[window_id] = location\\n            yield location\\n        expected = int(self.manifest["window_count"])\\n        if len(seen) != expected:\\n            raise RuntimeError(\\n                f"IHARQ_DERIVED_INDEX_COUNT_MISMATCH: expected={expected}; observed={len(seen)}"\\n            )\\n\\n    def location(self, window_id: str) -> WindowLocation:\\n        window_id = str(window_id)\\n        cached = self._location_cache.get(window_id)\\n        if cached is not None:\\n            return cached\\n        for location in self.iter_locations():\\n            if location.window_id == window_id:\\n                return location\\n        raise KeyError(window_id)\\n\\n    def load(self, window_id: str, *, verify_window_id: bool = True):\\n        import h5py\\n        import numpy as np\\n\\n        location = self.location(window_id)\\n        shard_path = self._resolve_and_verify_shard(location.shard_filename)\\n        with h5py.File(shard_path, "r") as handle:\\n            signals_path = f"{location.hdf5_group}/signals"\\n            ids_path = f"{location.hdf5_group}/window_ids"\\n            if signals_path not in handle or ids_path not in handle:\\n                raise RuntimeError(f"IHARQ_DERIVED_HDF5_GROUP_MISSING: {location.hdf5_group}")\\n            signals = handle[signals_path]\\n            identifiers = handle[ids_path]\\n            row = location.hdf5_row\\n            if row < 0 or row >= int(signals.shape[0]) or row >= int(identifiers.shape[0]):\\n                raise RuntimeError(f"IHARQ_DERIVED_HDF5_ROW_OUT_OF_RANGE: {window_id}; row={row}")\\n            stored_id = identifiers[row]\\n            if isinstance(stored_id, bytes):\\n                stored_id = stored_id.decode("utf-8")\\n            if verify_window_id and str(stored_id) != location.window_id:\\n                raise RuntimeError(\\n                    f"IHARQ_DERIVED_WINDOW_ID_MISMATCH: expected={location.window_id}; observed={stored_id}"\\n                )\\n            array = np.asarray(signals[row])\\n        if tuple(array.shape) != location.shape:\\n            raise RuntimeError(\\n                f"IHARQ_DERIVED_WINDOW_SHAPE_MISMATCH: expected={location.shape}; observed={array.shape}"\\n            )\\n        if str(array.dtype) != location.dtype:\\n            raise RuntimeError(\\n                f"IHARQ_DERIVED_WINDOW_DTYPE_MISMATCH: expected={location.dtype}; observed={array.dtype}"\\n            )\\n        return array\\n\\n    def load_many(self, window_ids: Iterable[str]) -> Iterator[tuple[str, Any]]:\\n        for window_id in window_ids:\\n            yield str(window_id), self.load(str(window_id))\\n\\n\\ndef _self_test() -> dict[str, Any]:\\n    import tempfile\\n    import h5py\\n    import numpy as np\\n\\n    with tempfile.TemporaryDirectory() as temporary:\\n        root = Path(temporary)\\n        array = np.arange(24, dtype=np.float32).reshape(3, 8)\\n        shard = root / "001_D_subject_001_windows.h5"\\n        with h5py.File(shard, "w") as handle:\\n            group = handle.require_group("window_groups/c3_t8")\\n            group.create_dataset("signals", data=array[None, ...], dtype="float32")\\n            group.create_dataset("window_ids", data=np.asarray(["window:test"], dtype=h5py.string_dtype("utf-8")))\\n        index = root / ("002_" + LOCATION_INDEX_NAME)\\n        index.write_text(json.dumps({\\n            "window_id": "window:test",\\n            "window_record_id": "WindowRecord:test",\\n            "shard_filename": "D_subject_001_windows.h5",\\n            "hdf5_group": "window_groups/c3_t8",\\n            "hdf5_row": 0,\\n            "shape": [3, 8],\\n            "dtype": "float32",\\n        }) + "\\\\n", encoding="utf-8")\\n        manifest = {\\n            "scientific_freeze": EXPECTED_FREEZE,\\n            "format": EXPECTED_FORMAT,\\n            "creation_status": "COMMITTED",\\n            "immutable_revision": 1,\\n            "window_count": 1,\\n            "window_location_index": {\\n                "dataset_filename": LOCATION_INDEX_NAME,\\n                "dataset_sha256": sha256_file(index),\\n            },\\n            "shards": [{\\n                "filename": "D_subject_001_windows.h5",\\n                "bytes": shard.stat().st_size,\\n                "sha256": sha256_file(shard),\\n            }],\\n        }\\n        (root / ("003_" + MANIFEST_NAME)).write_text(json.dumps(manifest), encoding="utf-8")\\n        dataset = DerivedWindowDataset(root)\\n        restored = dataset.load("window:test")\\n        return {\\n            "status": "PASS" if np.array_equal(array, restored) else "FAIL",\\n            "ordinal_prefix_resolution": True,\\n            "lossless_roundtrip": bool(np.array_equal(array, restored)),\\n            "shape": list(restored.shape),\\n            "dtype": str(restored.dtype),\\n        }\\n\\n\\nif __name__ == "__main__":\\n    print(json.dumps(_self_test(), indent=2))\\n\'\n\nPOLICY = {\n    "policy_id": "P01-L1-KAGGLE-DUAL-PERSISTENCE-BOUNDED-STREAMING-R3",\n    "runtime_revision": "R34",\n    "policy_kind": "RESOURCE_AND_PERSISTENCE_IMPLEMENTATION_AMENDMENT_ONLY",\n    "scientific_freeze_unchanged": "P01-L1-OFFICIAL-RUN-FREEZE-R2",\n    "controlling_authority": "IHARQ-IBB-R10-P01-L1-INDEPENDENT-AUDIT-REPAIRED",\n    "annex": "IHARQ-IBB-P01-L1-ANNEX-R4",\n    "passes": {\n        "pass_1": "SUBJECT_SCOPED_METADATA_EVENT_PROVENANCE_AND_RAW_FIT_STAT_SUMMARY",\n        "pass_2a": "DETERMINISTIC_COMBINATION_OF_PASS1_FIT_STATISTICS",\n        "pass_2b": "SUBJECT_SCOPED_TRANSFORM_QUALITY_WINDOW_MATERIALIZATION_AND_UPLOAD",\n    },\n    "maximum_concurrent_source_subjects": 8,\n    "disposable_subject_processes": True,\n    "child_rss_hard_limit_gib": 24.0,\n    "minimum_disk_free_gib": 4.0,\n    "progress_interval_seconds": 120,\n    "derived_window_storage": "PRIVATE_KAGGLE_DATASET_LOSSLESS_HDF5_SUBJECT_SHARDS",\n    "derived_kaggle_username": "csthv999z",\n    "derived_dataset_slug_prefix": "iharq-p01-l1-core",\n    "derived_dataset_version": 1,\n    "derived_dataset_private": True,\n    "local_raw_signal_retention": False,\n    "local_preprocessed_signal_retention": False,\n    "local_window_array_retention_after_upload": False,\n    "no_scientific_scope_reduction": True,\n    "active_sources_unchanged": ["PhysioNetMI", "BNCI2014_001", "Lee2019_MI"],\n    "labels_splits_budgets_preprocessing_windows_unchanged": True,\n    "official_joint_event_resampling_enforced": True,\n    "official_window_offset_and_one_window_policy_enforced": True,\n    "official_float32_window_dtype_enforced": True,\n    "finalization_stage_ordering_corrected": True,\n    "all_p01_gates_and_handoffs_preserved": True,\n    "dual_persistence": {\n        "compact_output": "GITHUB_READY_REPOSITORY_ZIP",\n        "large_numerical_output": "PRIVATE_KAGGLE_DERIVED_DATASET",\n        "manual_huggingface_roundtrip_required": False,\n        "manual_future_kaggle_reupload_required": False,\n    },\n    "github_ready_repository_max_gib": 0.95,\n    "github_ready_excludes_large_arrays": True,\n    "storage_forecast_compression_ratio_lower": 0.55,\n    "storage_forecast_compression_ratio_upper": 1.10,\n    "storage_forecast_safety_multiplier": 1.25,\n}\n\n_GIB = 1024 ** 3\n_CHILD_MODULE = "iharq_bounded_streaming"\n\n\nclass ShapeOnlySignal:\n    """A non-materializable signal descriptor used after streaming Pass 1."""\n    __slots__ = ("shape", "dtype", "ndim", "size", "nbytes")\n\n    def __init__(self, shape: Iterable[int], dtype: str | Any):\n        import numpy as np\n        self.shape = tuple(int(x) for x in shape)\n        self.dtype = np.dtype(dtype)\n        self.ndim = len(self.shape)\n        self.size = math.prod(self.shape)\n        self.nbytes = int(self.size * self.dtype.itemsize)\n\n    def __len__(self) -> int:\n        return self.shape[0]\n\n    def __array__(self, *args, **kwargs):\n        raise RuntimeError(\n            "SHAPE_ONLY_SIGNAL_MATERIALIZATION_PROHIBITED: R26 loads each subject "\n            "inside a disposable child process instead of retaining all signals."\n        )\n\n    def __getitem__(self, key):\n        raise RuntimeError("SHAPE_ONLY_SIGNAL_INDEXING_PROHIBITED")\n\n    def __repr__(self) -> str:\n        return f"ShapeOnlySignal(shape={self.shape!r}, dtype={str(self.dtype)!r})"\n\n\ndef _jsonable(value: Any) -> Any:\n    import numpy as np\n    if value is None or isinstance(value, (str, int, bool)):\n        return value\n    if isinstance(value, float):\n        return value if math.isfinite(value) else str(value)\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, np.generic):\n        return _jsonable(value.item())\n    if isinstance(value, np.ndarray):\n        return [_jsonable(x) for x in value.tolist()]\n    if is_dataclass(value):\n        return _jsonable(asdict(value))\n    if isinstance(value, dict):\n        return {str(k): _jsonable(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple, set)):\n        return [_jsonable(v) for v in value]\n    return str(value)\n\n\ndef _atomic_json(path: Path, payload: Any) -> None:\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(_jsonable(payload), indent=2) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef _append_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8") as stream:\n        for row in rows:\n            stream.write(json.dumps(_jsonable(row), separators=(",", ":")) + "\\n")\n\n\ndef _read_jsonl(path: Path) -> list[dict[str, Any]]:\n    if not Path(path).is_file():\n        return []\n    rows = []\n    with Path(path).open("r", encoding="utf-8") as stream:\n        for line in stream:\n            line = line.strip()\n            if line:\n                rows.append(json.loads(line))\n    return rows\n\n\ndef _sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef _safe(value: Any) -> str:\n    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_") or "unknown"\n\n\ndef _trim_memory() -> None:\n    gc.collect()\n    try:\n        ctypes.CDLL("libc.so.6").malloc_trim(0)\n    except Exception:\n        pass\n\n\ndef _rss_bytes(pid: int | None = None) -> int:\n    pid = int(pid or os.getpid())\n    try:\n        text = Path(f"/proc/{pid}/status").read_text(encoding="utf-8")\n        match = re.search(r"^VmRSS:\\s+(\\d+)\\s+kB", text, flags=re.MULTILINE)\n        return int(match.group(1)) * 1024 if match else 0\n    except Exception:\n        return 0\n\n\ndef _disk_snapshot(path: Path) -> dict[str, Any]:\n    usage = shutil.disk_usage(path)\n    return {\n        "total_bytes": usage.total,\n        "used_bytes": usage.used,\n        "free_bytes": usage.free,\n        "free_gib": round(usage.free / _GIB, 3),\n    }\n\n\ndef _resource_guard(work_root: Path, child_pid: int | None = None) -> None:\n    disk = _disk_snapshot(work_root)\n    if disk["free_bytes"] < int(POLICY["minimum_disk_free_gib"] * _GIB):\n        raise RuntimeError(\n            "R26_DISK_RESERVE_EXCEEDED: "\n            f"free_gib={disk[\'free_gib\']}; required_gib={POLICY[\'minimum_disk_free_gib\']}"\n        )\n    if child_pid:\n        rss = _rss_bytes(child_pid)\n        if rss > int(POLICY["child_rss_hard_limit_gib"] * _GIB):\n            raise MemoryError(\n                "R26_CHILD_RSS_LIMIT_EXCEEDED: "\n                f"pid={child_pid}; rss_gib={rss/_GIB:.3f}; limit_gib={POLICY[\'child_rss_hard_limit_gib\']}"\n            )\n\n\ndef _profile_dict(profile: Any) -> dict[str, Any]:\n    return _jsonable(asdict(profile) if is_dataclass(profile) else dict(profile))\n\n\ndef _subject_values(profile: Any) -> list[int]:\n    dataset = str(profile.dataset_id)\n    raw = dict(profile.adapter_options).get("subjects")\n    values: list[int] = []\n    if isinstance(raw, (list, tuple, set)):\n        values = [int(v) for v in raw]\n    elif isinstance(raw, range):\n        values = [int(v) for v in raw]\n    elif raw is not None:\n        try:\n            values = [int(raw)]\n        except Exception:\n            values = []\n    fallback = {\n        "PhysioNetMI": list(range(1, 110)),\n        "BNCI2014_001": list(range(1, 10)),\n        "Lee2019_MI": list(range(1, 55)),\n    }[dataset]\n    if not values:\n        values = fallback\n    expected = set(fallback)\n    if set(values) != expected:\n        raise RuntimeError(\n            "R26_SUBJECT_SCOPE_MISMATCH: "\n            f"dataset={dataset}; expected={sorted(expected)}; observed={sorted(set(values))}"\n        )\n    return sorted(set(values))\n\n\ndef _clone_profile_for_subject(profile_dict: dict[str, Any], subject: int):\n    from iharq.layer1_data_protocol.models import SourceProfile\n    payload = dict(profile_dict)\n    options = dict(payload.get("adapter_options", {}))\n    options["subjects"] = [int(subject)]\n    options["n_jobs"] = 1\n    payload["adapter_options"] = options\n    return SourceProfile(**payload)\n\n\ndef _recording_source_unit(recording: Any) -> str:\n    return f"{recording.dataset_id}:{recording.subject_id}:{recording.session_id}:{recording.run_id}"\n\n\ndef _descriptor_from_recording(recording: Any) -> dict[str, Any]:\n    return {\n        "dataset_id": str(recording.dataset_id),\n        "subject_id": _jsonable(recording.subject_id),\n        "session_id": _jsonable(recording.session_id),\n        "run_id": _jsonable(recording.run_id),\n        "source_file": str(recording.source_file),\n        "sampling_hz": float(recording.sampling_hz),\n        "channel_names": [str(x) for x in recording.channel_names],\n        "signal_shape": [int(x) for x in recording.signal.shape],\n        "signal_dtype": str(recording.signal.dtype),\n        "events": [\n            {\n                "event_id": str(event.event_id),\n                "start_sample": int(event.start_sample),\n                "stop_sample": int(event.stop_sample),\n                "original_label": str(event.original_label),\n                "metadata": _jsonable(getattr(event, "metadata", {})),\n            }\n            for event in recording.events\n        ],\n        "source_metadata": _jsonable(getattr(recording, "source_metadata", {})),\n        "source_unit": _recording_source_unit(recording),\n    }\n\n\ndef _recording_from_descriptor(row: dict[str, Any]):\n    from iharq.layer1_data_protocol.models import Event, RawRecording\n    events = [\n        Event(\n            event_id=str(event["event_id"]),\n            start_sample=int(event["start_sample"]),\n            stop_sample=int(event["stop_sample"]),\n            original_label=str(event["original_label"]),\n            metadata=dict(event.get("metadata", {})),\n        )\n        for event in row["events"]\n    ]\n    return RawRecording(\n        dataset_id=str(row["dataset_id"]),\n        subject_id=row["subject_id"],\n        session_id=row["session_id"],\n        run_id=row["run_id"],\n        source_file=str(row["source_file"]),\n        sampling_hz=float(row["sampling_hz"]),\n        channel_names=[str(x) for x in row["channel_names"]],\n        signal=ShapeOnlySignal(row["signal_shape"], row["signal_dtype"]),\n        events=events,\n        source_metadata=dict(row.get("source_metadata", {})),\n    )\n\n\ndef _child_runner_stub(task: dict[str, Any]):\n    report_root = Path(task["child_report_root"])\n    report_root.mkdir(parents=True, exist_ok=True)\n    return SimpleNamespace(\n        input_root=Path(task["input_root"]),\n        work_root=Path(task["child_work_root"]),\n        pipeline=SimpleNamespace(bundle_root=report_root),\n    )\n\n\ndef _load_subject_recordings(task: dict[str, Any]):\n    os.environ.update({\n        "OMP_NUM_THREADS": "1",\n        "MKL_NUM_THREADS": "1",\n        "OPENBLAS_NUM_THREADS": "1",\n        "NUMEXPR_NUM_THREADS": "1",\n        "IHARQ_STREAMING_CHILD": "1",\n    })\n    from iharq_acquisition_acceleration import install_acquisition_acceleration\n    stub = _child_runner_stub(task)\n    install_acquisition_acceleration(\n        stub,\n        child_mode=True,\n        resolution_file=task["source_resolution_file"],\n    )\n    from iharq.layer1_data_protocol import adapters as adapters_module\n    profile = _clone_profile_for_subject(task["profile"], int(task["subject"]))\n    adapter_class = adapters_module.ADAPTERS.get(profile.adapter)\n    if adapter_class is None:\n        raise RuntimeError(f"R26_CHILD_ADAPTER_UNKNOWN: {profile.adapter}")\n    cache_root = Path(task["child_work_root"]) / "source_cache" / profile.dataset_id\n    adapter = adapter_class(profile, Path(task["input_root"]), cache_root)\n    files = list(adapter.resolve_files())\n    recordings = list(adapter.load(files))\n    if not recordings:\n        raise RuntimeError(\n            f"R26_CHILD_NO_RECORDINGS: dataset={profile.dataset_id}; subject={task[\'subject\']}"\n        )\n    expected_subject = str(task["subject"])\n    observed = {str(recording.subject_id) for recording in recordings}\n    # Different adapters may zero-pad or prefix subject IDs. The profile-level\n    # process boundary is the authoritative subject selection; record IDs remain\n    # source-native and are not rewritten here.\n    return recordings, files, observed, expected_subject\n\n\ndef _deterministic_recording_role(recording: Any, split_profile: dict[str, Any]) -> str:\n    import hashlib as _hashlib\n    roles = list(split_profile["roles"]); ratios = dict(split_profile["ratios"]); keys = list(split_profile["group_keys"]); seed = split_profile["seed"]\n    unit = "|".join(str(getattr(recording, key)) for key in keys)\n    digest = _hashlib.sha256(f"{seed}|{unit}".encode()).hexdigest()\n    value = int(digest[:16], 16) / (16**16 - 1)\n    total = 0.0; selected = roles[-1]\n    for role in roles:\n        total += float(ratios[role])\n        if value <= total:\n            selected = role; break\n    return selected\n\n\ndef _child_descriptor(task: dict[str, Any]) -> dict[str, Any]:\n    recordings, files, observed, expected = _load_subject_recordings(task)\n    output = Path(task["output_dir"]); output.mkdir(parents=True, exist_ok=True)\n    descriptor_path = output / "descriptors.jsonl"\n    fit_stats_path = output / "pass1_fit_stats.jsonl"\n    for stale_path in (descriptor_path, fit_stats_path, output / "result.json"):\n        stale_path.unlink(missing_ok=True)\n    rows = [_descriptor_from_recording(recording) for recording in recordings]\n    _append_jsonl(descriptor_path, rows)\n    fit_rows = []\n    if bool(task.get("collect_fit_stats")):\n        import numpy as np\n        # Collect exact raw per-recording summaries for every recording. Stage 13\n        # later filters them by the actual frozen split assignment, which removes\n        # any dependency on a pre-split approximation and avoids an extra source\n        # reload. Blocks cap temporary memory while preserving float64 semantics.\n        block_samples = int(task.get("fit_stat_block_samples", 262144))\n        for recording in recordings:\n            signal = recording.signal\n            channel_count = int(signal.shape[0])\n            count = 0\n            mean = np.zeros(channel_count, dtype=np.float64)\n            m2 = np.zeros(channel_count, dtype=np.float64)\n            for start in range(0, int(signal.shape[1]), block_samples):\n                stop = min(int(signal.shape[1]), start + block_samples)\n                block = np.asarray(signal[:, start:stop], dtype=np.float64)\n                n_b = int(block.shape[1])\n                if n_b == 0:\n                    continue\n                mean_b = block.mean(axis=1, dtype=np.float64)\n                centered_b = block - mean_b[:, None]\n                m2_b = np.sum(centered_b * centered_b, axis=1, dtype=np.float64)\n                if count == 0:\n                    count = n_b\n                    mean = mean_b.copy()\n                    m2 = m2_b.copy()\n                else:\n                    delta = mean_b - mean\n                    total = count + n_b\n                    mean = mean + delta * (n_b / total)\n                    m2 = m2 + m2_b + delta * delta * (count * n_b / total)\n                    count = total\n                del block, mean_b, centered_b, m2_b\n            if count != int(signal.shape[1]):\n                raise RuntimeError(\n                    f"R26_PASS1_FIT_SAMPLE_COUNT_MISMATCH: "\n                    f"source={_recording_source_unit(recording)}; "\n                    f"expected={signal.shape[1]}; observed={count}"\n                )\n            fit_rows.append({\n                "source_unit": _recording_source_unit(recording),\n                "channel_names": [str(v) for v in recording.channel_names],\n                "count": count,\n                "mean": [float(v) for v in mean],\n                "m2": [float(v) for v in m2],\n                "shape": [int(v) for v in signal.shape],\n                "candidate_role": _deterministic_recording_role(recording, dict(task["split_profile"])),\n            })\n            del mean, m2\n            _trim_memory()\n        _append_jsonl(fit_stats_path, fit_rows)\n    result = {\n        "action": "descriptor",\n        "dataset_id": task["profile"]["dataset_id"],\n        "subject": int(task["subject"]),\n        "recordings": len(rows),\n        "events": sum(len(row["events"]) for row in rows),\n        "source_files_resolved": len(files),\n        "observed_subject_ids": sorted(observed),\n        "expected_profile_subject": expected,\n        "descriptor_path": str(descriptor_path),\n        "fit_stats_path": str(fit_stats_path),\n        "fit_stat_recordings": len(fit_rows),\n        "child_peak_rss_bytes": _rss_bytes(),\n    }\n    _atomic_json(output / "result.json", result)\n    return result\n\n\n\ndef _ensure_h5_group(handle: Any, shape: tuple[int, int], dtype: str = "float32"):\n    import h5py\n    import numpy as np\n\n    resolved_dtype = np.dtype(dtype)\n    if resolved_dtype != np.dtype("float32"):\n        raise RuntimeError(\n            "R34_HDF5_WINDOW_DTYPE_MUST_BE_FLOAT32: "\n            f"observed={resolved_dtype}"\n        )\n    key = f"c{shape[0]}_t{shape[1]}_f32"\n    group = handle.require_group(f"window_groups/{key}")\n    if "signals" not in group:\n        group.create_dataset(\n            "signals",\n            shape=(0, shape[0], shape[1]),\n            maxshape=(None, shape[0], shape[1]),\n            dtype="float32",\n            chunks=(1, shape[0], shape[1]),\n            compression="gzip",\n            compression_opts=1,\n            shuffle=True,\n            fletcher32=True,\n        )\n        group.create_dataset(\n            "window_ids",\n            shape=(0,),\n            maxshape=(None,),\n            dtype=h5py.string_dtype(encoding="utf-8"),\n            chunks=(256,),\n        )\n    elif str(group["signals"].dtype) != "float32":\n        raise RuntimeError(\n            "R34_EXISTING_HDF5_GROUP_DTYPE_MISMATCH: "\n            f"group={group.name}; dtype={group[\'signals\'].dtype}"\n        )\n    return key, group\n\n\n\n\ndef _append_h5_window(handle: Any, array: Any, window_id: str) -> tuple[str, int]:\n    import numpy as np\n\n    resolved = np.asarray(array)\n    if resolved.dtype != np.dtype("float32"):\n        raise RuntimeError(\n            "R34_WINDOW_ARRAY_DTYPE_MISMATCH: "\n            f"expected=float32; observed={resolved.dtype}"\n        )\n    if resolved.ndim != 2:\n        raise RuntimeError(\n            "R34_WINDOW_ARRAY_RANK_MISMATCH: "\n            f"expected=2; observed={resolved.ndim}"\n        )\n    key, group = _ensure_h5_group(\n        handle,\n        tuple(int(v) for v in resolved.shape),\n        str(resolved.dtype),\n    )\n    signals = group["signals"]\n    ids = group["window_ids"]\n    row = int(signals.shape[0])\n    signals.resize(row + 1, axis=0)\n    ids.resize(row + 1, axis=0)\n    signals[row] = resolved\n    ids[row] = window_id\n    return f"window_groups/{key}", row\n\n\n\n\ndef _child_materialize(task: dict[str, Any]) -> dict[str, Any]:\n    import h5py\n    import numpy as np\n    from iharq.canonical import semantic_hash\n    from iharq.layer1_data_protocol.preprocessing import FitState, transform_recording\n    from iharq.layer1_data_protocol.quality import annotate\n    from iharq.layer1_data_protocol.labels import map_event_label\n    from iharq.layer1_data_protocol.splits import recording_role\n    from iharq.layer1_data_protocol.records import make_record\n\n    recordings, files, observed, expected = _load_subject_recordings(task)\n    output = Path(task["output_dir"])\n    output.mkdir(parents=True, exist_ok=True)\n\n    fit_payload = task["fit_state"]\n    mean = None if fit_payload.get("mean") is None else np.asarray(fit_payload["mean"], dtype=np.float64)\n    std = None if fit_payload.get("std") is None else np.asarray(fit_payload["std"], dtype=np.float64)\n    fit_state = FitState(mean, std, list(fit_payload["source_ids"]), str(fit_payload["state_hash"]))\n    operations = list(task["operations"])\n    assignment = dict(task["assignment"])\n    split_keys = list(task["split_keys"])\n    label_record = dict(task["label_record"])\n    preprocessing_record = dict(task["preprocessing_record"])\n    split_record = dict(task["split_record"])\n    quality_profile = dict(task["quality_profile"])\n    window_profile = dict(task["window_profile"])\n    config_id = str(task["config_id"])\n    dataset_record_id = str(task["dataset_record_id"])\n    split_id = split_record["record_id"]\n\n    required_window_contract = {\n        "start_offset_samples": 80,\n        "duration_samples": 480,\n        "stride_samples": 480,\n        "target_sampling_hz": 160,\n        "last_window_policy": "ONE_WINDOW_PER_INCLUDED_SOURCE_EVENT",\n        "bounds_policy": "REJECT_OUT_OF_BOUNDS",\n    }\n    mismatches = {\n        key: {"expected": expected_value, "observed": window_profile.get(key)}\n        for key, expected_value in required_window_contract.items()\n        if window_profile.get(key) != expected_value\n    }\n    if mismatches:\n        raise RuntimeError(\n            "R34_FROZEN_WINDOW_CONTRACT_MISMATCH: "\n            + json.dumps(mismatches, sort_keys=True)\n        )\n\n    offset = 80\n    duration_samples = 480\n    stride_samples = 480\n    target_hz = 160.0\n\n    shard_filename = str(task["shard_filename"])\n    shard_path = output / shard_filename\n    window_records_path = output / "window_records.jsonl"\n    window_index_path = output / "window_index.jsonl"\n    quality_records_path = output / "quality_records.jsonl"\n    quality_summaries_path = output / "quality_summaries.jsonl"\n    location_path = output / "window_locations.jsonl"\n    invalid_windows_path = output / "invalid_windows.jsonl"\n    result_path = output / "result.json"\n\n    for stale_path in (\n        shard_path,\n        window_records_path,\n        window_index_path,\n        quality_records_path,\n        quality_summaries_path,\n        location_path,\n        invalid_windows_path,\n        result_path,\n    ):\n        stale_path.unlink(missing_ok=True)\n    for path in (\n        window_records_path,\n        window_index_path,\n        quality_records_path,\n        quality_summaries_path,\n        location_path,\n        invalid_windows_path,\n    ):\n        path.touch()\n\n    window_count = 0\n    logical_window_bytes = 0\n    event_ids: set[str] = set()\n    roles: set[str] = set()\n    h5_handle = None\n\n    quality_rows_all: list[dict[str, Any]] = []\n    quality_summaries: list[dict[str, Any]] = []\n    invalid_windows: list[dict[str, Any]] = []\n    window_records_buffer: list[dict[str, Any]] = []\n    index_buffer: list[dict[str, Any]] = []\n    location_buffer: list[dict[str, Any]] = []\n\n    try:\n        for source_recording in recordings:\n            recording = transform_recording(\n                source_recording,\n                operations,\n                fit_state,\n            )\n            if abs(float(recording.sampling_hz) - target_hz) > 1e-9:\n                raise RuntimeError(\n                    "R34_TRANSFORMED_SAMPLING_RATE_MISMATCH: "\n                    f"source={_recording_source_unit(recording)}; "\n                    f"observed={recording.sampling_hz}"\n                )\n            if np.asarray(recording.signal).dtype != np.dtype("float32"):\n                raise RuntimeError(\n                    "R34_TRANSFORMED_SIGNAL_DTYPE_MISMATCH: "\n                    f"source={_recording_source_unit(recording)}; "\n                    f"observed={np.asarray(recording.signal).dtype}"\n                )\n            if len(recording.channel_names) != int(recording.signal.shape[0]):\n                raise RuntimeError(\n                    "R34_TRANSFORMED_CHANNEL_GEOMETRY_MISMATCH: "\n                    f"source={_recording_source_unit(recording)}"\n                )\n\n            qrows, qsummary = annotate(\n                recording,\n                quality_profile,\n                config_id,\n                dataset_record_id,\n            )\n            quality_rows_all.extend(qrows)\n            quality_summaries.append(qsummary)\n\n            role = recording_role(recording, assignment, split_keys)\n            roles.add(role)\n\n            for event in recording.events:\n                normalized = map_event_label(event.original_label, label_record)\n                if normalized is None:\n                    continue\n\n                source_sample = event.metadata.get("original_source_event_sample")\n                resampled_sample = event.metadata.get("resampled_event_sample")\n                if source_sample is None or resampled_sample is None:\n                    invalid_windows.append(\n                        {\n                            "dataset_id": recording.dataset_id,\n                            "subject_id": recording.subject_id,\n                            "session_id": recording.session_id,\n                            "run_id": recording.run_id,\n                            "event_id": event.event_id,\n                            "reason": "MISSING_PARENT_EVENT_SAMPLE_LINEAGE",\n                        }\n                    )\n                    continue\n\n                start = int(resampled_sample) + offset\n                end = start + duration_samples\n                if start < 0 or end > int(recording.signal.shape[1]):\n                    invalid_windows.append(\n                        {\n                            "dataset_id": recording.dataset_id,\n                            "subject_id": recording.subject_id,\n                            "session_id": recording.session_id,\n                            "run_id": recording.run_id,\n                            "event_id": event.event_id,\n                            "original_source_event_sample": int(source_sample),\n                            "resampled_event_sample": int(resampled_sample),\n                            "start": start,\n                            "end": end,\n                            "samples": int(recording.signal.shape[1]),\n                            "reason": "WINDOW_OUT_OF_BOUNDS",\n                        }\n                    )\n                    continue\n\n                identity = {\n                    "dataset": recording.dataset_id,\n                    "subject": recording.subject_id,\n                    "session": recording.session_id,\n                    "run": recording.run_id,\n                    "event": event.event_id,\n                    "original_event_sample": int(source_sample),\n                    "resampled_event_sample": int(resampled_sample),\n                    "start": start,\n                    "stop": end,\n                    "split_record_id": split_id,\n                    "role": role,\n                    "config": config_id,\n                }\n                digest = semantic_hash(identity)\n                wid = "window:" + digest[:20]\n                if h5_handle is None:\n                    h5_handle = h5py.File(shard_path, "w")\n                    h5_handle.attrs["format"] = "IHARQ_P01_L1_LOSSLESS_WINDOW_SHARD_R34"\n                    h5_handle.attrs["scientific_freeze"] = POLICY["scientific_freeze_unchanged"]\n                    h5_handle.attrs["config_id"] = config_id\n                    h5_handle.attrs["dataset_id"] = str(recording.dataset_id)\n                    h5_handle.attrs["subject_profile"] = str(task["subject"])\n                    h5_handle.attrs["signal_dtype"] = "float32"\n                    h5_handle.attrs["window_start_offset_samples"] = offset\n                    h5_handle.attrs["window_duration_samples"] = duration_samples\n                    h5_handle.attrs["window_stride_samples"] = stride_samples\n                    h5_handle.attrs["event_resampling"] = "MNE_POLYPHASE_JOINT_EVENTS"\n\n                window = np.asarray(\n                    recording.signal[:, start:end],\n                    dtype=np.float32,\n                )\n                if window.shape[1] != duration_samples:\n                    raise RuntimeError(\n                        "R34_WINDOW_SHAPE_MISMATCH: "\n                        f"window={wid}; shape={window.shape}"\n                    )\n                group_path, row_number = _append_h5_window(\n                    h5_handle,\n                    window,\n                    wid,\n                )\n                local_pointer = (\n                    "external_artifact_pointers/window_to_shard.jsonl"\n                    f"#window_id={wid}"\n                )\n                payload = {\n                    "window_id": wid,\n                    "parent_event_id": event.event_id,\n                    "dataset_id": recording.dataset_id,\n                    "subject_id": recording.subject_id,\n                    "session_id": recording.session_id,\n                    "run_id": recording.run_id,\n                    "split_record_id": split_id,\n                    "preprocessing_record_id": preprocessing_record["record_id"],\n                    "label_map_record_id": label_record["record_id"],\n                    "original_source_event_sample": int(source_sample),\n                    "resampled_event_sample": int(resampled_sample),\n                    "start_offset_samples": offset,\n                    "start_sample": start,\n                    "stop_sample": end,\n                    "duration_samples": duration_samples,\n                    "stride_samples": stride_samples,\n                    "overlap_group_id": event.event_id,\n                    "normalized_label": normalized,\n                    "original_label": event.original_label,\n                    "role": role,\n                    "signal_pointer": local_pointer,\n                    "channel_mask_id": None,\n                }\n                source_ids = [\n                    dataset_record_id,\n                    split_id,\n                    preprocessing_record["record_id"],\n                    label_record["record_id"],\n                ]\n                record = make_record(\n                    "WindowRecord",\n                    payload,\n                    config_id,\n                    source_ids,\n                    lifecycle_status="VALIDATED",\n                )\n                sample_hash = semantic_hash(\n                    {\n                        "shape": list(window.shape),\n                        "head": [\n                            str(float(value))\n                            for value in window.reshape(-1)[:128]\n                        ],\n                    }\n                )\n                index_row = {\n                    "window_record_id": record["record_id"],\n                    "path": local_pointer,\n                    "role": role,\n                    "label": normalized,\n                    "event_id": event.event_id,\n                    "split_record_id": split_id,\n                    "dataset_id": recording.dataset_id,\n                    "subject_id": recording.subject_id,\n                    "session_id": recording.session_id,\n                    "run_id": recording.run_id,\n                    "overlap_group_id": event.event_id,\n                    "sample_hash": sample_hash,\n                    "external_shard_filename": shard_filename,\n                    "hdf5_group": group_path,\n                    "hdf5_row": row_number,\n                }\n                location_row = {\n                    "window_id": wid,\n                    "window_record_id": record["record_id"],\n                    "shard_filename": shard_filename,\n                    "hdf5_group": group_path,\n                    "hdf5_row": row_number,\n                    "shape": list(window.shape),\n                    "dtype": "float32",\n                }\n                window_records_buffer.append(record)\n                index_buffer.append(index_row)\n                location_buffer.append(location_row)\n                window_count += 1\n                logical_window_bytes += int(window.nbytes)\n                event_ids.add(str(event.event_id))\n\n                if len(window_records_buffer) >= 256:\n                    _append_jsonl(window_records_path, window_records_buffer)\n                    window_records_buffer.clear()\n                    _append_jsonl(window_index_path, index_buffer)\n                    index_buffer.clear()\n                    _append_jsonl(location_path, location_buffer)\n                    location_buffer.clear()\n                if window_count % 256 == 0 and h5_handle is not None:\n                    h5_handle.flush()\n\n            del recording\n            _trim_memory()\n\n        if window_records_buffer:\n            _append_jsonl(window_records_path, window_records_buffer)\n        if index_buffer:\n            _append_jsonl(window_index_path, index_buffer)\n        if location_buffer:\n            _append_jsonl(location_path, location_buffer)\n        if quality_rows_all:\n            _append_jsonl(quality_records_path, quality_rows_all)\n        if quality_summaries:\n            _append_jsonl(quality_summaries_path, quality_summaries)\n        if invalid_windows:\n            _append_jsonl(invalid_windows_path, invalid_windows)\n    finally:\n        if h5_handle is not None:\n            h5_handle.flush()\n            h5_handle.close()\n\n    shard = None\n    if shard_path.is_file():\n        verified_rows = 0\n        with h5py.File(shard_path, "r") as verify_handle:\n            format_value = str(verify_handle.attrs.get("format", ""))\n            if format_value != "IHARQ_P01_L1_LOSSLESS_WINDOW_SHARD_R34":\n                raise RuntimeError(f"R34_HDF5_FORMAT_MISMATCH: {format_value}")\n            if str(verify_handle.attrs.get("signal_dtype", "")) != "float32":\n                raise RuntimeError("R34_HDF5_SIGNAL_DTYPE_ATTRIBUTE_MISMATCH")\n            for group_name in sorted(verify_handle.get("window_groups", {})):\n                group = verify_handle[f"window_groups/{group_name}"]\n                signals = group["signals"]\n                identifiers = group["window_ids"]\n                if str(signals.dtype) != "float32":\n                    raise RuntimeError(\n                        f"R34_HDF5_GROUP_DTYPE_MISMATCH: group={group_name}; dtype={signals.dtype}"\n                    )\n                if int(signals.shape[0]) != int(identifiers.shape[0]):\n                    raise RuntimeError(\n                        f"R34_HDF5_SIGNAL_ID_COUNT_MISMATCH: group={group_name}; "\n                        f"signals={signals.shape[0]}; ids={identifiers.shape[0]}"\n                    )\n                if int(signals.shape[2]) != duration_samples:\n                    raise RuntimeError(\n                        f"R34_HDF5_WINDOW_DURATION_MISMATCH: group={group_name}; "\n                        f"observed={signals.shape[2]}"\n                    )\n                verified_rows += int(signals.shape[0])\n        if verified_rows != window_count:\n            raise RuntimeError(\n                f"R34_HDF5_WINDOW_COUNT_MISMATCH: expected={window_count}; observed={verified_rows}"\n            )\n        shard = {\n            "path": str(shard_path),\n            "filename": shard_filename,\n            "bytes": shard_path.stat().st_size,\n            "sha256": _sha256(shard_path),\n            "verified_window_rows": verified_rows,\n            "logical_window_bytes": logical_window_bytes,\n            "compression_ratio_to_logical": (\n                shard_path.stat().st_size / logical_window_bytes\n                if logical_window_bytes\n                else None\n            ),\n            "verification_status": "PASS",\n            "signal_dtype": "float32",\n            "window_duration_samples": duration_samples,\n        }\n\n    result = {\n        "action": "materialize",\n        "dataset_id": task["profile"]["dataset_id"],\n        "subject": int(task["subject"]),\n        "quality_records": len(quality_rows_all),\n        "quality_summaries": len(quality_summaries),\n        "hard_invalid": sum(int(row.get("hard_invalid", 0)) for row in quality_summaries),\n        "window_count": window_count,\n        "invalid_window_count": len(invalid_windows),\n        "logical_window_bytes": logical_window_bytes,\n        "event_count": len(event_ids),\n        "roles": sorted(roles),\n        "window_records_path": str(window_records_path),\n        "window_index_path": str(window_index_path),\n        "quality_records_path": str(quality_records_path),\n        "quality_summaries_path": str(quality_summaries_path),\n        "invalid_windows_path": str(invalid_windows_path),\n        "window_locations_path": str(location_path),\n        "shard": shard,\n        "child_peak_rss_bytes": _rss_bytes(),\n    }\n    _atomic_json(result_path, result)\n    return result\n\n\n\ndef _child_main(request_path: str) -> int:\n    try:\n        task = json.loads(Path(request_path).read_text(encoding="utf-8"))\n        action = str(task["action"])\n        if action == "descriptor": result = _child_descriptor(task)\n        elif action == "materialize": result = _child_materialize(task)\n        else: raise RuntimeError(f"R26_CHILD_ACTION_UNKNOWN: {action}")\n        print("__IHARQ_R26_CHILD_RESULT__" + json.dumps(_jsonable(result), separators=(",", ":")), flush=True)\n        return 0\n    except Exception as exc:\n        print("__IHARQ_R26_CHILD_ERROR__" + json.dumps({"error": repr(exc), "traceback": traceback.format_exc()}), flush=True)\n        return 1\n\n\ndef _progress_line(event: str, completed: int, total: int, started: float, current: str, child_pid: int | None, work_root: Path, extra: dict[str, Any] | None = None) -> dict[str, Any]:\n    elapsed = max(0.001, time.monotonic() - started)\n    rate = completed / elapsed if completed else 0.0\n    remaining = max(0, total - completed)\n    eta = remaining / rate if rate > 0 else None\n    payload = {\n        "event": event,\n        "completed": completed,\n        "total": total,\n        "percent": round(100.0 * completed / total, 2) if total else 100.0,\n        "elapsed_seconds": round(elapsed, 1),\n        "eta_seconds": None if eta is None else round(eta, 1),\n        "current": current,\n        "parent_rss_gib": round(_rss_bytes() / _GIB, 3),\n        "child_pid": child_pid,\n        "child_rss_gib": round(_rss_bytes(child_pid) / _GIB, 3) if child_pid else 0.0,\n        "disk": _disk_snapshot(work_root),\n    }\n    if extra: payload.update(_jsonable(extra))\n    print("[R26 PROGRESS] " + json.dumps(payload, separators=(",", ":")), flush=True)\n    return payload\n\n\ndef _run_child_task(task: dict[str, Any], work_root: Path, label: str, retries: int = 2) -> dict[str, Any]:\n    """Run one disposable subject worker while measuring its true peak RSS.\n\n    The measured peak is returned to the parent and is used after the real-loader\n    preflight to choose the fastest memory-safe parallelism. Scientific work and\n    retry semantics are unchanged.\n    """\n    task_root = Path(task["output_dir"])\n    task_root.mkdir(parents=True, exist_ok=True)\n    request_path = task_root / "request.json"\n    _atomic_json(request_path, task)\n    errors = []\n    for attempt in range(1, retries + 1):\n        output_log = task_root / f"child_attempt_{attempt}.log"\n        env = os.environ.copy()\n        env["IHARQ_STREAMING_CHILD"] = "1"\n        env.setdefault("PYTHONHASHSEED", "0")\n        command = [sys.executable, "-u", "-m", _CHILD_MODULE, "--child", str(request_path)]\n        peak_rss_bytes = 0\n        with output_log.open("w", encoding="utf-8", buffering=1) as log_stream:\n            process = subprocess.Popen(command, env=env, stdout=log_stream, stderr=subprocess.STDOUT, text=True)\n            started = time.monotonic(); last = 0.0\n            try:\n                while process.poll() is None:\n                    time.sleep(1.0)\n                    observed_rss = _rss_bytes(process.pid)\n                    peak_rss_bytes = max(peak_rss_bytes, observed_rss)\n                    _resource_guard(work_root, process.pid)\n                    if time.monotonic() - last >= float(POLICY["progress_interval_seconds"]):\n                        _progress_line(\n                            "SUBJECT_CHILD_HEARTBEAT", 0, 1, started, label,\n                            process.pid, work_root,\n                            {"attempt": attempt, "measured_peak_rss_bytes": peak_rss_bytes},\n                        )\n                        last = time.monotonic()\n            except Exception as exc:\n                process.kill(); process.wait(timeout=30)\n                errors.append({\n                    "attempt": attempt,\n                    "error": repr(exc),\n                    "log": str(output_log),\n                    "measured_peak_rss_bytes": peak_rss_bytes,\n                })\n            else:\n                peak_rss_bytes = max(peak_rss_bytes, _rss_bytes(process.pid))\n                returncode = process.wait()\n                if returncode == 0 and (task_root / "result.json").is_file():\n                    result = json.loads((task_root / "result.json").read_text(encoding="utf-8"))\n                    result["child_peak_rss_bytes"] = max(\n                        int(result.get("child_peak_rss_bytes", 0)),\n                        int(peak_rss_bytes),\n                    )\n                    result["child_runtime_seconds"] = round(time.monotonic() - started, 3)\n                    return result\n                errors.append({\n                    "attempt": attempt,\n                    "returncode": returncode,\n                    "log": str(output_log),\n                    "tail": output_log.read_text(encoding="utf-8", errors="replace")[-6000:],\n                    "measured_peak_rss_bytes": peak_rss_bytes,\n                })\n        _trim_memory()\n    raise RuntimeError(f"R27_SUBJECT_TASK_FAILED: label={label}; errors={json.dumps(errors, indent=2)}")\n\n\ndef _combine_fit_rows(rows: list[dict[str, Any]], legal_source_ids: set[str]):\n    import numpy as np\n    from iharq.canonical import semantic_hash\n    from iharq.layer1_data_protocol.preprocessing import FitState\n    found: set[str] = set(); count = 0; mean = None; m2 = None; channel_names = None\n    duplicates: set[str] = set()\n    for row in rows:\n        source_unit = str(row["source_unit"])\n        if source_unit not in legal_source_ids:\n            continue\n        if source_unit in found:\n            duplicates.add(source_unit)\n            continue\n        found.add(source_unit)\n        n_b = int(row["count"]); mean_b = np.asarray(row["mean"], dtype=np.float64); m2_b = np.asarray(row["m2"], dtype=np.float64)\n        names = [str(x) for x in row["channel_names"]]\n        if channel_names is None:\n            channel_names = names; mean = np.zeros_like(mean_b); m2 = np.zeros_like(m2_b)\n        if names != channel_names:\n            raise RuntimeError(f"R26_FIT_CHANNEL_ORDER_MISMATCH: source={source_unit}")\n        if mean_b.shape != mean.shape:\n            raise RuntimeError(f"R26_FIT_CHANNEL_COUNT_MISMATCH: source={source_unit}")\n        if count == 0:\n            count = n_b; mean = mean_b.copy(); m2 = m2_b.copy(); continue\n        delta = mean_b - mean; total = count + n_b\n        mean = mean + delta * (n_b / total)\n        m2 = m2 + m2_b + delta * delta * (count * n_b / total)\n        count = total\n    if duplicates:\n        raise RuntimeError(\n            f"R26_FIT_SOURCE_DUPLICATED: count={len(duplicates)}; "\n            f"sample={sorted(duplicates)[:20]}"\n        )\n    missing = sorted(set(legal_source_ids) - found)\n    if missing:\n        raise RuntimeError(f"R26_FIT_SOURCE_UNOBSERVED: count={len(missing)}; sample={missing[:20]}")\n    if count <= 0 or mean is None or m2 is None:\n        raise RuntimeError("R26_LEGAL_FIT_POPULATION_EMPTY")\n    std = np.sqrt(m2 / count); std = np.where(std < 1e-12, 1.0, std)\n    mean2 = mean[:, None]; std2 = std[:, None]\n    state_hash = semantic_hash({\n        "mean": [str(float(x)) for x in mean2[:, 0]],\n        "std": [str(float(x)) for x in std2[:, 0]],\n        "sources": sorted(legal_source_ids),\n    })\n    return FitState(mean2, std2, sorted(legal_source_ids), state_hash), {"sample_count": count, "channel_names": channel_names, "observed_fit_sources": len(found)}\n\n\ndef _ensure_kaggle_upload_api() -> dict[str, Any]:\n    expected = {"kagglehub": "1.0.2", "kagglesdk": "0.1.23"}\n    observed = {}\n    for package, version in expected.items():\n        try: observed[package] = importlib_metadata.version(package)\n        except importlib_metadata.PackageNotFoundError: observed[package] = "MISSING"\n    if observed != expected:\n        raise RuntimeError(f"R26_KAGGLE_UPLOAD_STACK_MISMATCH: expected={expected}; observed={observed}")\n    for name in list(sys.modules):\n        if name == "kagglehub" or name.startswith("kagglehub.") or name == "kagglesdk" or name.startswith("kagglesdk."):\n            sys.modules.pop(name, None)\n    if not os.environ.get("KAGGLE_API_TOKEN"):\n        try:\n            from kaggle_secrets import UserSecretsClient\n            token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")\n        except Exception as exc:\n            raise RuntimeError("R26_KAGGLE_API_TOKEN_SECRET_UNAVAILABLE") from exc\n        if not token: raise RuntimeError("R26_KAGGLE_API_TOKEN_SECRET_EMPTY")\n        os.environ["KAGGLE_API_TOKEN"] = token\n    import functools\n    import kagglehub.gcs_upload as gcs_upload_module\n    if not getattr(gcs_upload_module.tqdm, "_iharq_r26_silent", False):\n        silent = functools.partial(gcs_upload_module.tqdm, disable=True); silent._iharq_r26_silent = True; gcs_upload_module.tqdm = silent\n    from kagglesdk.kaggle_env import get_web_endpoint\n    from kagglehub.datasets_helpers import create_dataset_or_version\n    from kagglehub.gcs_upload import UploadDirectoryInfo, _upload_blob\n    from kagglehub.handle import parse_dataset_handle\n    from kagglesdk.blobs.types.blob_api_service import ApiBlobType\n    if not callable(get_web_endpoint): raise RuntimeError("R26_KAGGLESDK_ENDPOINT_API_UNAVAILABLE")\n    return {\n        "create_dataset_or_version": create_dataset_or_version,\n        "UploadDirectoryInfo": UploadDirectoryInfo,\n        "_upload_blob": _upload_blob,\n        "parse_dataset_handle": parse_dataset_handle,\n        "ApiBlobType": ApiBlobType,\n        "versions": expected,\n    }\n\n\ndef _upload_blob_with_retry(api: dict[str, Any], path: Path, retries: int = 3):\n    errors = []\n    for attempt in range(1, retries + 1):\n        try:\n            return api["_upload_blob"](str(path), api["ApiBlobType"].DATASET)\n        except Exception as exc:\n            errors.append({"attempt": attempt, "error": repr(exc)})\n            if attempt < retries: time.sleep(min(30, 2 ** attempt))\n    raise RuntimeError(f"R26_DERIVED_SHARD_UPLOAD_FAILED: path={path}; errors={errors}")\n\n\ndef _derived_identity(runner: Any) -> tuple[str, str]:\n    attempt = os.environ.get("IHARQ_EXECUTION_ATTEMPT_ID")\n    if not attempt:\n        attempt = time.strftime("%Y%m%d%H%M%S", time.gmtime()) + "-" + uuid.uuid4().hex[:8]\n        os.environ["IHARQ_EXECUTION_ATTEMPT_ID"] = attempt\n    suffix = re.sub(r"[^a-z0-9-]+", "-", attempt.lower()).strip("-")[-24:]\n    slug = f"{POLICY[\'derived_dataset_slug_prefix\']}-{runner.pipeline.config_id[:12]}-{suffix}"[:100].strip("-")\n    return attempt, f"{POLICY[\'derived_kaggle_username\']}/{slug}"\n\n\n\n\ndef _post_preprocessing_geometry(\n    recording: Any,\n    operations: list[dict[str, Any]],\n) -> tuple[int, int, float]:\n    """Infer the exact frozen output geometry without materializing a signal."""\n    names = [operation.get("name") for operation in operations]\n    official = [\n        "validate_units",\n        "capture_events",\n        "select_eeg",\n        "demean",\n        "rereference_average",\n        "resample_polyphase_with_events",\n        "bandpass_sos_zero_phase",\n        "cast",\n    ]\n    if names != official:\n        raise RuntimeError(\n            "R34_STORAGE_FORECAST_PREPROCESSING_GRAPH_MISMATCH: "\n            f"observed={names}"\n        )\n\n    channel_types = list(\n        recording.source_metadata.get("channel_types", [])\n    )\n    if len(channel_types) != len(recording.channel_names):\n        raise RuntimeError(\n            "R34_STORAGE_FORECAST_CHANNEL_METADATA_MISMATCH: "\n            f"source={_recording_source_unit(recording)}; "\n            f"channels={len(recording.channel_names)}; types={len(channel_types)}"\n        )\n    channels = sum(str(value).lower() == "eeg" for value in channel_types)\n    if channels <= 0:\n        raise RuntimeError(\n            "R34_STORAGE_FORECAST_NO_EEG_CHANNELS: "\n            f"source={_recording_source_unit(recording)}"\n        )\n\n    input_samples = int(recording.signal.shape[1])\n    input_hz = float(recording.sampling_hz)\n    output_hz = 160.0\n    output_samples = max(1, int(round(input_samples * output_hz / input_hz)))\n    return channels, output_samples, output_hz\n\n\n\n\ndef _count_windows_for_event(\n    *,\n    event: Any,\n    signal_samples: int,\n    input_sampling_hz: float,\n    output_sampling_hz: float,\n    start_offset_samples: int,\n    duration_samples: int,\n) -> tuple[int, dict[str, Any] | None]:\n    """Forecast the official one-window-per-event contract and invalid evidence."""\n    ratio = float(output_sampling_hz) / float(input_sampling_hz)\n    resampled_sample = min(\n        int(round(int(event.start_sample) * ratio)),\n        max(0, int(signal_samples) - 1),\n    )\n    start = resampled_sample + int(start_offset_samples)\n    end = start + int(duration_samples)\n    if start < 0 or end > int(signal_samples):\n        return 0, {\n            "event_id": str(event.event_id),\n            "original_source_event_sample": int(event.start_sample),\n            "resampled_event_sample": resampled_sample,\n            "start": start,\n            "end": end,\n            "samples": int(signal_samples),\n            "reason": "WINDOW_OUT_OF_BOUNDS",\n        }\n    return 1, None\n\n\n\n\ndef _estimate_derived_storage(pipeline: Any) -> dict[str, Any]:\n    """Forecast exact logical float32 window bytes under the official freeze."""\n    from iharq.layer1_data_protocol.labels import map_event_label\n\n    operations = list(pipeline.state["operations"])\n    window_profile = dict(pipeline.config.get("windows", {}))\n    required = {\n        "start_offset_samples": 80,\n        "duration_samples": 480,\n        "stride_samples": 480,\n        "target_sampling_hz": 160,\n        "last_window_policy": "ONE_WINDOW_PER_INCLUDED_SOURCE_EVENT",\n        "bounds_policy": "REJECT_OUT_OF_BOUNDS",\n    }\n    mismatches = {\n        key: {"expected": expected, "observed": window_profile.get(key)}\n        for key, expected in required.items()\n        if window_profile.get(key) != expected\n    }\n    if mismatches:\n        raise RuntimeError(\n            "R34_STORAGE_FORECAST_WINDOW_CONTRACT_MISMATCH: "\n            + json.dumps(mismatches, sort_keys=True)\n        )\n\n    label_by_dataset = {\n        row["payload"]["dataset_id"]: row\n        for row in pipeline.state["label_records"]\n    }\n    rows: list[dict[str, Any]] = []\n    dataset_totals: dict[str, dict[str, Any]] = {}\n    subject_totals: dict[str, int] = {}\n    forecast_invalid: list[dict[str, Any]] = []\n    total_windows = 0\n    total_logical_bytes = 0\n\n    for recording in pipeline.state["recordings"]:\n        channels, signal_samples, output_hz = _post_preprocessing_geometry(\n            recording,\n            operations,\n        )\n        accepted_events = 0\n        recording_windows = 0\n        recording_invalid = 0\n        for event in recording.events:\n            normalized = map_event_label(\n                event.original_label,\n                label_by_dataset[recording.dataset_id],\n            )\n            if normalized is None:\n                continue\n            accepted_events += 1\n            count, invalid = _count_windows_for_event(\n                event=event,\n                signal_samples=signal_samples,\n                input_sampling_hz=float(recording.sampling_hz),\n                output_sampling_hz=output_hz,\n                start_offset_samples=80,\n                duration_samples=480,\n            )\n            recording_windows += count\n            if invalid is not None:\n                recording_invalid += 1\n                forecast_invalid.append(\n                    {\n                        "dataset_id": recording.dataset_id,\n                        "subject_id": recording.subject_id,\n                        "session_id": recording.session_id,\n                        "run_id": recording.run_id,\n                        **invalid,\n                    }\n                )\n\n        logical_bytes = recording_windows * channels * 480 * 4\n        source_unit = _recording_source_unit(recording)\n        subject_key = f"{recording.dataset_id}:subject={recording.subject_id}"\n        subject_totals[subject_key] = subject_totals.get(subject_key, 0) + logical_bytes\n        dataset_row = dataset_totals.setdefault(\n            recording.dataset_id,\n            {\n                "dataset_id": recording.dataset_id,\n                "recordings": 0,\n                "accepted_events": 0,\n                "planned_windows": 0,\n                "forecast_invalid_windows": 0,\n                "logical_signal_bytes": 0,\n            },\n        )\n        dataset_row["recordings"] += 1\n        dataset_row["accepted_events"] += accepted_events\n        dataset_row["planned_windows"] += recording_windows\n        dataset_row["forecast_invalid_windows"] += recording_invalid\n        dataset_row["logical_signal_bytes"] += logical_bytes\n        rows.append(\n            {\n                "source_unit": source_unit,\n                "dataset_id": recording.dataset_id,\n                "subject_id": recording.subject_id,\n                "session_id": recording.session_id,\n                "run_id": recording.run_id,\n                "input_sampling_hz": float(recording.sampling_hz),\n                "output_sampling_hz": output_hz,\n                "eeg_channels": channels,\n                "output_signal_samples": signal_samples,\n                "start_offset_samples": 80,\n                "duration_samples": 480,\n                "stride_samples": 480,\n                "accepted_events": accepted_events,\n                "planned_windows": recording_windows,\n                "forecast_invalid_windows": recording_invalid,\n                "signal_dtype": "float32",\n                "logical_signal_bytes": logical_bytes,\n            }\n        )\n        total_windows += recording_windows\n        total_logical_bytes += logical_bytes\n\n    metadata_bytes_estimate = total_windows * 2048 + len(subject_totals) * 1024 * 1024\n    ratio_lower = float(POLICY["storage_forecast_compression_ratio_lower"])\n    ratio_upper = float(POLICY["storage_forecast_compression_ratio_upper"])\n    lower_bytes = int(total_logical_bytes * ratio_lower + metadata_bytes_estimate)\n    upper_bytes = int(total_logical_bytes * ratio_upper + metadata_bytes_estimate)\n    recommended_capacity = int(upper_bytes * float(POLICY["storage_forecast_safety_multiplier"]))\n    largest_subject_bytes = max(subject_totals.values(), default=0)\n    recommended_local_scratch = int(\n        largest_subject_bytes * 1.15\n        + float(POLICY["minimum_disk_free_gib"]) * _GIB\n    )\n\n    report = {\n        "artifact_id": f"P01-L1-DERIVED-STORAGE-FORECAST-{pipeline.config_id[:16]}",\n        "policy_id": POLICY["policy_id"],\n        "scientific_freeze": POLICY["scientific_freeze_unchanged"],\n        "calculation_status": "EXACT_LOGICAL_FLOAT32_BYTES_PLANNING_ENVELOPE_FOR_COMPRESSION",\n        "config_id": pipeline.config_id,\n        "window_profile": window_profile,\n        "preprocessing_operations": operations,\n        "event_resampling": "MNE_POLYPHASE_JOINT_EVENTS_EQUIVALENT_INDEX_FORECAST",\n        "planned_recordings": len(rows),\n        "planned_subject_profiles": len(subject_totals),\n        "planned_windows": total_windows,\n        "forecast_invalid_window_count": len(forecast_invalid),\n        "forecast_invalid_windows": forecast_invalid,\n        "signal_dtype": "float32",\n        "logical_float32_signal_bytes": total_logical_bytes,\n        "logical_float32_signal_gib": round(total_logical_bytes / _GIB, 3),\n        "estimated_index_and_hdf5_metadata_bytes": metadata_bytes_estimate,\n        "lossless_hdf5_planning_envelope": {\n            "lower_bytes": lower_bytes,\n            "lower_gib": round(lower_bytes / _GIB, 3),\n            "upper_bytes": upper_bytes,\n            "upper_gib": round(upper_bytes / _GIB, 3),\n            "compression_ratio_range": [ratio_lower, ratio_upper],\n            "warning": "Compression ratio is planning-only; Stage 14 records exact uploaded bytes.",\n        },\n        "recommended_private_kaggle_capacity": {\n            "bytes": recommended_capacity,\n            "gib": round(recommended_capacity / _GIB, 3),\n            "safety_multiplier": float(POLICY["storage_forecast_safety_multiplier"]),\n        },\n        "recommended_local_scratch": {\n            "bytes": recommended_local_scratch,\n            "gib": round(recommended_local_scratch / _GIB, 3),\n            "largest_subject_logical_bytes": largest_subject_bytes,\n        },\n        "dataset_totals": [\n            {\n                **row,\n                "logical_signal_gib": round(row["logical_signal_bytes"] / _GIB, 3),\n            }\n            for _, row in sorted(dataset_totals.items())\n        ],\n        "recording_rows": rows,\n    }\n    report_path = (\n        pipeline.bundle_root\n        / "reports"\n        / "phase_01"\n        / "storage"\n        / "derived_output_storage_forecast.json"\n    )\n    _atomic_json(report_path, report)\n    pipeline.state["r26_storage_forecast"] = report\n    pipeline.state["r26_storage_forecast_path"] = str(\n        report_path.relative_to(pipeline.bundle_root)\n    )\n    return report\n\n\n\ndef _subject_forecast_bytes(\n    pipeline: Any,\n    dataset_id: str,\n    subject: int,\n) -> int:\n    forecast = pipeline.state.get(\n        "r26_storage_forecast",\n        {},\n    )\n    rows = forecast.get("recording_rows", [])\n    return sum(\n        int(row.get("logical_signal_bytes", 0))\n        for row in rows\n        if (\n            str(row.get("dataset_id")) == str(dataset_id)\n            and str(row.get("subject_id")) == str(subject)\n        )\n    )\n\n\ndef _assert_subject_scratch_capacity(\n    pipeline: Any,\n    dataset_id: str,\n    subject: int,\n) -> None:\n    expected = _subject_forecast_bytes(\n        pipeline,\n        dataset_id,\n        subject,\n    )\n    fixed_reserve = int(\n        float(POLICY["minimum_disk_free_gib"]) * _GIB\n    )\n    required = int(expected * 1.15) + fixed_reserve\n    disk = _disk_snapshot(pipeline.work_root)\n\n    if disk["free_bytes"] < required:\n        raise RuntimeError(\n            "R26_SUBJECT_SHARD_SCRATCH_CAPACITY_INSUFFICIENT: "\n            f"dataset={dataset_id}; subject={subject}; "\n            f"expected_subject_logical_gib={expected/_GIB:.3f}; "\n            f"required_free_gib={required/_GIB:.3f}; "\n            f"observed_free_gib={disk[\'free_gib\']}"\n        )\n\n\ndef _copy_compact_path(\n    source: Path,\n    destination: Path,\n) -> None:\n    source = Path(source)\n    destination = Path(destination)\n\n    if not source.exists():\n        return\n\n    disallowed_suffixes = {\n        ".h5",\n        ".hdf5",\n        ".mat",\n        ".edf",\n        ".gdf",\n        ".fif",\n    }\n\n    if source.is_file():\n        if source.suffix.lower() in disallowed_suffixes:\n            return\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(source, destination)\n        return\n\n    for path in sorted(source.rglob("*")):\n        if not path.is_file():\n            continue\n        if path.suffix.lower() in disallowed_suffixes:\n            continue\n        relative = path.relative_to(source)\n        target = destination / relative\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(path, target)\n\n\ndef _write_deterministic_zip(\n    source_root: Path,\n    zip_path: Path,\n) -> None:\n    source_root = Path(source_root)\n    zip_path = Path(zip_path)\n    zip_path.parent.mkdir(parents=True, exist_ok=True)\n\n    with zipfile.ZipFile(\n        zip_path,\n        "w",\n        compression=zipfile.ZIP_DEFLATED,\n        compresslevel=9,\n    ) as archive:\n        for path in sorted(source_root.rglob("*")):\n            if not path.is_file():\n                continue\n            relative = path.relative_to(\n                source_root.parent\n            ).as_posix()\n            info = zipfile.ZipInfo(relative)\n            info.date_time = (1980, 1, 1, 0, 0, 0)\n            info.compress_type = zipfile.ZIP_DEFLATED\n            info.external_attr = 0o100644 << 16\n            archive.writestr(\n                info,\n                path.read_bytes(),\n                compress_type=zipfile.ZIP_DEFLATED,\n                compresslevel=9,\n            )\n\n\ndef _create_github_ready_repository(\n    runner: Any,\n) -> dict[str, Any]:\n    """\n    Create a compact repository package with no raw or derived EEG arrays.\n\n    This package is ready for later Git initialization/push, but the notebook\n    deliberately does not require GitHub credentials or publish automatically.\n    """\n    pipeline = runner.pipeline\n    attempt = pipeline.state["r26_execution_attempt_id"]\n    repository_name = (\n        "IHARQ_P01_L1_GitHub_Ready_Repository_"\n        f"{pipeline.config_id[:12]}_"\n        f"{_safe(attempt)}"\n    )\n    release_root = (\n        pipeline.work_root\n        / "github_ready_release"\n    )\n    repository_root = release_root / repository_name\n    shutil.rmtree(repository_root, ignore_errors=True)\n    repository_root.mkdir(parents=True)\n\n    # Reusable implementation and configuration.\n    _copy_compact_path(\n        pipeline.package_root / "src",\n        repository_root / "src",\n    )\n    _copy_compact_path(\n        pipeline.package_root / "configs",\n        repository_root / "configs",\n    )\n    for name in [\n        "requirements-lock.txt",\n        "pyproject.toml",\n        "README.md",\n    ]:\n        _copy_compact_path(\n            pipeline.package_root / name,\n            repository_root / name,\n        )\n\n    runtime_dir = repository_root / "runtime_overlays"\n    runtime_dir.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(\n        Path(__file__),\n        runtime_dir / "iharq_bounded_streaming.py",\n    )\n    try:\n        import iharq_acquisition_acceleration\n        acquisition_path = Path(\n            inspect.getsourcefile(\n                iharq_acquisition_acceleration\n            )\n            or ""\n        )\n        if acquisition_path.is_file():\n            shutil.copy2(\n                acquisition_path,\n                runtime_dir\n                / "iharq_acquisition_acceleration.py",\n            )\n    except Exception:\n        pass\n\n    (\n        runtime_dir\n        / "iharq_window_shard_reader.py"\n    ).write_text(\n        WINDOW_SHARD_READER_SOURCE,\n        encoding="utf-8",\n    )\n\n    # Compact scientific artifacts and evidence.\n    compact_bundle_paths = [\n        "authority_manifest.json",\n        "environment_manifest.json",\n        "notebook_manifest.json",\n        "environment_amendment.json",\n        "phase_execution_handoff.yaml",\n        "gate_decision.json",\n        "integration_patch_manifest.yaml",\n        "checksums.sha256",\n        "records",\n        "reports/phase_01",\n        "docs/cards",\n        "manifests/phase_01",\n        "derived_outputs/preprocessing_fit_state",\n        "analysis_inputs",\n        "protocol_v1_handoff",\n        "layer0_handoff",\n        "evidence_map_handoff",\n        "layer10_source_bundle",\n        "phase2_handoff",\n        "handoffs",\n        "external_artifact_pointers",\n        "negative_and_failed_results",\n        "figure_source_data",\n        "table_source_data",\n    ]\n\n    for relative in compact_bundle_paths:\n        _copy_compact_path(\n            pipeline.bundle_root / relative,\n            repository_root / "artifacts" / relative,\n        )\n\n    (repository_root / ".gitignore").write_text(\n        "\\n".join(\n            [\n                "__pycache__/",\n                "*.py[cod]",\n                ".pytest_cache/",\n                ".mypy_cache/",\n                ".venv/",\n                "venv/",\n                ".env",\n                "*.token",\n                "*.secret",\n                "*.h5",\n                "*.hdf5",\n                "*.mat",\n                "*.edf",\n                "*.gdf",\n                "*.fif",\n                "data/raw/",\n                "data/derived/",\n                "source_cache/",\n                "streaming_runtime/",\n                "",\n            ]\n        ),\n        encoding="utf-8",\n    )\n\n    data_dir = repository_root / "data"\n    data_dir.mkdir(parents=True, exist_ok=True)\n    (data_dir / "README.md").write_text(\n        textwrap.dedent(\n            f"""\\\n            # Data access\n\n            Large numerical arrays are intentionally not stored in this\n            GitHub-ready repository.\n\n            Attach the private Kaggle Dataset identified by:\n\n            `artifacts/external_artifact_pointers/derived_windows_dataset.json`\n\n            The original raw sources remain in the three source Kaggle\n            Datasets. The derived Dataset contains lossless HDF5 subject\n            shards, indexes, manifest, sidecar, storage reports, and the\n            future-phase reader.\n\n            Scientific freeze: {POLICY["scientific_freeze_unchanged"]}\n            Runtime policy: {POLICY["policy_id"]}\n            """\n        ),\n        encoding="utf-8",\n    )\n\n    (repository_root / "README.md").write_text(\n        textwrap.dedent(\n            f"""\\\n            # IHARQ Phase 01 / Layer 01\n\n            This repository is the compact code, configuration, governance,\n            records, evidence, cards, manifests, gates, and handoff companion\n            to the private Kaggle derived-window Dataset.\n\n            It contains no duplicate raw EEG files and no HDF5 window shards.\n\n            - Config ID: `{pipeline.config_id}`\n            - Execution attempt: `{attempt}`\n            - Scientific freeze: `{POLICY["scientific_freeze_unchanged"]}`\n            - Runtime policy: `{POLICY["policy_id"]}`\n            - Derived Dataset handle:\n              `{pipeline.state.get("r26_derived_handle")}`\n\n            ## Reproduction\n\n            1. Attach the original source Kaggle Datasets when raw-source\n               regeneration is required.\n            2. Attach the private derived-window Dataset for future phases.\n            3. Use `runtime_overlays/iharq_window_shard_reader.py` to resolve\n               window IDs to immutable HDF5 shard rows.\n            4. Verify manifests and SHA-256 values before use.\n\n            No later-phase model training is executed in Layer 1.\n            """\n        ),\n        encoding="utf-8",\n    )\n\n    (repository_root / "SECURITY.md").write_text(\n        (\n            "# Security\\n\\n"\n            "No Kaggle token or secret is stored here. Keep the derived "\n            "Dataset private unless a separate license and redistribution "\n            "review authorizes publication.\\n"\n        ),\n        encoding="utf-8",\n    )\n\n    (repository_root / "DATA_LICENSE_NOTICE.md").write_text(\n        (\n            "# Data and license notice\\n\\n"\n            "This compact repository does not redistribute the raw EEG "\n            "sources or derived numerical shards. Dataset-specific licenses "\n            "and redistribution constraints remain recorded in DatasetRecord "\n            "and card artifacts. The Kaggle derived Dataset inherits those "\n            "constraints and is private by default.\\n"\n        ),\n        encoding="utf-8",\n    )\n\n    # Validate exclusions and scan for obvious secret leakage.\n    files = [\n        path\n        for path in repository_root.rglob("*")\n        if path.is_file()\n    ]\n    forbidden = [\n        path\n        for path in files\n        if path.suffix.lower()\n        in {".h5", ".hdf5", ".mat", ".edf", ".gdf", ".fif"}\n    ]\n    if forbidden:\n        raise RuntimeError(\n            "R26_GITHUB_READY_LARGE_ARRAY_EXCLUSION_FAILED: "\n            f"{forbidden[:5]}"\n        )\n\n    # Reject actual credential files or the exact live secret value. Merely\n    # mentioning the environment-variable name in code/documentation is safe.\n    credential_files = [\n        path\n        for path in files\n        if path.name.lower() in {\n            "kaggle.json",\n            ".env",\n            "credentials.json",\n        }\n    ]\n    if credential_files:\n        raise RuntimeError(\n            "R26_GITHUB_READY_SECRET_SCAN_FAILED: "\n            f"credential_files={credential_files[:5]}"\n        )\n\n    live_token = os.environ.get("KAGGLE_API_TOKEN", "")\n    if live_token:\n        token_hits: list[str] = []\n        for path in files:\n            if path.stat().st_size > 5 * 1024 * 1024:\n                continue\n            try:\n                content = path.read_bytes()\n            except Exception:\n                continue\n            if live_token.encode("utf-8") in content:\n                token_hits.append(\n                    path.relative_to(\n                        repository_root\n                    ).as_posix()\n                )\n        if token_hits:\n            raise RuntimeError(\n                "R26_GITHUB_READY_SECRET_SCAN_FAILED: "\n                f"live_token_serialized_in={token_hits[:5]}"\n            )\n\n    manifest_rows = []\n    for path in sorted(files):\n        manifest_rows.append(\n            {\n                "path": path.relative_to(\n                    repository_root\n                ).as_posix(),\n                "bytes": path.stat().st_size,\n                "sha256": _sha256(path),\n            }\n        )\n\n    repository_manifest = {\n        "artifact_id": (\n            f"P01-L1-GITHUB-READY-"\n            f"{pipeline.config_id[:16]}-{attempt}"\n        ),\n        "schema_version": 1,\n        "scientific_freeze": (\n            POLICY["scientific_freeze_unchanged"]\n        ),\n        "policy_id": POLICY["policy_id"],\n        "repository_name": repository_name,\n        "derived_dataset_handle": (\n            pipeline.state.get("r26_derived_handle")\n        ),\n        "file_count": len(manifest_rows),\n        "files": manifest_rows,\n        "excluded_large_arrays": True,\n        "excluded_raw_sources": True,\n        "automatic_github_publish": False,\n        "reason": (\n            "GitHub credentials and repository ownership are not runtime "\n            "dependencies; this ZIP can be initialized/pushed later."\n        ),\n    }\n    manifest_path = (\n        repository_root\n        / "GITHUB_READY_REPOSITORY_MANIFEST.json"\n    )\n    _atomic_json(manifest_path, repository_manifest)\n\n    total_bytes = sum(\n        path.stat().st_size\n        for path in repository_root.rglob("*")\n        if path.is_file()\n    )\n    maximum = int(\n        float(POLICY["github_ready_repository_max_gib"])\n        * _GIB\n    )\n    if total_bytes > maximum:\n        raise RuntimeError(\n            "R26_GITHUB_READY_REPOSITORY_TOO_LARGE: "\n            f"bytes={total_bytes}; maximum={maximum}"\n        )\n\n    zip_path = (\n        pipeline.work_root\n        / f"{repository_name}.zip"\n    )\n    _write_deterministic_zip(\n        repository_root,\n        zip_path,\n    )\n    zip_hash = _sha256(zip_path)\n    sidecar_path = zip_path.with_suffix(\n        zip_path.suffix + ".sha256"\n    )\n    sidecar_path.write_text(\n        f"{zip_hash}  {zip_path.name}\\n",\n        encoding="utf-8",\n    )\n\n    pointer = {\n        "artifact_id": repository_manifest["artifact_id"],\n        "format": "GITHUB_READY_REPOSITORY_ZIP",\n        "path": str(zip_path),\n        "sha256": zip_hash,\n        "bytes": zip_path.stat().st_size,\n        "repository_uncompressed_bytes": total_bytes,\n        "repository_file_count": len(manifest_rows),\n        "contains_large_arrays": False,\n        "derived_dataset_pointer": (\n            "external_artifact_pointers/"\n            "derived_windows_dataset.json"\n        ),\n        "publication_status": (\n            "READY_FOR_OWNER_CONTROLLED_GITHUB_PUSH"\n        ),\n    }\n\n    pointer["detached_hash_boundary"] = (\n        "THE REPOSITORY ZIP IS CREATED AFTER THE EXECUTION BUNDLE FREEZE; "\n        "ITS ACTUAL SHA-256 IS RECORDED OUTSIDE BOTH SELF-REFERENTIAL ZIP FILES."\n    )\n    pointer_path = (\n        pipeline.work_root\n        / "github_ready_repository_pointer.json"\n    )\n    _atomic_json(pointer_path, pointer)\n    release_manifest_path = (\n        pipeline.work_root\n        / "github_ready_repository_manifest.json"\n    )\n    _atomic_json(release_manifest_path, repository_manifest)\n    pointer["external_pointer_path"] = str(pointer_path)\n    pointer["external_manifest_path"] = str(release_manifest_path)\n\n    pipeline.state["r26_github_ready"] = pointer\n    return pointer\n\ndef _streaming_load_sources(\n    self,\n    input_root: Path,\n):\n    from concurrent.futures import (\n        ThreadPoolExecutor,\n        as_completed,\n    )\n    from iharq.layer1_data_protocol import (\n        adapters as adapters_module,\n    )\n\n    recordings = []\n    inventories = {}\n    plan = []\n    pass1_fit_rows: list[\n        dict[str, Any]\n    ] = []\n\n    bundle = self.bundle_root\n    descriptor_index_path = (\n        bundle\n        / "inputs"\n        / "streaming_descriptor_index.jsonl"\n    )\n    fit_index_path = (\n        bundle\n        / "inputs"\n        / "streaming_pass1_fit_stats.jsonl"\n    )\n\n    for aggregate_path in [\n        descriptor_index_path,\n        fit_index_path,\n    ]:\n        aggregate_path.unlink(\n            missing_ok=True\n        )\n\n    source_resolution_file = Path(\n        self.state[\n            "r26_source_resolution_file"\n        ]\n    )\n    source_resolution_sha256 = _sha256(\n        source_resolution_file\n    )\n\n    profiles = list(\n        self.state.get("profiles", [])\n    )\n\n    collect_fit_stats = any(\n        operation.get("name")\n        == "standardize_train_fit"\n        for operation in (\n            self.config.get(\n                "preprocessing",\n                {},\n            ).get(\n                "operations",\n                [],\n            )\n        )\n    )\n\n    plan_by_dataset = {}\n\n    for profile in profiles:\n        adapter_class = (\n            adapters_module.ADAPTERS.get(\n                profile.adapter\n            )\n        )\n\n        if adapter_class is None:\n            self.blockers.append(\n                {\n                    "code": "P01_ADAPTER_UNKNOWN",\n                    "dataset_id": (\n                        profile.dataset_id\n                    ),\n                    "adapter": profile.adapter,\n                    "owner": "BUILD_BOOK",\n                }\n            )\n            continue\n\n        adapter = adapter_class(\n            profile,\n            input_root,\n            (\n                self.work_root\n                / "source_cache"\n                / profile.dataset_id\n            ),\n        )\n\n        try:\n            files = adapter.resolve_files()\n            inventory = adapter.verify_files(\n                files\n            )\n            inventories[\n                profile.dataset_id\n            ] = inventory\n        except Exception as exc:\n            self.blockers.append(\n                {\n                    "code": (\n                        "P01_SOURCE_LOAD_FAILED"\n                    ),\n                    "dataset_id": (\n                        profile.dataset_id\n                    ),\n                    "message": str(exc),\n                    "owner": "OWNER_OR_ADAPTER",\n                }\n            )\n            continue\n\n        subjects = _subject_values(\n            profile\n        )\n\n        for subject in subjects:\n            plan.append(\n                (profile, int(subject))\n            )\n\n        plan_by_dataset[\n            str(profile.dataset_id)\n        ] = (\n            profile,\n            [int(value) for value in subjects],\n        )\n\n    if self.blockers:\n        raise RuntimeError(\n            "R26_PASS1_SOURCE_INTAKE_BLOCKED: "\n            + json.dumps(\n                self.blockers,\n                indent=2,\n            )\n        )\n\n    checkpoint_root = (\n        self.work_root\n        / "streaming_runtime"\n        / "pass1_subject_checkpoints"\n        / _safe(self.config_id)\n    )\n    checkpoint_root.mkdir(\n        parents=True,\n        exist_ok=True,\n    )\n\n    child_root = (\n        self.work_root\n        / "streaming_runtime"\n        / "pass1_descriptors"\n    )\n\n    cpu_count = max(\n        1,\n        int(os.cpu_count() or 1),\n    )\n\n    # Upper bounds only. The final worker count is resolved *after* a real\n    # conversion preflight using measured peak RSS and currently available RAM.\n    parallelism_upper_bounds = {\n        "PhysioNetMI": min(8, cpu_count),\n        "BNCI2014_001": min(6, cpu_count),\n        "Lee2019_MI": min(4, cpu_count),\n    }\n    configured_parallelism = dict(parallelism_upper_bounds)\n    resolved_parallelism: dict[str, int] = {}\n    parallelism_evidence: dict[str, dict[str, Any]] = {}\n\n    def available_memory_bytes() -> int:\n        try:\n            import psutil\n            return int(psutil.virtual_memory().available)\n        except Exception:\n            try:\n                return int(os.sysconf("SC_AVPHYS_PAGES")) * int(os.sysconf("SC_PAGE_SIZE"))\n            except Exception:\n                # Conservative fallback for environments without either API.\n                return 8 * _GIB\n\n    def resolve_parallelism(dataset_id: str, preflight_result: dict[str, Any]) -> int:\n        available = max(1, available_memory_bytes())\n        measured_peak = max(\n            int(preflight_result.get("child_peak_rss_bytes", 0)),\n            512 * 1024 * 1024,\n        )\n        # Keep a large parent/filesystem reserve and inflate the measured worker\n        # footprint. This makes acceleration adaptive without risking OOM-driven\n        # corruption or repeated subject failures.\n        reserve = max(4 * _GIB, int(available * 0.20))\n        usable = max(measured_peak, available - reserve)\n        guarded_per_worker = max(1 * _GIB, int(measured_peak * 1.60))\n        memory_cap = max(1, int(usable // guarded_per_worker))\n        upper = max(1, int(parallelism_upper_bounds.get(dataset_id, 1)))\n        selected = max(1, min(cpu_count, upper, memory_cap))\n        resolved_parallelism[dataset_id] = selected\n        parallelism_evidence[dataset_id] = {\n            "selected_workers": selected,\n            "cpu_count": cpu_count,\n            "dataset_upper_bound": upper,\n            "available_memory_bytes": available,\n            "reserved_memory_bytes": reserve,\n            "measured_preflight_peak_rss_bytes": measured_peak,\n            "guarded_per_worker_bytes": guarded_per_worker,\n            "memory_worker_cap": memory_cap,\n            "safety_multiplier": 1.60,\n            "scientific_scope_changed": False,\n        }\n        return selected\n\n    result_by_key: dict[\n        tuple[str, int],\n        tuple[\n            list[dict[str, Any]],\n            list[dict[str, Any]],\n        ],\n    ] = {}\n\n    started = time.monotonic()\n    last_progress = 0.0\n    completed = 0\n    completed_recordings = 0\n    reused_checkpoints = 0\n    generated_checkpoints = 0\n    total_subjects = len(plan)\n\n    def profile_sha256(\n        profile,\n    ) -> str:\n        payload = json.dumps(\n            _jsonable(\n                _profile_dict(profile)\n            ),\n            sort_keys=True,\n            separators=(",", ":"),\n        ).encode("utf-8")\n        return hashlib.sha256(\n            payload\n        ).hexdigest()\n\n    def checkpoint_paths(\n        dataset_id: str,\n        subject: int,\n    ) -> tuple[Path, Path, Path]:\n        root = (\n            checkpoint_root\n            / _safe(dataset_id)\n            / f"subject_{subject:03d}"\n        )\n        root.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n        return (\n            root / "descriptors.jsonl",\n            root / "fit_stats.jsonl",\n            root / "checkpoint_manifest.json",\n        )\n\n    def load_checkpoint(\n        profile,\n        subject: int,\n    ):\n        dataset_id = str(\n            profile.dataset_id\n        )\n        descriptor_path, fit_path, manifest_path = (\n            checkpoint_paths(\n                dataset_id,\n                subject,\n            )\n        )\n\n        if not (\n            descriptor_path.is_file()\n            and fit_path.is_file()\n            and manifest_path.is_file()\n        ):\n            return None\n\n        try:\n            manifest = json.loads(\n                manifest_path.read_text(\n                    encoding="utf-8"\n                )\n            )\n\n            expected_inventory = str(\n                inventories[\n                    dataset_id\n                ].get(\n                    "observed_checksum",\n                    inventories[\n                        dataset_id\n                    ].get(\n                        "aggregate_sha256",\n                        "",\n                    ),\n                )\n            )\n\n            required = {\n                "checkpoint_policy_id": (\n                    "P01-L1-R26-PASS1-"\n                    "SUBJECT-CHECKPOINT-R1"\n                ),\n                "config_id": (\n                    str(self.config_id)\n                ),\n                "dataset_id": dataset_id,\n                "subject": int(subject),\n                "source_resolution_sha256": (\n                    source_resolution_sha256\n                ),\n                "profile_sha256": (\n                    profile_sha256(profile)\n                ),\n                "source_inventory_sha256": (\n                    expected_inventory\n                ),\n                "collect_fit_stats": bool(\n                    collect_fit_stats\n                ),\n            }\n\n            for key, expected in (\n                required.items()\n            ):\n                if (\n                    manifest.get(key)\n                    != expected\n                ):\n                    raise RuntimeError(\n                        "CHECKPOINT_FIELD_MISMATCH: "\n                        f"{key}; "\n                        f"expected={expected!r}; "\n                        f"observed="\n                        f"{manifest.get(key)!r}"\n                    )\n\n            if (\n                _sha256(descriptor_path)\n                != manifest.get(\n                    "descriptor_sha256"\n                )\n            ):\n                raise RuntimeError(\n                    "CHECKPOINT_DESCRIPTOR_SHA256_"\n                    "MISMATCH"\n                )\n\n            if (\n                _sha256(fit_path)\n                != manifest.get(\n                    "fit_stats_sha256"\n                )\n            ):\n                raise RuntimeError(\n                    "CHECKPOINT_FIT_SHA256_MISMATCH"\n                )\n\n            rows = _read_jsonl(\n                descriptor_path\n            )\n            fit_rows = _read_jsonl(\n                fit_path\n            )\n\n            if not rows:\n                raise RuntimeError(\n                    "CHECKPOINT_EMPTY_DESCRIPTORS"\n                )\n\n            if any(\n                str(row.get("dataset_id"))\n                != dataset_id\n                or str(row.get("subject_id"))\n                != str(subject)\n                for row in rows\n            ):\n                raise RuntimeError(\n                    "CHECKPOINT_SUBJECT_SCOPE_"\n                    "MISMATCH"\n                )\n\n            if (\n                collect_fit_stats\n                and len(fit_rows)\n                != len(rows)\n            ):\n                raise RuntimeError(\n                    "CHECKPOINT_FIT_ROW_COUNT_"\n                    "MISMATCH"\n                )\n\n            return rows, fit_rows\n\n        except Exception as exc:\n            rejection_path = (\n                manifest_path.parent\n                / "checkpoint_rejection.json"\n            )\n            _atomic_json(\n                rejection_path,\n                {\n                    "event": (\n                        "PASS1_CHECKPOINT_REJECTED"\n                    ),\n                    "dataset_id": dataset_id,\n                    "subject": int(subject),\n                    "reason": repr(exc),\n                    "recorded_unix": time.time(),\n                },\n            )\n\n            for path in [\n                descriptor_path,\n                fit_path,\n                manifest_path,\n            ]:\n                path.unlink(\n                    missing_ok=True\n                )\n\n            return None\n\n    def commit_checkpoint(\n        profile,\n        subject: int,\n        rows,\n        fit_rows,\n    ):\n        dataset_id = str(\n            profile.dataset_id\n        )\n        descriptor_path, fit_path, manifest_path = (\n            checkpoint_paths(\n                dataset_id,\n                subject,\n            )\n        )\n\n        descriptor_temp = (\n            descriptor_path\n            .with_suffix(\n                ".jsonl.tmp"\n            )\n        )\n        fit_temp = (\n            fit_path\n            .with_suffix(\n                ".jsonl.tmp"\n            )\n        )\n\n        for path in [\n            descriptor_temp,\n            fit_temp,\n        ]:\n            path.unlink(\n                missing_ok=True\n            )\n\n        _append_jsonl(\n            descriptor_temp,\n            rows,\n        )\n        _append_jsonl(\n            fit_temp,\n            fit_rows,\n        )\n\n        descriptor_temp.replace(\n            descriptor_path\n        )\n        fit_temp.replace(\n            fit_path\n        )\n\n        inventory_sha256 = str(\n            inventories[\n                dataset_id\n            ].get(\n                "observed_checksum",\n                inventories[\n                    dataset_id\n                ].get(\n                    "aggregate_sha256",\n                    "",\n                ),\n            )\n        )\n\n        _atomic_json(\n            manifest_path,\n            {\n                "checkpoint_policy_id": (\n                    "P01-L1-R26-PASS1-"\n                    "SUBJECT-CHECKPOINT-R1"\n                ),\n                "scientific_freeze": (\n                    POLICY[\n                        "scientific_freeze_"\n                        "unchanged"\n                    ]\n                ),\n                "config_id": (\n                    str(self.config_id)\n                ),\n                "dataset_id": dataset_id,\n                "subject": int(subject),\n                "source_resolution_sha256": (\n                    source_resolution_sha256\n                ),\n                "profile_sha256": (\n                    profile_sha256(profile)\n                ),\n                "source_inventory_sha256": (\n                    inventory_sha256\n                ),\n                "collect_fit_stats": bool(\n                    collect_fit_stats\n                ),\n                "descriptor_rows": len(rows),\n                "fit_stat_rows": len(\n                    fit_rows\n                ),\n                "descriptor_sha256": (\n                    _sha256(descriptor_path)\n                ),\n                "fit_stats_sha256": (\n                    _sha256(fit_path)\n                ),\n                "status": (\n                    "ATOMICALLY_COMMITTED"\n                ),\n                "created_unix": time.time(),\n            },\n        )\n\n    def make_task(\n        profile,\n        subject: int,\n    ):\n        dataset_id = str(\n            profile.dataset_id\n        )\n        task_dir = (\n            child_root\n            / _safe(dataset_id)\n            / f"subject_{subject:03d}"\n        )\n\n        task = {\n            "action": "descriptor",\n            "profile": (\n                _profile_dict(profile)\n            ),\n            "subject": int(subject),\n            "input_root": str(input_root),\n            "child_work_root": str(\n                task_dir / "work"\n            ),\n            "child_report_root": str(\n                task_dir / "report"\n            ),\n            "output_dir": str(\n                task_dir / "output"\n            ),\n            "source_resolution_file": str(\n                source_resolution_file\n            ),\n            "collect_fit_stats": (\n                collect_fit_stats\n            ),\n            "split_profile": (\n                self.config.get(\n                    "split",\n                    {},\n                )\n            ),\n            "fit_roles": (\n                self.config.get(\n                    "preprocessing",\n                    {},\n                ).get(\n                    "fit_roles",\n                    ["train"],\n                )\n            ),\n        }\n\n        return task_dir, task\n\n    def record_progress(\n        current: str,\n        active_workers: int,\n    ):\n        nonlocal last_progress\n\n        now = time.monotonic()\n        if (\n            now - last_progress\n            >= float(\n                POLICY[\n                    "progress_interval_seconds"\n                ]\n            )\n            or completed == total_subjects\n        ):\n            _progress_line(\n                "PASS1_DESCRIPTOR_PROGRESS",\n                completed,\n                total_subjects,\n                started,\n                current,\n                None,\n                self.work_root,\n                {\n                    "recordings": (\n                        completed_recordings\n                    ),\n                    "reused_checkpoints": (\n                        reused_checkpoints\n                    ),\n                    "generated_checkpoints": (\n                        generated_checkpoints\n                    ),\n                    "active_parallel_workers": (\n                        active_workers\n                    ),\n                    "parallelism_policy": (\n                        configured_parallelism\n                    ),\n                },\n            )\n            last_progress = now\n\n    def consume_child_result(\n        profile,\n        dataset_id: str,\n        subject: int,\n        result: dict[str, Any],\n    ) -> None:\n        nonlocal generated_checkpoints\n        nonlocal completed_recordings\n\n        rows = _read_jsonl(\n            Path(result["descriptor_path"])\n        )\n        fit_rows = _read_jsonl(\n            Path(result["fit_stats_path"])\n        )\n\n        if not rows:\n            raise RuntimeError(\n                "R26_EMPTY_DESCRIPTOR_RESULT: "\n                f"{dataset_id}:subject={subject}"\n            )\n\n        if (\n            collect_fit_stats\n            and len(fit_rows) != len(rows)\n        ):\n            raise RuntimeError(\n                "R26_PASS1_FIT_ROW_COUNT_MISMATCH: "\n                f"{dataset_id}:subject={subject}; "\n                f"descriptors={len(rows)}; "\n                f"fit_rows={len(fit_rows)}"\n            )\n\n        commit_checkpoint(\n            profile,\n            subject,\n            rows,\n            fit_rows,\n        )\n        result_by_key[(dataset_id, subject)] = (\n            rows,\n            fit_rows,\n        )\n        generated_checkpoints += 1\n        completed_recordings += len(rows)\n\n    def failure_row(\n        dataset_id: str,\n        subject: int,\n        exc: BaseException,\n        phase: str,\n    ) -> dict[str, Any]:\n        return {\n            "dataset_id": dataset_id,\n            "subject": int(subject),\n            "phase": phase,\n            "error_type": type(exc).__name__,\n            "error": repr(exc),\n            "traceback": traceback.format_exc(),\n        }\n\n    def write_dataset_failure(\n        dataset_id: str,\n        errors: list[dict[str, Any]],\n    ) -> Path:\n        failure_path = (\n            bundle\n            / "negative_and_failed_results"\n            / (\n                "pass1_dataset_failures_"\n                + _safe(dataset_id)\n                + ".json"\n            )\n        )\n        _atomic_json(\n            failure_path,\n            {\n                "dataset_id": dataset_id,\n                "errors": errors,\n                "successful_subject_checkpoints": sum(\n                    1\n                    for current_dataset, _\n                    in result_by_key\n                    if current_dataset == dataset_id\n                ),\n                "checkpoint_root": str(\n                    checkpoint_root\n                ),\n                "failure_policy": (\n                    "REAL_LOADER_PREFLIGHT_THEN_"\n                    "BOUNDED_PARALLEL_EXECUTION"\n                ),\n            },\n        )\n        return failure_path\n\n    for dataset_id, (\n        profile,\n        subjects,\n    ) in plan_by_dataset.items():\n        # Stay serial until the real-loader preflight measures this dataset\'s\n        # actual memory footprint. Parallelism is then raised to the fastest\n        # guarded value for the remaining subjects.\n        max_workers = 1\n\n        subjects_to_run: list[int] = []\n        dataset_errors: list[dict[str, Any]] = []\n\n        # Resolve valid checkpoints before creating any process. A clean run\n        # has no checkpoints; a resumed run skips completed subjects exactly.\n        for subject in subjects:\n            checkpoint = load_checkpoint(\n                profile,\n                subject,\n            )\n            if checkpoint is None:\n                subjects_to_run.append(subject)\n                continue\n\n            rows, fit_rows = checkpoint\n            result_by_key[(dataset_id, subject)] = (\n                rows,\n                fit_rows,\n            )\n            reused_checkpoints += 1\n            completed += 1\n            completed_recordings += len(rows)\n            record_progress(\n                (\n                    "CHECKPOINT_REUSED:"\n                    f"{dataset_id}:subject={subject}"\n                ),\n                max_workers,\n            )\n\n        # Real conversion preflight: not merely a filename/path probe. This\n        # catches downloader leakage, MAT incompatibility, MOABB API drift,\n        # event/channel errors, and adapter errors before mass submission.\n        if subjects_to_run:\n            preflight_subject = subjects_to_run.pop(0)\n            task_dir, task = make_task(\n                profile,\n                preflight_subject,\n            )\n            try:\n                preflight_result = _run_child_task(\n                    task,\n                    self.work_root,\n                    (\n                        f"PREFLIGHT:{dataset_id}:"\n                        f"subject={preflight_subject}"\n                    ),\n                )\n                consume_child_result(\n                    profile,\n                    dataset_id,\n                    preflight_subject,\n                    preflight_result,\n                )\n                max_workers = resolve_parallelism(\n                    dataset_id,\n                    preflight_result,\n                )\n            except Exception as exc:\n                row = failure_row(\n                    dataset_id,\n                    preflight_subject,\n                    exc,\n                    "REAL_LOADER_PREFLIGHT",\n                )\n                dataset_errors.append(row)\n                failure_path = write_dataset_failure(\n                    dataset_id,\n                    dataset_errors,\n                )\n                raise RuntimeError(\n                    "R26_PASS1_REAL_LOADER_PREFLIGHT_FAILED: "\n                    f"dataset={dataset_id}; "\n                    f"subject={preflight_subject}; "\n                    f"error={repr(exc)}; "\n                    f"evidence={failure_path}"\n                ) from exc\n            finally:\n                completed += 1\n                record_progress(\n                    (\n                        f"PREFLIGHT:{dataset_id}:"\n                        f"subject={preflight_subject}"\n                    ),\n                    1,\n                )\n                shutil.rmtree(\n                    task_dir,\n                    ignore_errors=True,\n                )\n                _trim_memory()\n\n        # Only subjects that passed the dataset-level real-loader preflight\n        # reach bounded parallel execution. A fully checkpoint-reused dataset\n        # performs no new child work.\n        if dataset_id not in resolved_parallelism:\n            resolved_parallelism[dataset_id] = 1\n            parallelism_evidence[dataset_id] = {\n                "selected_workers": 1,\n                "reason": "NO_PENDING_SUBJECTS_AFTER_CHECKPOINT_VALIDATION",\n                "scientific_scope_changed": False,\n            }\n        pending = {}\n        with ThreadPoolExecutor(\n            max_workers=max_workers,\n            thread_name_prefix=(\n                "iharq-pass1-"\n                + _safe(dataset_id)\n            ),\n        ) as executor:\n            for subject in subjects_to_run:\n                task_dir, task = make_task(\n                    profile,\n                    subject,\n                )\n                future = executor.submit(\n                    _run_child_task,\n                    task,\n                    self.work_root,\n                    (\n                        f"PASS1:{dataset_id}:"\n                        f"subject={subject}"\n                    ),\n                )\n                pending[future] = (\n                    subject,\n                    task_dir,\n                )\n\n            for future in as_completed(pending):\n                subject, task_dir = pending[future]\n                try:\n                    consume_child_result(\n                        profile,\n                        dataset_id,\n                        subject,\n                        future.result(),\n                    )\n                except Exception as exc:\n                    dataset_errors.append(\n                        failure_row(\n                            dataset_id,\n                            subject,\n                            exc,\n                            "BOUNDED_PARALLEL_SUBJECT",\n                        )\n                    )\n                finally:\n                    completed += 1\n                    record_progress(\n                        (\n                            f"PASS1:{dataset_id}:"\n                            f"subject={subject}"\n                        ),\n                        max_workers,\n                    )\n                    shutil.rmtree(\n                        task_dir,\n                        ignore_errors=True,\n                    )\n                    _trim_memory()\n\n        if dataset_errors:\n            failure_path = write_dataset_failure(\n                dataset_id,\n                dataset_errors,\n            )\n            sample = [\n                {\n                    "subject": row["subject"],\n                    "error_type": row["error_type"],\n                    "error": row["error"],\n                }\n                for row in dataset_errors[:2]\n            ]\n            raise RuntimeError(\n                "R26_PASS1_DATASET_SUBJECT_FAILURES: "\n                f"dataset={dataset_id}; "\n                f"failed={len(dataset_errors)}; "\n                f"sample={json.dumps(sample, default=str)}; "\n                f"evidence={failure_path}"\n            )\n\n    aggregate_descriptor_rows = []\n    aggregate_fit_rows = []\n\n    for profile, subject in plan:\n        key = (\n            str(profile.dataset_id),\n            int(subject),\n        )\n\n        if key not in result_by_key:\n            raise RuntimeError(\n                "R26_PASS1_RESULT_MISSING: "\n                f"{key}"\n            )\n\n        rows, fit_rows = (\n            result_by_key[key]\n        )\n\n        recordings.extend(\n            _recording_from_descriptor(row)\n            for row in rows\n        )\n        pass1_fit_rows.extend(\n            fit_rows\n        )\n        aggregate_descriptor_rows.extend(\n            rows\n        )\n        aggregate_fit_rows.extend(\n            fit_rows\n        )\n\n    descriptor_temp = (\n        descriptor_index_path\n        .with_suffix(\n            ".jsonl.tmp"\n        )\n    )\n    fit_temp = (\n        fit_index_path\n        .with_suffix(\n            ".jsonl.tmp"\n        )\n    )\n\n    for path in [\n        descriptor_temp,\n        fit_temp,\n    ]:\n        path.unlink(\n            missing_ok=True\n        )\n\n    _append_jsonl(\n        descriptor_temp,\n        aggregate_descriptor_rows,\n    )\n    _append_jsonl(\n        fit_temp,\n        aggregate_fit_rows,\n    )\n\n    descriptor_index_path.parent.mkdir(\n        parents=True,\n        exist_ok=True,\n    )\n    descriptor_temp.replace(\n        descriptor_index_path\n    )\n    fit_temp.replace(\n        fit_index_path\n    )\n\n    self.state[\n        "recordings"\n    ] = recordings\n    self.state[\n        "inventories"\n    ] = inventories\n    self.state[\n        "r26_subject_plan"\n    ] = [\n        (\n            profile.dataset_id,\n            subject,\n        )\n        for profile, subject in plan\n    ]\n    self.state[\n        "r26_pass1_fit_rows"\n    ] = pass1_fit_rows\n    self.state[\n        "r26_descriptor_index_path"\n    ] = str(\n        descriptor_index_path.relative_to(\n            bundle\n        )\n    )\n    self.state[\n        "r26_pass1_fit_index_path"\n    ] = str(\n        fit_index_path.relative_to(\n            bundle\n        )\n    )\n\n    _atomic_json(\n        (\n            bundle\n            / "reports"\n            / "phase_01"\n            / "runtime"\n            / "bounded_streaming"\n            / "pass1_summary.json"\n        ),\n        {\n            "policy_id": (\n                POLICY["policy_id"]\n            ),\n            "checkpoint_policy_id": (\n                "P01-L1-R26-PASS1-"\n                "SUBJECT-CHECKPOINT-R1"\n            ),\n            "subjects_processed": (\n                len(plan)\n            ),\n            "recordings": (\n                len(recordings)\n            ),\n            "descriptor_index": (\n                self.state[\n                    "r26_descriptor_index_path"\n                ]\n            ),\n            "fit_stats_index": (\n                self.state[\n                    "r26_pass1_fit_index_path"\n                ]\n            ),\n            "pass1_fit_stat_rows": (\n                len(pass1_fit_rows)\n            ),\n            "signals_retained_in_parent": (\n                False\n            ),\n            "reused_checkpoints": (\n                reused_checkpoints\n            ),\n            "generated_checkpoints": (\n                generated_checkpoints\n            ),\n            "checkpoint_hash_validation": (\n                True\n            ),\n            "parallelism_upper_bounds": (\n                configured_parallelism\n            ),\n            "resolved_parallelism": (\n                resolved_parallelism\n            ),\n            "parallelism_evidence": (\n                parallelism_evidence\n            ),\n            "cpu_count": cpu_count,\n        },\n    )\n\n    return recordings\n\n\n\n\ndef _split_budget_without_signal_fit(self):\n    """Construct the frozen split/budget/preprocessing plan without loading signals.\n\n    This is the accelerated equivalent of the planning half of authoritative\n    R6 ``Layer1Pipeline.split_budget_preprocess``.  Event rows intentionally\n    use the exact authoritative schema consumed by ``budgets.allocate`` and\n    later lineage surfaces.\n    """\n    from iharq.layer1_data_protocol.splits import (\n        construct,\n        recording_role,\n        validate_disjointness,\n        validate_role_coverage,\n    )\n    from iharq.layer1_data_protocol.budgets import allocate\n    from iharq.layer1_data_protocol.preprocessing import compile_operations\n    from iharq.layer1_data_protocol.labels import map_event_label\n\n    split_profile = self.config.get("split", {})\n    dataset_record_ids = [\n        row["record_id"]\n        for row in self.state["dataset_records"]\n    ]\n    assignment, split_record = construct(\n        self.state["recordings"],\n        split_profile,\n        self.config_id,\n        dataset_record_ids,\n    )\n    self.state["assignment"] = assignment\n    self.state["split_record"] = split_record\n    self.state["records"].append(split_record)\n\n    group_keys = list(split_profile["group_keys"])\n    required_roles = list(split_profile["roles"])\n    self.state["split_disjointness"] = validate_disjointness(\n        self.state["recordings"],\n        assignment,\n        group_keys,\n    )\n    self.state["split_role_coverage"] = validate_role_coverage(\n        assignment,\n        required_roles,\n    )\n\n    label_by_dataset = {\n        row["payload"]["dataset_id"]: row\n        for row in self.state["label_records"]\n    }\n    event_rows: list[dict[str, Any]] = []\n    for recording in self.state["recordings"]:\n        dataset_id = recording.dataset_id\n        if dataset_id not in label_by_dataset:\n            raise RuntimeError(\n                "R35_STAGE11_LABEL_RECORD_MISSING: "\n                f"dataset_id={dataset_id!r}"\n            )\n        role = recording_role(\n            recording,\n            assignment,\n            group_keys,\n        )\n        label_record = label_by_dataset[dataset_id]\n        source_unit = _recording_source_unit(recording)\n        for event in recording.events:\n            # Exact authoritative R6 row contract.  These identity fields are\n            # required by deterministic low-calibration budget allocation.\n            event_rows.append(\n                {\n                    "event_id": event.event_id,\n                    "dataset_id": recording.dataset_id,\n                    "subject_id": recording.subject_id,\n                    "session_id": recording.session_id,\n                    "run_id": recording.run_id,\n                    "role": role,\n                    "normalized_label": map_event_label(\n                        event.original_label,\n                        label_record,\n                    ),\n                    "source_unit": source_unit,\n                }\n            )\n\n    required_event_fields = {\n        "event_id",\n        "dataset_id",\n        "subject_id",\n        "session_id",\n        "run_id",\n        "role",\n        "normalized_label",\n        "source_unit",\n    }\n    malformed = [\n        {\n            "index": index,\n            "missing": sorted(\n                required_event_fields.difference(row)\n            ),\n        }\n        for index, row in enumerate(event_rows)\n        if required_event_fields.difference(row)\n    ]\n    if malformed:\n        raise RuntimeError(\n            "R35_STAGE11_EVENT_ROW_SCHEMA_INVALID: "\n            + json.dumps(malformed[:20], indent=2)\n        )\n    if not event_rows:\n        raise RuntimeError("R35_STAGE11_EVENT_ROWS_EMPTY")\n\n    self.state["event_rows"] = event_rows\n    budgets, budget_report = allocate(\n        event_rows,\n        self.config.get("budgets", {}),\n    )\n    self.state["budgets"] = budgets\n    self.state["budget_report"] = budget_report\n    split_record["payload"]["budget_ids"] = [\n        row["budget_id"]\n        for row in budgets\n    ]\n    split_record["payload"]["source_event_ids"] = sorted(\n        row["event_id"]\n        for row in event_rows\n    )\n\n    operations = compile_operations(\n        self.config.get("preprocessing", {})\n    )\n    self.state["operations"] = operations\n    fit_roles = set(\n        self.config["preprocessing"].get(\n            "fit_roles",\n            ["train"],\n        )\n    )\n    legal = {\n        row["source_unit"]\n        for row in event_rows\n        if row["role"] in fit_roles\n    }\n    if not legal:\n        raise RuntimeError("R26_LEGAL_FIT_POPULATION_EMPTY")\n    self.state["r26_legal_fit_source_ids"] = sorted(legal)\n\n    _atomic_json(\n        self.bundle_root\n        / "reports"\n        / "phase_01"\n        / "runtime"\n        / "bounded_streaming"\n        / "split_budget_fit_plan.json",\n        {\n            "policy_id": POLICY["policy_id"],\n            "repair_id": (\n                "P01-L1-R35-STAGE11-"\n                "AUTHORITATIVE-EVENT-ROW-CONTRACT-R1"\n            ),\n            "split_record_id": split_record["record_id"],\n            "event_row_count": len(event_rows),\n            "event_row_required_fields": sorted(\n                required_event_fields\n            ),\n            "event_row_contract": (\n                "AUTHORITATIVE_R6_DATASET_SUBJECT_"\n                "SESSION_RUN_EVENT_ROLE_LABEL_SOURCE_UNIT"\n            ),\n            "budget_status": budget_report.get("status"),\n            "budget_allocation_count": len(budgets),\n            "legal_fit_sources": sorted(legal),\n            "operations": operations,\n            "signal_arrays_loaded": False,\n            "scientific_scope_changed": False,\n        },\n    )\n\n\n\n\ndef _streaming_fit(runner: Any) -> dict[str, Any]:\n    import numpy as np\n    from iharq.canonical import semantic_hash\n    from iharq.layer1_data_protocol.preprocessing import FitState, build_preprocessing_record\n    pipeline = runner.pipeline; operations = list(pipeline.state["operations"]); legal = set(pipeline.state["r26_legal_fit_source_ids"])\n    need = any(op["name"] == "standardize_train_fit" for op in operations)\n    fit_rows: list[dict[str, Any]] = list(pipeline.state.get("r26_pass1_fit_rows", []))\n    if need:\n        fit_state, detail = _combine_fit_rows(fit_rows, legal)\n        detail["fit_stats_collected_during_pass1"] = True\n        detail["additional_source_reload_for_fit"] = False\n    else:\n        fit_state = FitState(None, None, sorted(legal), semantic_hash({"fit": "not-required", "sources": sorted(legal)})); detail = {"fit_required": False}\n    fit_dir = pipeline.bundle_root / "derived_outputs" / "preprocessing_fit_state"; fit_dir.mkdir(parents=True, exist_ok=True)\n    npz_path = fit_dir / "fit_state.npz"\n    np.savez_compressed(npz_path, mean=np.asarray([]) if fit_state.mean is None else fit_state.mean, std=np.asarray([]) if fit_state.std is None else fit_state.std, source_ids=np.asarray(fit_state.source_ids), state_hash=np.asarray(fit_state.state_hash))\n    fit_manifest = {\n        "policy_id": POLICY["policy_id"], "state_hash": fit_state.state_hash, "source_ids": fit_state.source_ids, "mean_shape": None if fit_state.mean is None else list(fit_state.mean.shape), "std_shape": None if fit_state.std is None else list(fit_state.std.shape),\n        "npz_path": str(npz_path.relative_to(pipeline.bundle_root)), "npz_sha256": _sha256(npz_path), "npz_bytes": npz_path.stat().st_size, "reduction": "DETERMINISTIC_FLOAT64_CHAN_PARALLEL_VARIANCE_IN_FROZEN_SUBJECT_ORDER", **detail,\n    }\n    _atomic_json(fit_dir / "fit_state_manifest.json", fit_manifest)\n    source_ids = [r["record_id"] for r in pipeline.state["dataset_records"]] + [pipeline.state["split_record"]["record_id"]]\n    preproc = build_preprocessing_record(pipeline.config["preprocessing"], operations, fit_state, source_ids, pipeline.config_id, "external_artifact_pointers/derived_windows_dataset.json")\n    preproc["payload"]["split_record_id"] = pipeline.state["split_record"]["record_id"]\n    preproc["payload"]["fit_state_pointer"] = str((fit_dir / "fit_state_manifest.json").relative_to(pipeline.bundle_root))\n    pipeline.state["preprocessing_record"] = preproc; pipeline.state["records"].append(preproc); pipeline.state["fit_source_ids"] = fit_state.source_ids; pipeline.state["r26_fit_state"] = fit_state\n    storage_forecast = _estimate_derived_storage(pipeline)\n    return {\n        "preprocessing_record": preproc["record_id"],\n        "fit_state": fit_manifest,\n        "storage_forecast": {\n            "planned_windows": storage_forecast["planned_windows"],\n            "logical_float32_signal_gib": storage_forecast[\n                "logical_float32_signal_gib"\n            ],\n            "planning_lower_gib": storage_forecast[\n                "lossless_hdf5_planning_envelope"\n            ]["lower_gib"],\n            "planning_upper_gib": storage_forecast[\n                "lossless_hdf5_planning_envelope"\n            ]["upper_gib"],\n            "recommended_private_kaggle_capacity_gib": storage_forecast[\n                "recommended_private_kaggle_capacity"\n            ]["gib"],\n            "report_path": pipeline.state[\n                "r26_storage_forecast_path"\n            ],\n        },\n    }\n\n\n\ndef _streaming_materialize(runner: Any) -> dict[str, Any]:\n    pipeline = runner.pipeline\n    api = _ensure_kaggle_upload_api()\n    attempt, handle = _derived_identity(runner)\n\n    plan = list(pipeline.state["r26_subject_plan"])\n    profile_by_dataset = {\n        profile.dataset_id: profile\n        for profile in pipeline.state["profiles"]\n    }\n    label_by_dataset = {\n        row["payload"]["dataset_id"]: row\n        for row in pipeline.state["label_records"]\n    }\n    dataset_record_by_dataset = {\n        row["payload"]["dataset_id"]: row["record_id"]\n        for row in pipeline.state["dataset_records"]\n    }\n\n    fit_state = pipeline.state["r26_fit_state"]\n    fit_payload = {\n        "mean": (\n            None\n            if fit_state.mean is None\n            else fit_state.mean.tolist()\n        ),\n        "std": (\n            None\n            if fit_state.std is None\n            else fit_state.std.tolist()\n        ),\n        "source_ids": fit_state.source_ids,\n        "state_hash": fit_state.state_hash,\n    }\n\n    root = (\n        pipeline.work_root\n        / "streaming_runtime"\n        / "pass2b_materialize"\n    )\n    location_target = (\n        pipeline.bundle_root\n        / "external_artifact_pointers"\n        / "window_to_shard.jsonl"\n    )\n    if location_target.exists():\n        location_target.unlink()\n\n    tokens: list[str] = []\n    shard_rows: list[dict[str, Any]] = []\n    window_records: list[dict[str, Any]] = []\n    window_index: list[dict[str, Any]] = []\n    quality_records: list[dict[str, Any]] = []\n    quality_summaries: list[dict[str, Any]] = []\n    invalid_windows: list[dict[str, Any]] = []\n\n    started = time.monotonic()\n    last_progress = 0.0\n    completed = 0\n    uploaded_bytes = 0\n    logical_window_bytes = 0\n    dataset_actual: dict[str, dict[str, Any]] = {}\n\n    for dataset, subject in plan:\n        _assert_subject_scratch_capacity(\n            pipeline,\n            dataset,\n            int(subject),\n        )\n\n        label = f"PASS2B:{dataset}:subject={subject}"\n        task_dir = (\n            root\n            / _safe(dataset)\n            / f"subject_{int(subject):03d}"\n        )\n        shard_filename = (\n            f"{_safe(dataset)}_"\n            f"subject_{int(subject):03d}_windows.h5"\n        )\n        profile = profile_by_dataset[dataset]\n\n        task = {\n            "action": "materialize",\n            "profile": _profile_dict(profile),\n            "subject": int(subject),\n            "input_root": str(runner.input_root),\n            "child_work_root": str(task_dir / "work"),\n            "child_report_root": str(task_dir / "report"),\n            "output_dir": str(task_dir / "output"),\n            "source_resolution_file": str(\n                pipeline.state[\n                    "r26_source_resolution_file"\n                ]\n            ),\n            "operations": pipeline.state["operations"],\n            "fit_state": fit_payload,\n            "assignment": pipeline.state["assignment"],\n            "split_keys": list(\n                pipeline.config["split"]["group_keys"]\n            ),\n            "label_record": label_by_dataset[dataset],\n            "preprocessing_record": (\n                pipeline.state["preprocessing_record"]\n            ),\n            "split_record": pipeline.state["split_record"],\n            "quality_profile": pipeline.config.get(\n                "quality",\n                {},\n            ),\n            "window_profile": pipeline.config.get(\n                "windows",\n                {},\n            ),\n            "config_id": pipeline.config_id,\n            "dataset_record_id": (\n                dataset_record_by_dataset[dataset]\n            ),\n            "shard_filename": shard_filename,\n        }\n\n        result = _run_child_task(\n            task,\n            pipeline.work_root,\n            label,\n        )\n\n        subject_logical = int(\n            result.get("logical_window_bytes", 0)\n        )\n        logical_window_bytes += subject_logical\n\n        quality_records.extend(\n            _read_jsonl(\n                Path(result["quality_records_path"])\n            )\n        )\n        quality_summaries.extend(\n            _read_jsonl(\n                Path(result["quality_summaries_path"])\n            )\n        )\n        invalid_windows.extend(\n            _read_jsonl(\n                Path(result["invalid_windows_path"])\n            )\n        )\n        window_records.extend(\n            _read_jsonl(\n                Path(result["window_records_path"])\n            )\n        )\n        window_index.extend(\n            _read_jsonl(\n                Path(result["window_index_path"])\n            )\n        )\n\n        locations = _read_jsonl(\n            Path(result["window_locations_path"])\n        )\n        for row in locations:\n            row.update(\n                {\n                    "provider": "Kaggle",\n                    "dataset_handle": handle,\n                    "immutable_revision": 1,\n                }\n            )\n        _append_jsonl(location_target, locations)\n\n        shard = result.get("shard")\n        if shard:\n            shard_path = Path(shard["path"])\n            _resource_guard(pipeline.work_root)\n            token = _upload_blob_with_retry(\n                api,\n                shard_path,\n            )\n            tokens.append(token)\n\n            actual_bytes = int(shard["bytes"])\n            uploaded_bytes += actual_bytes\n\n            if shard.get("signal_dtype") != "float32":\n                raise RuntimeError(\n                    "R34_UPLOADED_SHARD_DTYPE_MISMATCH: "\n                    f"subject={subject}; observed={shard.get(\'signal_dtype\')}"\n                )\n            shard_row = {\n                "filename": shard["filename"],\n                "bytes": actual_bytes,\n                "sha256": shard["sha256"],\n                "dataset_id": dataset,\n                "subject_profile": int(subject),\n                "window_count": int(\n                    result["window_count"]\n                ),\n                "logical_window_bytes": (\n                    subject_logical\n                ),\n                "compression_ratio_to_logical": (\n                    actual_bytes / subject_logical\n                    if subject_logical\n                    else None\n                ),\n                "provider": "Kaggle",\n                "dataset_handle": handle,\n                "immutable_revision": 1,\n                "format": "HDF5",\n                "compression": "gzip-1-lossless",\n                "signal_dtype": "float32",\n                "window_duration_samples": 480,\n            }\n            shard_rows.append(shard_row)\n\n            dataset_row = dataset_actual.setdefault(\n                dataset,\n                {\n                    "dataset_id": dataset,\n                    "subjects": 0,\n                    "shards": 0,\n                    "windows": 0,\n                    "logical_window_bytes": 0,\n                    "actual_hdf5_bytes": 0,\n                },\n            )\n            dataset_row["subjects"] += 1\n            dataset_row["shards"] += 1\n            dataset_row["windows"] += int(\n                result["window_count"]\n            )\n            dataset_row["logical_window_bytes"] += (\n                subject_logical\n            )\n            dataset_row["actual_hdf5_bytes"] += (\n                actual_bytes\n            )\n\n            # No local large-array retention after verified token.\n            shard_path.unlink(missing_ok=True)\n\n        completed += 1\n        if (\n            time.monotonic() - last_progress\n            >= float(POLICY["progress_interval_seconds"])\n            or completed == len(plan)\n        ):\n            _progress_line(\n                "PASS2B_MATERIALIZE_UPLOAD_PROGRESS",\n                completed,\n                len(plan),\n                started,\n                label,\n                None,\n                pipeline.work_root,\n                {\n                    "windows": len(window_records),\n                    "shards_uploaded": len(shard_rows),\n                    "uploaded_gib": round(\n                        uploaded_bytes / _GIB,\n                        3,\n                    ),\n                    "logical_window_gib": round(\n                        logical_window_bytes / _GIB,\n                        3,\n                    ),\n                },\n            )\n            last_progress = time.monotonic()\n\n        shutil.rmtree(task_dir, ignore_errors=True)\n        _trim_memory()\n\n    pipeline.state["quality_records"] = quality_records\n    pipeline.state["quality_summaries"] = quality_summaries\n    pipeline.state["invalid_windows"] = invalid_windows\n    pipeline.state["records"].extend(quality_records)\n    invalid_path = (\n        pipeline.bundle_root\n        / "negative_and_failed_results"\n        / "invalid_windows_streaming.json"\n    )\n    _atomic_json(invalid_path, invalid_windows)\n    pipeline.state["r34_invalid_windows_path"] = str(\n        invalid_path.relative_to(pipeline.bundle_root)\n    )\n    pipeline.state["r26_pending_window_records"] = (\n        window_records\n    )\n    pipeline.state["r26_pending_window_index"] = (\n        window_index\n    )\n    pipeline.state["r26_derived_tokens"] = tokens\n    pipeline.state["r26_derived_shards"] = shard_rows\n    pipeline.state["r26_derived_handle"] = handle\n    pipeline.state["r26_execution_attempt_id"] = attempt\n    pipeline.state["r26_upload_api"] = api\n\n    actual_report = {\n        "artifact_id": (\n            f"P01-L1-DERIVED-STORAGE-ACTUAL-"\n            f"{pipeline.config_id[:16]}-{attempt}"\n        ),\n        "policy_id": POLICY["policy_id"],\n        "scientific_freeze": (\n            POLICY["scientific_freeze_unchanged"]\n        ),\n        "status": "PRECOMMIT_ALL_SHARD_TOKENS_OBTAINED",\n        "quality_records": len(quality_records),\n        "quality_summaries": len(quality_summaries),\n        "windows_materialized": len(window_records),\n        "invalid_window_count": len(invalid_windows),\n        "invalid_windows_path": pipeline.state["r34_invalid_windows_path"],\n        "signal_dtype": "float32",\n        "shards_uploaded": len(shard_rows),\n        "logical_float32_window_bytes": (\n            logical_window_bytes\n        ),\n        "logical_float32_window_gib": round(\n            logical_window_bytes / _GIB,\n            3,\n        ),\n        "actual_hdf5_uploaded_bytes": uploaded_bytes,\n        "actual_hdf5_uploaded_gib": round(\n            uploaded_bytes / _GIB,\n            3,\n        ),\n        "actual_compression_ratio": (\n            uploaded_bytes / logical_window_bytes\n            if logical_window_bytes\n            else None\n        ),\n        "derived_dataset_handle": handle,\n        "execution_attempt_id": attempt,\n        "forecast_report": pipeline.state.get(\n            "r26_storage_forecast_path"\n        ),\n        "dataset_totals": [\n            {\n                **row,\n                "logical_window_gib": round(\n                    row["logical_window_bytes"]\n                    / _GIB,\n                    3,\n                ),\n                "actual_hdf5_gib": round(\n                    row["actual_hdf5_bytes"]\n                    / _GIB,\n                    3,\n                ),\n                "compression_ratio": (\n                    row["actual_hdf5_bytes"]\n                    / row["logical_window_bytes"]\n                    if row["logical_window_bytes"]\n                    else None\n                ),\n            }\n            for _, row in sorted(dataset_actual.items())\n        ],\n    }\n    actual_path = (\n        pipeline.bundle_root\n        / "reports"\n        / "phase_01"\n        / "storage"\n        / "derived_output_storage_actual_precommit.json"\n    )\n    _atomic_json(actual_path, actual_report)\n    pipeline.state["r26_storage_actual"] = actual_report\n    pipeline.state["r26_storage_actual_path"] = str(\n        actual_path.relative_to(pipeline.bundle_root)\n    )\n\n    summary = {\n        "quality_records": len(quality_records),\n        "quality_summaries": len(quality_summaries),\n        "windows_materialized": len(window_records),\n        "invalid_window_count": len(invalid_windows),\n        "invalid_windows_path": pipeline.state["r34_invalid_windows_path"],\n        "signal_dtype": "float32",\n        "shards_uploaded": len(shard_rows),\n        "logical_window_bytes": logical_window_bytes,\n        "uploaded_bytes": uploaded_bytes,\n        "actual_compression_ratio": (\n            uploaded_bytes / logical_window_bytes\n            if logical_window_bytes\n            else None\n        ),\n        "derived_dataset_handle": handle,\n        "execution_attempt_id": attempt,\n        "storage_actual_report": (\n            pipeline.state["r26_storage_actual_path"]\n        ),\n    }\n    _atomic_json(\n        pipeline.bundle_root\n        / "reports"\n        / "phase_01"\n        / "runtime"\n        / "bounded_streaming"\n        / "pass2b_summary_precommit.json",\n        summary,\n    )\n    return summary\n\n\n\ndef _finalize_derived_dataset(runner: Any) -> dict[str, Any]:\n    pipeline = runner.pipeline\n    invalid_windows = list(pipeline.state.get("invalid_windows", []))\n    if invalid_windows:\n        raise RuntimeError(\n            "R34_DERIVED_DATASET_COMMIT_REFUSED_INVALID_WINDOWS: "\n            f"count={len(invalid_windows)}; "\n            f"evidence={pipeline.state.get(\'r34_invalid_windows_path\')}"\n        )\n    api = pipeline.state["r26_upload_api"]\n    handle = pipeline.state["r26_derived_handle"]\n    shards = list(pipeline.state["r26_derived_shards"])\n    tokens = list(pipeline.state["r26_derived_tokens"])\n    pending_records = list(pipeline.state["r26_pending_window_records"])\n    pending_index = list(pipeline.state["r26_pending_window_index"])\n    if not pending_records or len(pending_records) != len(pending_index):\n        raise RuntimeError(\n            "R34_DERIVED_WINDOW_RECORD_INDEX_COUNT_MISMATCH: "\n            f"records={len(pending_records)}; index={len(pending_index)}"\n        )\n    record_ids = [row["record_id"] for row in pending_records]\n    index_ids = [row["window_record_id"] for row in pending_index]\n    if len(record_ids) != len(set(record_ids)) or set(record_ids) != set(index_ids):\n        raise RuntimeError("R34_DERIVED_WINDOW_RECORD_ID_CLOSURE_FAILED")\n    if sum(int(row.get("window_count", 0)) for row in shards) != len(pending_records):\n        raise RuntimeError("R34_DERIVED_SHARD_WINDOW_COUNT_CLOSURE_FAILED")\n    locations_path = pipeline.bundle_root / "external_artifact_pointers" / "window_to_shard.jsonl"\n    location_hash = _sha256(locations_path); location_bytes = locations_path.stat().st_size\n    manifest = {\n        "artifact_id": f"P01-L1-DERIVED-WINDOWS-{pipeline.config_id[:16]}-{pipeline.state[\'r26_execution_attempt_id\']}",\n        "schema_version": 1,\n        "policy_id": POLICY["policy_id"],\n        "scientific_freeze": POLICY["scientific_freeze_unchanged"],\n        "provider": "Kaggle",\n        "repository_or_dataset": handle,\n        "immutable_revision": 1,\n        "access": "PRIVATE",\n        "format": "LOSSLESS_HDF5_SUBJECT_SHARDS",\n        "signal_dtype": "float32",\n        "event_resampling": "MNE_POLYPHASE_JOINT_EVENTS",\n        "window_policy": {\n            "start_offset_samples": 80,\n            "duration_samples": 480,\n            "stride_samples": 480,\n            "last_window_policy": "ONE_WINDOW_PER_INCLUDED_SOURCE_EVENT",\n            "bounds_policy": "REJECT_OUT_OF_BOUNDS",\n        },\n        "source_dataset_ids": POLICY["active_sources_unchanged"],\n        "source_inventory_record_ids": [r["record_id"] for r in pipeline.state["dataset_records"]],\n        "split_record_id": pipeline.state["split_record"]["record_id"],\n        "preprocessing_record_id": pipeline.state["preprocessing_record"]["record_id"],\n        "window_count": len(\n            pipeline.state["r26_pending_window_records"]\n        ),\n        "storage_forecast": pipeline.state.get(\n            "r26_storage_forecast_path"\n        ),\n        "storage_actual": pipeline.state.get(\n            "r26_storage_actual_path"\n        ),\n        "dual_persistence": POLICY["dual_persistence"],\n        "shards": shards,\n        "window_location_index": {"bundle_path": str(locations_path.relative_to(pipeline.bundle_root)), "sha256": location_hash, "bytes": location_bytes},\n        "local_copy_status": "SHARDS_DELETED_AFTER_VERIFIED_KAGGLE_BLOB_UPLOAD",\n        "retrieval_instructions": "Attach the private Kaggle Dataset at immutable version 1; load IHARQ_P01_L1_WINDOW_TO_SHARD_INDEX.jsonl (or an ordinal-prefixed suffix match); resolve the shard filename; read the declared HDF5 group and row.",\n        "license": "INHERIT_EACH_SOURCE_LICENSE_AND_REDISTRIBUTION_CONSTRAINT_FROM_DATASET_RECORDS",\n        "consumer_phases": [f"P{i:02d}" for i in range(2, 16)],\n    }\n    scratch = pipeline.work_root / "streaming_runtime" / "derived_dataset_commit"; scratch.mkdir(parents=True, exist_ok=True)\n\n    # Persist the compact scientific indexes inside the derived Dataset itself,\n    # so later phases can attach one immutable Dataset and do not need to\n    # manually download/re-upload local shard metadata.\n    location_dataset_name = "IHARQ_P01_L1_WINDOW_TO_SHARD_INDEX.jsonl"\n    location_dataset_path = scratch / location_dataset_name\n    shutil.copy2(locations_path, location_dataset_path)\n\n    window_records_name = "IHARQ_P01_L1_WINDOW_RECORDS.jsonl"\n    window_records_path = scratch / window_records_name\n    _append_jsonl(window_records_path, pipeline.state["r26_pending_window_records"])\n\n    window_index_name = "IHARQ_P01_L1_WINDOW_INDEX.jsonl"\n    window_index_path = scratch / window_index_name\n    _append_jsonl(window_index_path, pipeline.state["r26_pending_window_index"])\n\n    sidecar_name = "IHARQ_P01_L1_DERIVED_DATASET_SIDECAR.json"\n    sidecar_path = scratch / sidecar_name\n    _atomic_json(sidecar_path, {\n        "schema_version": 1,\n        "policy_id": POLICY["policy_id"],\n        "scientific_freeze": POLICY["scientific_freeze_unchanged"],\n        "config_id": pipeline.config_id,\n        "dataset_records": pipeline.state["dataset_records"],\n        "label_records": pipeline.state["label_records"],\n        "split_record": pipeline.state["split_record"],\n        "preprocessing_record": pipeline.state["preprocessing_record"],\n        "quality_summaries": pipeline.state.get("quality_summaries", []),\n        "window_count": len(pipeline.state["r26_pending_window_records"]),\n        "invalid_window_count": 0,\n        "signal_dtype": "float32",\n        "event_resampling": "MNE_POLYPHASE_JOINT_EVENTS",\n        "window_contract": {\n            "start_offset_samples": 80,\n            "duration_samples": 480,\n            "stride_samples": 480,\n            "one_window_per_included_event": True,\n        },\n        "shard_count": len(shards),\n    })\n\n    reader_name = "iharq_window_shard_reader.py"\n    reader_path = scratch / reader_name\n    reader_path.write_text(\n        WINDOW_SHARD_READER_SOURCE,\n        encoding="utf-8",\n    )\n\n    forecast_name = (\n        "IHARQ_P01_L1_DERIVED_OUTPUT_STORAGE_FORECAST.json"\n    )\n    forecast_path = scratch / forecast_name\n    shutil.copy2(\n        pipeline.bundle_root\n        / pipeline.state["r26_storage_forecast_path"],\n        forecast_path,\n    )\n\n    actual_name = (\n        "IHARQ_P01_L1_DERIVED_OUTPUT_STORAGE_ACTUAL.json"\n    )\n    actual_path = scratch / actual_name\n    shutil.copy2(\n        pipeline.bundle_root\n        / pipeline.state["r26_storage_actual_path"],\n        actual_path,\n    )\n\n    compact_files = [\n        location_dataset_path,\n        window_records_path,\n        window_index_path,\n        sidecar_path,\n        reader_path,\n        forecast_path,\n        actual_path,\n    ]\n    manifest["dataset_local_indexes"] = {\n        path.name: {\n            "sha256": _sha256(path),\n            "bytes": path.stat().st_size,\n            "format": "JSONL" if path.suffix == ".jsonl" else "JSON",\n        }\n        for path in compact_files\n    }\n    manifest["window_location_index"].update({\n        "dataset_filename": location_dataset_name,\n        "dataset_sha256": _sha256(location_dataset_path),\n        "dataset_bytes": location_dataset_path.stat().st_size,\n    })\n    manifest["window_records_filename"] = window_records_name\n    manifest["window_index_filename"] = window_index_name\n    manifest["derived_dataset_sidecar_filename"] = sidecar_name\n    manifest["future_phase_reader_filename"] = reader_name\n    manifest["storage_forecast_filename"] = forecast_name\n    manifest["storage_actual_filename"] = actual_name\n\n    for compact_path in compact_files:\n        tokens.append(_upload_blob_with_retry(api, compact_path))\n\n    manifest_name = "IHARQ_P01_L1_DERIVED_WINDOW_DATASET_MANIFEST.json"\n    manifest_path = scratch / manifest_name; _atomic_json(manifest_path, manifest)\n    manifest_token = _upload_blob_with_retry(api, manifest_path); tokens.append(manifest_token)\n    expected_token_count = len(shards) + len(compact_files) + 1\n    if len(tokens) != expected_token_count:\n        raise RuntimeError(\n            f"R26_DERIVED_DATASET_TOKEN_COUNT_MISMATCH: "\n            f"expected={expected_token_count}; observed={len(tokens)}"\n        )\n    upload_dir = api["UploadDirectoryInfo"](name="", files=tokens, directories=[])\n    response = api["create_dataset_or_version"](\n        api["parse_dataset_handle"](handle), upload_dir,\n        f"IHARQ P01/L1 exact derived windows for {POLICY[\'scientific_freeze_unchanged\']} under {POLICY[\'policy_id\']}",\n    )\n    pointer_path = pipeline.bundle_root / "external_artifact_pointers" / "derived_windows_dataset.json"\n    manifest["creation_status"] = "COMMITTED"\n    actual_report = dict(\n        pipeline.state.get("r26_storage_actual", {})\n    )\n    actual_report["status"] = "COMMITTED"\n    actual_report["dataset_handle"] = handle\n    actual_report["dataset_version"] = 1\n    _atomic_json(\n        pipeline.bundle_root\n        / pipeline.state["r26_storage_actual_path"],\n        actual_report,\n    )\n    pipeline.state["r26_storage_actual"] = actual_report\n    manifest["creation_response_type"] = type(response).__name__\n    manifest["dataset_version"] = 1\n    manifest["manifest_filename_in_dataset"] = manifest_name\n    _atomic_json(pointer_path, manifest)\n    pipeline.state["preprocessing_record"]["payload"]["output_pointer"] = str(pointer_path.relative_to(pipeline.bundle_root))\n    window_records = pipeline.state.pop("r26_pending_window_records"); window_index = pipeline.state.pop("r26_pending_window_index")\n    pipeline.state["window_records"] = window_records; pipeline.state["window_index"] = window_index; pipeline.state["records"].extend(window_records)\n    pipeline.state["window_report"] = {\n        "window_count": len(window_records),\n        "event_count": len({row["event_id"] for row in window_index}),\n        "invalid_window_count": 0,\n        "invalid_windows": [],\n        "roles": sorted({row["role"] for row in window_index}),\n        "start_offset_samples": 80,\n        "duration_samples": 480,\n        "stride_samples": 480,\n        "duration_seconds": pipeline.config.get("windows", {}).get("duration_seconds"),\n        "stride_seconds": pipeline.config.get("windows", {}).get("stride_seconds"),\n        "last_window_policy": "ONE_WINDOW_PER_INCLUDED_SOURCE_EVENT",\n        "bounds_policy": "REJECT_OUT_OF_BOUNDS",\n        "event_resampling": "MNE_POLYPHASE_JOINT_EVENTS",\n        "signal_dtype": "float32",\n        "overlap_group": "PARENT_EVENT",\n        "storage": "PRIVATE_KAGGLE_DATASET_LOSSLESS_HDF5_SUBJECT_SHARDS",\n        "dataset_handle": handle,\n        "immutable_revision": 1,\n    }\n    pipeline.state["r26_derived_pointer"] = str(pointer_path.relative_to(pipeline.bundle_root))\n    pipeline.state.pop("r26_derived_tokens", None); pipeline.state.pop("r26_upload_api", None)\n    shutil.rmtree(scratch, ignore_errors=True)\n    return {"window_report": pipeline.state["window_report"], "pointer": pipeline.state["r26_derived_pointer"], "shards": len(shards)}\n\n\n\n\n\ndef install_bounded_streaming(\n    runner: Any,\n    *,\n    source_resolution_file: str | Path,\n) -> dict[str, Any]:\n    pipeline = runner.pipeline\n    pipeline.state["r26_source_resolution_file"] = str(source_resolution_file)\n    attempt, handle = _derived_identity(runner)\n    pipeline.state["r26_execution_attempt_id"] = attempt\n    pipeline.state["r26_derived_handle"] = handle\n    pipeline.load_sources = MethodType(_streaming_load_sources, pipeline)\n    pipeline.split_budget_preprocess = MethodType(_split_budget_without_signal_fit, pipeline)\n\n    def _sync_stage_result(self, result: Any) -> None:\n        rows = self.pipeline.state.get("stage_results", [])\n        for index in range(len(rows) - 1, -1, -1):\n            if rows[index].get("stage") == result.stage:\n                rows[index] = dict(result.__dict__)\n                break\n        status_path = self.work_root / "stage_status" / f"{result.stage}.json"\n        status_path.parent.mkdir(parents=True, exist_ok=True)\n        status_path.write_text(\n            json.dumps(result.__dict__, indent=2, default=str),\n            encoding="utf-8",\n        )\n\n    def stage_13(self):\n        if not self.pipeline.state.get("r26_legal_fit_source_ids"):\n            return self._record("13", "BLOCKED", blockers=self.pipeline.blockers)\n        try:\n            observations = _streaming_fit(self)\n            return self._record("13", "PASS", observations=observations)\n        except Exception as exc:\n            blocker = {\n                "code": "P01_STREAMING_FIT_FAILED",\n                "message": str(exc),\n                "owner": "L1_PREPROCESSING_RUNTIME",\n            }\n            self.pipeline.blockers.append(blocker)\n            return self._record(\n                "13",\n                "BLOCKED",\n                blockers=self.pipeline.blockers,\n                observations={"traceback": traceback.format_exc()},\n            )\n\n    def stage_14(self):\n        if not self.pipeline.state.get("preprocessing_record"):\n            return self._record("14", "BLOCKED", blockers=self.pipeline.blockers)\n        try:\n            observations = _streaming_materialize(self)\n            hard_invalid = sum(\n                int(row.get("hard_invalid", 0))\n                for row in self.pipeline.state.get("quality_summaries", [])\n            )\n            blockers = []\n            if hard_invalid:\n                blocker = {\n                    "code": "P01_QUALITY_HARD_INVALID",\n                    "hard_invalid": hard_invalid,\n                    "owner": "L1_QUALITY_OR_SOURCE_BYTES",\n                }\n                self.pipeline.blockers.append(blocker)\n                blockers.append(blocker)\n            status = "PASS" if not blockers else "FAIL"\n            return self._record(\n                "14",\n                status,\n                observations={\n                    "quality_records": observations["quality_records"],\n                    "quality_summaries": observations["quality_summaries"],\n                    "hard_invalid": hard_invalid,\n                    "coverage": self.pipeline.state.get("quality_summaries", []),\n                    "streaming_window_candidates": observations["windows_materialized"],\n                    "invalid_window_count": observations["invalid_window_count"],\n                    "invalid_windows_path": observations["invalid_windows_path"],\n                    "signal_dtype": observations["signal_dtype"],\n                    "shards_uploaded_precommit": observations["shards_uploaded"],\n                    "logical_window_gib": round(observations["logical_window_bytes"] / _GIB, 3),\n                    "uploaded_hdf5_gib": round(observations["uploaded_bytes"] / _GIB, 3),\n                    "actual_compression_ratio": observations["actual_compression_ratio"],\n                    "storage_actual_report": observations["storage_actual_report"],\n                },\n                blockers=blockers,\n            )\n        except Exception as exc:\n            blocker = {\n                "code": "P01_STREAMING_MATERIALIZATION_FAILED",\n                "message": str(exc),\n                "owner": "L1_QUALITY_WINDOW_RUNTIME",\n            }\n            self.pipeline.blockers.append(blocker)\n            return self._record(\n                "14",\n                "BLOCKED",\n                blockers=self.pipeline.blockers,\n                observations={"traceback": traceback.format_exc()},\n            )\n\n    def stage_15(self):\n        if "r26_derived_tokens" not in self.pipeline.state:\n            return self._record("15", "BLOCKED", blockers=self.pipeline.blockers)\n        invalid_windows = list(self.pipeline.state.get("invalid_windows", []))\n        if invalid_windows:\n            blocker = {\n                "code": "P01_WINDOW_INVALID_OR_MISSING",\n                "invalid_window_count": len(invalid_windows),\n                "evidence": self.pipeline.state.get("r34_invalid_windows_path"),\n                "owner": "L1_WINDOWS_OR_SOURCE_BYTES",\n            }\n            self.pipeline.blockers.append(blocker)\n            return self._record(\n                "15",\n                "FAIL",\n                observations={\n                    "invalid_window_count": len(invalid_windows),\n                    "invalid_windows_path": self.pipeline.state.get("r34_invalid_windows_path"),\n                },\n                blockers=[blocker],\n            )\n        try:\n            observations = _finalize_derived_dataset(self)\n            window_report = self.pipeline.state.get("window_report", {})\n            valid = (\n                bool(self.pipeline.state.get("window_records"))\n                and int(window_report.get("invalid_window_count", -1)) == 0\n                and window_report.get("signal_dtype") == "float32"\n                and int(window_report.get("start_offset_samples", -1)) == 80\n                and int(window_report.get("duration_samples", -1)) == 480\n                and int(window_report.get("stride_samples", -1)) == 480\n            )\n            blockers = [] if valid else [{\n                "code": "P01_WINDOW_COMMIT_CONTRACT_MISMATCH",\n                "window_report": window_report,\n                "owner": "L1_WINDOWS_OR_PERSISTENCE",\n            }]\n            if blockers:\n                self.pipeline.blockers.extend(blockers)\n            return self._record(\n                "15",\n                "PASS" if valid else "FAIL",\n                outputs=[observations["pointer"]],\n                observations={\n                    **window_report,\n                    "storage_forecast": self.pipeline.state.get("r26_storage_forecast_path"),\n                    "storage_actual": self.pipeline.state.get("r26_storage_actual_path"),\n                    "future_phase_reader_in_dataset": "iharq_window_shard_reader.py",\n                },\n                blockers=blockers,\n            )\n        except Exception as exc:\n            blocker = {\n                "code": "P01_DERIVED_WINDOW_DATASET_COMMIT_FAILED",\n                "message": str(exc),\n                "owner": "KAGGLE_ARTIFACT_PERSISTENCE",\n            }\n            self.pipeline.blockers.append(blocker)\n            return self._record(\n                "15",\n                "BLOCKED",\n                blockers=self.pipeline.blockers,\n                observations={"traceback": traceback.format_exc()},\n            )\n\n    def stage_26(self):\n        expected_zip = self.work_root / f"{self.pipeline.bundle_root.name}.zip"\n        expected_sha = Path(str(expected_zip) + ".sha256")\n        repository_name = (\n            "IHARQ_P01_L1_GitHub_Ready_Repository_"\n            f"{self.pipeline.config_id[:12]}_"\n            f"{_safe(self.pipeline.state[\'r26_execution_attempt_id\'])}"\n        )\n        expected_repo_zip = self.work_root / f"{repository_name}.zip"\n        expected_repo_sha = expected_repo_zip.with_suffix(\n            expected_repo_zip.suffix + ".sha256"\n        )\n        release_plan_rel = "reports/phase_01/repository_release_plan.json"\n        release_plan = {\n            "artifact_type": "GITHUB_READY_REPOSITORY_ZIP",\n            "expected_path": str(expected_repo_zip),\n            "expected_detached_sha256_path": str(expected_repo_sha),\n            "creation_order": "AFTER_EXECUTION_BUNDLE_FINAL_FREEZE",\n            "actual_hash_location": "EXTERNAL_DETACHED_VERIFICATION",\n            "circularity_rule": (\n                "THE ACTUAL REPOSITORY ZIP HASH MUST NOT BE WRITTEN BACK INTO "\n                "THE FROZEN EXECUTION BUNDLE OR INTO THE REPOSITORY ZIP ITSELF."\n            ),\n        }\n        _atomic_json(\n            self.pipeline.bundle_root / release_plan_rel,\n            release_plan,\n        )\n        pre = {\n            "phase": "P01",\n            "layer": "L1",\n            "notebook_revision": "R49",\n            "config_id": self.pipeline.config_id,\n            "pre_package_decision": self.pipeline.state.get(\n                "preliminary_decision", {}\n            ).get("status", "BLOCKED"),\n            "blockers": self.pipeline.blockers,\n            "bundle_target": str(expected_zip),\n            "detached_checksum_target": str(expected_sha),\n            "github_ready_repository_target": str(expected_repo_zip),\n            "github_ready_repository_detached_checksum_target": str(\n                expected_repo_sha\n            ),\n            "repository_release_plan": release_plan_rel,\n            "external_hash_boundary": (\n                "BOTH ZIP SHA-256 VALUES ARE DETACHED EXTERNAL VERIFICATION "\n                "SURFACES AND ARE NOT EMBEDDED INTO THEIR OWN FROZEN BYTES."\n            ),\n            "next_step": (\n                "Create Protocol v1.0 P01 annex"\n                if self.pipeline.state.get("preliminary_decision", {}).get(\n                    "status"\n                ) == "ACCEPTED"\n                else "Preserve failed bundle; repair the exact governed defect and rerun"\n            ),\n        }\n        result = self._record(\n            "26",\n            "PASS",\n            outputs=[\n                str(expected_zip),\n                str(expected_sha),\n                str(expected_repo_zip),\n                str(expected_repo_sha),\n                release_plan_rel,\n            ],\n            observations=pre,\n        )\n        try:\n            # Stage 26 already exists, so the exact 00-26 identity is frozen once.\n            # No file inside the bundle is modified after this call.\n            final_decision = self.pipeline.prepare_final_artifacts()\n            repository = _create_github_ready_repository(self)\n            package = self.pipeline.package_bundle()\n            final = {\n                **pre,\n                "decision": final_decision["status"],\n                "final_manifest_closure": self.pipeline.state.get(\n                    "final_manifest_closure"\n                ),\n                "compact_phase_execution_bundle": package,\n                "github_ready_repository": repository,\n                "private_derived_windows_dataset": {\n                    "handle": self.pipeline.state.get("r26_derived_handle"),\n                    "version": 1,\n                    "pointer": self.pipeline.state.get("r26_derived_pointer"),\n                    "storage_forecast": self.pipeline.state.get(\n                        "r26_storage_forecast_path"\n                    ),\n                    "storage_actual": self.pipeline.state.get(\n                        "r26_storage_actual_path"\n                    ),\n                    "signal_dtype": "float32",\n                    "window_contract": (\n                        "OFFSET_80_DURATION_480_ONE_PER_INCLUDED_EVENT"\n                    ),\n                },\n                "future_phase_retrieval": (\n                    "Attach the private derived Dataset directly; use the included "\n                    "shard reader and verify manifest/shard hashes."\n                ),\n                "next_step": (\n                    "Create Protocol v1.0 Phase 1 annex"\n                    if final_decision["status"] == "ACCEPTED"\n                    else "Resolve blockers and rerun"\n                ),\n            }\n            # The internal Stage 26 record remains the frozen pre-export contract.\n            # Actual external ZIP hashes are emitted only outside the bundle.\n            result.observations = final\n            result.outputs = [\n                package["zip"],\n                package["sha256_file"],\n                repository["path"],\n                repository["path"] + ".sha256",\n                repository["external_pointer_path"],\n                repository["external_manifest_path"],\n                "external_artifact_pointers/derived_windows_dataset.json",\n            ]\n            status_path = self.work_root / "stage_status" / "26.json"\n            status_path.parent.mkdir(parents=True, exist_ok=True)\n            status_path.write_text(\n                json.dumps(result.__dict__, indent=2, default=str),\n                encoding="utf-8",\n            )\n            external = self.work_root / "final_external_package_verification.json"\n            external.write_text(\n                json.dumps(\n                    {\n                        "stage": "26",\n                        "package": package,\n                        "github_ready_repository": repository,\n                        "decision": final_decision["status"],\n                        "bundle_internal_stage_evidence": (\n                            "reports/phase_01/tests/stage_results.json"\n                        ),\n                        "bundle_frozen_before_external_zip_creation": True,\n                        "self_hash_boundary": (\n                            "ZIP SHA-256 VALUES ARE DETACHED AND CANNOT BE "\n                            "EMBEDDED INSIDE THE SAME FROZEN ZIP BYTES."\n                        ),\n                    },\n                    indent=2,\n                    default=str,\n                ),\n                encoding="utf-8",\n            )\n            print(json.dumps(final, indent=2, default=str))\n            return result\n        except Exception as exc:\n            blocker = {\n                "code": "P01_FINAL_EXPORT_OR_DUAL_PERSISTENCE_FAILED",\n                "message": str(exc),\n                "owner": "L1_PACKAGING_OR_EXTERNAL_PERSISTENCE",\n            }\n            if blocker not in self.pipeline.blockers:\n                self.pipeline.blockers.append(blocker)\n            result.status = "BLOCKED"\n            result.blockers = [blocker]\n            result.observations = {\n                **pre,\n                "error": repr(exc),\n                "traceback": traceback.format_exc(),\n            }\n            _sync_stage_result(self, result)\n            try:\n                failed_decision = self.pipeline.prepare_final_artifacts()\n                failed_package = self.pipeline.package_bundle()\n                result.outputs = [\n                    failed_package["zip"],\n                    failed_package["sha256_file"],\n                ]\n                result.observations["failed_bundle_package"] = failed_package\n                result.observations["failed_final_decision"] = failed_decision[\n                    "status"\n                ]\n            except Exception as preserve_exc:\n                result.observations["failed_bundle_preservation_error"] = repr(\n                    preserve_exc\n                )\n            status_path = self.work_root / "stage_status" / "26.json"\n            status_path.parent.mkdir(parents=True, exist_ok=True)\n            status_path.write_text(\n                json.dumps(result.__dict__, indent=2, default=str),\n                encoding="utf-8",\n            )\n            return result\n\n    runner.stage_13 = MethodType(stage_13, runner)\n    runner.stage_14 = MethodType(stage_14, runner)\n    runner.stage_15 = MethodType(stage_15, runner)\n    # Stages 24 and 25 intentionally retain the authoritative R6 ordering.\n    # Repository creation and final manifest freezing occur only in Stage 26.\n    runner.stage_26 = MethodType(stage_26, runner)\n\n    installation = {\n        "policy": POLICY,\n        "installed_at_unix": time.time(),\n        "execution_attempt_id": attempt,\n        "derived_dataset_handle": handle,\n        "source_resolution_file": str(source_resolution_file),\n        "patched_surfaces": [\n            "Layer1Pipeline.load_sources",\n            "Layer1Pipeline.split_budget_preprocess:R35_AUTHORITATIVE_EVENT_ROW_SCHEMA",\n            "StageRunner.stage_13",\n            "StageRunner.stage_14",\n            "StageRunner.stage_15",\n            "StageRunner.stage_26",\n        ],\n        "authoritative_stage_order_retained": ["24", "25"],\n        "downstream_contract_closure": {\n            "joint_event_resampling": True,\n            "eeg_only_channel_selection": True,\n            "float32_output_dtype": True,\n            "window_start_offset_samples": 80,\n            "window_duration_samples": 480,\n            "window_stride_samples": 480,\n            "one_window_per_included_event": True,\n            "invalid_windows_preserved_and_block_commit": True,\n            "quality_hard_invalid_blocks_stage14": True,\n            "stage24_no_premature_finalization": True,\n            "stage26_complete_identity_before_finalization": True,\n            "compact_bundle_packaged": True,\n            "github_ready_repository_packaged": True,\n        },\n        "preservation": {\n            "all_three_sources": True,\n            "all_subjects_sessions_runs": True,\n            "all_labels": True,\n            "split_budget_profiles": True,\n            "stage11_authoritative_event_row_contract": True,\n            "preprocessing_operations": True,\n            "quality_annotation": True,\n            "window_identity_and_sample_hash": True,\n            "canonical_records": True,\n            "validation_and_leakage": True,\n            "cards_manifests_readiness": True,\n            "p01_gates": True,\n            "phase2_and_later_handoffs": True,\n            "negative_evidence": True,\n            "final_bundle": True,\n        },\n        "resource_properties": {\n            "parent_retains_signal_arrays": False,\n            "maximum_subject_processes": 8,\n            "subject_process_disposable": True,\n            "stage07_parallelism": "REAL_PREFLIGHT_MEASURED_RSS_ADAPTIVE",\n            "temporary_shard_deleted_after_blob_upload": True,\n            "exact_float32_storage_forecast_before_materialization": True,\n            "actual_storage_report_after_upload": True,\n        },\n        "persistence_outputs": {\n            "compact_github_ready_repository": True,\n            "large_private_kaggle_dataset": True,\n            "future_phase_reader_in_dataset": True,\n        },\n    }\n    _atomic_json(\n        pipeline.bundle_root\n        / "reports"\n        / "phase_01"\n        / "runtime"\n        / "bounded_streaming"\n        / "installation.json",\n        installation,\n    )\n    return installation\n\n\n\n\ndef _synthetic_self_test() -> dict[str, Any]:\n    import numpy as np\n    import h5py\n    import types\n    from dataclasses import dataclass\n    from tempfile import TemporaryDirectory\n    # The test can run before the R6 base package is attached. Minimal private\n    # stubs exercise only the exact contracts consumed by _combine_fit_rows.\n    inserted_modules: list[str] = []\n    try:\n        from iharq.canonical import semantic_hash as _probe_semantic_hash\n        from iharq.layer1_data_protocol.preprocessing import FitState as _probe_fit_state\n    except Exception:\n        iharq_mod = types.ModuleType("iharq")\n        canonical_mod = types.ModuleType("iharq.canonical")\n        layer1_mod = types.ModuleType("iharq.layer1_data_protocol")\n        prep_mod = types.ModuleType("iharq.layer1_data_protocol.preprocessing")\n        def semantic_hash(value):\n            raw = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")\n            return hashlib.sha256(raw).hexdigest()\n        @dataclass\n        class FitState:\n            mean: Any\n            std: Any\n            source_ids: list[str]\n            state_hash: str\n        canonical_mod.semantic_hash = semantic_hash\n        prep_mod.FitState = FitState\n        iharq_mod.canonical = canonical_mod\n        iharq_mod.layer1_data_protocol = layer1_mod\n        layer1_mod.preprocessing = prep_mod\n        for name, module in {\n            "iharq": iharq_mod,\n            "iharq.canonical": canonical_mod,\n            "iharq.layer1_data_protocol": layer1_mod,\n            "iharq.layer1_data_protocol.preprocessing": prep_mod,\n        }.items():\n            if name not in sys.modules:\n                sys.modules[name] = module\n                inserted_modules.append(name)\n    rng = np.random.default_rng(20260806)\n    arrays = [rng.normal(size=(4, n)).astype(np.float64) for n in (37, 53, 29)]\n    rows = []\n    for i, x in enumerate(arrays):\n        mean = x.mean(axis=1); centered = x - mean[:, None]\n        rows.append({"source_unit": f"D:{i}:S:R", "channel_names": [f"C{k}" for k in range(4)], "count": x.shape[1], "mean": mean.tolist(), "m2": np.sum(centered*centered, axis=1).tolist()})\n    state, detail = _combine_fit_rows(rows, {row["source_unit"] for row in rows})\n    cat = np.concatenate(arrays, axis=1)\n    fit_ok = np.allclose(state.mean[:, 0], cat.mean(axis=1), rtol=1e-13, atol=1e-13) and np.allclose(state.std[:, 0], cat.std(axis=1), rtol=1e-12, atol=1e-12)\n    with TemporaryDirectory() as tmp:\n        path = Path(tmp) / "test.h5"; window = rng.normal(size=(4, 64)).astype(np.float32)\n        with h5py.File(path, "w") as handle: group, row = _append_h5_window(handle, window, "window:test")\n        with h5py.File(path, "r") as handle: restored = handle[group + "/signals"][row]\n        h5_ok = np.array_equal(window, restored)\n    import ast as _ast\n    no_h5_proxy = not any(isinstance(node, _ast.ClassDef) and node.name == "H5Signal" for node in _ast.walk(_ast.parse(Path(__file__).read_text(encoding="utf-8"))))\n    for name in reversed(inserted_modules):\n        sys.modules.pop(name, None)\n    return {"status": "PASS" if fit_ok and h5_ok and no_h5_proxy and str(restored.dtype) == "float32" else "FAIL", "streaming_fit_matches_concatenate": fit_ok, "lossless_hdf5_roundtrip": h5_ok, "no_whole_array_h5_proxy": no_h5_proxy, "detail": detail}\n\n\nif __name__ == "__main__":\n    if len(sys.argv) == 3 and sys.argv[1] == "--child":\n        raise SystemExit(_child_main(sys.argv[2]))\n    if len(sys.argv) == 2 and sys.argv[1] == "--self-test":\n        print(json.dumps(_synthetic_self_test(), indent=2)); raise SystemExit(0)\n    raise SystemExit("Use --child <request.json> or --self-test")\n'

# R42 source-level repairs and additive A4 extension.
r42_a4_child_source = '# =====================================================================\n# R42 additive A4 evidence-window implementation\n# =====================================================================\n\nR42_A4_WINDOW_FAMILY = {\n    "window_family_id": os.environ.get(\n        "IHARQ_A4_WINDOW_FAMILY_ID",\n        "P01-L1-A4-WINDOW-FAMILY-FREEZE-R1",\n    ),\n    "protocol_status": os.environ.get(\n        "IHARQ_A4_PROTOCOL_STATUS",\n        "DATA_READY_PROTOCOL_SYNC_REQUIRED",\n    ),\n    "target_sampling_hz": 160,\n    "materialized_profile": {\n        "profile_id": "A4_LONG_FULL_4S_R1",\n        "event_anchor": "MI_CUE_ONSET",\n        "start_offset_samples": 0,\n        "duration_samples": 640,\n        "duration_seconds": 4.0,\n        "view_kind": "MATERIALIZED",\n    },\n    "multi_window_profile": {\n        "profile_id": "A4_MULTI_3X2S_R1",\n        "member_count": 3,\n        "member_duration_samples": 320,\n        "member_duration_seconds": 2.0,\n        "member_stride_samples": 160,\n        "member_stride_seconds": 1.0,\n        "member_slices": [\n            {\n                "member_index": 1,\n                "condition_id": "A4_MULTI_3X2S_M1_R1",\n                "slice_start": 0,\n                "slice_stop": 320,\n                "start_offset_seconds": 0.0,\n                "stop_offset_seconds": 2.0,\n            },\n            {\n                "member_index": 2,\n                "condition_id": "A4_MULTI_3X2S_M2_R1",\n                "slice_start": 160,\n                "slice_stop": 480,\n                "start_offset_seconds": 1.0,\n                "stop_offset_seconds": 3.0,\n            },\n            {\n                "member_index": 3,\n                "condition_id": "A4_MULTI_3X2S_M3_R1",\n                "slice_start": 320,\n                "slice_stop": 640,\n                "start_offset_seconds": 2.0,\n                "stop_offset_seconds": 4.0,\n            },\n        ],\n        "view_kind": "REGISTERED_VIRTUAL_SLICE",\n        "storage_rule": (\n            "VIEWS_REFERENCE_THE_LOSSLESS_FULL_4S_EVENT_TENSOR; "\n            "OVERLAPPING_NUMERICAL_BYTES_ARE_NOT_DUPLICATED"\n        ),\n    },\n}\n\n\nR42_A4_READER_SOURCE = r"""from __future__ import annotations\n\nfrom pathlib import Path\nimport json\nimport h5py\nimport numpy as np\n\n\ndef read_jsonl(path: str | Path):\n    with Path(path).open("r", encoding="utf-8") as stream:\n        for line in stream:\n            if line.strip():\n                yield json.loads(line)\n\n\ndef resolve_window(\n    dataset_root: str | Path,\n    location_row: dict,\n) -> np.ndarray:\n    root = Path(dataset_root)\n    shard = root / location_row["shard_filename"]\n    with h5py.File(shard, "r") as handle:\n        array = np.asarray(\n            handle[\n                location_row["hdf5_group"] + "/signals"\n            ][int(location_row["hdf5_row"])],\n            dtype=np.float32,\n        )\n\n    start = int(location_row.get("slice_start", 0))\n    stop = int(\n        location_row.get(\n            "slice_stop",\n            array.shape[1],\n        )\n    )\n    resolved = np.asarray(array[:, start:stop], dtype=np.float32)\n\n    expected = list(location_row["shape"])\n    if list(resolved.shape) != expected:\n        raise RuntimeError(\n            f"A4 view shape mismatch: expected={expected}; "\n            f"observed={list(resolved.shape)}"\n        )\n    return resolved\n"""\n\n\ndef _r42_a4_config_id(base_config_id: str) -> str:\n    from iharq.canonical import semantic_hash\n\n    return semantic_hash(\n        {\n            "base_config_id": base_config_id,\n            "a4_window_family": R42_A4_WINDOW_FAMILY,\n        }\n    )\n\n\ndef _r42_child_materialize_a4(\n    task: dict[str, Any],\n) -> dict[str, Any]:\n    import h5py\n    import numpy as np\n\n    from iharq.canonical import semantic_hash\n    from iharq.layer1_data_protocol.preprocessing import (\n        FitState,\n        transform_recording,\n    )\n    from iharq.layer1_data_protocol.quality import annotate\n    from iharq.layer1_data_protocol.labels import map_event_label\n    from iharq.layer1_data_protocol.splits import recording_role\n    from iharq.layer1_data_protocol.records import make_record\n\n    recordings, files, observed, expected = _load_subject_recordings(\n        task\n    )\n    output = Path(task["output_dir"])\n    output.mkdir(parents=True, exist_ok=True)\n\n    fit_payload = task["fit_state"]\n    mean = (\n        None\n        if fit_payload.get("mean") is None\n        else np.asarray(fit_payload["mean"], dtype=np.float64)\n    )\n    std = (\n        None\n        if fit_payload.get("std") is None\n        else np.asarray(fit_payload["std"], dtype=np.float64)\n    )\n    fit_state = FitState(\n        mean,\n        std,\n        list(fit_payload["source_ids"]),\n        str(fit_payload["state_hash"]),\n    )\n\n    operations = list(task["operations"])\n    assignment = dict(task["assignment"])\n    split_keys = list(task["split_keys"])\n    label_record = dict(task["label_record"])\n    preprocessing_record = dict(task["preprocessing_record"])\n    split_record = dict(task["split_record"])\n    quality_profile = dict(task["quality_profile"])\n    base_config_id = str(task["base_config_id"])\n    a4_config_id = str(task["a4_config_id"])\n    dataset_record_id = str(task["dataset_record_id"])\n    split_id = split_record["record_id"]\n    family = dict(task["a4_window_family"])\n\n    materialized = dict(family["materialized_profile"])\n    multi = dict(family["multi_window_profile"])\n\n    target_hz = float(family["target_sampling_hz"])\n    full_offset = int(materialized["start_offset_samples"])\n    full_duration = int(materialized["duration_samples"])\n\n    if target_hz != 160.0:\n        raise RuntimeError(\n            f"R42_A4_TARGET_RATE_MISMATCH: {target_hz}"\n        )\n    if full_offset != 0 or full_duration != 640:\n        raise RuntimeError(\n            "R42_A4_FULL4S_CONTRACT_MISMATCH"\n        )\n    if int(multi["member_count"]) != 3:\n        raise RuntimeError(\n            "R42_A4_MULTI_MEMBER_COUNT_MUST_BE_THREE"\n        )\n\n    shard_filename = str(task["shard_filename"])\n    shard_path = output / shard_filename\n\n    records_path = output / "a4_window_records.jsonl"\n    index_path = output / "a4_window_index.jsonl"\n    groups_path = output / "a4_group_index.jsonl"\n    locations_path = output / "a4_window_locations.jsonl"\n    quality_records_path = output / "quality_records.jsonl"\n    quality_summaries_path = output / "quality_summaries.jsonl"\n    invalid_path = output / "a4_invalid_windows.jsonl"\n    result_path = output / "result.json"\n\n    for stale in (\n        shard_path,\n        records_path,\n        index_path,\n        groups_path,\n        locations_path,\n        quality_records_path,\n        quality_summaries_path,\n        invalid_path,\n        result_path,\n    ):\n        stale.unlink(missing_ok=True)\n\n    for path in (\n        records_path,\n        index_path,\n        groups_path,\n        locations_path,\n        quality_records_path,\n        quality_summaries_path,\n        invalid_path,\n    ):\n        path.touch()\n\n    record_buffer: list[dict[str, Any]] = []\n    index_buffer: list[dict[str, Any]] = []\n    group_buffer: list[dict[str, Any]] = []\n    location_buffer: list[dict[str, Any]] = []\n    quality_rows_all: list[dict[str, Any]] = []\n    quality_summaries: list[dict[str, Any]] = []\n    invalid_rows: list[dict[str, Any]] = []\n\n    materialized_event_count = 0\n    window_record_count = 0\n    logical_stored_bytes = 0\n    source_event_ids: set[str] = set()\n    roles: set[str] = set()\n    h5_handle = None\n\n    try:\n        for source_recording in recordings:\n            recording = transform_recording(\n                source_recording,\n                operations,\n                fit_state,\n            )\n\n            signal = np.asarray(recording.signal)\n            if abs(float(recording.sampling_hz) - target_hz) > 1e-9:\n                raise RuntimeError(\n                    "R42_A4_TRANSFORMED_RATE_MISMATCH: "\n                    f"{_recording_source_unit(recording)}"\n                )\n            if signal.dtype != np.dtype("float32"):\n                raise RuntimeError(\n                    "R42_A4_SIGNAL_DTYPE_MISMATCH: "\n                    f"{signal.dtype}"\n                )\n            if signal.ndim != 2:\n                raise RuntimeError(\n                    f"R42_A4_SIGNAL_RANK_MISMATCH: {signal.ndim}"\n                )\n            if len(recording.channel_names) != int(signal.shape[0]):\n                raise RuntimeError(\n                    "R42_A4_CHANNEL_GEOMETRY_MISMATCH"\n                )\n\n            qrows, qsummary = annotate(\n                recording,\n                quality_profile,\n                base_config_id,\n                dataset_record_id,\n            )\n            quality_rows_all.extend(qrows)\n            quality_summaries.append(qsummary)\n\n            role = recording_role(\n                recording,\n                assignment,\n                split_keys,\n            )\n            roles.add(role)\n\n            for event in recording.events:\n                normalized = map_event_label(\n                    event.original_label,\n                    label_record,\n                )\n                if normalized is None:\n                    continue\n\n                source_sample = event.metadata.get(\n                    "original_source_event_sample"\n                )\n                resampled_sample = event.metadata.get(\n                    "resampled_event_sample"\n                )\n                if source_sample is None or resampled_sample is None:\n                    invalid_rows.append(\n                        {\n                            "dataset_id": recording.dataset_id,\n                            "subject_id": recording.subject_id,\n                            "session_id": recording.session_id,\n                            "run_id": recording.run_id,\n                            "event_id": event.event_id,\n                            "reason": (\n                                "MISSING_PARENT_EVENT_SAMPLE_LINEAGE"\n                            ),\n                        }\n                    )\n                    continue\n\n                full_start = int(resampled_sample) + full_offset\n                full_stop = full_start + full_duration\n\n                if (\n                    full_start < 0\n                    or full_stop > int(signal.shape[1])\n                ):\n                    invalid_rows.append(\n                        {\n                            "dataset_id": recording.dataset_id,\n                            "subject_id": recording.subject_id,\n                            "session_id": recording.session_id,\n                            "run_id": recording.run_id,\n                            "event_id": event.event_id,\n                            "resampled_event_sample": int(\n                                resampled_sample\n                            ),\n                            "start": full_start,\n                            "stop": full_stop,\n                            "available_samples": int(signal.shape[1]),\n                            "reason": "A4_FULL4S_OUT_OF_BOUNDS",\n                        }\n                    )\n                    continue\n\n                full_window = np.asarray(\n                    signal[:, full_start:full_stop],\n                    dtype=np.float32,\n                )\n                if list(full_window.shape) != [\n                    int(signal.shape[0]),\n                    640,\n                ]:\n                    raise RuntimeError(\n                        "R42_A4_FULL_WINDOW_SHAPE_MISMATCH: "\n                        f"{list(full_window.shape)}"\n                    )\n                if not np.isfinite(full_window).all():\n                    raise RuntimeError(\n                        "R42_A4_FULL_WINDOW_NONFINITE"\n                    )\n\n                storage_identity = {\n                    "window_family_id": family["window_family_id"],\n                    "dataset": recording.dataset_id,\n                    "subject": recording.subject_id,\n                    "session": recording.session_id,\n                    "run": recording.run_id,\n                    "event": event.event_id,\n                    "original_event_sample": int(source_sample),\n                    "resampled_event_sample": int(resampled_sample),\n                    "full_start": full_start,\n                    "full_stop": full_stop,\n                    "split_record_id": split_id,\n                    "role": role,\n                    "a4_config_id": a4_config_id,\n                }\n                storage_id = (\n                    "a4-storage:"\n                    + semantic_hash(storage_identity)[:20]\n                )\n\n                if h5_handle is None:\n                    h5_handle = h5py.File(shard_path, "w")\n                    h5_handle.attrs["format"] = (\n                        "IHARQ_P01_L1_A4_FULL4S_SHARD_R1"\n                    )\n                    h5_handle.attrs["base_scientific_freeze"] = (\n                        POLICY["scientific_freeze_unchanged"]\n                    )\n                    h5_handle.attrs["a4_window_family_id"] = (\n                        family["window_family_id"]\n                    )\n                    h5_handle.attrs["base_config_id"] = base_config_id\n                    h5_handle.attrs["a4_config_id"] = a4_config_id\n                    h5_handle.attrs["dataset_id"] = str(\n                        recording.dataset_id\n                    )\n                    h5_handle.attrs["subject_profile"] = str(\n                        task["subject"]\n                    )\n                    h5_handle.attrs["signal_dtype"] = "float32"\n                    h5_handle.attrs["materialized_duration_samples"] = 640\n                    h5_handle.attrs["event_resampling"] = (\n                        "MNE_POLYPHASE_JOINT_EVENTS"\n                    )\n\n                group_path, hdf5_row = _append_h5_window(\n                    h5_handle,\n                    full_window,\n                    storage_id,\n                )\n\n                common_payload = {\n                    "parent_event_id": event.event_id,\n                    "dataset_id": recording.dataset_id,\n                    "subject_id": recording.subject_id,\n                    "session_id": recording.session_id,\n                    "run_id": recording.run_id,\n                    "split_record_id": split_id,\n                    "preprocessing_record_id": (\n                        preprocessing_record["record_id"]\n                    ),\n                    "label_map_record_id": (\n                        label_record["record_id"]\n                    ),\n                    "original_source_event_sample": int(source_sample),\n                    "resampled_event_sample": int(resampled_sample),\n                    "normalized_label": normalized,\n                    "original_label": event.original_label,\n                    "role": role,\n                    "overlap_group_id": event.event_id,\n                    "window_family_id": family["window_family_id"],\n                    "protocol_status": family["protocol_status"],\n                    "a4_group_id": (\n                        "a4-group:"\n                        + semantic_hash(\n                            {\n                                "event": event.event_id,\n                                "family": family["window_family_id"],\n                                "config": a4_config_id,\n                            }\n                        )[:20]\n                    ),\n                }\n\n                source_ids = [\n                    dataset_record_id,\n                    split_id,\n                    preprocessing_record["record_id"],\n                    label_record["record_id"],\n                ]\n\n                # Full 4-second longer-window control.\n                long_window_id = (\n                    "a4-window:"\n                    + semantic_hash(\n                        {\n                            **storage_identity,\n                            "condition": materialized["profile_id"],\n                        }\n                    )[:20]\n                )\n                long_pointer = (\n                    "external_artifact_pointers/"\n                    "a4_window_to_shard.jsonl"\n                    f"#window_id={long_window_id}"\n                )\n                long_payload = {\n                    **common_payload,\n                    "window_id": long_window_id,\n                    "evidence_condition_id": (\n                        materialized["profile_id"]\n                    ),\n                    "a4_component": "LONGER_WINDOW",\n                    "view_kind": "MATERIALIZED_FULL_4S",\n                    "start_offset_samples": 0,\n                    "start_sample": full_start,\n                    "stop_sample": full_stop,\n                    "duration_samples": 640,\n                    "stride_samples": 640,\n                    "signal_pointer": long_pointer,\n                    "channel_mask_id": None,\n                    "evidence_availability_seconds": 4.0,\n                }\n                long_record = make_record(\n                    "WindowRecord",\n                    long_payload,\n                    a4_config_id,\n                    source_ids,\n                    evidence_mode="IMPLEMENTATION",\n                    lifecycle_status="VALIDATED",\n                    evidence_role="DERIVED",\n                    ablation_id="A4",\n                    limitation_tags=[\n                        "PUBLIC_EEG_ONLY",\n                        "NON_CLINICAL",\n                        "NO_DEPLOYMENT_CLAIM",\n                        "A4_PROTOCOL_SYNC_REQUIRED",\n                    ],\n                )\n\n                long_sample_hash = semantic_hash(\n                    {\n                        "shape": list(full_window.shape),\n                        "head": [\n                            str(float(value))\n                            for value in full_window.reshape(-1)[:128]\n                        ],\n                    }\n                )\n\n                record_buffer.append(long_record)\n                index_buffer.append(\n                    {\n                        "window_record_id": long_record["record_id"],\n                        "window_id": long_window_id,\n                        "event_id": event.event_id,\n                        "dataset_id": recording.dataset_id,\n                        "subject_id": recording.subject_id,\n                        "session_id": recording.session_id,\n                        "run_id": recording.run_id,\n                        "role": role,\n                        "label": normalized,\n                        "split_record_id": split_id,\n                        "a4_group_id": common_payload["a4_group_id"],\n                        "a4_component": "LONGER_WINDOW",\n                        "evidence_condition_id": (\n                            materialized["profile_id"]\n                        ),\n                        "member_index": 0,\n                        "sample_hash": long_sample_hash,\n                        "external_shard_filename": shard_filename,\n                        "hdf5_group": group_path,\n                        "hdf5_row": hdf5_row,\n                        "slice_start": 0,\n                        "slice_stop": 640,\n                        "view_kind": "MATERIALIZED_FULL_4S",\n                    }\n                )\n                location_buffer.append(\n                    {\n                        "window_id": long_window_id,\n                        "window_record_id": long_record["record_id"],\n                        "shard_filename": shard_filename,\n                        "hdf5_group": group_path,\n                        "hdf5_row": hdf5_row,\n                        "slice_start": 0,\n                        "slice_stop": 640,\n                        "shape": [int(signal.shape[0]), 640],\n                        "dtype": "float32",\n                        "view_kind": "MATERIALIZED_FULL_4S",\n                        "a4_group_id": common_payload["a4_group_id"],\n                    }\n                )\n\n                multi_record_ids: list[str] = []\n                multi_window_ids: list[str] = []\n\n                for member in multi["member_slices"]:\n                    slice_start = int(member["slice_start"])\n                    slice_stop = int(member["slice_stop"])\n                    view = np.asarray(\n                        full_window[:, slice_start:slice_stop],\n                        dtype=np.float32,\n                    )\n                    if list(view.shape) != [\n                        int(signal.shape[0]),\n                        320,\n                    ]:\n                        raise RuntimeError(\n                            "R42_A4_MULTI_VIEW_SHAPE_MISMATCH: "\n                            f"{list(view.shape)}"\n                        )\n\n                    multi_window_id = (\n                        "a4-window:"\n                        + semantic_hash(\n                            {\n                                **storage_identity,\n                                "condition": member["condition_id"],\n                                "slice": [slice_start, slice_stop],\n                            }\n                        )[:20]\n                    )\n                    pointer = (\n                        "external_artifact_pointers/"\n                        "a4_window_to_shard.jsonl"\n                        f"#window_id={multi_window_id}"\n                    )\n                    payload = {\n                        **common_payload,\n                        "window_id": multi_window_id,\n                        "evidence_condition_id": (\n                            member["condition_id"]\n                        ),\n                        "a4_component": "MULTI_WINDOW_MEMBER",\n                        "view_kind": (\n                            "REGISTERED_VIRTUAL_SLICE_OF_FULL_4S"\n                        ),\n                        "member_index": int(member["member_index"]),\n                        "start_offset_samples": slice_start,\n                        "start_sample": full_start + slice_start,\n                        "stop_sample": full_start + slice_stop,\n                        "duration_samples": 320,\n                        "stride_samples": 160,\n                        "slice_start": slice_start,\n                        "slice_stop": slice_stop,\n                        "signal_pointer": pointer,\n                        "channel_mask_id": None,\n                        "evidence_availability_seconds": (\n                            float(member["stop_offset_seconds"])\n                        ),\n                    }\n                    record = make_record(\n                        "WindowRecord",\n                        payload,\n                        a4_config_id,\n                        source_ids,\n                        evidence_mode="IMPLEMENTATION",\n                        lifecycle_status="VALIDATED",\n                        evidence_role="DERIVED",\n                        ablation_id="A4",\n                        limitation_tags=[\n                            "PUBLIC_EEG_ONLY",\n                            "NON_CLINICAL",\n                            "NO_DEPLOYMENT_CLAIM",\n                            "A4_PROTOCOL_SYNC_REQUIRED",\n                        ],\n                    )\n\n                    sample_hash = semantic_hash(\n                        {\n                            "shape": list(view.shape),\n                            "head": [\n                                str(float(value))\n                                for value in view.reshape(-1)[:128]\n                            ],\n                        }\n                    )\n\n                    record_buffer.append(record)\n                    index_buffer.append(\n                        {\n                            "window_record_id": record["record_id"],\n                            "window_id": multi_window_id,\n                            "event_id": event.event_id,\n                            "dataset_id": recording.dataset_id,\n                            "subject_id": recording.subject_id,\n                            "session_id": recording.session_id,\n                            "run_id": recording.run_id,\n                            "role": role,\n                            "label": normalized,\n                            "split_record_id": split_id,\n                            "a4_group_id": common_payload["a4_group_id"],\n                            "a4_component": "MULTI_WINDOW_MEMBER",\n                            "evidence_condition_id": (\n                                member["condition_id"]\n                            ),\n                            "member_index": int(\n                                member["member_index"]\n                            ),\n                            "sample_hash": sample_hash,\n                            "external_shard_filename": shard_filename,\n                            "hdf5_group": group_path,\n                            "hdf5_row": hdf5_row,\n                            "slice_start": slice_start,\n                            "slice_stop": slice_stop,\n                            "view_kind": (\n                                "REGISTERED_VIRTUAL_SLICE_OF_FULL_4S"\n                            ),\n                        }\n                    )\n                    location_buffer.append(\n                        {\n                            "window_id": multi_window_id,\n                            "window_record_id": record["record_id"],\n                            "shard_filename": shard_filename,\n                            "hdf5_group": group_path,\n                            "hdf5_row": hdf5_row,\n                            "slice_start": slice_start,\n                            "slice_stop": slice_stop,\n                            "shape": [int(signal.shape[0]), 320],\n                            "dtype": "float32",\n                            "view_kind": (\n                                "REGISTERED_VIRTUAL_SLICE_OF_FULL_4S"\n                            ),\n                            "a4_group_id": common_payload["a4_group_id"],\n                        }\n                    )\n                    multi_record_ids.append(record["record_id"])\n                    multi_window_ids.append(multi_window_id)\n\n                group_buffer.append(\n                    {\n                        "a4_group_id": common_payload["a4_group_id"],\n                        "parent_event_id": event.event_id,\n                        "dataset_id": recording.dataset_id,\n                        "subject_id": recording.subject_id,\n                        "session_id": recording.session_id,\n                        "run_id": recording.run_id,\n                        "role": role,\n                        "normalized_label": normalized,\n                        "split_record_id": split_id,\n                        "long_window_record_id": (\n                            long_record["record_id"]\n                        ),\n                        "long_window_id": long_window_id,\n                        "multi_member_window_record_ids": (\n                            multi_record_ids\n                        ),\n                        "multi_member_window_ids": multi_window_ids,\n                        "expected_multi_member_count": 3,\n                        "observed_multi_member_count": len(\n                            multi_record_ids\n                        ),\n                        "complete": len(multi_record_ids) == 3,\n                        "member_duration_samples": 320,\n                        "member_stride_samples": 160,\n                        "member_overlap_samples": 160,\n                        "unique_source_event_span_samples": 640,\n                        "evidence_availability_seconds": 4.0,\n                        "causal_mode": (\n                            "CAUSAL_AFTER_COMPLETE_4S_EVIDENCE"\n                        ),\n                        "protocol_status": family["protocol_status"],\n                    }\n                )\n\n                materialized_event_count += 1\n                window_record_count += 4\n                logical_stored_bytes += int(full_window.nbytes)\n                source_event_ids.add(str(event.event_id))\n\n                if len(record_buffer) >= 256:\n                    _append_jsonl(records_path, record_buffer)\n                    record_buffer.clear()\n                    _append_jsonl(index_path, index_buffer)\n                    index_buffer.clear()\n                    _append_jsonl(locations_path, location_buffer)\n                    location_buffer.clear()\n\n                if len(group_buffer) >= 256:\n                    _append_jsonl(groups_path, group_buffer)\n                    group_buffer.clear()\n\n                if (\n                    materialized_event_count % 256 == 0\n                    and h5_handle is not None\n                ):\n                    h5_handle.flush()\n\n            del recording\n            _trim_memory()\n\n        if record_buffer:\n            _append_jsonl(records_path, record_buffer)\n        if index_buffer:\n            _append_jsonl(index_path, index_buffer)\n        if group_buffer:\n            _append_jsonl(groups_path, group_buffer)\n        if location_buffer:\n            _append_jsonl(locations_path, location_buffer)\n        if quality_rows_all:\n            _append_jsonl(\n                quality_records_path,\n                quality_rows_all,\n            )\n        if quality_summaries:\n            _append_jsonl(\n                quality_summaries_path,\n                quality_summaries,\n            )\n        if invalid_rows:\n            _append_jsonl(invalid_path, invalid_rows)\n    finally:\n        if h5_handle is not None:\n            h5_handle.flush()\n            h5_handle.close()\n\n    shard = None\n    if shard_path.is_file():\n        verified_rows = 0\n        with h5py.File(shard_path, "r") as verify:\n            if str(verify.attrs.get("format", "")) != (\n                "IHARQ_P01_L1_A4_FULL4S_SHARD_R1"\n            ):\n                raise RuntimeError(\n                    "R42_A4_HDF5_FORMAT_MISMATCH"\n                )\n            if str(verify.attrs.get("signal_dtype", "")) != "float32":\n                raise RuntimeError(\n                    "R42_A4_HDF5_DTYPE_ATTRIBUTE_MISMATCH"\n                )\n            for group_name in sorted(\n                verify.get("window_groups", {})\n            ):\n                group = verify[f"window_groups/{group_name}"]\n                signals = group["signals"]\n                identifiers = group["window_ids"]\n                if str(signals.dtype) != "float32":\n                    raise RuntimeError(\n                        "R42_A4_HDF5_GROUP_DTYPE_MISMATCH"\n                    )\n                if int(signals.shape[2]) != 640:\n                    raise RuntimeError(\n                        "R42_A4_HDF5_DURATION_MISMATCH"\n                    )\n                if int(signals.shape[0]) != int(\n                    identifiers.shape[0]\n                ):\n                    raise RuntimeError(\n                        "R42_A4_HDF5_SIGNAL_ID_COUNT_MISMATCH"\n                    )\n                verified_rows += int(signals.shape[0])\n\n        if verified_rows != materialized_event_count:\n            raise RuntimeError(\n                "R42_A4_HDF5_EVENT_COUNT_MISMATCH: "\n                f"expected={materialized_event_count}; "\n                f"observed={verified_rows}"\n            )\n\n        shard = {\n            "path": str(shard_path),\n            "filename": shard_filename,\n            "bytes": shard_path.stat().st_size,\n            "sha256": _sha256(shard_path),\n            "verified_materialized_event_rows": verified_rows,\n            "logical_stored_bytes": logical_stored_bytes,\n            "compression_ratio_to_logical": (\n                shard_path.stat().st_size / logical_stored_bytes\n                if logical_stored_bytes\n                else None\n            ),\n            "verification_status": "PASS",\n            "signal_dtype": "float32",\n            "materialized_duration_samples": 640,\n            "virtual_views_per_event": 3,\n        }\n\n    result = {\n        "status": "PASS",\n        "dataset_id": task["profile"]["dataset_id"],\n        "subject": int(task["subject"]),\n        "source_files": [str(path) for path in files],\n        "observed_recordings": int(observed),\n        "expected_recordings": int(expected),\n        "materialized_event_count": materialized_event_count,\n        "window_record_count": window_record_count,\n        "a4_group_count": materialized_event_count,\n        "logical_stored_bytes": logical_stored_bytes,\n        "source_event_count": len(source_event_ids),\n        "roles": sorted(roles),\n        "shard": shard,\n        "a4_window_records_path": str(records_path),\n        "a4_window_index_path": str(index_path),\n        "a4_group_index_path": str(groups_path),\n        "a4_window_locations_path": str(locations_path),\n        "quality_records_path": str(quality_records_path),\n        "quality_summaries_path": str(quality_summaries_path),\n        "invalid_windows_path": str(invalid_path),\n        "invalid_window_count": len(invalid_rows),\n        "signal_dtype": "float32",\n        "a4_config_id": a4_config_id,\n        "window_family_id": family["window_family_id"],\n    }\n    _atomic_json(result_path, result)\n    return result\n\n\ndef _r42_download_dataset_file(\n    handle: str,\n    version: int,\n    filename: str,\n    output_root: Path,\n) -> Path:\n    import kagglehub\n\n    output_root.mkdir(parents=True, exist_ok=True)\n    resolved = kagglehub.dataset_download(\n        f"{handle}/versions/{int(version)}",\n        path=filename,\n        output_dir=str(output_root),\n        force_download=True,\n    )\n    path = Path(resolved)\n\n    if path.is_dir():\n        matches = list(path.rglob(filename))\n        if len(matches) != 1:\n            raise RuntimeError(\n                "R42_DATASET_FILE_RESOLUTION_AMBIGUOUS: "\n                f"filename={filename}; matches={len(matches)}"\n            )\n        path = matches[0]\n\n    if not path.is_file():\n        raise RuntimeError(\n            f"R42_DATASET_FILE_MISSING: {filename}"\n        )\n    return path\n\n\ndef _r42_assert_semantically_equal(\n    current: list[dict[str, Any]],\n    adopted: list[dict[str, Any]],\n    label: str,\n) -> None:\n    current_hashes = sorted(\n        str(row.get("semantic_hash"))\n        for row in current\n    )\n    adopted_hashes = sorted(\n        str(row.get("semantic_hash"))\n        for row in adopted\n    )\n    if current_hashes != adopted_hashes:\n        raise RuntimeError(\n            f"R42_CORE_ADOPTION_{label}_SEMANTIC_MISMATCH: "\n            f"current={current_hashes}; adopted={adopted_hashes}"\n        )\n\n\ndef _r42_adopt_existing_core_dataset(\n    runner: Any,\n) -> dict[str, Any]:\n    pipeline = runner.pipeline\n\n    handle = os.environ.get(\n        "IHARQ_EXISTING_CORE_DATASET_HANDLE",\n        "",\n    ).strip()\n    version = int(\n        os.environ.get(\n            "IHARQ_EXISTING_CORE_DATASET_VERSION",\n            "0",\n        )\n    )\n    expected_manifest_sha = os.environ.get(\n        "IHARQ_EXISTING_CORE_MANIFEST_SHA256",\n        "",\n    ).strip().lower()\n\n    if not handle or "/" not in handle or version < 1:\n        raise RuntimeError(\n            "R42_EXISTING_CORE_DATASET_CONFIGURATION_INVALID"\n        )\n    if not re.fullmatch(r"[0-9a-f]{64}", expected_manifest_sha):\n        raise RuntimeError(\n            "R42_EXISTING_CORE_MANIFEST_SHA256_INVALID"\n        )\n\n    root = (\n        pipeline.work_root\n        / "r42_core_adoption"\n        / f"provider_version_{version}"\n    )\n    if root.exists():\n        shutil.rmtree(root)\n    root.mkdir(parents=True)\n\n    filenames = [\n        "IHARQ_P01_L1_DERIVED_WINDOW_DATASET_MANIFEST.json",\n        "IHARQ_P01_L1_DERIVED_DATASET_SIDECAR.json",\n        "IHARQ_P01_L1_WINDOW_RECORDS.jsonl",\n        "IHARQ_P01_L1_WINDOW_INDEX.jsonl",\n        "IHARQ_P01_L1_WINDOW_TO_SHARD_INDEX.jsonl",\n        "IHARQ_P01_L1_DERIVED_OUTPUT_STORAGE_ACTUAL.json",\n        "IHARQ_P01_L1_DERIVED_OUTPUT_STORAGE_FORECAST.json",\n        "iharq_window_shard_reader.py",\n    ]\n    paths = {\n        name: _r42_download_dataset_file(\n            handle,\n            version,\n            name,\n            root,\n        )\n        for name in filenames\n    }\n\n    manifest_path = paths[\n        "IHARQ_P01_L1_DERIVED_WINDOW_DATASET_MANIFEST.json"\n    ]\n    observed_manifest_sha = _sha256(manifest_path)\n    if observed_manifest_sha != expected_manifest_sha:\n        raise RuntimeError(\n            "R42_EXISTING_CORE_MANIFEST_SHA256_MISMATCH: "\n            f"expected={expected_manifest_sha}; "\n            f"observed={observed_manifest_sha}"\n        )\n\n    manifest = json.loads(\n        manifest_path.read_text(encoding="utf-8")\n    )\n    sidecar = json.loads(\n        paths[\n            "IHARQ_P01_L1_DERIVED_DATASET_SIDECAR.json"\n        ].read_text(encoding="utf-8")\n    )\n    window_records = _read_jsonl(\n        paths["IHARQ_P01_L1_WINDOW_RECORDS.jsonl"]\n    )\n    window_index = _read_jsonl(\n        paths["IHARQ_P01_L1_WINDOW_INDEX.jsonl"]\n    )\n    locations = _read_jsonl(\n        paths["IHARQ_P01_L1_WINDOW_TO_SHARD_INDEX.jsonl"]\n    )\n\n    if manifest.get("repository_or_dataset") != handle:\n        raise RuntimeError(\n            "R42_EXISTING_CORE_HANDLE_MISMATCH"\n        )\n    if manifest.get("signal_dtype") != "float32":\n        raise RuntimeError(\n            "R42_EXISTING_CORE_DTYPE_MISMATCH"\n        )\n    if int(manifest.get("window_count", -1)) != 12_910:\n        raise RuntimeError(\n            "R42_EXISTING_CORE_WINDOW_COUNT_MISMATCH"\n        )\n    if len(manifest.get("shards", [])) != 172:\n        raise RuntimeError(\n            "R42_EXISTING_CORE_SHARD_COUNT_MISMATCH"\n        )\n\n    policy = manifest.get("window_policy", {})\n    expected_policy = {\n        "start_offset_samples": 80,\n        "duration_samples": 480,\n        "stride_samples": 480,\n        "last_window_policy": (\n            "ONE_WINDOW_PER_INCLUDED_SOURCE_EVENT"\n        ),\n        "bounds_policy": "REJECT_OUT_OF_BOUNDS",\n    }\n    for key, expected in expected_policy.items():\n        if policy.get(key) != expected:\n            raise RuntimeError(\n                "R42_EXISTING_CORE_WINDOW_POLICY_MISMATCH: "\n                f"{key}={policy.get(key)!r}"\n            )\n\n    if not (\n        len(window_records)\n        == len(window_index)\n        == len(locations)\n        == 12_910\n    ):\n        raise RuntimeError(\n            "R42_EXISTING_CORE_INDEX_COUNT_MISMATCH: "\n            f"records={len(window_records)}; "\n            f"index={len(window_index)}; "\n            f"locations={len(locations)}"\n        )\n\n    record_ids = {\n        row["record_id"] for row in window_records\n    }\n    index_record_ids = {\n        row["window_record_id"] for row in window_index\n    }\n    location_record_ids = {\n        row["window_record_id"] for row in locations\n    }\n    if (\n        len(record_ids) != 12_910\n        or record_ids != index_record_ids\n        or record_ids != location_record_ids\n    ):\n        raise RuntimeError(\n            "R42_EXISTING_CORE_RECORD_ID_CLOSURE_FAILED"\n        )\n\n    event_ids = {\n        row["event_id"] for row in window_index\n    }\n    if len(event_ids) != 12_910:\n        raise RuntimeError(\n            "R42_EXISTING_CORE_ONE_WINDOW_PER_EVENT_FAILED"\n        )\n\n    # Verify every compact file against the remote manifest.\n    for name, row in manifest.get(\n        "dataset_local_indexes",\n        {},\n    ).items():\n        if name not in paths:\n            continue\n        observed = _sha256(paths[name])\n        if observed != row.get("sha256"):\n            raise RuntimeError(\n                "R42_EXISTING_CORE_COMPACT_HASH_MISMATCH: "\n                f"{name}"\n            )\n\n    # Current run must reproduce the same scientific records before the\n    # existing window artifact can be adopted. IDs contain a run date, so\n    # semantic hashes—not date-bearing record IDs—are compared.\n    _r42_assert_semantically_equal(\n        list(pipeline.state.get("dataset_records", [])),\n        list(sidecar["dataset_records"]),\n        "DATASET_RECORDS",\n    )\n    _r42_assert_semantically_equal(\n        list(pipeline.state.get("label_records", [])),\n        list(sidecar["label_records"]),\n        "LABEL_RECORDS",\n    )\n    _r42_assert_semantically_equal(\n        [pipeline.state["split_record"]],\n        [sidecar["split_record"]],\n        "SPLIT_RECORD",\n    )\n    _r42_assert_semantically_equal(\n        [pipeline.state["preprocessing_record"]],\n        [sidecar["preprocessing_record"]],\n        "PREPROCESSING_RECORD",\n    )\n\n    # Adopt the already released record identities so the core windows keep\n    # exactly the lineage under which they were generated.\n    replaced_types = {\n        "DatasetRecord",\n        "LabelMapRecord",\n        "SplitRecord",\n        "PreprocessingRecord",\n        "WindowRecord",\n    }\n    pipeline.state["records"] = [\n        row\n        for row in pipeline.state.get("records", [])\n        if row.get("record_type") not in replaced_types\n    ]\n\n    pipeline.state["dataset_records"] = list(\n        sidecar["dataset_records"]\n    )\n    pipeline.state["label_records"] = list(\n        sidecar["label_records"]\n    )\n    pipeline.state["split_record"] = dict(\n        sidecar["split_record"]\n    )\n    pipeline.state["preprocessing_record"] = dict(\n        sidecar["preprocessing_record"]\n    )\n    pipeline.state["window_records"] = window_records\n    pipeline.state["window_index"] = window_index\n    pipeline.state["quality_summaries"] = list(\n        sidecar.get("quality_summaries", [])\n    )\n    pipeline.state["records"].extend(\n        pipeline.state["dataset_records"]\n        + pipeline.state["label_records"]\n        + [pipeline.state["split_record"]]\n        + [pipeline.state["preprocessing_record"]]\n        + window_records\n    )\n\n    # Preserve compact remote evidence in the new execution bundle.\n    compact_root = (\n        pipeline.bundle_root\n        / "external_artifact_pointers"\n        / "adopted_core_dataset"\n    )\n    if compact_root.exists():\n        shutil.rmtree(compact_root)\n    compact_root.mkdir(parents=True)\n    for name, path in paths.items():\n        shutil.copy2(path, compact_root / name)\n\n    pointer = {\n        **manifest,\n        "creation_status": "ADOPTED_VERIFIED_EXISTING_DATASET",\n        "dataset_handle": handle,\n        "repository_or_dataset": handle,\n        "dataset_version": version,\n        "provider_dataset_version": version,\n        "logical_immutable_revision": 1,\n        "manifest_sha256": observed_manifest_sha,\n        "adoption_run_config_id": pipeline.config_id,\n        "adoption_mode": (\n            "SEMANTIC_RECORD_EQUIVALENCE_PLUS_EXACT_REMOTE_HASH"\n        ),\n        "scientific_artifact_recomputed": False,\n        "stage14_core_reexecuted": False,\n        "core_hdf5_shards_reuploaded": False,\n        "provider_version_amendment": {\n            "shell_provider_version": 1,\n            "scientific_provider_version": 2,\n            "reason": (\n                "Provider version 1 is the short-title shell; "\n                "provider version 2 is the verified scientific artifact."\n            ),\n        },\n    }\n    pointer_path = (\n        pipeline.bundle_root\n        / "external_artifact_pointers"\n        / "derived_windows_dataset.json"\n    )\n    _atomic_json(pointer_path, pointer)\n\n    pipeline.state["window_report"] = {\n        "window_count": 12_910,\n        "event_count": 12_910,\n        "invalid_window_count": 0,\n        "invalid_windows": [],\n        "roles": sorted(\n            {row["role"] for row in window_index}\n        ),\n        "start_offset_samples": 80,\n        "duration_samples": 480,\n        "stride_samples": 480,\n        "duration_seconds": 3.0,\n        "stride_seconds": 3.0,\n        "last_window_policy": (\n            "ONE_WINDOW_PER_INCLUDED_SOURCE_EVENT"\n        ),\n        "bounds_policy": "REJECT_OUT_OF_BOUNDS",\n        "event_resampling": "MNE_POLYPHASE_JOINT_EVENTS",\n        "signal_dtype": "float32",\n        "overlap_group": "PARENT_EVENT",\n        "storage": (\n            "ADOPTED_PRIVATE_KAGGLE_DATASET_"\n            "LOSSLESS_HDF5_SUBJECT_SHARDS"\n        ),\n        "dataset_handle": handle,\n        "provider_dataset_version": version,\n        "logical_immutable_revision": 1,\n        "manifest_sha256": observed_manifest_sha,\n    }\n\n    pipeline.state["r42_core_adopted"] = True\n    pipeline.state["r42_core_manifest"] = manifest\n    pipeline.state["r42_core_pointer"] = str(\n        pointer_path.relative_to(pipeline.bundle_root)\n    )\n    pipeline.state["r26_derived_handle"] = handle\n    pipeline.state["r26_derived_pointer"] = (\n        pipeline.state["r42_core_pointer"]\n    )\n\n    adoption_report = {\n        "status": "PASS",\n        "dataset_handle": handle,\n        "provider_dataset_version": version,\n        "logical_immutable_revision": 1,\n        "manifest_sha256": observed_manifest_sha,\n        "window_count": 12_910,\n        "unique_parent_event_count": 12_910,\n        "shard_count": 172,\n        "signal_dtype": "float32",\n        "scientific_artifact_recomputed": False,\n        "core_stage14_reexecuted": False,\n        "core_hdf5_shards_reuploaded": False,\n        "compact_evidence_root": str(\n            compact_root.relative_to(pipeline.bundle_root)\n        ),\n    }\n    _atomic_json(\n        pipeline.bundle_root\n        / "reports"\n        / "phase_01"\n        / "storage"\n        / "existing_core_dataset_adoption.json",\n        adoption_report,\n    )\n    return adoption_report\n\n\ndef _r42_a4_identity(\n    runner: Any,\n) -> tuple[str, str]:\n    attempt = os.environ.get(\n        "IHARQ_EXECUTION_ATTEMPT_ID",\n        datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S"),\n    )\n    owner = os.environ.get(\n        "IHARQ_KAGGLE_USERNAME",\n        "csthv999z",\n    ).strip()\n    suffix = hashlib.sha256(\n        attempt.encode("utf-8")\n    ).hexdigest()[:8]\n    slug = (\n        f"iharq-p01-l1-a4-"\n        f"{runner.pipeline.config_id[:8]}-{suffix}"\n    )\n    if len(slug) > 50:\n        raise RuntimeError(\n            f"R42_A4_SLUG_TOO_LONG: {len(slug)}"\n        )\n    return attempt, f"{owner}/{slug}"\n\n\ndef _r42_streaming_materialize_a4(\n    runner: Any,\n) -> dict[str, Any]:\n    pipeline = runner.pipeline\n    api = _ensure_kaggle_upload_api()\n    attempt, handle = _r42_a4_identity(runner)\n    a4_config_id = _r42_a4_config_id(pipeline.config_id)\n\n    plan = list(pipeline.state["r26_subject_plan"])\n    profile_by_dataset = {\n        profile.dataset_id: profile\n        for profile in pipeline.state["profiles"]\n    }\n    label_by_dataset = {\n        row["payload"]["dataset_id"]: row\n        for row in pipeline.state["label_records"]\n    }\n    dataset_record_by_dataset = {\n        row["payload"]["dataset_id"]: row["record_id"]\n        for row in pipeline.state["dataset_records"]\n    }\n\n    fit_state = pipeline.state["r26_fit_state"]\n    fit_payload = {\n        "mean": (\n            None\n            if fit_state.mean is None\n            else fit_state.mean.tolist()\n        ),\n        "std": (\n            None\n            if fit_state.std is None\n            else fit_state.std.tolist()\n        ),\n        "source_ids": fit_state.source_ids,\n        "state_hash": fit_state.state_hash,\n    }\n\n    root = (\n        pipeline.work_root\n        / "streaming_runtime"\n        / "r42_a4_materialize"\n    )\n    if root.exists():\n        shutil.rmtree(root)\n    root.mkdir(parents=True)\n\n    pointer_root = (\n        pipeline.bundle_root\n        / "external_artifact_pointers"\n    )\n    pointer_root.mkdir(parents=True, exist_ok=True)\n\n    location_target = pointer_root / "a4_window_to_shard.jsonl"\n    records_target = pointer_root / "a4_window_records.jsonl"\n    index_target = pointer_root / "a4_window_index.jsonl"\n    groups_target = pointer_root / "a4_group_index.jsonl"\n\n    for target in (\n        location_target,\n        records_target,\n        index_target,\n        groups_target,\n    ):\n        target.unlink(missing_ok=True)\n\n    tokens: list[str] = []\n    shard_rows: list[dict[str, Any]] = []\n    a4_records: list[dict[str, Any]] = []\n    a4_index: list[dict[str, Any]] = []\n    a4_groups: list[dict[str, Any]] = []\n    quality_records: list[dict[str, Any]] = []\n    quality_summaries: list[dict[str, Any]] = []\n    invalid_windows: list[dict[str, Any]] = []\n\n    materialized_events = 0\n    logical_stored_bytes = 0\n    uploaded_bytes = 0\n    completed = 0\n    started = time.monotonic()\n    last_progress = 0.0\n    dataset_actual: dict[str, dict[str, Any]] = {}\n\n    for dataset, subject in plan:\n        _assert_subject_scratch_capacity(\n            pipeline,\n            dataset,\n            int(subject),\n        )\n\n        task_dir = (\n            root\n            / _safe(dataset)\n            / f"subject_{int(subject):03d}"\n        )\n        shard_filename = (\n            f"{_safe(dataset)}_"\n            f"subject_{int(subject):03d}_a4_full4s.h5"\n        )\n        profile = profile_by_dataset[dataset]\n        label = f"R42-A4:{dataset}:subject={subject}"\n\n        task = {\n            "action": "materialize_a4",\n            "profile": _profile_dict(profile),\n            "subject": int(subject),\n            "input_root": str(runner.input_root),\n            "child_work_root": str(task_dir / "work"),\n            "child_report_root": str(task_dir / "report"),\n            "output_dir": str(task_dir / "output"),\n            "source_resolution_file": str(\n                pipeline.state["r26_source_resolution_file"]\n            ),\n            "operations": pipeline.state["operations"],\n            "fit_state": fit_payload,\n            "assignment": pipeline.state["assignment"],\n            "split_keys": list(\n                pipeline.config["split"]["group_keys"]\n            ),\n            "label_record": label_by_dataset[dataset],\n            "preprocessing_record": (\n                pipeline.state["preprocessing_record"]\n            ),\n            "split_record": pipeline.state["split_record"],\n            "quality_profile": pipeline.config.get(\n                "quality",\n                {},\n            ),\n            "base_config_id": pipeline.config_id,\n            "a4_config_id": a4_config_id,\n            "dataset_record_id": (\n                dataset_record_by_dataset[dataset]\n            ),\n            "a4_window_family": R42_A4_WINDOW_FAMILY,\n            "shard_filename": shard_filename,\n        }\n\n        result = _run_child_task(\n            task,\n            pipeline.work_root,\n            label,\n        )\n\n        subject_records = _read_jsonl(\n            Path(result["a4_window_records_path"])\n        )\n        subject_index = _read_jsonl(\n            Path(result["a4_window_index_path"])\n        )\n        subject_groups = _read_jsonl(\n            Path(result["a4_group_index_path"])\n        )\n        subject_locations = _read_jsonl(\n            Path(result["a4_window_locations_path"])\n        )\n\n        a4_records.extend(subject_records)\n        a4_index.extend(subject_index)\n        a4_groups.extend(subject_groups)\n        quality_records.extend(\n            _read_jsonl(Path(result["quality_records_path"]))\n        )\n        quality_summaries.extend(\n            _read_jsonl(Path(result["quality_summaries_path"]))\n        )\n        invalid_windows.extend(\n            _read_jsonl(Path(result["invalid_windows_path"]))\n        )\n\n        _append_jsonl(records_target, subject_records)\n        _append_jsonl(index_target, subject_index)\n        _append_jsonl(groups_target, subject_groups)\n\n        for row in subject_locations:\n            row.update(\n                {\n                    "provider": "Kaggle",\n                    "dataset_handle": handle,\n                    "provider_dataset_version": 1,\n                    "window_family_id": (\n                        R42_A4_WINDOW_FAMILY[\n                            "window_family_id"\n                        ]\n                    ),\n                }\n            )\n        _append_jsonl(location_target, subject_locations)\n\n        materialized_events += int(\n            result["materialized_event_count"]\n        )\n        subject_logical = int(\n            result["logical_stored_bytes"]\n        )\n        logical_stored_bytes += subject_logical\n\n        shard = result.get("shard")\n        if not shard:\n            raise RuntimeError(\n                f"R42_A4_SUBJECT_SHARD_MISSING: {label}"\n            )\n\n        shard_path = Path(shard["path"])\n        token = _upload_blob_with_retry(api, shard_path)\n        tokens.append(token)\n\n        actual_bytes = int(shard["bytes"])\n        uploaded_bytes += actual_bytes\n\n        shard_row = {\n            "filename": shard["filename"],\n            "bytes": actual_bytes,\n            "sha256": shard["sha256"],\n            "dataset_id": dataset,\n            "subject_profile": int(subject),\n            "materialized_event_count": int(\n                result["materialized_event_count"]\n            ),\n            "window_record_count": int(\n                result["window_record_count"]\n            ),\n            "a4_group_count": int(\n                result["a4_group_count"]\n            ),\n            "logical_stored_bytes": subject_logical,\n            "compression_ratio_to_logical": (\n                actual_bytes / subject_logical\n                if subject_logical\n                else None\n            ),\n            "provider": "Kaggle",\n            "dataset_handle": handle,\n            "format": "HDF5",\n            "compression": "gzip-1-lossless",\n            "signal_dtype": "float32",\n            "materialized_duration_samples": 640,\n            "virtual_multi_views_per_event": 3,\n        }\n        shard_rows.append(shard_row)\n\n        dataset_row = dataset_actual.setdefault(\n            dataset,\n            {\n                "dataset_id": dataset,\n                "subjects": 0,\n                "shards": 0,\n                "materialized_events": 0,\n                "window_records": 0,\n                "logical_stored_bytes": 0,\n                "actual_hdf5_bytes": 0,\n            },\n        )\n        dataset_row["subjects"] += 1\n        dataset_row["shards"] += 1\n        dataset_row["materialized_events"] += int(\n            result["materialized_event_count"]\n        )\n        dataset_row["window_records"] += int(\n            result["window_record_count"]\n        )\n        dataset_row["logical_stored_bytes"] += (\n            subject_logical\n        )\n        dataset_row["actual_hdf5_bytes"] += actual_bytes\n\n        shard_path.unlink(missing_ok=True)\n        shutil.rmtree(task_dir, ignore_errors=True)\n        _trim_memory()\n\n        completed += 1\n        if (\n            time.monotonic() - last_progress\n            >= float(POLICY["progress_interval_seconds"])\n            or completed == len(plan)\n        ):\n            _progress_line(\n                "R42_A4_MATERIALIZE_UPLOAD_PROGRESS",\n                completed,\n                len(plan),\n                started,\n                label,\n                None,\n                pipeline.work_root,\n                {\n                    "materialized_events": materialized_events,\n                    "a4_window_records": len(a4_records),\n                    "a4_groups": len(a4_groups),\n                    "shards_uploaded": len(shard_rows),\n                    "uploaded_gib": round(\n                        uploaded_bytes / _GIB,\n                        3,\n                    ),\n                    "logical_stored_gib": round(\n                        logical_stored_bytes / _GIB,\n                        3,\n                    ),\n                },\n            )\n            last_progress = time.monotonic()\n\n    core_event_rows = {\n        str(row["event_id"]): row\n        for row in pipeline.state["window_index"]\n    }\n    expected_events = len(core_event_rows)\n    expected_records = expected_events * 4\n\n    a4_group_rows = {\n        str(row["parent_event_id"]): row\n        for row in a4_groups\n    }\n    if len(a4_group_rows) != len(a4_groups):\n        raise RuntimeError(\n            "R42_A4_DUPLICATE_PARENT_EVENT_GROUP"\n        )\n    if set(a4_group_rows) != set(core_event_rows):\n        missing = sorted(set(core_event_rows) - set(a4_group_rows))\n        unexpected = sorted(set(a4_group_rows) - set(core_event_rows))\n        raise RuntimeError(\n            "R42_A4_PARENT_EVENT_SET_MISMATCH: "\n            f"missing={missing[:20]}; unexpected={unexpected[:20]}"\n        )\n\n    for event_id, group in a4_group_rows.items():\n        core = core_event_rows[event_id]\n        comparisons = {\n            "dataset_id": (\n                str(group["dataset_id"]),\n                str(core["dataset_id"]),\n            ),\n            "subject_id": (\n                str(group["subject_id"]),\n                str(core["subject_id"]),\n            ),\n            "session_id": (\n                str(group["session_id"]),\n                str(core["session_id"]),\n            ),\n            "run_id": (\n                str(group["run_id"]),\n                str(core["run_id"]),\n            ),\n            "role": (\n                str(group["role"]),\n                str(core["role"]),\n            ),\n            "normalized_label": (\n                str(group["normalized_label"]),\n                str(core["label"]),\n            ),\n        }\n        mismatches = {\n            key: {"a4": left, "core": right}\n            for key, (left, right) in comparisons.items()\n            if left != right\n        }\n        if mismatches:\n            raise RuntimeError(\n                "R42_A4_CORE_LINEAGE_MISMATCH: "\n                f"event_id={event_id}; mismatches={mismatches}"\n            )\n\n        member_ids = list(group["multi_member_window_ids"])\n        if (\n            len(member_ids) != 3\n            or len(set(member_ids)) != 3\n            or not bool(group.get("complete"))\n        ):\n            raise RuntimeError(\n                "R42_A4_MULTI_MEMBER_CLOSURE_FAILED: "\n                f"event_id={event_id}"\n            )\n\n    component_counts = {\n        "LONGER_WINDOW": sum(\n            row.get("a4_component") == "LONGER_WINDOW"\n            for row in a4_index\n        ),\n        "MULTI_WINDOW_MEMBER": sum(\n            row.get("a4_component") == "MULTI_WINDOW_MEMBER"\n            for row in a4_index\n        ),\n    }\n    if component_counts != {\n        "LONGER_WINDOW": expected_events,\n        "MULTI_WINDOW_MEMBER": expected_events * 3,\n    }:\n        raise RuntimeError(\n            "R42_A4_COMPONENT_COUNT_MISMATCH: "\n            f"{component_counts}"\n        )\n\n    if materialized_events != expected_events:\n        raise RuntimeError(\n            "R42_A4_EVENT_COVERAGE_MISMATCH: "\n            f"expected={expected_events}; "\n            f"observed={materialized_events}"\n        )\n    if len(a4_groups) != expected_events:\n        raise RuntimeError(\n            "R42_A4_GROUP_COUNT_MISMATCH"\n        )\n    if len(a4_records) != expected_records:\n        raise RuntimeError(\n            "R42_A4_WINDOW_RECORD_COUNT_MISMATCH: "\n            f"expected={expected_records}; "\n            f"observed={len(a4_records)}"\n        )\n    if len(a4_index) != expected_records:\n        raise RuntimeError(\n            "R42_A4_WINDOW_INDEX_COUNT_MISMATCH"\n        )\n    if invalid_windows:\n        invalid_path = (\n            pipeline.bundle_root\n            / "negative_and_failed_results"\n            / "a4_invalid_windows.json"\n        )\n        _atomic_json(invalid_path, invalid_windows)\n        raise RuntimeError(\n            "R42_A4_INVALID_WINDOWS_PRESENT: "\n            f"count={len(invalid_windows)}; "\n            f"path={invalid_path}"\n        )\n\n    if len(quality_summaries) != 489:\n        raise RuntimeError(\n            "R42_A4_QUALITY_SUMMARY_COVERAGE_MISMATCH: "\n            f"{len(quality_summaries)}"\n        )\n    hard_invalid = sum(\n        int(row.get("hard_invalid", 0))\n        for row in quality_summaries\n    )\n    if hard_invalid:\n        raise RuntimeError(\n            f"R42_A4_QUALITY_HARD_INVALID: {hard_invalid}"\n        )\n\n    # Use the fresh quality evidence generated during the A4 raw-source pass.\n    pipeline.state["quality_records"] = quality_records\n    pipeline.state["quality_summaries"] = quality_summaries\n    pipeline.state["records"].extend(quality_records)\n\n    pipeline.state["r42_a4_records"] = a4_records\n    pipeline.state["r42_a4_index"] = a4_index\n    pipeline.state["r42_a4_groups"] = a4_groups\n    pipeline.state["r42_a4_shards"] = shard_rows\n    pipeline.state["r42_a4_tokens"] = tokens\n    pipeline.state["r42_a4_upload_api"] = api\n    pipeline.state["r42_a4_handle"] = handle\n    pipeline.state["r42_a4_attempt"] = attempt\n    pipeline.state["r42_a4_config_id"] = a4_config_id\n    pipeline.state["r42_a4_location_path"] = str(\n        location_target.relative_to(pipeline.bundle_root)\n    )\n    pipeline.state["r42_a4_records_path"] = str(\n        records_target.relative_to(pipeline.bundle_root)\n    )\n    pipeline.state["r42_a4_index_path"] = str(\n        index_target.relative_to(pipeline.bundle_root)\n    )\n    pipeline.state["r42_a4_groups_path"] = str(\n        groups_target.relative_to(pipeline.bundle_root)\n    )\n\n    report = {\n        "artifact_id": (\n            f"P01-L1-A4-STORAGE-ACTUAL-"\n            f"{a4_config_id[:16]}-{attempt}"\n        ),\n        "status": "PRECOMMIT_ALL_A4_SHARD_TOKENS_OBTAINED",\n        "base_scientific_freeze": (\n            POLICY["scientific_freeze_unchanged"]\n        ),\n        "a4_window_family": R42_A4_WINDOW_FAMILY,\n        "base_config_id": pipeline.config_id,\n        "a4_config_id": a4_config_id,\n        "core_dataset_handle": os.environ[\n            "IHARQ_EXISTING_CORE_DATASET_HANDLE"\n        ],\n        "core_provider_dataset_version": int(\n            os.environ["IHARQ_EXISTING_CORE_DATASET_VERSION"]\n        ),\n        "a4_dataset_handle": handle,\n        "subject_shards": len(shard_rows),\n        "materialized_full4s_events": materialized_events,\n        "a4_window_records": len(a4_records),\n        "a4_groups": len(a4_groups),\n        "virtual_multi_views": expected_events * 3,\n        "invalid_window_count": 0,\n        "signal_dtype": "float32",\n        "logical_materialized_float32_bytes": (\n            logical_stored_bytes\n        ),\n        "logical_materialized_float32_gib": round(\n            logical_stored_bytes / _GIB,\n            3,\n        ),\n        "actual_hdf5_uploaded_bytes": uploaded_bytes,\n        "actual_hdf5_uploaded_gib": round(\n            uploaded_bytes / _GIB,\n            3,\n        ),\n        "actual_compression_ratio": (\n            uploaded_bytes / logical_stored_bytes\n            if logical_stored_bytes\n            else None\n        ),\n        "overlap_storage_policy": (\n            "STORE_FULL4S_ONCE_AND_REGISTER_3X2S_VIEWS"\n        ),\n        "dataset_totals": [\n            {\n                **row,\n                "logical_stored_gib": round(\n                    row["logical_stored_bytes"] / _GIB,\n                    3,\n                ),\n                "actual_hdf5_gib": round(\n                    row["actual_hdf5_bytes"] / _GIB,\n                    3,\n                ),\n            }\n            for _, row in sorted(dataset_actual.items())\n        ],\n    }\n    report_path = (\n        pipeline.bundle_root\n        / "reports"\n        / "phase_01"\n        / "storage"\n        / "a4_derived_output_storage_actual_precommit.json"\n    )\n    _atomic_json(report_path, report)\n    pipeline.state["r42_a4_storage_report"] = report\n    pipeline.state["r42_a4_storage_report_path"] = str(\n        report_path.relative_to(pipeline.bundle_root)\n    )\n    return report\n\n\ndef _r42_poll_dataset_version(\n    handle: str,\n    minimum_version: int = 1,\n) -> int:\n    from kagglehub.clients import build_kaggle_client\n    from kagglesdk.datasets.types.dataset_api_service import (\n        ApiGetDatasetRequest,\n    )\n\n    owner, slug = handle.split("/", 1)\n    observed = 0\n    for _ in range(80):\n        try:\n            with build_kaggle_client() as client:\n                request = ApiGetDatasetRequest()\n                request.owner_slug = owner\n                request.dataset_slug = slug\n                response = (\n                    client.datasets.dataset_api_client.get_dataset(\n                        request\n                    )\n                )\n                observed = int(\n                    getattr(\n                        response,\n                        "current_version_number",\n                        0,\n                    )\n                    or 0\n                )\n                if observed >= minimum_version:\n                    return observed\n        except Exception:\n            pass\n        time.sleep(5)\n    raise RuntimeError(\n        "R42_A4_DATASET_VERSION_NOT_VISIBLE: "\n        f"handle={handle}; observed={observed}"\n    )\n\n\ndef _r42_finalize_a4_dataset(\n    runner: Any,\n) -> dict[str, Any]:\n    pipeline = runner.pipeline\n    api = pipeline.state["r42_a4_upload_api"]\n    handle = pipeline.state["r42_a4_handle"]\n    tokens = list(pipeline.state["r42_a4_tokens"])\n    shards = list(pipeline.state["r42_a4_shards"])\n    a4_records = list(pipeline.state["r42_a4_records"])\n    a4_index = list(pipeline.state["r42_a4_index"])\n    a4_groups = list(pipeline.state["r42_a4_groups"])\n    a4_config_id = pipeline.state["r42_a4_config_id"]\n\n    expected_events = 12_910\n    if len(a4_groups) != expected_events:\n        raise RuntimeError(\n            "R42_A4_FINALIZE_GROUP_COUNT_MISMATCH"\n        )\n    if len(a4_records) != expected_events * 4:\n        raise RuntimeError(\n            "R42_A4_FINALIZE_RECORD_COUNT_MISMATCH"\n        )\n    if any(\n        not bool(row.get("complete"))\n        for row in a4_groups\n    ):\n        raise RuntimeError(\n            "R42_A4_INCOMPLETE_MULTI_WINDOW_GROUP"\n        )\n\n    scratch = (\n        pipeline.work_root\n        / "streaming_runtime"\n        / "r42_a4_dataset_commit"\n    )\n    if scratch.exists():\n        shutil.rmtree(scratch)\n    scratch.mkdir(parents=True)\n\n    from iharq.layer1_data_protocol.validation import (\n        validate_records,\n    )\n\n    validation_context = (\n        list(pipeline.state["dataset_records"])\n        + list(pipeline.state["label_records"])\n        + [pipeline.state["split_record"]]\n        + [pipeline.state["preprocessing_record"]]\n        + a4_records\n    )\n    a4_validation_errors, a4_validation_report = validate_records(\n        validation_context,\n        pipeline.package_root / "schemas" / "phase_01" / "records",\n        a4_config_id,\n    )\n    validation_path = (\n        scratch\n        / "IHARQ_P01_L1_A4_RECORD_VALIDATION_REPORT.json"\n    )\n    _atomic_json(\n        validation_path,\n        {\n            "validation_scope": (\n                "A4_WINDOW_RECORDS_WITH_REQUIRED_PARENT_CONTEXT"\n            ),\n            "a4_window_record_count": len(a4_records),\n            "parent_context_record_count": (\n                len(validation_context) - len(a4_records)\n            ),\n            "errors": a4_validation_errors,\n            "report": a4_validation_report,\n        },\n    )\n    if a4_validation_errors:\n        raise RuntimeError(\n            "R42_A4_RECORD_SCHEMA_OR_LINEAGE_VALIDATION_FAILED: "\n            + json.dumps(a4_validation_errors[:20], indent=2)\n        )\n\n    source_map = {\n        "IHARQ_P01_L1_A4_WINDOW_RECORDS.jsonl": (\n            pipeline.bundle_root\n            / pipeline.state["r42_a4_records_path"]\n        ),\n        "IHARQ_P01_L1_A4_WINDOW_INDEX.jsonl": (\n            pipeline.bundle_root\n            / pipeline.state["r42_a4_index_path"]\n        ),\n        "IHARQ_P01_L1_A4_GROUP_INDEX.jsonl": (\n            pipeline.bundle_root\n            / pipeline.state["r42_a4_groups_path"]\n        ),\n        "IHARQ_P01_L1_A4_WINDOW_TO_SHARD_INDEX.jsonl": (\n            pipeline.bundle_root\n            / pipeline.state["r42_a4_location_path"]\n        ),\n        "IHARQ_P01_L1_A4_OUTPUT_STORAGE_ACTUAL.json": (\n            pipeline.bundle_root\n            / pipeline.state["r42_a4_storage_report_path"]\n        ),\n    }\n\n    compact_paths: list[Path] = [validation_path]\n    for name, source in source_map.items():\n        target = scratch / name\n        shutil.copy2(source, target)\n        compact_paths.append(target)\n\n    freeze_path = (\n        scratch\n        / "IHARQ_P01_L1_A4_WINDOW_FAMILY_FREEZE_R1.json"\n    )\n    _atomic_json(\n        freeze_path,\n        {\n            "freeze_id": (\n                R42_A4_WINDOW_FAMILY["window_family_id"]\n            ),\n            "base_scientific_freeze": (\n                POLICY["scientific_freeze_unchanged"]\n            ),\n            "base_config_id": pipeline.config_id,\n            "a4_config_id": a4_config_id,\n            "authority_class": (\n                "ADDITIVE_LAYER1_WINDOW_FAMILY_EXTENSION"\n            ),\n            "protocol_status": (\n                R42_A4_WINDOW_FAMILY["protocol_status"]\n            ),\n            "profiles": R42_A4_WINDOW_FAMILY,\n            "core_dataset_dependency": {\n                "handle": os.environ[\n                    "IHARQ_EXISTING_CORE_DATASET_HANDLE"\n                ],\n                "provider_version": int(\n                    os.environ[\n                        "IHARQ_EXISTING_CORE_DATASET_VERSION"\n                    ]\n                ),\n                "manifest_sha256": os.environ[\n                    "IHARQ_EXISTING_CORE_MANIFEST_SHA256"\n                ],\n            },\n            "mutation_rule": (\n                "THE_EXISTING_CORE_DATASET_IS_NOT_CHANGED"\n            ),\n        },\n    )\n    compact_paths.append(freeze_path)\n\n    reader_path = scratch / "iharq_a4_window_shard_reader.py"\n    reader_path.write_text(\n        R42_A4_READER_SOURCE,\n        encoding="utf-8",\n    )\n    compact_paths.append(reader_path)\n\n    sidecar_path = (\n        scratch\n        / "IHARQ_P01_L1_A4_DERIVED_DATASET_SIDECAR.json"\n    )\n    _atomic_json(\n        sidecar_path,\n        {\n            "schema_version": 1,\n            "base_scientific_freeze": (\n                POLICY["scientific_freeze_unchanged"]\n            ),\n            "window_family": R42_A4_WINDOW_FAMILY,\n            "base_config_id": pipeline.config_id,\n            "a4_config_id": a4_config_id,\n            "dataset_records": (\n                pipeline.state["dataset_records"]\n            ),\n            "label_records": pipeline.state["label_records"],\n            "split_record": pipeline.state["split_record"],\n            "preprocessing_record": (\n                pipeline.state["preprocessing_record"]\n            ),\n            "core_dataset_pointer": (\n                pipeline.state["r42_core_pointer"]\n            ),\n            "materialized_full4s_event_count": 12_910,\n            "a4_window_record_count": 51_640,\n            "a4_group_count": 12_910,\n            "materialized_arrays_per_event": 1,\n            "registered_virtual_views_per_event": 3,\n            "signal_dtype": "float32",\n            "storage_efficiency_rule": (\n                "OVERLAPPING_MULTI_WINDOW_BYTES_NOT_DUPLICATED"\n            ),\n        },\n    )\n    compact_paths.append(sidecar_path)\n\n    compact_manifest = {\n        path.name: {\n            "sha256": _sha256(path),\n            "bytes": path.stat().st_size,\n            "format": (\n                "JSONL"\n                if path.suffix == ".jsonl"\n                else "PYTHON"\n                if path.suffix == ".py"\n                else "JSON"\n            ),\n        }\n        for path in compact_paths\n    }\n\n    manifest = {\n        "artifact_id": (\n            f"P01-L1-A4-DERIVED-WINDOWS-"\n            f"{a4_config_id[:16]}-"\n            f"{pipeline.state[\'r42_a4_attempt\']}"\n        ),\n        "schema_version": 1,\n        "provider": "Kaggle",\n        "repository_or_dataset": handle,\n        "access": "PRIVATE",\n        "format": (\n            "LOSSLESS_HDF5_FULL4S_SUBJECT_SHARDS_"\n            "WITH_REGISTERED_VIRTUAL_MULTIWINDOW_VIEWS"\n        ),\n        "base_scientific_freeze": (\n            POLICY["scientific_freeze_unchanged"]\n        ),\n        "a4_window_family": R42_A4_WINDOW_FAMILY,\n        "base_config_id": pipeline.config_id,\n        "a4_config_id": a4_config_id,\n        "core_dataset_dependency": {\n            "handle": os.environ[\n                "IHARQ_EXISTING_CORE_DATASET_HANDLE"\n            ],\n            "provider_version": int(\n                os.environ[\n                    "IHARQ_EXISTING_CORE_DATASET_VERSION"\n                ]\n            ),\n            "manifest_sha256": os.environ[\n                "IHARQ_EXISTING_CORE_MANIFEST_SHA256"\n            ],\n        },\n        "source_dataset_ids": POLICY[\n            "active_sources_unchanged"\n        ],\n        "split_record_id": (\n            pipeline.state["split_record"]["record_id"]\n        ),\n        "preprocessing_record_id": (\n            pipeline.state["preprocessing_record"]["record_id"]\n        ),\n        "materialized_full4s_event_count": 12_910,\n        "longer_window_record_count": 12_910,\n        "multi_window_member_record_count": 38_730,\n        "a4_window_record_count": 51_640,\n        "a4_group_count": 12_910,\n        "shard_count": 172,\n        "signal_dtype": "float32",\n        "record_schema_id_lineage_validation": "PASS",\n        "core_parent_event_set_match": "PASS",\n        "a4_component_counts": {\n            "longer_window": 12_910,\n            "multi_window_members": 38_730,\n        },\n        "shards": shards,\n        "dataset_local_indexes": compact_manifest,\n        "local_copy_status": (\n            "A4_SHARDS_DELETED_AFTER_VERIFIED_KAGGLE_BLOB_UPLOAD"\n        ),\n        "reader_filename": reader_path.name,\n        "consumer": "PHASE_02_LAYER_02_A4",\n        "protocol_status": (\n            R42_A4_WINDOW_FAMILY["protocol_status"]\n        ),\n        "confirmatory_use_rule": (\n            "EXACT_A4_PROFILE_MUST_BE_SYNCHRONIZED_IN_PROTOCOL_V1 "\n            "BEFORE_CONFIRMATORY_CLAIMS"\n        ),\n        "scientific_scope_changed_for_core": False,\n        "core_dataset_mutated": False,\n    }\n\n    manifest_path = (\n        scratch\n        / "IHARQ_P01_L1_A4_DERIVED_WINDOW_DATASET_MANIFEST.json"\n    )\n    _atomic_json(manifest_path, manifest)\n    compact_paths.append(manifest_path)\n\n    for path in compact_paths:\n        tokens.append(_upload_blob_with_retry(api, path))\n\n    upload_dir = api["UploadDirectoryInfo"](\n        name="",\n        files=tokens,\n        directories=[],\n    )\n    api["create_dataset_or_version"](\n        api["parse_dataset_handle"](handle),\n        upload_dir,\n        (\n            "IHARQ P01 L1 A4 full-4s and registered "\n            "3x2s multi-window evidence extension"\n        ),\n    )\n\n    provider_version = _r42_poll_dataset_version(\n        handle,\n        minimum_version=1,\n    )\n\n    manifest["creation_status"] = "COMMITTED"\n    manifest["provider_dataset_version"] = provider_version\n    manifest["logical_window_family_revision"] = 1\n    manifest["manifest_filename_in_dataset"] = (\n        manifest_path.name\n    )\n    manifest["manifest_sha256"] = _sha256(manifest_path)\n\n    pointer_path = (\n        pipeline.bundle_root\n        / "external_artifact_pointers"\n        / "a4_window_family_dataset.json"\n    )\n    _atomic_json(pointer_path, manifest)\n\n    freeze_bundle_path = (\n        pipeline.bundle_root\n        / "config_snapshot"\n        / "p01_l1_a4_window_family_freeze_R1.json"\n    )\n    shutil.copy2(freeze_path, freeze_bundle_path)\n\n    amendment_path = (\n        pipeline.bundle_root\n        / "reports"\n        / "phase_01"\n        / "amendments"\n        / "P01_L1_A4_Window_Family_Amendment_R1.md"\n    )\n    amendment_path.parent.mkdir(parents=True, exist_ok=True)\n    amendment_path.write_text(\n        textwrap.dedent(\n            f"""\\\n            # P01/L1 A4 Window-Family Additive Amendment R1\n\n            ## Status\n\n            `{R42_A4_WINDOW_FAMILY["protocol_status"]}`\n\n            ## Unchanged official core\n\n            The verified core Dataset remains immutable:\n\n            - Handle: `{os.environ["IHARQ_EXISTING_CORE_DATASET_HANDLE"]}`\n            - Provider version: `{os.environ["IHARQ_EXISTING_CORE_DATASET_VERSION"]}`\n            - Manifest SHA-256:\n              `{os.environ["IHARQ_EXISTING_CORE_MANIFEST_SHA256"]}`\n            - Official core window: cue +0.5 s to +3.5 s.\n\n            ## Additive A4 data substrate\n\n            A separate Dataset stores one lossless cue +0.0 s to +4.0 s\n            tensor for every included source event. It also registers three\n            exact same-event 2-second views:\n\n            1. +0.0 s to +2.0 s\n            2. +1.0 s to +3.0 s\n            3. +2.0 s to +4.0 s\n\n            The views reference slices of the full 4-second tensor; their\n            overlapping bytes are not duplicated.\n\n            ## Governance boundary\n\n            This execution makes the data substrate available to Layer 2.\n            Confirmatory A4 claims require the exact profile and group\n            configuration to be synchronized into Protocol v1.0 and the\n            applicable Build Book. Until then, the data are valid for\n            implementation readiness and diagnostic execution, not silent\n            confirmatory promotion.\n\n            ## Dataset\n\n            - Handle: `{handle}`\n            - Provider version: `{provider_version}`\n            - A4 config ID: `{a4_config_id}`\n            """\n        ),\n        encoding="utf-8",\n    )\n\n    pipeline.state["r42_a4_pointer"] = str(\n        pointer_path.relative_to(pipeline.bundle_root)\n    )\n    pipeline.state["r42_a4_manifest"] = manifest\n    pipeline.state["r42_a4_provider_version"] = (\n        provider_version\n    )\n    pipeline.state["r42_a4_committed"] = True\n\n    # Clear only upload tokens and compact scratch.\n    pipeline.state.pop("r42_a4_tokens", None)\n    pipeline.state.pop("r42_a4_upload_api", None)\n    shutil.rmtree(scratch, ignore_errors=True)\n\n    return {\n        "status": "PASS",\n        "pointer": pipeline.state["r42_a4_pointer"],\n        "dataset_handle": handle,\n        "provider_dataset_version": provider_version,\n        "a4_config_id": a4_config_id,\n        "materialized_full4s_events": 12_910,\n        "a4_window_records": 51_640,\n        "a4_groups": 12_910,\n        "core_dataset_reused": True,\n        "core_hdf5_shards_reuploaded": False,\n    }'
r42_a4_install_wrapper_source = '# =====================================================================\n# R42 runner/pipeline integration\n# =====================================================================\n\n_R42_BASE_INSTALL_BOUNDED_STREAMING = install_bounded_streaming\n\n\ndef install_bounded_streaming(\n    runner: Any,\n    *,\n    source_resolution_file: str | Path,\n) -> dict[str, Any]:\n    installation = _R42_BASE_INSTALL_BOUNDED_STREAMING(\n        runner,\n        source_resolution_file=source_resolution_file,\n    )\n\n    pipeline = runner.pipeline\n    base_readiness_cards_manifests = (\n        pipeline.readiness_cards_manifests\n    )\n    base_p02_handoff = pipeline.p02_handoff\n    base_write_downstream_handoffs = (\n        pipeline.write_downstream_handoffs\n    )\n\n    def r42_readiness_cards_manifests(self):\n        result = base_readiness_cards_manifests()\n\n        a4_pointer = self.state.get("r42_a4_pointer")\n        if not a4_pointer:\n            raise RuntimeError(\n                "R42_A4_POINTER_MISSING_AT_READINESS"\n            )\n\n        for row in self.state.get("readiness", []):\n            if row.get("ablation_id") != "A4":\n                continue\n            row["status"] = "FOUNDATION_READY"\n            row["foundation_ready"] = True\n            row["data_substrate_status"] = (\n                "READY_WITH_PROTOCOL_SYNC_REQUIRED"\n            )\n            row["a4_components"] = {\n                "ordinary_cross_model_ensemble": {\n                    "status": (\n                        "DOWNSTREAM_MODEL_PREDICTIONS_REQUIRED"\n                    ),\n                    "data_ready": True,\n                },\n                "longer_window_full4s": {\n                    "status": (\n                        "DATA_READY_PROTOCOL_SYNC_REQUIRED"\n                    ),\n                    "window_profile_id": (\n                        "A4_LONG_FULL_4S_R1"\n                    ),\n                    "pointer": a4_pointer,\n                },\n                "same_event_multi_window_3x2s": {\n                    "status": (\n                        "DATA_READY_PROTOCOL_SYNC_REQUIRED"\n                    ),\n                    "window_profile_id": (\n                        "A4_MULTI_3X2S_R1"\n                    ),\n                    "member_count": 3,\n                    "pointer": a4_pointer,\n                },\n            }\n            row["confirmatory_use_rule"] = (\n                "PROTOCOL_V1_EXACT_PROFILE_SYNC_REQUIRED"\n            )\n            row["activated_in_p01"] = False\n            row["executed_in_p01"] = False\n\n        from iharq.layer1_data_protocol.manifests import write_json\n\n        write_json(\n            self.bundle_root\n            / "manifests"\n            / "phase_01"\n            / "layer1_ablation_readiness_l1_v1.json",\n            {\n                "rows": self.state["readiness"],\n                "a14_absence": self.state["a14"],\n                "a4_window_family_pointer": a4_pointer,\n                "a4_window_family": R42_A4_WINDOW_FAMILY,\n                "core_dataset_pointer": (\n                    self.state["r42_core_pointer"]\n                ),\n            },\n        )\n\n        for row in self.state.get("matched_key_rows", []):\n            if row.get("ablation_id") == "A4":\n                row["keys_complete"] = True\n                row["status"] = "FOUNDATION_READY"\n                row["required_keys"] += (\n                    "|a4_group_id|evidence_condition_id"\n                )\n        return result\n\n    def r42_p02_handoff(self):\n        ready = base_p02_handoff()\n        path = (\n            self.bundle_root\n            / "handoffs"\n            / "phase_01_to_phase_02.yaml"\n        )\n        import yaml\n\n        payload = yaml.safe_load(\n            path.read_text(encoding="utf-8")\n        )\n        payload["core_dataset_pointer"] = (\n            self.state["r42_core_pointer"]\n        )\n        payload["a4_window_family_pointer"] = (\n            self.state["r42_a4_pointer"]\n        )\n        payload["a4_data_readiness"] = {\n            "ordinary_ensemble": (\n                "DOWNSTREAM_MODEL_PREDICTIONS_REQUIRED"\n            ),\n            "longer_window_full4s": (\n                "DATA_READY_PROTOCOL_SYNC_REQUIRED"\n            ),\n            "same_event_multi_window_3x2s": (\n                "DATA_READY_PROTOCOL_SYNC_REQUIRED"\n            ),\n            "confirmatory_use_rule": (\n                "SYNC_EXACT_A4_PROFILE_IN_PROTOCOL_V1"\n            ),\n        }\n        payload["limitations"] = sorted(\n            set(\n                list(payload.get("limitations", []))\n                + ["A4_EXACT_PROFILE_PROTOCOL_SYNC_REQUIRED"]\n            )\n        )\n        data = yaml.safe_dump(payload, sort_keys=False)\n        path.write_text(data, encoding="utf-8")\n        (\n            self.bundle_root\n            / "phase2_handoff"\n            / "phase_01_to_phase_02.yaml"\n        ).write_text(data, encoding="utf-8")\n        self.state["p02_handoff"] = payload\n        return ready\n\n    def r42_write_downstream_handoffs(self):\n        result = base_write_downstream_handoffs()\n        import yaml\n\n        for path in (\n            self.bundle_root\n            / "handoffs"\n        ).rglob("*.yaml"):\n            try:\n                payload = yaml.safe_load(\n                    path.read_text(encoding="utf-8")\n                )\n            except Exception:\n                continue\n            if not isinstance(payload, dict):\n                continue\n            payload["core_dataset_pointer"] = (\n                self.state["r42_core_pointer"]\n            )\n            payload["a4_window_family_pointer"] = (\n                self.state["r42_a4_pointer"]\n            )\n            payload["a4_protocol_status"] = (\n                R42_A4_WINDOW_FAMILY["protocol_status"]\n            )\n            path.write_text(\n                yaml.safe_dump(payload, sort_keys=False),\n                encoding="utf-8",\n            )\n        return result\n\n    pipeline.readiness_cards_manifests = MethodType(\n        r42_readiness_cards_manifests,\n        pipeline,\n    )\n    pipeline.p02_handoff = MethodType(\n        r42_p02_handoff,\n        pipeline,\n    )\n    pipeline.write_downstream_handoffs = MethodType(\n        r42_write_downstream_handoffs,\n        pipeline,\n    )\n\n    def r42_stage_14(self):\n        try:\n            core = _r42_adopt_existing_core_dataset(self)\n            a4 = _r42_streaming_materialize_a4(self)\n            observations = {\n                "core_adoption": core,\n                "a4_precommit": a4,\n                "quality_records": len(\n                    self.pipeline.state.get(\n                        "quality_records",\n                        [],\n                    )\n                ),\n                "quality_summaries": len(\n                    self.pipeline.state.get(\n                        "quality_summaries",\n                        [],\n                    )\n                ),\n                "hard_invalid": 0,\n                "core_stage14_reexecuted": False,\n                "core_hdf5_shards_reuploaded": False,\n                "a4_materialization_executed": True,\n            }\n            return self._record(\n                "14",\n                "PASS",\n                outputs=[\n                    self.pipeline.state["r42_core_pointer"],\n                    self.pipeline.state[\n                        "r42_a4_storage_report_path"\n                    ],\n                ],\n                observations=observations,\n            )\n        except Exception as exc:\n            blocker = {\n                "code": (\n                    "P01_R42_CORE_ADOPTION_OR_A4_"\n                    "MATERIALIZATION_FAILED"\n                ),\n                "message": str(exc),\n                "owner": (\n                    "L1_A4_WINDOW_EXTENSION_OR_EXTERNAL_BYTES"\n                ),\n            }\n            self.pipeline.blockers.append(blocker)\n            return self._record(\n                "14",\n                "BLOCKED",\n                blockers=self.pipeline.blockers,\n                observations={\n                    "traceback": traceback.format_exc(),\n                    "core_stage14_reexecuted": False,\n                },\n            )\n\n    def r42_stage_15(self):\n        if not self.pipeline.state.get("r42_core_adopted"):\n            return self._record(\n                "15",\n                "BLOCKED",\n                blockers=self.pipeline.blockers,\n            )\n        if "r42_a4_tokens" not in self.pipeline.state:\n            return self._record(\n                "15",\n                "BLOCKED",\n                blockers=self.pipeline.blockers,\n            )\n        try:\n            a4 = _r42_finalize_a4_dataset(self)\n            valid = (\n                bool(\n                    self.pipeline.state.get("window_records")\n                )\n                and len(\n                    self.pipeline.state.get(\n                        "window_records",\n                        [],\n                    )\n                )\n                == 12_910\n                and bool(\n                    self.pipeline.state.get(\n                        "r42_a4_committed"\n                    )\n                )\n                and len(\n                    self.pipeline.state.get(\n                        "r42_a4_groups",\n                        [],\n                    )\n                )\n                == 12_910\n            )\n            blockers = []\n            if not valid:\n                blockers = [\n                    {\n                        "code": (\n                            "P01_R42_CORE_A4_POINTER_"\n                            "CLOSURE_FAILED"\n                        ),\n                        "owner": "L1_PERSISTENCE",\n                    }\n                ]\n                self.pipeline.blockers.extend(blockers)\n\n            return self._record(\n                "15",\n                "PASS" if valid else "FAIL",\n                outputs=[\n                    self.pipeline.state["r42_core_pointer"],\n                    self.pipeline.state["r42_a4_pointer"],\n                ],\n                observations={\n                    "core_dataset": (\n                        self.pipeline.state["window_report"]\n                    ),\n                    "a4_dataset": a4,\n                    "core_recomputed": False,\n                    "core_hdf5_shards_reuploaded": False,\n                    "a4_full4s_materialized": True,\n                    "a4_multiwindow_views_registered": True,\n                    "a4_protocol_status": (\n                        R42_A4_WINDOW_FAMILY[\n                            "protocol_status"\n                        ]\n                    ),\n                },\n                blockers=blockers,\n            )\n        except Exception as exc:\n            blocker = {\n                "code": "P01_R42_A4_DATASET_COMMIT_FAILED",\n                "message": str(exc),\n                "owner": "KAGGLE_ARTIFACT_PERSISTENCE",\n            }\n            self.pipeline.blockers.append(blocker)\n            return self._record(\n                "15",\n                "BLOCKED",\n                blockers=self.pipeline.blockers,\n                observations={"traceback": traceback.format_exc()},\n            )\n\n    runner.stage_14 = MethodType(r42_stage_14, runner)\n    runner.stage_15 = MethodType(r42_stage_15, runner)\n\n    installation["r42_core_mode"] = (\n        "ADOPT_EXISTING_VERIFIED_DATASET"\n    )\n    installation["r42_a4_window_family"] = (\n        R42_A4_WINDOW_FAMILY\n    )\n    installation["r42_core_hdf5_reupload"] = False\n\n    _atomic_json(\n        pipeline.bundle_root\n        / "reports"\n        / "phase_01"\n        / "runtime"\n        / "r42_core_reuse_a4_extension_installation.json",\n        installation,\n    )\n    return installation'
bounded_streaming_source = bounded_streaming_source.replace(
    'elif action == "materialize": result = _child_materialize(task)\n        else:',
    'elif action == "materialize": result = _child_materialize(task)\n        elif action == "materialize_a4": result = _r42_child_materialize_a4(task)\n        else:',
)
bounded_streaming_source = bounded_streaming_source.replace(
    "\ndef _child_main(request_path: str) -> int:",
    "\n" + r42_a4_child_source + "\n\ndef _child_main(request_path: str) -> int:",
)
bounded_streaming_source = bounded_streaming_source.replace(
    "\ndef _synthetic_self_test() -> dict[str, Any]:",
    "\n" + r42_a4_install_wrapper_source + "\n\ndef _synthetic_self_test() -> dict[str, Any]:",
)
# Clean Stage 15 compact scratch before any fallback write, avoiding
# append-mode duplicate JSONL rows across retries.
bounded_streaming_source = bounded_streaming_source.replace(
    'scratch = pipeline.work_root / "streaming_runtime" / "derived_dataset_commit"; scratch.mkdir(parents=True, exist_ok=True)',
    'scratch = pipeline.work_root / "streaming_runtime" / "derived_dataset_commit"\n    if scratch.exists(): shutil.rmtree(scratch)\n    scratch.mkdir(parents=True, exist_ok=True)',
)

bounded_streaming_source = bounded_streaming_source.replace(
    '    manifest_rows = []\n',
    '    a4_pointer_rel = pipeline.state.get("r42_a4_pointer")\n    if a4_pointer_rel:\n        data_readme = repository_root / "data" / "README.md"\n        with data_readme.open("a", encoding="utf-8") as stream:\n            stream.write(\n                "\\n## A4 evidence extension\\n\\n"\n                "The official 3-second core Dataset remains immutable and is "\n                "referenced by:\\n\\n"\n                "`artifacts/external_artifact_pointers/"\n                "derived_windows_dataset.json`\\n\\n"\n                "The separate A4 longer/multi-window evidence Dataset is "\n                "referenced by:\\n\\n"\n                "`artifacts/external_artifact_pointers/"\n                "a4_window_family_dataset.json`\\n\\n"\n                "The A4 Dataset stores one full 4-second tensor per event and "\n                "registers three exact 2-second views without duplicating "\n                "their overlapping numerical bytes. Confirmatory A4 use "\n                "requires exact Protocol v1.0 profile synchronization.\\n"\n            )\n\n        with (repository_root / "README.md").open(\n            "a",\n            encoding="utf-8",\n        ) as stream:\n            stream.write(\n                "\\n## A4 data readiness\\n\\n"\n                "- Existing official core: reused and not regenerated.\\n"\n                "- A4 full-4-second evidence: separate private Dataset.\\n"\n                "- A4 same-event 3x2-second members: registered immutable "\n                "views into each full-4-second tensor.\\n"\n                "- Governance status: "\n                "`DATA_READY_PROTOCOL_SYNC_REQUIRED`.\\n"\n                "- No A14 ablation is introduced.\\n"\n            )\n\n        docs_dir = repository_root / "docs"\n        docs_dir.mkdir(parents=True, exist_ok=True)\n        (docs_dir / "A4_DATA_ACCESS_AND_GOVERNANCE.md").write_text(\n            "# A4 Data Access and Governance\\n\\n"\n            "Use the two immutable Dataset pointers under "\n            "`artifacts/external_artifact_pointers/`.\\n\\n"\n            "The core Dataset contains the official cue +0.5 to +3.5 second "\n            "window family. The A4 extension contains cue +0.0 to +4.0 second "\n            "tensors and three registered 2-second views: 0-2, 1-3 and "\n            "2-4 seconds.\\n\\n"\n            "Do not reinterpret these profiles or promote them to "\n            "confirmatory A4 evidence until the exact profile is synchronized "\n            "into Protocol v1.0 and the applicable Build Book.\\n",\n            encoding="utf-8",\n        )\n\n    manifest_rows = []\n',
    1,
)

# ---------------------------------------------------------------------
# R49 complete matched A4 R2 overlay override
# ---------------------------------------------------------------------
import base64 as _r49_b64
import gzip as _r49_gzip

_R49_OVERLAY_SHA256 = "9bb6ddbb9aa7d64b78c909b5685dfdc81bf120ae3085f8fe635c12839d643185"
_R49_OVERLAY_GZIP_B64 = """..."""

bounded_streaming_source = _r49_gzip.decompress(
    _r49_b64.b64decode(
        _R49_OVERLAY_GZIP_B64.encode("ascii")
    )
).decode("utf-8")

if hashlib.sha256(
    bounded_streaming_source.encode("utf-8")
).hexdigest() != _R49_OVERLAY_SHA256:
    raise RuntimeError(
        "R49 embedded A4 R2 overlay SHA-256 mismatch."
    )

compile(
    bounded_streaming_source,
    "iharq_bounded_streaming_R49.py",
    "exec",
)

bounded_streaming_path = OVERLAY_ROOT / "iharq_bounded_streaming.py"
bounded_streaming_path.write_text(bounded_streaming_source, encoding="utf-8")
bounded_streaming_policy_path = (
    OVERLAY_ROOT / "notebook_support" / "IHARQ_P01_L1_Bounded_Streaming_Policy.json"
)
bounded_streaming_policy_path.write_text(
    json.dumps({
        "policy_id": "P01-L1-KAGGLE-DUAL-PERSISTENCE-BOUNDED-STREAMING-R3",
        "runtime_revision": "R26",
        "scientific_freeze": "P01-L1-OFFICIAL-RUN-FREEZE-R2",
        "scientific_scope_reduction": False,
        "parent_signal_arrays_retained": False,
        "disposable_subject_processes": True,
        "maximum_concurrent_subjects": 1,
        "persistent_derived_storage": "PRIVATE_KAGGLE_DATASET_LOSSLESS_HDF5_SUBJECT_SHARDS",
        "local_shards_deleted_after_verified_blob_upload": True,
        "derived_kaggle_username": "csthv999z",
        "kaggle_secret_name": "KAGGLE_API_TOKEN",
        "progress_interval_seconds": 60,
        "active_sources": ["PhysioNetMI", "BNCI2014_001", "Lee2019_MI"],
        "all_27_sections_preserved": True,
        "all_p01_gates_handoffs_preserved": True
    }, indent=2),
    encoding="utf-8",
)
acquisition_policy_path = (
    OVERLAY_ROOT
    / "notebook_support"
    / "IHARQ_P01_L1_Kaggle_Accelerated_Acquisition_Policy.json"
)
acquisition_policy_path.write_text(
    json.dumps({'policy_id': 'P01-L1-KAGGLE-THREE-SOURCE-DATASETS-R6', 'display_runtime_successor': 'R26', 'controlling_build_book': 'IHARQ-IBB-R10-P01-L1-INDEPENDENT-AUDIT-REPAIRED', 'controlling_annex': 'IHARQ-IBB-P01-L1-ANNEX-R4', 'scientific_freeze': 'P01-L1-OFFICIAL-RUN-FREEZE-R2', 'scientific_change': False, 'active_sources': ['PhysioNetMI', 'BNCI2014_001', 'Lee2019_MI'], 'dataset_order_unchanged': True, 'cross_dataset_parallelism': False, 'loading_remains_sequential': True, 'parallel_prefetch_workers': {'PhysioNetMI': 8, 'BNCI2014_001': 4, 'Lee2019_MI': 2}, 'physionet_exact_scope': {'subjects': '1-109', 'runs': [4, 8, 12], 'expected_edf_files': 327, 'baseline_runs_downloaded': False, 'fists_or_feet_imagery_runs_downloaded': False}, 'lee_exact_scope': {'subjects': '1-54', 'sessions': [1, 2], 'train_run': True, 'test_run': False}, 'quality_controls': ['MOABB 1.5.0 remains the final source-path resolver and loader', 'each subject prefetch uses a separate MOABB dataset object', 'provider bytes remain in the normal MOABB cache', 'original per-file and aggregate checksum verification remains mandatory', 'original exact per-subject provenance logic remains mandatory', 'PhysioNet subject 88 is still loaded separately by the original adapter', 'verified cache eviction remains after successful load', 'failed subjects retry with bounded backoff and remain blocking after final failure', 'all completed subjects and files receive a machine-readable acquisition ledger', 'no source, subject, session, gate, artifact, or later notebook stage is removed'], 'observability': {'aggregate_progress': True, 'subjects_completed': True, 'bytes_resolved': True, 'aggregate_rate': True, 'retry_failures': True, 'per_dataset_json_report': True, 'provider_progress_bar_noise_suppressed': True}, 'connection_contract': 'P01-L1-KAGGLE-REVISION-NEUTRAL-CONNECTION-R1'}, indent=2),
    encoding="utf-8",
)

acquisition_policy_path.write_text(
    json.dumps({'policy_id': 'P01-L1-KAGGLE-THREE-SOURCE-DATASETS-R6', 'display_runtime_successor': 'R26', 'controlling_build_book': 'IHARQ-IBB-R10-P01-L1-INDEPENDENT-AUDIT-REPAIRED', 'controlling_annex': 'IHARQ-IBB-P01-L1-ANNEX-R4', 'scientific_freeze': 'P01-L1-OFFICIAL-RUN-FREEZE-R2', 'scientific_change': False, 'scope_reduction': False, 'active_sources': ['PhysioNetMI', 'BNCI2014_001', 'Lee2019_MI'], 'physionet': {'provider': 'PhysioNet', 'version': 'eegmmidb/1.0.0', 'subjects': '1-109', 'runs': [4, 8, 12], 'expected_files': 327, 'primary_access_route': 'official public AWS S3 HTTPS endpoint', 'fallback_access_route': 'official PhysioNet HTTPS endpoint', 'file_level_workers': 20, 'resumable_partial_files': True, 'content_length_validation': True, 'etag_md5_validation_when_available': True, 'streamed_sha256_recording': True, 'original_moabb_final_resolution': True}, 'generic_subject_prefetch': {'BNCI2014_001_workers': 6, 'Lee2019_MI_workers': 4}, 'unchanged_controls': ['dataset order', 'all active datasets', 'all governed subjects and sessions', 'official provider version', 'MOABB final path resolution', 'project checksum inventory', 'exact subject provenance', 'PhysioNet subject 88 handling', 'sequential loading', 'adaptive disk telemetry', 'verified cache eviction', 'all records, artifacts, gates and handoffs'], 'connection_contract': 'P01-L1-KAGGLE-REVISION-NEUTRAL-CONNECTION-R1'}, indent=2),
    encoding="utf-8",
)

acquisition_policy_path.write_text(
    json.dumps({'policy_id': 'P01-L1-KAGGLE-THREE-SOURCE-DATASETS-R6', 'display_runtime_successor': 'R26', 'scientific_freeze': 'P01-L1-OFFICIAL-RUN-FREEZE-R2', 'cache_priority': ['verified environment cache archive', 'online exact dependency fallback', 'verified attached raw cache union', 'official PhysioNet AWS fast lane for missing files', 'official PhysioNet HTTPS fallback', 'MOABB provider route fallback for BNCI and Lee'], 'environment_cache_marker': 'IHARQ_P01_L1_ENVIRONMENT_CACHE_MANIFEST.json', 'raw_cache_marker': 'IHARQ_P01_L1_RAW_CACHE_MANIFEST.json', 'base_dataset_reupload_required': False, 'connection_contract': 'P01-L1-KAGGLE-REVISION-NEUTRAL-CONNECTION-R1', 'scientific_change': False, 'scope_reduction': False, 'network_free_when_all_caches_complete': True}, indent=2),
    encoding="utf-8",
)

# Record the overlay independently. Do not overwrite any inherited input-bundle manifest.
overlay_manifest_path = OVERLAY_ROOT / "runtime_overlay_manifest.json"
overlay_rows = []
for path in sorted(OVERLAY_ROOT.rglob("*")):
    if path.is_file() and path != overlay_manifest_path:
        overlay_rows.append({
            "path": path.relative_to(OVERLAY_ROOT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": _sha256(path),
        })
overlay_manifest = {
    "manifest_id": "IHARQ-P01-L1-KAGGLE-RUNTIME-OVERLAY-R1",
    "connection_contract": CONNECTION_CONTRACT_ID,
    "display_revision": DISPLAY_REVISION,
    "notebook_repair_id": NOTEBOOK_REPAIR_ID,
    "base_manifest_path": str(source_manifest_path.relative_to(SOURCE_ROOT)),
    "base_manifest_id": source_manifest.get("manifest_id"),
    "base_manifest_sha256": selected["manifest_sha256"],
    "base_bundle_root_name": SOURCE_ROOT.name,
    "base_archive": str(SOURCE_BUNDLE_ARCHIVE) if SOURCE_BUNDLE_ARCHIVE else None,
    "base_files_modified": [],
    "base_files_renamed": [],
    "base_files_deleted": [],
    "files": overlay_rows,
}
overlay_manifest_path.write_text(json.dumps(overlay_manifest, indent=2), encoding="utf-8")

# Recheck the inherited base after overlay construction.
post_overlay_errors = []
for row in source_manifest.get("files", source_manifest.get("entries", [])):
    rel = row.get("path") or row.get("relative_path")
    expected = row.get("sha256")
    if not rel or not expected:
        continue
    target = RUNTIME_ROOT / rel
    if not target.is_file() or _sha256(target) != expected:
        post_overlay_errors.append(rel)
if post_overlay_errors:
    raise RuntimeError("Overlay construction changed inherited base files: " + json.dumps(post_overlay_errors[:20]))

runtime_environment = {
    "environment_amendment_id": "P01-L1-KAGGLE-ENV-COMPAT-PY312-R1",
    "connection_contract": CONNECTION_CONTRACT_ID,
    "amendment_type": "KAGGLE_RUNTIME_COMPATIBILITY_AND_CONNECTION_ONLY",
    "scientific_freeze_unchanged": "P01-L1-OFFICIAL-RUN-FREEZE-R2",
    "build_book_unchanged": "IHARQ-IBB-R10-P01-L1-INDEPENDENT-AUDIT-REPAIRED",
    "annex_unchanged": "IHARQ-IBB-P01-L1-ANNEX-R4",
    "expected_python_major_minor": "3.12",
    "observed_python": platform.python_version(),
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "kaggle_environment": {
        key: (
            "REDACTED_SECRET_PRESENT_AT_RUNTIME"
            if any(
                marker in key.upper()
                for marker in (
                    "TOKEN",
                    "SECRET",
                    "PASSWORD",
                    "CREDENTIAL",
                    "API_KEY",
                    "ACCESS_KEY",
                    "PRIVATE_KEY",
                )
            )
            else value
        )
        for key, value in sorted(os.environ.items())
        if key.startswith("KAGGLE_")
    },
    "scientific_values_changed": [],
    "base_bundle_files_modified": [],
    "connection_critical_paths_revisioned": False,
}

PACKAGE_ROOT = RUNTIME_ROOT

# Deterministic runtime variables.
for key, value in {
    "PYTHONHASHSEED": "20260804",
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
}.items():
    os.environ[key] = value

# Exact project-direct package pins retained from R6. These releases provide
# Python 3.12-compatible distributions; the notebook still verifies every
# imported version and the MOABB wheel SHA-256 before source acquisition.
required = {
    "moabb": "1.5.0",
    "mne": "1.12.1",
    "numpy": "2.2.6",
    "scipy": "1.15.3",
    "pandas": "2.3.1",
    "scikit-learn": "1.7.1",
    "h5py": "3.14.0",
    "pooch": "1.8.2",
    "PyYAML": "6.0.2",
    "pydantic": "2.11.7",
    "jsonschema": "4.25.0",
    "nbformat": "5.10.4",
    "pytest": "8.4.1",
}

def _version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "MISSING"

# Kaggle kernels may preload NumPy or other compiled packages before Cell 1.
# R26 never imports the IHARQ scientific stack in the notebook kernel. It builds
# a private dependency target, verifies it in a fresh process, and runs one
# persistent StageRunner worker in that isolated child process. Sections 00-26
# communicate with the worker over a JSON-lines RPC channel.

from packaging.markers import default_environment
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from packaging.version import Version
import atexit


# Cache-first dependency bootstrap. A verified environment cache removes all
# PyPI resolution/download/install work from routine R26 runs. The R14 online
# path remains the deterministic fallback when no compatible cache is attached.
import tarfile

ONLINE_DEPS_ROOT = WORK_ROOT / "python312_deps"
DEPS_ROOT = ONLINE_DEPS_ROOT
ENVIRONMENT_CACHE_REPORT = {"status": "NOT_FOUND", "marker": "IHARQ_P01_L1_ENVIRONMENT_CACHE_MANIFEST.json"}


def _safe_extract_tar(archive_path: Path, destination: Path) -> None:
    with tarfile.open(archive_path, "r:*") as archive:
        destination_resolved = destination.resolve()
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            try:
                target.relative_to(destination_resolved)
            except ValueError as exc:
                raise RuntimeError(
                    f"UNSAFE_ENVIRONMENT_CACHE_ARCHIVE_MEMBER: {member.name}"
                ) from exc
        archive.extractall(destination)


def _environment_fingerprint_matches(manifest: dict) -> tuple[bool, list[str]]:
    failures = []
    expected_python = str(manifest.get("python_major_minor"))
    observed_python = f"{sys.version_info.major}.{sys.version_info.minor}"
    if expected_python != observed_python:
        failures.append(
            f"PYTHON_MAJOR_MINOR expected={expected_python} observed={observed_python}"
        )
    expected_machine = str(manifest.get("machine"))
    observed_machine = platform.machine()
    if expected_machine != observed_machine:
        failures.append(
            f"MACHINE expected={expected_machine} observed={observed_machine}"
        )
    expected_system = str(manifest.get("system", "Linux"))
    observed_system = platform.system()
    if expected_system != observed_system:
        failures.append(
            f"SYSTEM expected={expected_system} observed={observed_system}"
        )
    direct = manifest.get("direct_requirements")
    if direct != required:
        failures.append("DIRECT_REQUIREMENT_FREEZE_MISMATCH")
    return not failures, failures


def _discover_environment_cache() -> dict | None:
    markers = sorted(INPUT_ROOT.rglob("IHARQ_P01_L1_ENVIRONMENT_CACHE_MANIFEST.json")) if INPUT_ROOT.exists() else []
    compatible = []
    rejected = []
    for marker in markers:
        try:
            manifest = json.loads(marker.read_text(encoding="utf-8"))
            if manifest.get("cache_kind") != "IHARQ_P01_L1_ENVIRONMENT_CACHE":
                raise RuntimeError("UNEXPECTED_CACHE_KIND")
            if manifest.get("scientific_freeze") != "P01-L1-OFFICIAL-RUN-FREEZE-R2":
                raise RuntimeError("SCIENTIFIC_FREEZE_MISMATCH")
            compatible_fingerprint, failures = _environment_fingerprint_matches(manifest)
            if not compatible_fingerprint:
                rejected.append({"marker": str(marker), "reasons": failures})
                continue
            archive_row = manifest.get("archive", {})
            archive_rel = archive_row.get("path")
            archive_hash = archive_row.get("sha256")
            archive_bytes = archive_row.get("bytes")
            if not archive_rel or not archive_hash or archive_bytes is None:
                raise RuntimeError("ARCHIVE_METADATA_INCOMPLETE")
            archive_path = marker.parent / archive_rel
            if not archive_path.is_file():
                raise RuntimeError("ARCHIVE_MISSING")
            if archive_path.stat().st_size != int(archive_bytes):
                raise RuntimeError("ARCHIVE_SIZE_MISMATCH")
            observed_hash = _sha256(archive_path)
            if observed_hash != archive_hash:
                raise RuntimeError("ARCHIVE_SHA256_MISMATCH")
            compatible.append({
                "marker": marker,
                "manifest": manifest,
                "archive": archive_path,
                "archive_sha256": observed_hash,
            })
        except Exception as exc:
            rejected.append({"marker": str(marker), "reasons": [repr(exc)]})
    if not compatible:
        ENVIRONMENT_CACHE_REPORT.update({"status": "NOT_FOUND_OR_INCOMPATIBLE", "rejected": rejected})
        return None
    identities = {entry["archive_sha256"] for entry in compatible}
    if len(identities) != 1:
        raise RuntimeError(
            "MULTIPLE_DISTINCT_COMPATIBLE_ENVIRONMENT_CACHES: "
            + json.dumps([str(entry["marker"]) for entry in compatible], indent=2)
        )
    selected = compatible[0]
    ENVIRONMENT_CACHE_REPORT.update({
        "status": "SELECTED",
        "marker": str(selected["marker"]),
        "archive": str(selected["archive"]),
        "archive_sha256": selected["archive_sha256"],
        "rejected": rejected,
    })
    return selected


def _materialize_environment_cache(selected: dict) -> Path:
    extraction_root = WORK_ROOT / "environment_cache_extract"
    target = extraction_root / "python312_deps"
    marker = extraction_root / ".environment_cache_sha256"
    expected = selected["archive_sha256"]
    if target.is_dir() and marker.is_file() and marker.read_text().strip() == expected:
        ENVIRONMENT_CACHE_REPORT["materialization"] = "REUSED_EXISTING_EXTRACTION"
        return target
    if extraction_root.exists():
        shutil.rmtree(extraction_root)
    extraction_root.mkdir(parents=True)
    started = time.monotonic()
    print("[CACHE] extracting verified Python 3.12 dependency cache", flush=True)
    _safe_extract_tar(selected["archive"], extraction_root)
    if not target.is_dir():
        raise RuntimeError(
            "ENVIRONMENT_CACHE_ARCHIVE_MISSING_python312_deps_ROOT"
        )
    marker.write_text(expected, encoding="utf-8")
    ENVIRONMENT_CACHE_REPORT.update({
        "materialization": "EXTRACTED_VERIFIED_ARCHIVE",
        "target": str(target),
        "elapsed_seconds": round(time.monotonic() - started, 3),
    })
    print(
        f"[PASS] environment cache extracted in "
        f"{ENVIRONMENT_CACHE_REPORT['elapsed_seconds']}s",
        flush=True,
    )
    return target


_selected_environment_cache = _discover_environment_cache()
if _selected_environment_cache is not None:
    DEPS_ROOT = _materialize_environment_cache(_selected_environment_cache)


def _target_distribution_index(target: Path):
    distributions = {}
    for distribution in importlib_metadata.distributions(path=[str(target)]):
        raw_name = distribution.metadata.get("Name")
        if raw_name:
            distributions[canonicalize_name(raw_name)] = distribution
    return distributions

def _validate_isolated_dependency_layer(target: Path):
    distributions = _target_distribution_index(target)
    versions = {name: distribution.version for name, distribution in distributions.items()}
    direct_mismatches = {}
    for raw_name, expected_version in required.items():
        normalized = canonicalize_name(raw_name)
        observed_version = versions.get(normalized, "MISSING")
        if observed_version != expected_version:
            direct_mismatches[raw_name] = {"expected": expected_version, "observed": observed_version}
    marker_environment = default_environment()
    marker_environment["extra"] = ""
    closure_errors = []
    for owner_name, distribution in distributions.items():
        for requirement_text in distribution.requires or []:
            requirement = Requirement(requirement_text)
            if requirement.marker is not None:
                try:
                    applicable = requirement.marker.evaluate(marker_environment)
                except Exception as exc:
                    closure_errors.append({"owner": owner_name, "requirement": requirement_text, "reason": f"MARKER_EVALUATION_FAILED: {exc!r}"})
                    continue
                if not applicable:
                    continue
            dependency_name = canonicalize_name(requirement.name)
            dependency_version = versions.get(dependency_name)
            if dependency_version is None:
                closure_errors.append({"owner": owner_name, "requirement": requirement_text, "reason": "MISSING_FROM_PRIVATE_TARGET"})
                continue
            if requirement.specifier and Version(dependency_version) not in requirement.specifier:
                closure_errors.append({"owner": owner_name, "requirement": requirement_text, "observed": dependency_version, "reason": "VERSION_OUTSIDE_REQUIRED_SPECIFIER"})
    return {"distribution_count": len(distributions), "versions": versions, "direct_mismatches": direct_mismatches, "closure_errors": closure_errors, "pass": not direct_mismatches and not closure_errors}

def _run_install_command(command, label):
    completed = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if completed.returncode != 0:
        print(completed.stdout)
        raise RuntimeError(f"{label} failed with exit code {completed.returncode}")
    print(f"[PASS] {label}")
    return completed.stdout

dependency_layer_report = _validate_isolated_dependency_layer(DEPS_ROOT) if DEPS_ROOT.is_dir() else {"distribution_count": 0, "versions": {}, "direct_mismatches": {name: {"expected": version, "observed": "MISSING"} for name, version in required.items()}, "closure_errors": [], "pass": False}

if not dependency_layer_report["pass"]:
    if DEPS_ROOT != ONLINE_DEPS_ROOT:
        ENVIRONMENT_CACHE_REPORT["validation"] = {"status": "FAILED", "report": dependency_layer_report}
        ENVIRONMENT_CACHE_REPORT["status"] = "SELECTED_BUT_INVALID_FALLBACK_ONLINE"
        DEPS_ROOT = ONLINE_DEPS_ROOT
        dependency_layer_report = _validate_isolated_dependency_layer(DEPS_ROOT) if DEPS_ROOT.is_dir() else {"distribution_count": 0, "versions": {}, "direct_mismatches": {name: {"expected": version, "observed": "MISSING"} for name, version in required.items()}, "closure_errors": [], "pass": False}
    if DEPS_ROOT.exists():
        shutil.rmtree(DEPS_ROOT)
    DEPS_ROOT.mkdir(parents=True)
    wheel_dir = WORK_ROOT / "verified_wheels"
    if wheel_dir.exists():
        shutil.rmtree(wheel_dir)
    wheel_dir.mkdir(parents=True)
    _run_install_command([sys.executable, "-m", "pip", "download", "--disable-pip-version-check", "--no-deps", "--only-binary=:all:", "moabb==1.5.0", "-d", str(wheel_dir)], "download MOABB 1.5.0 wheel")
    wheels = sorted(wheel_dir.glob("moabb-1.5.0-*.whl"))
    if len(wheels) != 1:
        raise RuntimeError(f"Expected one MOABB 1.5.0 wheel, found {wheels}")
    expected_moabb = "8856067aba66fa4389f86e1fbc5cd9c5d52343538ad59193cef103a48a41c297"
    if _sha256(wheels[0]) != expected_moabb:
        raise RuntimeError("MOABB 1.5.0 wheel SHA-256 mismatch.")
    exact_direct_requirements = [f"{name}=={version}" for name, version in required.items() if name != "moabb"]
    _run_install_command([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--quiet", "--upgrade", "--ignore-installed", "--prefer-binary", "--target", str(DEPS_ROOT), str(wheels[0]), *exact_direct_requirements], "install private IHARQ dependency layer")
    dependency_layer_report = _validate_isolated_dependency_layer(DEPS_ROOT)
    if not dependency_layer_report["pass"]:
        raise RuntimeError("The private IHARQ dependency layer did not close: " + json.dumps(dependency_layer_report, indent=2)[:12000])

# R26 exact private-Kaggle derived-artifact upload stack. It is installed
# without dependency replacement so the already verified scientific stack is untouched.
upload_required = {"kagglehub": "1.0.2", "kagglesdk": "0.1.23"}
upload_observed = {}
for package, expected in upload_required.items():
    try:
        upload_observed[package] = next(
            d.version for d in importlib_metadata.distributions(path=[str(DEPS_ROOT)])
            if canonicalize_name(d.metadata.get("Name", "")) == canonicalize_name(package)
        )
    except (StopIteration, importlib_metadata.PackageNotFoundError):
        upload_observed[package] = "MISSING"
if upload_observed != upload_required:
    _run_install_command([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "--quiet", "--upgrade", "--force-reinstall", "--no-deps",
        "--target", str(DEPS_ROOT),
        "kagglehub==1.0.2", "kagglesdk==0.1.23"
    ], "install pinned private Kaggle artifact upload stack")

required_imports = ["moabb", "mne", "numpy", "scipy", "pandas", "sklearn", "h5py", "pymatreader", "mne_bids", "pyriemann", "edfio", "filelock", "pooch", "requests", "kagglehub", "kagglesdk"]
isolated_import_report = {
    "status": "DEFERRED_TO_PERSISTENT_WORKER_STAGE_01",
    "target": str(DEPS_ROOT),
    "modules": {},
    "errors": [],
    "quality_note": "The exact import-path and upload-API probes run once inside the persistent worker before Stage 01; duplicate cold subprocess imports are intentionally removed.",
}
observed = {raw_name: dependency_layer_report["versions"].get(canonicalize_name(raw_name), "MISSING") for raw_name in required}
mismatches = {name: {"expected": expected, "observed": observed[name]} for name, expected in required.items() if observed[name] != expected}
if mismatches:
    raise RuntimeError(f"Exact project-direct versions were not resolved in the private target: {mismatches}")

runtime_environment["environment_amendment_id"] = "P01-L1-KAGGLE-ENV-FREEZE-R5"
runtime_environment["execution_isolation"] = "PERSISTENT_CHILD_PROCESS"
runtime_environment["display_revision"] = DISPLAY_REVISION
runtime_environment["notebook_kernel_scientific_imports"] = False
environment_report_path = WORK_ROOT / "environment_amendment.json"
environment_report_path.write_text(json.dumps(runtime_environment, indent=2), encoding="utf-8")
dependency_report = {"installation_mode": ("VERIFIED_ATTACHED_ENVIRONMENT_CACHE" if ENVIRONMENT_CACHE_REPORT.get("status") == "SELECTED" else "PRIVATE_PIP_TARGET"), "environment_cache": ENVIRONMENT_CACHE_REPORT, "execution_mode": "PERSISTENT_CHILD_PROCESS", "target": str(DEPS_ROOT), "global_pip_check_used": False, "notebook_kernel_scientific_imports": False, "dependency_closure": dependency_layer_report, "import_probe": isolated_import_report}
dependency_report_path = WORK_ROOT / "isolated_dependency_layer.json"
dependency_report_path.write_text(json.dumps(dependency_report, indent=2), encoding="utf-8")
notebook_manifest = {"notebook_id": "P01-L1-KAGGLE-NOTEBOOK-R26", "filename": "IHARQ_Phase_01_Layer_01_Full_Bounded_Streaming_Execution_R26.ipynb", "build_book": "IHARQ-IBB-R10-P01-L1-INDEPENDENT-AUDIT-REPAIRED", "annex": "IHARQ-IBB-P01-L1-ANNEX-R4", "scientific_freeze": "P01-L1-OFFICIAL-RUN-FREEZE-R2", "environment_freeze": "P01-L1-KAGGLE-ENV-FREEZE-R5", "runtime_repair": "R26", "bounded_streaming_policy": "P01-L1-KAGGLE-DUAL-PERSISTENCE-BOUNDED-STREAMING-R3", "derived_artifact_persistence": "PRIVATE_KAGGLE_DATASET_LOSSLESS_HDF5_SUBJECT_SHARDS", "acquisition_amendment": "P01-L1-KAGGLE-THREE-SOURCE-DATASETS-R6", "cache_first": True, "environment_cache_marker": "IHARQ_P01_L1_ENVIRONMENT_CACHE_MANIFEST.json", "raw_cache_marker": "IHARQ_P01_L1_RAW_CACHE_MANIFEST.json", "acquisition_parallelism": "ONE_DISPOSABLE_SUBJECT_PROCESS", "loading_parallelism": "SUBJECT_SCOPED_SEQUENTIAL", "python_required": "3.12", "python_observed": platform.python_version(), "source_input_bundle": SOURCE_BUNDLE_NAME, "source_manifest": str(source_manifest_path.relative_to(SOURCE_ROOT)), "source_manifest_sha256": selected["manifest_sha256"], "runtime_input_bundle": RUNTIME_BUNDLE_NAME, "connection_contract": CONNECTION_CONTRACT_ID, "connection_critical_paths_revisioned": False, "sections": [f"{i:02d}" for i in range(27)], "official_execution_saved": False, "direct_run_all": True, "scientific_edit_required": False, "execution_isolation": "PERSISTENT_CHILD_PROCESS", "stage00_fastboot": "R29-LAZY-STAGE-RUNTIME-HOTFIX", "source_verification_strategy": "ONE_SCAN_BOUNDED_PARALLEL_SHA256"}
notebook_manifest_path = WORK_ROOT / "notebook_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")
worker_script = WORK_ROOT / "worker" / "iharq_stage_worker.py"
worker_script.parent.mkdir(parents=True, exist_ok=True)
worker_script.write_text('from pathlib import Path\nimport contextlib\nimport importlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError\n\nBOOT_MARKER = "__IHARQ_WORKER_BOOT__"\nRPC_MARKER = "__IHARQ_STAGE_RPC__"\nEVENT_MARKER = "__IHARQ_STAGE_EVENT__"\nREADY_MARKER = "__IHARQ_WORKER_READY__"\n\n\ndef _emit(marker, payload):\n    print(marker + json.dumps(payload, default=str, separators=(",", ":")), flush=True)\n\n\ndef _boot(step, **extra):\n    _emit(BOOT_MARKER, {"event": step, "unix": time.time(), **extra})\n\n\npackage_root = Path(sys.argv[1])\nwork_root = Path(sys.argv[2])\ninput_root = Path(sys.argv[3])\nconfig_path = Path(sys.argv[4])\nnotebook_manifest_path = Path(sys.argv[5])\nenvironment_report_path = Path(sys.argv[6])\ndependency_report_path = Path(sys.argv[7])\n\noverlay_root = Path(os.environ["IHARQ_OVERLAY_ROOT"])\ndeps_root = Path(os.environ["IHARQ_DEPS_ROOT"]).resolve()\nrequired_imports = json.loads(os.environ.get("IHARQ_REQUIRED_IMPORTS_JSON", "[]"))\n\n_boot("IMPORT_STAGE_RUNNER_STARTED")\nfrom iharq.layer1_data_protocol.kaggle_adapter import StageRunner\n_boot("IMPORT_STAGE_RUNNER_COMPLETED")\n\n_boot("RUNNER_CONSTRUCTION_STARTED")\nrunner = StageRunner(package_root, work_root, input_root, config_path)\n_boot("RUNNER_CONSTRUCTION_COMPLETED", bundle_root=str(runner.pipeline.bundle_root))\n\n_boot("RESOURCE_POLICY_INSTALL_STARTED")\nfrom iharq_resource_policy_adaptive import (\n    install_resource_policy, disk_snapshot, emergency_floor_bytes, write_emergency_record\n)\nresource_policy_installation = install_resource_policy(runner)\n_boot("RESOURCE_POLICY_INSTALL_COMPLETED")\n\nenvironment_validation = None\nacquisition_acceleration_installation = None\nbounded_streaming_installation = None\n\n\ndef _copy_runtime_evidence():\n    sources = [\n        (notebook_manifest_path, "notebook_manifest.json"),\n        (environment_report_path, "environment_amendment.json"),\n        (dependency_report_path, "isolated_dependency_layer.json"),\n        (overlay_root / "notebook_support" / "IHARQ_P01_L1_Kaggle_Adaptive_Disk_Policy.json", "adaptive_disk_policy_input.json"),\n        (overlay_root / "notebook_support" / "IHARQ_P01_L1_Revision_Neutral_Connection_Contract.json", "revision_neutral_connection_contract.json"),\n        (overlay_root / "notebook_support" / "IHARQ_P01_L1_Kaggle_Accelerated_Acquisition_Policy.json", "accelerated_acquisition_policy_input.json"),\n        (overlay_root / "iharq_acquisition_acceleration.py", "acquisition_acceleration_runtime.py"),\n        (overlay_root / "iharq_bounded_streaming.py", "bounded_streaming_runtime.py"),\n        (overlay_root / "notebook_support" / "IHARQ_P01_L1_Bounded_Streaming_Policy.json", "bounded_streaming_policy_input.json"),\n        (overlay_root / "runtime_overlay_manifest.json", "runtime_overlay_manifest.json"),\n    ]\n    for source_path, destination_name in sources:\n        destination = runner.pipeline.bundle_root / destination_name\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        destination.write_bytes(source_path.read_bytes())\n\n\n_copy_runtime_evidence()\n_boot("BASE_RUNTIME_EVIDENCE_COPIED")\n\n\ndef _validate_environment_once():\n    global environment_validation\n    if environment_validation is not None:\n        return environment_validation\n    started = time.monotonic()\n    report = {"target": str(deps_root), "modules": {}, "errors": []}\n    for name in required_imports:\n        try:\n            module = importlib.import_module(name)\n            module_file = getattr(module, "__file__", None)\n            resolved = str(Path(module_file).resolve()) if module_file is not None else None\n            under_target = resolved is None or Path(resolved) == deps_root or deps_root in Path(resolved).parents\n            report["modules"][name] = {"file": resolved, "under_target": under_target}\n            if not under_target:\n                report["errors"].append({"module": name, "reason": "IMPORTED_OUTSIDE_PRIVATE_TARGET", "file": resolved})\n        except Exception as exc:\n            report["errors"].append({"module": name, "reason": "IMPORT_FAILED", "error": repr(exc)})\n    try:\n        from kagglesdk.kaggle_env import get_web_endpoint\n        from kagglehub.datasets_helpers import create_dataset_or_version\n        from kagglehub.gcs_upload import UploadDirectoryInfo, _upload_blob\n        from kagglehub.handle import parse_dataset_handle\n        from kagglesdk.blobs.types.blob_api_service import ApiBlobType\n        report["upload_api_probe"] = "PASS"\n    except Exception as exc:\n        report["upload_api_probe"] = "FAIL"\n        report["errors"].append({"module": "kaggle_upload_api", "reason": "IMPORT_FAILED", "error": repr(exc)})\n    report["status"] = "PASS" if not report["errors"] else "FAIL"\n    report["elapsed_seconds"] = round(time.monotonic() - started, 3)\n    dependency = json.loads(dependency_report_path.read_text(encoding="utf-8"))\n    dependency["import_probe"] = report\n    dependency["import_probe_execution"] = "PERSISTENT_WORKER_SINGLE_IMPORT_PASS"\n    temporary = dependency_report_path.with_suffix(dependency_report_path.suffix + ".tmp")\n    temporary.write_text(json.dumps(dependency, indent=2, default=str) + "\\n", encoding="utf-8")\n    temporary.replace(dependency_report_path)\n    destination = runner.pipeline.bundle_root / "isolated_dependency_layer.json"\n    destination.write_bytes(dependency_report_path.read_bytes())\n    if report["errors"]:\n        raise RuntimeError("The private IHARQ import/upload API validation failed: " + json.dumps(report["errors"], indent=2))\n    environment_validation = report\n    return report\n\n\ndef _ensure_stage_runtime(stage):\n    global acquisition_acceleration_installation, bounded_streaming_installation\n    number = int(stage)\n    # Stage 01 is the environment stage. Import validation belongs here, not\n    # on Stage 00\'s critical path.\n    if number >= 1:\n        _validate_environment_once()\n    # Source discovery and full physical-byte SHA-256 verification are first\n    # needed by Stage 05. Deferring them makes Stage 00 genuinely fast while\n    # preserving verification before any source is resolved or consumed.\n    if number >= 5 and acquisition_acceleration_installation is None:\n        from iharq_acquisition_acceleration import install_acquisition_acceleration\n        acquisition_acceleration_installation = install_acquisition_acceleration(runner)\n    # Bounded streaming modifies Stage 07+ only and is installed just in time.\n    if number >= 7 and bounded_streaming_installation is None:\n        from iharq_acquisition_acceleration import source_resolution_file\n        from iharq_bounded_streaming import install_bounded_streaming\n        bounded_streaming_installation = install_bounded_streaming(\n            runner, source_resolution_file=source_resolution_file()\n        )\n\n\ndef _process_snapshot():\n    def run_ps(arguments):\n        try:\n            completed = subprocess.run(\n                arguments,\n                text=True,\n                stdout=subprocess.PIPE,\n                stderr=subprocess.STDOUT,\n                timeout=10,\n            )\n            return completed.stdout.strip()\n        except Exception as exc:\n            return f"PROCESS_SNAPSHOT_FAILED: {exc!r}"\n    return {\n        "worker": run_ps([\n            "ps", "-o", "pid=,ppid=,stat=,etime=,%cpu=,%mem=,wchan=,cmd=",\n            "-p", str(os.getpid()),\n        ]),\n        "children": run_ps([\n            "ps", "-o", "pid=,ppid=,stat=,etime=,%cpu=,%mem=,wchan=,cmd=",\n            "--ppid", str(os.getpid()),\n        ]),\n    }\n\n\ndef _execute_stage(stage, log_path):\n    # Truncate the current attempt\'s log. Old worker attempts remain available\n    # in their own persisted evidence, but cannot contaminate the live excerpt.\n    with log_path.open("w", encoding="utf-8", buffering=1) as log_stream:\n        with contextlib.redirect_stdout(log_stream), contextlib.redirect_stderr(log_stream):\n            print(json.dumps({\n                "event": "STAGE_WORK_STARTED",\n                "stage": stage,\n                "worker_pid": os.getpid(),\n                "started_unix": time.time(),\n            }), flush=True)\n            setup_started = time.monotonic()\n            _ensure_stage_runtime(stage)\n            print(json.dumps({\n                "event": "STAGE_RUNTIME_READY",\n                "stage": stage,\n                "setup_elapsed_seconds": round(time.monotonic() - setup_started, 3),\n            }), flush=True)\n            return runner.run_stage(stage)\n\n\n_emit(READY_MARKER, {\n    "status": "READY",\n    "worker_pid": os.getpid(),\n    "bundle_root": str(runner.pipeline.bundle_root),\n    "stage00_critical_path": "NO_SOURCE_HASHING_NO_BOUNDED_STREAMING_NO_DUPLICATE_IMPORT_PROBE",\n})\n\nfor raw_line in sys.stdin:\n    raw_line = raw_line.strip()\n    if not raw_line:\n        continue\n    try:\n        command = json.loads(raw_line)\n        stage = str(command["stage"]).zfill(2)\n        heartbeat_seconds = max(2.0, float(command.get("heartbeat_seconds", 60.0)))\n    except Exception as exc:\n        _emit(RPC_MARKER, {"ok": False, "stage": None, "error": f"INVALID_COMMAND: {exc!r}"})\n        continue\n\n    log_dir = runner.pipeline.bundle_root / "reports" / "phase_01" / "worker_logs"\n    log_dir.mkdir(parents=True, exist_ok=True)\n    log_path = log_dir / f"stage_{stage}.log"\n    started_monotonic = time.monotonic()\n    _emit(EVENT_MARKER, {\n        "event": "STARTED",\n        "stage": stage,\n        "worker_pid": os.getpid(),\n        "log_path": str(log_path),\n        "heartbeat_seconds": heartbeat_seconds,\n    })\n\n    pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix=f"iharq-stage-{stage}")\n    future = pool.submit(_execute_stage, stage, log_path)\n    try:\n        while True:\n            try:\n                result = future.result(timeout=heartbeat_seconds)\n                break\n            except FutureTimeoutError:\n                try:\n                    stat = log_path.stat()\n                    log_bytes = stat.st_size\n                    log_mtime = stat.st_mtime\n                except FileNotFoundError:\n                    log_bytes = 0\n                    log_mtime = None\n                resource_snapshot = disk_snapshot(work_root, runner.pipeline.bundle_root, f"STAGE_{stage}_HEARTBEAT")\n                if resource_snapshot["disk_free_bytes"] < emergency_floor_bytes():\n                    emergency_path = write_emergency_record(\n                        work_root, runner.pipeline.bundle_root, stage, resource_snapshot\n                    )\n                    _emit(EVENT_MARKER, {\n                        "event": "RESOURCE_EMERGENCY",\n                        "stage": stage,\n                        "worker_pid": os.getpid(),\n                        "resource_snapshot": resource_snapshot,\n                        "emergency_record": str(emergency_path),\n                    })\n                    os._exit(88)\n                _emit(EVENT_MARKER, {\n                    "event": "HEARTBEAT",\n                    "stage": stage,\n                    "worker_pid": os.getpid(),\n                    "elapsed_seconds": round(time.monotonic() - started_monotonic, 1),\n                    "log_path": str(log_path),\n                    "log_bytes": log_bytes,\n                    "log_mtime": log_mtime,\n                    "resource_snapshot": resource_snapshot,\n                    "process_snapshot": _process_snapshot(),\n                })\n        serializable_result = dict(result.__dict__) if hasattr(result, "__dict__") else result\n        payload = {\n            "ok": True,\n            "stage": stage,\n            "result": serializable_result,\n            "log_path": str(log_path),\n            "elapsed_seconds": round(time.monotonic() - started_monotonic, 3),\n        }\n    except Exception as exc:\n        payload = {\n            "ok": False,\n            "stage": stage,\n            "error": repr(exc),\n            "traceback": traceback.format_exc(),\n            "log_path": str(log_path),\n            "elapsed_seconds": round(time.monotonic() - started_monotonic, 3),\n            "process_snapshot": _process_snapshot(),\n        }\n    finally:\n        pool.shutdown(wait=True, cancel_futures=False)\n    _emit(RPC_MARKER, payload)\n', encoding="utf-8")
STAGE_TIMEOUT_SECONDS = {
    "00": 20 * 60, "01": 30 * 60, "02": 20 * 60, "03": 30 * 60,
    "04": 60 * 60, "05": 45 * 60, "06": 45 * 60, "07": 8 * 60 * 60,
    "08": 2 * 60 * 60, "09": 2 * 60 * 60, "10": 2 * 60 * 60,
    "11": 2 * 60 * 60, "12": 60 * 60, "13": 8 * 60 * 60,
    "14": 4 * 60 * 60, "15": 4 * 60 * 60, "16": 2 * 60 * 60,
    "17": 2 * 60 * 60, "18": 60 * 60, "19": 60 * 60,
    "20": 60 * 60, "21": 60 * 60, "22": 60 * 60,
    "23": 60 * 60, "24": 60 * 60, "25": 2 * 60 * 60,
    "26": 2 * 60 * 60,
}
WORKER_HEARTBEAT_SECONDS = 120.0
LOCAL_STATUS_SECONDS = 120.0
LOG_POLL_SECONDS = 120.0
RPC_MARKER = "__IHARQ_STAGE_RPC__"
EVENT_MARKER = "__IHARQ_STAGE_EVENT__"
BOOT_MARKER = "__IHARQ_WORKER_BOOT__"
READY_MARKER = "__IHARQ_WORKER_READY__"

import atexit
import queue
import threading
import time

# Construct the persistent worker environment before adding IHARQ-specific
# variables. The R28 fast-boot refactor removed the old subprocess import
# probe together with the block that originally created worker_env. Keeping
# this lightweight environment construction restores correct ordering without
# restoring the slow duplicate probe.
worker_env = os.environ.copy()
worker_env["PYTHONNOUSERSITE"] = "1"
worker_env["PYTHONPATH"] = os.pathsep.join([
    str(DEPS_ROOT),
    str(PACKAGE_ROOT / "src"),
    str(OVERLAY_ROOT),
])
worker_env["IHARQ_OVERLAY_ROOT"] = str(OVERLAY_ROOT)
worker_env["IHARQ_DEPS_ROOT"] = str(DEPS_ROOT)
worker_env["IHARQ_REQUIRED_IMPORTS_JSON"] = json.dumps(required_imports)
worker_env["IHARQ_STAGE00_FASTBOOT"] = "R35-LAZY-STAGE-RUNTIME-STAGE11-CORRECTED"
worker_env["IHARQ_NOTEBOOK_REVISION"] = DISPLAY_REVISION
worker_env["IHARQ_MOABB_SUBJECT_KEY_COMPAT"] = "R35"
worker_env["IHARQ_EXECUTION_ATTEMPT_ID"] = (
    datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S") + "-" + hashlib.sha256(os.urandom(32)).hexdigest()[:8]
)
os.environ["IHARQ_EXECUTION_ATTEMPT_ID"] = worker_env["IHARQ_EXECUTION_ATTEMPT_ID"]
worker_env["IHARQ_CONNECTION_CONTRACT"] = CONNECTION_CONTRACT_ID

worker_env["IHARQ_KAGGLE_USERNAME"] = os.environ[
    "IHARQ_KAGGLE_USERNAME"
]
worker_env["IHARQ_EXISTING_CORE_DATASET_HANDLE"] = os.environ[
    "IHARQ_EXISTING_CORE_DATASET_HANDLE"
]
worker_env["IHARQ_EXISTING_CORE_DATASET_VERSION"] = os.environ[
    "IHARQ_EXISTING_CORE_DATASET_VERSION"
]
worker_env["IHARQ_EXISTING_CORE_MANIFEST_SHA256"] = os.environ[
    "IHARQ_EXISTING_CORE_MANIFEST_SHA256"
]
worker_env["IHARQ_CORE_DATASET_MODE"] = os.environ[
    "IHARQ_CORE_DATASET_MODE"
]
worker_env["IHARQ_ENABLE_A4_EXTENSION"] = os.environ[
    "IHARQ_ENABLE_A4_EXTENSION"
]
worker_env["IHARQ_A4_WINDOW_FAMILY_ID"] = os.environ[
    "IHARQ_A4_WINDOW_FAMILY_ID"
]
worker_env["IHARQ_A4_PROTOCOL_STATUS"] = os.environ[
    "IHARQ_A4_PROTOCOL_STATUS"
]


worker_process = subprocess.Popen(
    [
        sys.executable, "-u", str(worker_script), str(PACKAGE_ROOT),
        str(WORK_ROOT), str(INPUT_ROOT), str(CONFIG_PATH),
        str(notebook_manifest_path), str(environment_report_path),
        str(dependency_report_path),
    ],
    env=worker_env,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
worker_output_queue = queue.Queue()


def _worker_stdout_reader():
    if worker_process.stdout is None:
        worker_output_queue.put({"kind": "EOF", "line": "worker stdout unavailable"})
        return
    try:
        for line in worker_process.stdout:
            worker_output_queue.put({"kind": "LINE", "line": line.rstrip("\n")})
    finally:
        worker_output_queue.put({"kind": "EOF", "line": "worker stdout closed"})


worker_reader_thread = threading.Thread(
    target=_worker_stdout_reader,
    name="iharq-worker-stdout-reader",
    daemon=True,
)
worker_reader_thread.start()


def _process_snapshot(pid):
    commands = [
        ["ps", "-o", "pid=,ppid=,stat=,etime=,%cpu=,%mem=,wchan=,cmd=", "-p", str(pid)],
        ["ps", "-o", "pid=,ppid=,stat=,etime=,%cpu=,%mem=,wchan=,cmd=", "--ppid", str(pid)],
    ]
    outputs = []
    for command in commands:
        try:
            completed = subprocess.run(
                command, text=True, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, timeout=10,
            )
            if completed.stdout.strip():
                outputs.append(completed.stdout.strip())
        except Exception as exc:
            outputs.append(f"PROCESS_SNAPSHOT_FAILED: {exc!r}")
    return "\n".join(outputs) or "[no process rows returned]"


def _tail_log(log_path, offset, max_bytes=128_000):
    if log_path is None:
        return offset, ""
    path = Path(log_path)
    if not path.is_file():
        return offset, ""
    size = path.stat().st_size
    if offset > size:
        offset = 0
    start = offset if size - offset <= max_bytes else size - max_bytes
    with path.open("rb") as stream:
        stream.seek(start)
        data = stream.read()
    return size, data.decode("utf-8", errors="replace")


def _stop_worker(reason):
    if worker_process.poll() is not None:
        return
    print(f"[WORKER STOP] {reason}")
    try:
        worker_process.terminate()
        worker_process.wait(timeout=15)
    except Exception:
        try:
            worker_process.kill()
            worker_process.wait(timeout=10)
        except Exception:
            pass


def show_worker_status():
    print(json.dumps({
        "worker_pid": worker_process.pid,
        "return_code": worker_process.poll(),
        "reader_thread_alive": worker_reader_thread.is_alive(),
    }, indent=2))
    print(_process_snapshot(worker_process.pid))


def _resolved_timeout(stage, explicit_timeout):
    if explicit_timeout is not None:
        return float(explicit_timeout)
    env_key = f"IHARQ_STAGE_TIMEOUT_{stage}_SECONDS"
    if os.environ.get(env_key):
        return float(os.environ[env_key])
    return float(STAGE_TIMEOUT_SECONDS.get(stage, 2 * 60 * 60))



MAX_CONSOLE_LOG_CHARS = 12000
MAX_CONSOLE_LOG_LINES = 80


def _bounded_console_text(text: str) -> str:
    """Bound visible output; full worker logs remain persisted on disk."""
    if not text:
        return ""

    lines = text.splitlines()
    truncated = False

    if len(lines) > MAX_CONSOLE_LOG_LINES:
        lines = lines[-MAX_CONSOLE_LOG_LINES:]
        truncated = True

    result = "\n".join(lines)

    if len(result) > MAX_CONSOLE_LOG_CHARS:
        result = result[-MAX_CONSOLE_LOG_CHARS:]
        truncated = True

    if truncated:
        result = (
            "[VISIBLE EXCERPT TRUNCATED — full text remains in the worker log]\n"
            + result
        )

    return result


def _compact_result_value(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, dict):
        preferred = (
            "status",
            "count",
            "record_count",
            "file_count",
            "checksum_status",
            "aggregate_sha256",
            "recordings",
            "dataset_records",
            "label_records",
            "quality_records",
            "quality_summaries",
            "window_count",
            "windows_materialized",
            "shards_uploaded",
            "uploaded_bytes",
            "derived_dataset_handle",
            "execution_attempt_id",
            "terminal_decision",
            "decision",
            "pointer",
            "split_record",
            "preprocessing_record",
        )
        compact = {
            key: _compact_result_value(value[key])
            for key in preferred
            if key in value
        }
        if compact:
            return compact
        return {
            "key_count": len(value),
            "keys": sorted(str(key) for key in value)[:30],
        }

    if isinstance(value, (list, tuple, set)):
        sequence = list(value)
        return {
            "count": len(sequence),
            "sample": [
                _compact_result_value(item)
                for item in sequence[:3]
            ],
        }

    return str(value)


def _compact_stage_result(result: dict) -> dict:
    observations = result.get("observations", {}) or {}
    blockers = result.get("blockers", []) or []

    summary = {
        "stage": str(result.get("stage", "")).zfill(2),
        "status": result.get("status"),
        "output_count": len(result.get("outputs", []) or []),
        "blocker_count": len(blockers),
    }

    outputs = result.get("outputs", []) or []
    if outputs:
        summary["outputs"] = [str(value) for value in outputs[:10]]

    if observations:
        summary["observations"] = _compact_result_value(observations)

    if blockers:
        summary["blockers"] = [
            {
                "code": blocker.get("code"),
                "owner": blocker.get("owner"),
                "message": str(blocker.get("message", ""))[:500],
            }
            for blocker in blockers[:10]
        ]

    return summary



_worker_ready = False
_worker_boot_report = []


def _wait_for_worker_ready(timeout_seconds=15 * 60):
    global _worker_ready, _worker_boot_report, dependency_report
    if _worker_ready:
        return
    started = time.monotonic()
    transcript = []
    print(f"[WORKER BOOT] PID={worker_process.pid}; Stage 00 has not been submitted yet.")
    while True:
        if time.monotonic() - started > float(timeout_seconds):
            _stop_worker("worker boot timeout")
            raise TimeoutError(
                "The isolated worker did not become ready within the boot timeout.\n"
                + _process_snapshot(worker_process.pid)
                + "\nRecent boot transcript:\n"
                + _bounded_console_text("\n".join(transcript[-80:]))
            )
        if worker_process.poll() is not None:
            raise RuntimeError(
                f"The isolated worker terminated during boot; return code={worker_process.poll()}.\n"
                + _bounded_console_text("\n".join(transcript[-80:]))
            )
        try:
            item = worker_output_queue.get(timeout=1.0)
        except queue.Empty:
            continue
        if item.get("kind") == "EOF":
            transcript.append(item.get("line", "worker EOF"))
            continue
        line = item.get("line", "")
        transcript.append(line)
        if line.startswith(BOOT_MARKER):
            event = json.loads(line[len(BOOT_MARKER):])
            _worker_boot_report.append(event)
            print(json.dumps({"event": "WORKER_BOOT_PROGRESS", **event}, separators=(",", ":")))
        elif line.startswith(READY_MARKER):
            ready = json.loads(line[len(READY_MARKER):])
            _worker_ready = True
            print(json.dumps({"event": "WORKER_READY", **ready}, indent=2))
            if dependency_report_path.is_file():
                dependency_report = json.loads(dependency_report_path.read_text(encoding="utf-8"))
            return
        else:
            print("[WORKER BOOT] " + _bounded_console_text(line))


def run_remote_stage(
    stage: str,
    *,
    timeout_seconds=None,
    heartbeat_seconds=WORKER_HEARTBEAT_SECONDS,
):
    stage = str(stage).zfill(2)
    timeout_seconds = _resolved_timeout(stage, timeout_seconds)

    if not _worker_ready:
        _wait_for_worker_ready()

    if worker_process.poll() is not None:
        raise RuntimeError(
            f"The isolated worker is not running before stage {stage}; "
            f"return code={worker_process.poll()}. Restart and use Run All."
        )

    if worker_process.stdin is None:
        raise RuntimeError("The worker stdin channel is unavailable.")

    while True:
        try:
            pending = worker_output_queue.get_nowait()
        except queue.Empty:
            break

        if pending.get("kind") == "LINE" and pending.get("line"):
            print(
                "[WORKER PRE-STAGE] "
                + _bounded_console_text(pending["line"])
            )

    worker_process.stdin.write(
        json.dumps(
            {
                "stage": stage,
                "heartbeat_seconds": float(heartbeat_seconds),
            }
        )
        + "\n"
    )
    worker_process.stdin.flush()

    started = time.monotonic()
    last_worker_event = started
    last_local_status = started
    last_log_poll = started
    log_path = None
    log_offset = 0
    transcript = []
    payload = None

    print(
        f"[STAGE {stage}] submitted to isolated worker "
        f"PID={worker_process.pid}; timeout={timeout_seconds:.0f}s"
    )

    try:
        while payload is None:
            now = time.monotonic()
            elapsed = now - started

            if elapsed > timeout_seconds:
                log_offset, log_text = _tail_log(log_path, log_offset)
                diagnostic = _process_snapshot(worker_process.pid)
                _stop_worker(
                    f"stage {stage} exceeded {timeout_seconds:.0f}s"
                )
                raise TimeoutError(
                    f"Stage {stage} exceeded {timeout_seconds:.0f} seconds.\n"
                    f"Worker/process snapshot:\n{diagnostic}\n"
                    "Latest stage-log excerpt:\n"
                    + _bounded_console_text(log_text)
                )

            if worker_process.poll() is not None:
                log_offset, log_text = _tail_log(log_path, log_offset)
                joined = "\n".join(transcript[-40:])
                raise RuntimeError(
                    f"Worker terminated during stage {stage}; "
                    f"return code={worker_process.poll()}.\n"
                    "Recent worker transcript:\n"
                    + _bounded_console_text(joined)
                    + "\nLatest stage-log excerpt:\n"
                    + _bounded_console_text(log_text)
                )

            try:
                item = worker_output_queue.get(timeout=1.0)
            except queue.Empty:
                item = None

            if item is not None:
                if item.get("kind") == "EOF":
                    transcript.append(item.get("line", "worker EOF"))
                else:
                    line = item.get("line", "")

                    if line.startswith(EVENT_MARKER):
                        event = json.loads(line[len(EVENT_MARKER):])
                        last_worker_event = time.monotonic()
                        log_path = event.get("log_path") or log_path

                        if event.get("event") == "STARTED":
                            print(
                                f"[STAGE {stage}] worker started; "
                                f"log={log_path}"
                            )

                        elif event.get("event") == "HEARTBEAT":
                            log_offset, log_text = _tail_log(
                                log_path,
                                log_offset,
                            )
                            excerpt = _bounded_console_text(log_text)

                            if excerpt:
                                print(
                                    f"[STAGE {stage} LIVE LOG]\n{excerpt}"
                                )

                            resource = (
                                event.get("resource_snapshot", {})
                                or {}
                            )
                            process = (
                                event.get("process_snapshot", {})
                                or {}
                            )

                            print(
                                json.dumps(
                                    {
                                        "event": "ONE_MINUTE_STAGE_STATUS",
                                        "stage": stage,
                                        "elapsed_seconds": event.get(
                                            "elapsed_seconds"
                                        ),
                                        "worker_pid": event.get(
                                            "worker_pid"
                                        ),
                                        "worker_alive": (
                                            worker_process.poll() is None
                                        ),
                                        "log_bytes": event.get(
                                            "log_bytes"
                                        ),
                                        "disk_free_gib": resource.get(
                                            "disk_free_gib"
                                        ),
                                        "source_cache_gib": round(
                                            resource.get(
                                                "source_cache_bytes",
                                                0,
                                            )
                                            / (1024**3),
                                            3,
                                        ),
                                        "bundle_gib": round(
                                            resource.get(
                                                "bundle_bytes",
                                                0,
                                            )
                                            / (1024**3),
                                            3,
                                        ),
                                        "worker_process": process.get(
                                            "worker",
                                            "",
                                        )[:800],
                                    },
                                    separators=(",", ":"),
                                )
                            )

                    elif line.startswith(RPC_MARKER):
                        payload = json.loads(line[len(RPC_MARKER):])

                    else:
                        transcript.append(line)
                        if line:
                            print(
                                "[WORKER] "
                                + _bounded_console_text(line)
                            )

            now = time.monotonic()

            if (
                log_path is not None
                and now - last_log_poll >= LOG_POLL_SECONDS
            ):
                log_offset, log_text = _tail_log(log_path, log_offset)
                excerpt = _bounded_console_text(log_text)
                if excerpt:
                    print(f"[STAGE {stage} LIVE LOG]\n{excerpt}")
                last_log_poll = now

            # This fallback appears only when no worker heartbeat has arrived
            # during the same interval; it does not double-print routine status.
            if (
                now - last_local_status >= LOCAL_STATUS_SECONDS
                and now - last_worker_event >= LOCAL_STATUS_SECONDS
            ):
                print(
                    json.dumps(
                        {
                            "event": "LOCAL_STAGE_WAIT_STATUS",
                            "stage": stage,
                            "elapsed_seconds": round(now - started, 1),
                            "seconds_since_worker_event": round(
                                now - last_worker_event,
                                1,
                            ),
                            "worker_alive": worker_process.poll() is None,
                        },
                        separators=(",", ":"),
                    )
                )
                last_local_status = now

        log_offset, final_log_text = _tail_log(log_path, log_offset)
        final_excerpt = _bounded_console_text(final_log_text)
        if final_excerpt:
            print(f"[STAGE {stage} FINAL LOG]\n{final_excerpt}")

        if not payload.get("ok"):
            error_text = str(payload.get("error", ""))
            evidence_excerpt = ""
            evidence_matches = re.findall(
                r"evidence=([^\s;]+)",
                error_text,
            )
            for raw_candidate in evidence_matches[:3]:
                candidate = Path(
                    raw_candidate.strip("'\"),]")
                )
                if candidate.is_file():
                    try:
                        evidence_excerpt += (
                            f"\nPersisted evidence: {candidate}\n"
                            + _bounded_console_text(
                                candidate.read_text(
                                    encoding="utf-8",
                                    errors="replace",
                                )
                            )
                        )
                    except Exception as evidence_exc:
                        evidence_excerpt += (
                            "\nEvidence read failed: "
                            + repr(evidence_exc)
                        )
            raise RuntimeError(
                f"Stage {stage} failed.\n"
                f"Error: {error_text}\n"
                f"Log: {payload.get('log_path')}\n"
                "Traceback excerpt:\n"
                + _bounded_console_text(payload.get("traceback", ""))
                + evidence_excerpt
            )

        result = payload.get("result")
        print(
            json.dumps(
                {
                    "event": "STAGE_COMPLETED",
                    "worker_log": payload.get("log_path"),
                    "elapsed_seconds": payload.get("elapsed_seconds"),
                    **_compact_stage_result(result),
                },
                indent=2,
                default=str,
            )
        )
        return result

    except KeyboardInterrupt as exc:
        log_offset, log_text = _tail_log(log_path, log_offset)
        diagnostic = _process_snapshot(worker_process.pid)
        _stop_worker(f"user interrupted stage {stage}")
        raise RuntimeError(
            f"Stage {stage} was interrupted. "
            "The persistent worker was terminated to prevent hidden "
            "background work. Restart the Kaggle session and use Run All.\n"
            f"Worker/process snapshot:\n{diagnostic}\n"
            "Latest stage-log excerpt:\n"
            + _bounded_console_text(log_text)
        ) from exc



def _terminate_worker():
    _stop_worker("notebook/kernel shutdown")


atexit.register(_terminate_worker)
_wait_for_worker_ready()
_stage_00_result = run_remote_stage("00", heartbeat_seconds=15.0)

# 01 — Environment

The exact Python 3.12 runtime, package versions, deterministic variables, resource preflight, and environment-amendment identity are recorded before source acquisition.

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_01_result = run_remote_stage("01")


# 02 — Project and input intake

The verified R6 bundle is discovered under `/kaggle/input`, validated before amendment, and copied into a writable R10 runtime root under `/kaggle/working`.

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_02_result = run_remote_stage("02")


# 03 — Authority and configuration

The controlling R10/R4 implementation authority and scientific Freeze R2 are preserved. Only the Kaggle environment freeze is amended to R3 for Python 3.12.

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_03_result = run_remote_stage("03")


# 04 — Phase 0 regression

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_04_result = run_remote_stage("04")


# 05 — Source resolution

The three active sources and source-specific adapters remain exactly frozen: PhysioNetMI, BNCI2014_001, and Lee2019_MI.

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_05_result = run_remote_stage("05")


# 06 — Dataset registry

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_06_result = run_remote_stage("06")


# 07 — Pass 1: verified source acquisition and bounded loading

This is the complete corrected Stage 07 path. Its output is needed for the new A4 raw-source extension.

In [ ]:
# Fully corrected extreme-speed Stage 07 execution. All scientific and evidentiary contracts remain active.
if DISPLAY_REVISION != "R54-MATCHED-A4-R2-FINAL-EXPORT-SECRET-SAFE":
    raise RuntimeError(f"R54 bootstrap required; observed {DISPLAY_REVISION!r}")
_stage_07_result = run_remote_stage("07", heartbeat_seconds=15.0)
_compact_stage_result(_stage_07_result)


# 08 — Metadata normalization

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_08_result = run_remote_stage("08")


# 09 — Label mapping

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_09_result = run_remote_stage("09")


# 10 — Preprocessing compilation

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_10_result = run_remote_stage("10")


# 11 — Split construction and frozen fit population

The original group-safe split, role coverage, event inventory, low-calibration allocation, and operation compilation are executed from the complete lightweight descriptor set. The lawful preprocessing-fit population is frozen before any signal is reloaded.

R35 correction: the accelerated planner emits the exact authoritative R6 event-row identity contract required for deterministic budget allocation.

In [ ]:
_stage_11_result = run_remote_stage("11")


# 12 — Low-calibration budgets

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_12_result = run_remote_stage("12")


# 13 — Pass 2A: bounded preprocessing fit

Each subject is reloaded in a disposable process only when it contributes to the lawful fit population. Exact float64 per-channel count/mean/M2 statistics are combined deterministically without concatenating all recordings. The compact fit state, source IDs, hash, and record are persisted in the execution bundle.

In [ ]:
_stage_13_result = run_remote_stage("13")


# 14 — Adopt verified core and materialize matched A4 R2

This stage does **not** regenerate the existing core HDF5 shards.

Before reading the raw sources for A4, it:

1. verifies and adopts the existing provider-version-2 core Dataset;
2. proves the released core contains exactly 12,910 unique valid parent events;
3. proves every core event obeys the frozen +80/480 core timing and therefore
   reaches cue +560 samples;
4. runs the synthetic A4 child/HDF5/schema/reader preflight;
5. only then materializes the new +0.0…+3.5 s A4 R2 tensors.

The exact core parent-event set is passed into every subject child. The child
must reproduce that exact set; it may not silently add/drop events.


In [ ]:
_stage_14_result = run_remote_stage("14")


# 15 — Validate and commit the separate A4 R2 Dataset

The existing core pointer remains unchanged.

This stage validates all A4 R2 records, commits the separate short-named private
A4 Dataset as provider version 1, downloads the remote A4 manifest, verifies its
exact SHA-256, and only then removes local resumable A4 subject checkpoints.


In [ ]:
_stage_15_result = run_remote_stage("15")


# 16 — Record validation

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_16_result = run_remote_stage("16")


# 17 — Leakage audit

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_17_result = run_remote_stage("17")


# 18 — A0–A13 readiness

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_18_result = run_remote_stage("18")


# 19 — Cards

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_19_result = run_remote_stage("19")


# 20 — Manifests

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_20_result = run_remote_stage("20")


# 21 — Negative register

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_21_result = run_remote_stage("21")


# 22 — P02 and later compatibility

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_22_result = run_remote_stage("22")


# 23 — Evidence sufficiency

All P01-G01 through P01-G16 are evaluated from serialized evidence. Narrative text cannot override a failed gate.

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_23_result = run_remote_stage("23")


# 24 — Repair metadata

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_24_result = run_remote_stage("24")


# 25 — Final export preparation

Every pre-export artifact must physically exist before terminal export. No reduced or alternate official path exists.

This section belongs to the one full-depth official P01/L1 path. It does not define a smoke, fast, reduced, demo, or fixture-only official execution mode.

In [ ]:
_stage_25_result = run_remote_stage("25")


# 26 — Terminal decision and bundle export

This section closes the complete 00–26 ledger, negative/blocker register, manifests, checksums, private derived-Dataset pointer, all handoffs, and terminal decision. The compact execution ZIP deliberately excludes raw source duplicates, temporary subject workspaces, upload tokens, and already externalized HDF5 shards.

In [ ]:
_stage_26_result = run_remote_stage("26")
